# E27 NBDF: DFlash2 vs MTP K3 (Qwen3.8-Flash-Next W4, Merlin v3.0.0, 4 x L4), 8k to 116k

Arms: Q (NB4T: MTP K3 + Q-defer) > D4 > D3 > D4t (conv torch, short) > D7 (ring) > D4b (FP8 drafter KV, optional), + one
isolation / INGRAPH_META_FALLBACK rerun. Readable code: nbdf_cell7_body.py (the CELL 7 stub execs it). Commit run only;
a competition rerun writes CELL 0's identity submission (E18 R2 format).


In [ ]:
# ==================================================================== CELL 0 — mode + fallback submission (stdlib only)
import os, sys, json, time, random, hashlib, re, shutil, subprocess, threading, traceback, zipfile, math
from pathlib import Path
from datetime import datetime, timezone
import concurrent.futures as cf

NOTEBOOK_T0 = time.perf_counter()
# S1 (CLOCK): Kaggle's 12 h counts from kernel start; R5's cell 0 began 7 s later. Anchor the clock at the kernel
# process start (capped at 10 min in case the value is odd).
try:
    import psutil as _ps
    _since_kernel_start = max(0.0, min(600.0, time.time() - _ps.Process(os.getpid()).create_time()))
except Exception:
    _since_kernel_start = 0.0
NOTEBOOK_T0 -= _since_kernel_start
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
ARC_ROOT = Path(os.environ.get("Q27X_ARC_ROOT", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"))
if not ARC_ROOT.exists():
    for _alt in ("/kaggle/input/arc-prize-2026-arc-agi-2",):
        if Path(_alt).exists():
            ARC_ROOT = Path(_alt)
WORKING = Path(os.environ.get("Q27X_WORKING", "/kaggle/working")); WORKING.mkdir(parents=True, exist_ok=True)
OUT = WORKING / "q27x"; OUT.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = WORKING / "submission.json"

def _load_json(p):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

test_challenges = _load_json(ARC_ROOT / "arc-agi_test_challenges.json")

def write_json_atomic(path, obj):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f)
    os.replace(tmp, path)

def identity_submission(challenges):
    return {tid: [{"attempt_1": q["input"], "attempt_2": q["input"]} for q in task["test"]]
            for tid, task in challenges.items()}

write_json_atomic(SUBMISSION_PATH, identity_submission(test_challenges))
print("MODE:", "COMPETITION RERUN (hidden test)" if RERUN else "COMMIT / INTERACTIVE (evaluation)",
      "| fallback submission.json:", len(test_challenges), "tasks | RUN_ID", RUN_ID,
      "| clock anchored", round(_since_kernel_start, 1), "s before cell 0")


In [ ]:
# ==================================================================== CELL 1 — config, inputs, lab module (RC2 verbatim + E25 NB1 / NB4S / NB4T / E27 NBDF marked edits)
import os, sys, json, math, time, re, shutil, subprocess, hashlib, tarfile, threading, traceback
from pathlib import Path
T_START = time.time()
MODE = "FULL"
RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
FAKE = os.environ.get("E23_FAKE_SERVER")                          # local CPU harness only
BUDGET = int(os.environ.get("RC2_BUDGET_SEC") or 150 * 60)     # E27 NBDF: hard stop 150 min (user: Q + D4 + D3 guaranteed; D4t if time)
HARD_STOP = T_START + BUDGET
MAX_LEN = 131072                                                  # user 2026-10-29: thinking up to 120k
OUT = Path(os.environ.get("E23_OUT", "/kaggle/working/nbdf"))   # E27 NBDF
OUT.mkdir(parents=True, exist_ok=True)
LOGS = OUT / "logs"
LOGS.mkdir(exist_ok=True)
TRACES = OUT / "traces"                                           # memory rule: the full text of every long generation
TRACES.mkdir(exist_ok=True)
TMP = Path(os.environ.get("E23_TMP", "/tmp"))
INP = Path(os.environ.get("E23_INPUT", "/kaggle/input"))
R = {"what": "E27 NBDF: DFlash2 K4 / K3 / K7 vs MTP K3 on the Q-defer base, 8k-116k (Flash-Next W4 on Merlin)",   # E27 NBDF
      "budget_min": BUDGET // 60, "max_len": MAX_LEN,
     "start_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "setup": {}, "sections": {}, "errors": [], "order": []}
_SAVE_LOCK = threading.Lock()
NB1_EXTRA, NB1_ENV, NB1_KV = {}, {}, {}   # E25 NB1: per-boot trailing vllm flags, env and KV pin (filled by the boot plan)

# ---- E27 NBDF (user 2026-11-01; the E26 K100Q pattern): the commit's ABSOLUTE DEADMAN, 5 min after the 150-min hard stop. A
# daemon thread sleeps until NOTEBOOK_T0 + NBDF_DEADMAN_SEC, then: one marker line (stdout fd + stderr fd +
# /kaggle/working/nbdf_deadman.txt), SIGKILL every child (psutil tree; else os.killpg on the live servers' groups + pkill -9 -f
# vllm), os._exit(0). No lock, no main thread, no cell. The competition rerun has no deadman (CELL 0 wrote its submission).
NBDF_DEADMAN_SEC = 9300
NBDF_DEADMAN = {"armed": False, "fire_pc": None}


def nbdf_deadman(t_fire_pc, marker, poll=5.0, use_psutil=True):
    """Sleep until perf_counter >= t_fire_pc, then the marker, SIGKILL every child process (group), os._exit(0)."""
    while True:
        d = t_fire_pc - time.perf_counter()
        if d <= 0:
            break
        time.sleep(min(poll, d))
    line = f"@@NBDF_DEADMAN_FIRED@@ t={round(time.perf_counter() - NOTEBOOK_T0, 1)}s pid={os.getpid()}\n"
    for fd in (1, 2):
        try:
            os.write(fd, line.encode())
        except Exception:
            pass
    try:
        with open(marker, "a", encoding="utf-8") as f:
            f.write(line)
            f.flush()
            os.fsync(f.fileno())
    except Exception:
        pass
    killed = False
    if use_psutil:
        try:
            import psutil
            for c in psutil.Process(os.getpid()).children(recursive=True):
                try:
                    c.kill()
                except Exception:
                    pass
            killed = True
        except Exception:
            killed = False
    lab = globals().get("L")
    for s in list(getattr(lab, "LIVE", None) or []):
        pid = getattr(getattr(s, "proc", None), "pid", None)
        if pid and hasattr(os, "killpg"):
            try:
                os.killpg(int(pid), 9)                    # e23_lab.Server starts each server in its own session (pgid = pid)
            except Exception:
                pass
    if not killed:
        try:
            subprocess.run(["pkill", "-9", "-f", "vllm"], timeout=10)
        except Exception:
            pass
    os._exit(0)


if not RERUN and not os.environ.get("RC2_UNIT"):
    NBDF_DEADMAN.update(fire_pc=NOTEBOOK_T0 + NBDF_DEADMAN_SEC, marker=str(WORKING / "nbdf_deadman.txt"))
    NBDF_DEADMAN["thread"] = threading.Thread(target=nbdf_deadman, args=(NBDF_DEADMAN["fire_pc"], NBDF_DEADMAN["marker"]),
                                              daemon=True, name="nbdf_deadman")
    NBDF_DEADMAN["thread"].start()
    NBDF_DEADMAN["armed"] = True
    print(f"NBDF DEADMAN armed: fires at +{NBDF_DEADMAN_SEC / 60:.0f} min after the notebook start (in "
          f"{round((NBDF_DEADMAN['fire_pc'] - time.perf_counter()) / 60, 1)} min)", flush=True)


def save():
    with _SAVE_LOCK:
        tmp = OUT / "nbdf_results.json.tmp"                      # E27 NBDF
        tmp.write_text(json.dumps(R, indent=1, default=str))
        tmp.replace(OUT / "nbdf_results.json")                   # E27 NBDF


def left():
    return HARD_STOP - time.time()


def log(*a):
    print(f"[{(time.time() - T_START) / 60:6.1f} min]", *a, flush=True)


COMP = INP / "competitions" / "arc-prize-2026-arc-agi-2"


def write_fallback_submission():
    """E25 NB4T: the proven E18 R2 format (CELL 0's writer): the identity submission of every test task (attempt_1 =
    attempt_2 = the test input), written atomically to CELL 0's SUBMISSION_PATH; never a [[0]] placeholder."""
    write_json_atomic(SUBMISSION_PATH, identity_submission(test_challenges))


if RERUN:                                                         # E25 NB4T: CELL 0 already wrote the identity file
    print("E27 NBDF DFlash2: competition rerun -> CELL 0's identity fallback submission.json only (E18 R2 format)")   # E27 NBDF
else:
    LAB_SRC = (lambda _t: _t if hashlib.sha256(_t.encode()).hexdigest() == "ef393fbb2ff7669ee4c2c99ee841b60da66514a3b86b7c0ce81fd5f64689ab5e" else (_ for _ in ()).throw(RuntimeError("LAB_SRC sha mismatch")))(__import__("lzma").decompress(__import__("base64").b85decode("{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;K_bFNnHRDXbna~F$u0Pgj)JG27#5$Z?y(-4y`r~GNitw^S8-9YLUa|c(I<zlpY<#1Ng|!{;sE_(y@QAm{+XUC0$9Wo_B4PIC3RK!O+U5`<kC926s2nq?Cuejv$rZzjOYWPz;7sm6LeHL>>ku(i)C6zVWq&QZ3++F01hkKCjG3Y8Qr>u+$I?;|RzAz_a-*#*=T%cj*&N(f=+mO5{RzcHUwwXtH=6@r*#gO@*DKa(MoOAUs<xf9vRT?dRmlvaz0K`rv?kh@g!*yL6F-Ja$;&(Z%oqX~JE!VX^ld5QUSE*gCLD$b04(iit-UTcBL{IQ3l5aRaC7<-RR)cBN>$vI06t4@J!7PON?87C+1OQEa`Xe4@|HS->F*k)^on>Zx_ByK{buu+>$saj~O2fg2Cm@PI8w1Tt>}7#1=WTte>?bewTyEcssU4oWzrpT6sUZ4y)-C89dPUMvk|JGq`oKlPm<l@LJ9NAsnNpgVWO8#i)*p1?{i(%zFqP>h@%XCf$ebF}%~wUtM4HLJ|zZ)dWj*NnDzG3*%HqwrqwX2oc2H|{xcmmt<pNgzTLs_nfh$Aw_{Gfv8jKUw7!Qy!SsFMJzG)iL2c*j<*Bj4G>-cATvvR0yl8VOJVrH`d+0vgRzT#8QA)P#(L9iy7;1oyNXvk@7=lgXPq(fiSeKw|~thn)lr|R(I)q33SEMq4P~Xj*$r&Nf>>cJEX8YU+pYagqtbSWUHpm%GPQx;}$c=U)heFNl`MSsqaU284+ZuNRQZwXBFAVdjimY=EYI1<iuv5xep-WNIE-(qd%$>|89+iAX<uUY(?7wm-;*lr`ZbJe_q!It}{QvQlbsNsx3SJ@~g1+sjutCRNM`qzeV_D?|pLC8AcPxf>7}cZSRx~DdZI*Z%NO!`;o<3Y=d9XguJ&C_q#6yMwirL2L@L}QW&*oAPU+$d`SF#f=F?nJ;KcnhR&nrFEV9L*0^hS_oambm3rZq*pA4#-PV&+I{^DUh{mT)G}r@bbd%o-MnUTv5w;pDnAFPN5Ofn&gaL7GSZqxm@GP==7f}r3bXRK9rpQ`YB)t<cK+&S&oBFy{B4i5`grOo>yKT@PP>itj%Sb(@kO4gx-J+}k&9;@J#`R;7R&zO`T&)#AjW1Y3``zKf0wfnIv01x#+f{rMa)<$W&BJ1f*L_VkUmfJej9xZw{aewieJ=$bh!kG8iP${~jOVJs`wCsd^*w4O{iiDol)#s(OtQi4P8@Z_p{o5{_<wS&z4f2Rk4>3qlv_z)Xg5x$LOAK7{}tO>;|8f!AWi){?+9lxwd3Ua%GQPuE?wUN;?5;oHXG?;-V<CVwFJPpiv~<NRYDKEDtD}$Y?*axa;gB0#%%+_O?j8y<;@5Rde7G<x__0}oE?sb`qfXH4VoJ4C(lKTmh{mK4U1s@>#S<sW9k4=--gP(<3A+Bn`JD}>ay!~8rW*FNyoqfIWM<&2Hq)4Py|Eir4uqr;|j_vFW@2e#OLmD%$oOL5R5!Z1uwd`f)%4xNoDZ#vpezDD)Uup=HAJg>_;7<qW=18W-F1*%Of8OO(e;ZoTF<%**})bMDQF%AMp`)Rzx*6jaPg*1X1F#Gy%^$(^K9mA+=GG8|$t8I$R7?TQ%`ijcll=9wI4C?zavWu=h9}GB2kVXqm$rN=5*IlRL#L=lwz*vSB5)8S+uu_SO2goBZ4Wr6xze)iBC`Y|5JM4)yN=!Cx<m>CUSRh%y0`9<~e+5q^SEP{7vb-IHc*5P?kWkzkFBD}vt613H)T2YSpFIcxZrJ`<H+Jl^;LVkTb)bK$&OFKL<`m)knW0e*$5IX~$vjiFUkySY?dzscY~z5R-x>fAoe9b)^xK#3sQ2eH<F57nrrlRaLFG$N71k|rdNp~5Ie-%0|WI;@>1JisN%3r^kLR*jjkD7h=uhi=Xs*R^lr5PDq{wb6v^vKnz4gMUlkTQrTr=S2Lr5Qo~t|7{#tMh`9lqXYgn*k&!y?wDiKX!tSaaT{OCfEa`<GJd?ArEw`*vyQT~lxAR&n3pi${8z!j%I$urV8#q<)%)aU-|(IKShVmZvznf3#wRWkiif`|M>zb}WS%X%q*38Y^0<pf`wO6!3#Zmf3JLjN!4`oD^aY>9l*uK%7nDA7;F%s91M)pMp#-8KKFwTa6#4!HQ`h8z)<1j0r&DsxYX@>%2G+=eYnoj17<0CjDF`vjGMa?MQVdKGA<H&Uj_-a|<mdHx2@{UUa5EU1gbAGLUP2(hB3WoHQk+lXu!AWybN?|An+x^nBP*-gKqTi!-n)sBh~zH)6oeU?9Qzt~fxP|K&V?pDi$3EYFA|`2yZ3mn9h<!_-Es!%4)jpU#3Vg1E?-z)WRzTa#X~@mlidY=^EGW3X0HgE7U{1V)9hx`a+kw1`1!Mu>Y`o_M9MwKp~zOaNoHD@jNpj|fH*ScDCke>;7A+jo-sR5iFSqjuXWV#V!qs0<<*KJ3iMz_j|cu~+0TE$`Kv4f-q-0Hk$D0ux5;(>JFK!tCg<!1MA`FUXbA$A0O`vh5js1?hkfI?Y(q|!6ZMg-d^yb4J4JFwtfAI9@R3~goZ5nMVMC0cBYbp&RTA#pjm>HhQ+Y+p(*V+Qv`V^}nO=2N47>nIe;l$CQ&w-a2J;$|IVAvG4GJa@sh=}*EWw8;wwP!bP6gI@4hCLqJn++;(9yVL0);C-REaGVW>*S=&6$fQzIj+XeyUi4wLA%1tG(|JVONJT%(l_j723*r5TR%0%tc=1X2*0HBP8u5a8#S%iq&0hMizvp8V>t#h%OW|S-*4@R9tjx?>_2OtPAMBjDMD}=y|w<vkv$w$IuURamGvBXLOnImP2}!&T#Y)10er_NX-|^RS_6cvac+sD&P1@HS^!Vp6_>tWBwz|7MhffTCK|48Hz!ObuU8KcW2maa`XRd3Lc}#{af^kJk^4^m6TfFFA_Fd_QmQcDP>LnmocihCX?x4$^;re6SY}^LKdoy>q()ubWDk~`yd15wpOn0)h1q?mqa-=34aU6g5`-gun>1huM8EiHZgxdKlV1eGfY=c95(G94Zs7~X3n(^EmnJgD0vSNQ}CSBM{xeV+uG)IcH0AxmB)S2^VT||ELDo%s;x|`aW_g;9;Bi6p~&;^ij99OXj9I6?8ZFVcnJEFc&km~H3OOf{RR%Tpyo~h>y&%!P_%5fS~45&p-2Oa^B?1=#kpU*<BB;_FsWniNv4tzOEyaCXRBXxH&3(G<>9C6LbQ>kmQ*?K<t5X4H8)2=i^^CuR=}>h&k=1pVnUd|b?~cJj$%W%LH1{Aii8wyjIxQ>FaNhNpe&ytZSZ!h)S>#HXWaT5tu2RigG_V#XO2CSP2b)pmVz@0Ik|{7?VHl_IAVdEY(I_{GhC-J=%jmC+bZt*dRj{8`N;2AjhY+i+(svJ8;le8$#Hf$8xLOU9K6&c;<)LeoA#W?PDO+_dzkagRki<jEn9bEUrGa1Xqi%eG3oa7dU9>Z$NJdT)@(qq;o(}#J&GwwYm`R^r$$Dw_h)_uJJHgv^UwNGpkP6Y%e$gutnNcH;I8N03ZU(OHZ5>J_tHj>H;}||>Y@c!D+dA~O#BZSMg<1>@GyvL6WXck%=y!yS~#?*kYj{_FO=N-a7KwTB%#w;G^$zZ_-QSLlgtp86Cipp>OI9MC}J=(M3lsW6|($~2e#0jK*T7~5KAdDGkNNB>8vR6D?U3k5rBSwjx5rgJ2@zluB~I79AulhZdc_bw_FSuQC!;Na|oX7@S_E%mBpohG4GbL?3LPgUo(ke6IkOk4GUNGO}BKYmk~O#<>479Fyh1S5>E#^7@o(gVNwOQm9hnFL%X-WWKjgm^<PJWV(b5KM1(sMfr0xVnI99ct$xh0wS6F7ajbN;Enx`#{6j;wGRa1}^z+gW|8jIIK|2*inANVw@t(1EeJ(CmR5R-V!b?-dl%}2+)tQ$chy*W0&%y|2&5?j;pw==6+ATb*>CkZHWSyRjAjH*qr`I=$xG{!)3iOKMyxh9^YEEaBQJg}K`hD>+>ILHTD(;*(<VH5<LMO=x`mfr~5wQK*e)ji-jx{A6B+0&)P3Qua8F=)ADff>0dn6T@VMSF0Iznz3iQnLGa$#^m6tez)-pFVYx^AeZx*<l?`}xt7jGE%_(Ih5`FuqL1ci|6DpsjV0^#XdRn;#-mjj+AesK5*cwx45MW?Lr~r)aS&@b8zsV3!-tj27CekGv)?ss@b)(ZB^kF0yj!h~UHJyl}z>bOMbi&=y+}Q(Avj%c%i8Z4|sb5CX4@^JEf=1fd^0Wd<7Fv#(XT3U<lmlQ>t<CM3AGISf=roM|_x*$mwXDEf8OPzB*QaqH=Hnt0_rjVR<1uCD{5M%lkAIGP3tVn+UU-V|s7f07kwJ_4s6JuCU)#kp={>jr*{bP2tzl7-1f;(cB8ixjvm2bY23&Rk$GV`VvMgipxAAMZtTdz9H8*$@8@x>_!_H}~h*I~*xJILj|g5!`fUG*Bf^-=qc!4%*8BPd87}@$idnxAYe-jk+zmgFHPBGFn`7z6WhbwSUK0dpdl}5O?j5b&;J3?#~k{pA&8)|0K9f7Cmofs3@8e-})9iJfE6mQ?n_I#4t@mFI0vF8VBj7=QY8I?0`XXTXoGV_!;RX2>d<pf5$K?;4A_ZbUn)1DvoV02D8l_3@=K~j6(7O|B#A>J3l^YWYU{<xfT`ScJVwq@h@!pP<M=+y?08hWlU!e+fL2@tio)R)!CDKnReI&A9CyW|I}9O-NsKZOoth{M<c`u3seaOA-G>s7Ihx~bYcHEv+CQP6G<1O<<(Z1p)UWhPOEg4jbCp+)*7x2E#ieD`XfgkGBRzFra8fD)fe(~m8_T7{B&iBK!?c@2&?65qw#(SRVF%3R3~$scAxA@M6OlMLp;oS|Lv(X<-m{qoTS3n_S}3(DPdLL;DYs*DXq-dxP^{(h*88gVPafyW6kE~N03()BXO6wEygYMB7OW$jT$FD7h3`^Xm$qA5;<pP*;`6ieC`^D#p~euNgOXZ)uF_^eKIEy1dkS00rmR!^o9{F)b{R3r`#2cWj1|e3N@{g7?NFjxwhZ!{76cHa%M1_L;!U%eyn?`PKSqYOiit+mna+k${g)Ukwmx4;}{hZQpMQlNk3aj|J$L#<d_4nCMG*DS~oqJFh276ITDX&?Kp{<S>!EGiwDK?6_|Hvg)@FF)#BB90$%Gk<Jdha3sPsDN43Ds=~MG&JFnSpw(xg?@9x=HNW0TroYIGGxP(dn0U#S3VD!v5_q0WZ@hlAo?I~Wu-~o&7LG$KVjV+P;e?w;vEn-bNyqf*B<p<mJz5j-BHnylp#$G(eiST`3j;XjJpm+}e0&<XBR|Mb+*DtRGp^`=}D@iqlc_<hIXOEI#Ja5iS%?VbK%OvRIof+L$=nw&JB~ADxC^4<|*XJB+|6~ZidkJvqDL#woWN2f^aYaiesokhI!~Uqy;K!cE$kqytBI;R{av8T)R4A6xduK;GTyuFOob~8Ip(a);b9iZ>4yCG~nK)06PVNzw#9x=Q1UH2<YNf>He0No*07PlCyT$`EQ&!utV^FUK_+ig$2cPWa;XF38m<hVWHK+rI7-+10cE0Xq7Bvw>M&!hd&o@aCv`eqw$pZ5~OaOIQvEq^!;)R1>N3giNO5pRa%*vP`L_BkUa(j(OxF^p(B~I}~;x^~*m|ZjlO)rRtM7TH}!Uc^Az(u&}+!s~6qY``(5U-;xw_n=GnTQOlTwwhr*M|w&!XEPyt)<7)znrK3q~FA!&GP68uyQdA17M{%r+c+JbvV|r7mKsz;S+=+9MoABuFoVqy;X8$*TY?rE>rL#22f15pls#uwqJ6qDR;Z^JJitI3?^taXZTT3?}hfh6CUvwW@U<`gn<U&#hs)N;h(Mw?gc%5s`#Z>H7kL`8SGZsivXLr`$DIr-RCULbd{OMat1s%Ae5jWiOL(>`tEp4cS=EFrs6CiZ-E=9#gM6g{x;8mRy|$-H5^Iq6(eOl{Q8(b6`L-H$hciA*rj!DJ1!zD*!&U|tzk4rO8(pY#HG5(Af=X9uaXDmBaMC?Oxk3PMVk!$$-`yAf_j267T4nGoK8GWL$TncCq*JQ*c{VOpS#6|rTj|S;cbYq*9_~|5fsS8=m1OMfnln}>$9N%-{vVPgH!pK>w-o<l_2`zj$F_r2ZQHO{RHII?v1|I5D3JSr;<?ra;^L9-LFI(nJA;fI}m*`$J0m}D%|coZA-p+f1Jg)ioBFU4k$4#1@-p}%Bta4-?DQQ?=GMSnH|d}V*#w0e90^WA2)^c+0JnU@l-{s^33VZgqzkN#JBRQD5z?v-$@YA?z?JBus0h}gHz9)qFW3uw2`~15(RppFuCj9XZIQ<E`@4;8nk6rY+CA8lL)*paOp)&V=Hv~FZ^WXW?A8D0N?{M54e;M%?7oNXGJ1r0|iI)QuZ>h%rx8&Hrg`G46Zm!Q~{ROElO*P^A86UtFSWr=kIz2&iU<u;y^uJ`bnBhILN4(U}d}m7RpKFz={fL`jxKNS+)a5xjthG*Qh9NzB#+J+cH4x*#r1U!TK%)1_tO5P!oSeg1~mU6)>VVpdsmMazLD9WcHiD#b2hmvQ;`|XQS+#CHUDDnTTo_6agX11NExo*AMN=SfT63uq``cA;boP)NT(0c^nsUNAWY%)I-QkOq2T68f~<tF_;*gThh{8k#BLEhe6_FR${E*(zHMSUN|YmLEt7hugyI0Zp)*g+K}ZnjK+ljmYEH)p-rfZU7p4k4KTk`;u(iLGTjI57oy!(`%|tZ*^v{?V{uG6NzDXuqja!8DdIe)_U}1{n-h7(Q{aIf!sTwoY^|V<{MPi*KkgE`5srPrQ7$o7aCZ0Y-t^S3aSWnhS1Y!nk|Y~ay`n+tSr|*5V7w}D{j{sCtJq;^g!{#QM||N%mgt|MT@O8w4ruN%^iqoD&W}nRJNxL&RVY*mI-<v9!lP$_4U=ylK2>BR{cnwW&T!|mL*URKcq+c&o*DpC?i&+_@;Ng_lx;-L3vfY&6W5oWCm5k>d!_v`%Uf>}A-vPM?7xkZE<e69>?UCgeF*KN^r=F?d3UbGVhW{yeZfKBhQ;rz#EDM+w2&5!-m)4Coybx}uyH?!1vT(-2`He-n38Z&2Fr!sX8mRx8@UeHt!OS0X(j&|ME*>&O6VYTo7$sGtXW2vsA@!3m?&OZ(2og9?I+YW0)*j7qw~AE^=^q~z;tBf_GzzvO0o}IX7o?G@ro?JG82H4UeARHx`_;f(R&(F?73s+>*c)N&BJiWD=OPK4d`tY^K!CBL#k8ApNALF3Gy2gpuScpT!sYIE}Z845gl+X>l+C-Sg`t>;0cUDlEC`D=qZ~>Y?=w>{V63F2b{jLJQ{qc4Dz2Q(INfQA?z!({jC0d;KyaB9piCe$+)QCgY4v7GfpR~0=pjk*DZEvOS!SaSrqhf-1i!HzQ>AA6)U2r8&87qOZp_H-h(>33D7S*SIYQpQm+i9=Fd3-l8oFH!IB=!P4A__4aviR#nKl~u+7cXI@>ZGEK7X*r*W=sUUySU%#<EtZLxn8`nYWMg4z*Jv8-H)Ty_I_!(HIIA*^^)e{#_#kl?Dl@+DKc9Q3*oDU*Tl1>W>k!0;6Kh=7S%v8Q*_Ha7S$R|#?)W1he%%hy$Ff@vD4C%gRFpiL_QWGabH<>4#$0{TXZecGhguava>bb_U~gN^6Itq3Kc`;Vz!A7gUd#Y>ZlwBz2JX0!L4?b|*fOSx)4Gv+rCn=F>%0ZYDlOl=}}w)Qovlx^y&lRB{6BdG5fu+Pjng$`9Qy2_Z#T61Hsf0w?K1s}ZWkhm<e%ASyc@Q)XHr1JIBAm$*)A+Q3fkqD<_z^J0~+yfAcnoB-M`)a>X@LsgSA0uGnN+R3usA#Z7=1e;!@9>KA;u300I!5OfRhx+N!BkT!ky7Z|DNVG_2OOf7iU^ZJ@j!bdLx+=JXw4G51<u`hOcOw_8mLiVjzmL8obM&4*Kpqh3J;_bp6}G11)_-kqMUgQ`$W5n7@3R}jl$13kq>Cuh2c==9yGU6U#c^d&nJIe4SkoR>;du{07dOgLQVNU8OQ6bn7ON<vK9&Zeyb*(KrE=jg+wel?v~VSIM@?qaCj`ejQIMuw=42GC-i>|{WK<^tk|ESK`>J>O0jpN0TLRMx92yg4hx43>1|?6F{J|4f84nd@53h<w@D%ye^L$k!*GjAGss39o{C1Cenuj7-A`?9Jc~9rG4`b6HhYbT1Cl08gddUG6;If$68o%(__NsW-k0o=1dcorJDYDh#$FY7ysq7iswU>u#v#Lvron8RhUW}5UOYp@!BbPD18-_rwy1s+gXYl``}JG9;>T+fd1o24CBmR1PW@E$N6~&{``5t}IM9!^(Wr*W8t-l@9L7Z`pCwuv`2`~pqeE{*JB9L<*l8Wu;0e-Z#S_q;=+|$NU}kP(T&2ICcTgQ91PTNf7LM1=RuC{mEzoWUg^RwVXvhoy!Fb)264VJA8N_d5Vwz1{eH)bFjz}7PRqOR#iLRe#O>@$%ei*A!Y~Hb7_1VxV4wre5zKM&FGiV7l*}{A_y=~Z^s;|ZJPXYEKx(2usR92kb6-{mYRPsbPu|B<)KmK+dfzz7a1Oi9B0!b=DIpi5kx24rDyE4x?&_AUI!l&2F0ch&R4K+=~A5l0b-lz1kSUgR#YqAP3!0s6&E;{QDd+)M!M)zR(*EG27GcSSQNhVKq$y^l0xJ(v~GuMnQIvuMhG5&B+GruxKs`#f$UsuxLqrea#9D<Bz+(Wz@O6iqferE;^nKIx7ud*9`V*|pz^{4ix6AyJ4){4O=t_L4paV3Ienw(=}GuZA&sm@bvoPbarGy<Y)Ri?fG%x|?_8k&aVo#8k+3WR~m5$YL&Wo1K11IEKG>Cb)SJIQ|;ZjxBmze(5J7vdfU+!d4}J5vquXHQ*O47oO>rFQX7PXxpOY=RRpR5V7CR0tXKfnPzSmX&IC#4dE!^kgT>`M$bFZ#<iHkDMhql(tohQzllI%6e_+*Imdb$_pv%@~mD;M<j3YF=0=R^|D}W`Gl_4fL<cwk@M2<;MBMoTFCsuHpzg5>EHZ8LScU1c#+*|20f5OUD%a$Jq(H&hkpmA#$kul3|{2b*HiFwQ$XvdlYDzL&hy^n=~!s)*l1sI5`YVFHb|~A?-_orp1;wq<Qv*M+v`q`5Y|tRb=-lQJ6F<?k>3@>9fu7nPv5=RUt9YxZ&Vzt^lcYn@e*4>%Wzr7ddyz+PW*);MW09QH6F8W`Po1DiPn?~K3%QKD`6mxN*!PTHkD52wO=74zxvf0v01i+5w4NPP*I`4F#puHY;KVk!wIVD<t|V&KrYFQ_fiAnftY5X?Di(5ed3gl=w;mTYQZk(Kt0NSx!5#~EaApY2);R0;0<p;uNY9a2OI1|PRBP4*7oRQ%|~vwq=5$E4yn}!0bw4xF|^3B$#f8N`pOw25f$3p#}H7K<Z%2?!?(gs4m2>@Xtl%UL#&)3FHC7V{?~!p_`uu`*^0ZIz1Z$U;&x-Bl~9=JKc#xdA9qWM%r)g0iBGv??j*)uG)zEq95B&@4fawLYHrrh=!{)vJutH{JmpWmV&5he{9!8jnc^!3(;R*K7n^a4-Elw0GbW7?c>RmTkLkETu3)!T60E|Sj@|aA7bEncEp5nMC2{DAi?Ceoc%I7VPZ_Ntq;jJ4pxQ9g#dKxt$(H&<njy@5y9v3uk@eP{ZXzadBFH$LjP)Ge>%vb}Dwvi+p`n|5g7<@V;7HAIt-@g9gyz3Ha^a%93R0lQWXb?`@&%O)8vC6btJ|S`L`S4el~>>68E0GfYd4U=#drmXO`}L}9X@fX2=8_&Tptph$dV0S(cpeiPWB8?N_2(%0$xCP={2s^52X1nc6<H4>)^DCcfF8kYG;C7_aqNx0z<;(wHf0)>;i2T)fq0Ly>LYB8hQy2QB{5*Z@Qn4S#o5pgQ?)xu^HV^j${-YcCHvJGm&x9R!7xN7-P(`8ufHztj<}&y-$sd9!Zy@w|Qobx_5kae1yXf0)<e_J<2{}OXoujSn_2YeL(KsbSbyGDBjZB!f7stz0EN@YPf{4t~V2vy#nB)-^9vtPP{o-cum$u<uWu#l?#!|QEwgX^veX6C1?6F_;5%>e*5O{z2nHm-{#hrkvl31<LOi1Bq^@&PUf=iBKke(xlmbZG6ufAJIOT(`y>vUoauvzRqN<%>t`=@r`g178dag0eC&bRQ~l7_Eb*XR$@xx_oo?kSJ5Aoh2sqkx_%KGY0$0{jlp-R`XI#{5M1|i4l>#bQj!^N5#pucdh7Q{{yOkoPtu#XeCpuhy+DmXBvoc=JC1RRnpixzN4Ohv#pD4hFyw~hmfmnj`-gw7g3s<>cpyWy)f*-4~$QkiUqs_^58z1r@FqX&bf>LRRM1Z>t@<ta1L7@s^{4obnqU|sZg7p5}h^{p%Znu9>F@UR=K#O&Y-O&0wH?F%U+5*TcWmZ1rjX$^p_`(5!rkwm$Dncmt>iENT?Sb!a$uG6!3^Us3<6~s@6#4sR8h90fHyfu+FMjpFikO6vrea7TpL#}m#K`Nbp?cm_SMCBa#ysm5)T`sazg6>KXHF3Y^wF?i>3#`D4Ozqj0henm+TI!P^GBU28EXXlsDS0Y<X`q^09|btkrrnWgX#yJtE!3JvjQUi=|<LKmwK`D|JFaeePpRIb07Ww5^sW$7w@F^&uXp#a)v1<dP^}(DK3aSLYLf-ILI4L`C5z@@<;^5@tt>ObXF0Wv!~#^XA{};ahvP5?*=?+uGxSY1KGI|8*J?PZw3tDB)42A5ScD~s^jmG+82wO<dNkiMC5Qzjyr4sX<q-dLEUL*W!m}OE_h4adm}{#?^zg{p3bbZZsimvU6Od@@qO|lJFiX|W3>b3kYXHg&`#;4=66FSDB~+s`+GI@D|cIZ=HnP9J*SUG>VoY_M!ZaT_(6M@)m29OTq0H6npN%VC8{eCmA#3KD0-m2SKyPhVGvOH(WRTt%kyUWit4wFN*;#v(SZZXQ&_Hxzl|rg)&8U2xlw0kO4w2_b?A;K26WR>Bv=juoDwNkmVGUN(O?W72mJ)z#XqMhCc`=wf9pMt%7oO5Mh+R8n(QK2%|BHz)ZL$hhx$w$Pl-X})F6%fq_KN1Drc*}Lq~jqBN$wqO4_N~x}^7o%4Kb^&isEd=Gwb4yON(zvmKNvgVaEWVwI8-b0F_we7(n!f|N`MWiuZ}`I4L&{=0Y#k|kYmINiqlj(MYu#-GAs_RGBef9aC)lmpA-L$ab|pIdP?>PAlF)?#E~?bi!sfR2o?26X}fHt2UB1gujnN#VM#b_kpwG9+(7l2~kto`4`Hk&dsuuw^X4u#vX@1TZe1C;PrtDa?~w=Y$n!?>>7Hl;<CedCV78^Ky53JpapYroR}!r#v-d*4LQ}T1S9`lK=>_G_{o&oK}M4fO$s<oA=k29NUIK={h$4*XI$9NS<Z8!G}W3yRW@YXY}vN{_ykSj&zwJf2Bwr%lM7CNn2dgigNt>e)zN?-P%47gV(V?tsgzdR3Q0Pk>T1yUJm^NzR|$&WF8P(CzK2P{E^=g@tgz|@%Ps0apXf;P-vLOt{}_3S#Ad4lBrP%rRNHhV7G#}-F9p}`~pyp66GTrY1g3Dhm%1P8lI{$CJGJZO|5W{xmmxMN$2x*!ca!Jpa3$WB|Y1mIu^mwFO(()SBUebGl6>~NC=%AZdF<nfX+t;vtNUqCRC)eHqoCb0mQr;Hi8gos{x*C#AbY3pz#6|bP0p`8Eb9j4aX_9JW9hyaQs1z%|UB=5n6d0-Unt5un(+3G-6h`breyCIb<tf1mCBJ61?vBL^=^Ejo;#K{H@PjmZ8v=%Y0jI2(p-9RmGZI2=|L<J3>T^pIN|}_A3`7Bt}v=$D2Wst*GxShKSg~s;byA_Ax8x=!`m?hW|c+Z&kt_N<xfyf|>CAaDRSE0%ZiuRagBP{Y`oYP(E;5P@?KtOh-f<y2Y7J(x&+JvL!h$CexVfzKJG1&C?}R=~D*Ds{*u1%V#@A82Pym;ZNG?rXc%U_TO^snkC4`x(+YEkt>Y=4UY%S(p)EexRa}U_2<-S*YUgh=`KqW!`dHsI_7MDpl;7_$RgZN772Qp@VgkZx$_?ir!b){n=Xl{j#SNb&;W2CVttGIAZe3nGv(S{M4m2q;R)=C{ZU$<Vdg?Zk<1Nva8-Agwu;aZT@iPG+wM08h&$u<`*>e-rmM3Z!++#QL+O*}?ScwtIVA?H*Zg7HV-(^jC3m*a<+mP#ch5J=tsG!jj{IXq1NGtp3WvDnscr$QuEa1^iWNt>#z;w5$VpK@Rnx;6IPyOwC_fW+=OzB<asj&TR_k;1L-CkCVKx(Ns1r^&jiT8aTvIVOs_<VtKmVvvbIuo0w9I!ObBVlt-Az@9{4>?ew*uVS<Yq|tKfCo1jt%p>ytt6NcM8wdLG@>rVRb1l!xm7HUu3}9)`R(!YWT2!fg?hGS8(Z6jjdmZ3>jkacE9ZJ@@6Uj%cegpd-&Yhf5u{GArgt=shJ2le=?P&?u^Eqf~G9_+wRx64@i9A(V2Ie;#i@I`v**53r11()2fKE22Mh}xq1dAlbOI-xdNuNF7ZToAHfEmphb;YiA9$m(~;9WGyF*ImtxnsygEw4)CUk>rT0?qxM!G_Jl78*(|v7gyWj0hl@9&Y2@Zr`-Ku?Ck=hMq9<wg{m)AFAOQ_{mi<t(T2=|9f;4N861d*H>wrJw1MK!~)B?ZJMGUVcOYUTk5@Fr7cpDYklM>p&Ex7D+_SQaC^(pgLfr)sIF0?fH01knm%%Tub`S+VHr2zD<ktcD?7K?1#7=53*1IV_R%Q)ZB1^Q%`}$7XSci1HI0ObzMI@Mku-fQn#e`zRW`l*{kABxRCbB^9Ire<b<WVNKHx9FL#TaI<fia1W%O^=CIR=a4W}RmS46M+UXJ%of4<qb-f*fbsXD<{>e1)A#jq_5^5##g}6i*&5e+mmbzP)TKM_A#K$yqBpyn^)FAMxy*@7N*fVjZD_Yb6l-SdHETH|t2rjA@l_@krb|~87Ptx)y>T<u5fgxohLm)sCE4a<80Khgpm=JHK@Y+M{30reJ&yi~rt+?jB2e+BzbQ27K2nt%PJyfSd`9XIBU%L|_>5rDlQX2>$(dmMJ^+{J0rLX4Gbj0jNvKo_=d9S;NAGH#PdJv1{MlI#g1oZ^KdHx`i;0G?V4IZE?`8Q>Iaz$G>?&e&IbDmG45@x9JDvV)D*_hIz0V5F<)^#lkvuSQjSpB$$b};~rCM`Ln*4+|Y?S8X!x1wEyhU7H|HB;E#Mu)w*&vg8`xgO>B6T^3dLesy0%8;fw2li&I$|KAeb@$$q>>SZ(SVoBY2{K&ywxOyC8`*p$@kKqiw$}R3%8+s{S|3E3(;j1k~D0dayki80;j|*E#S4=W#iGyf|9r%C&AawTqn`Ki_VSK&p?5_k1j5Xx7zz?Gl}*jVpT5oXOmq=>XZb9E47ALkrN`C95c1sq9W$jF@1j@%rujuKFTw$qtj*S7IDa^O1X3VIl;eg!32)k{eD9<Yv7ad7gonS=+*p$HX#f}B-a*`jlm^x<9+Si80>IZc6P_rf6{XM2Dj$wclWtx<QE%|{$CzYfmg6rOg}NCrF&eAvSFHerI>5w7wmaeNv|7uYsUX<>2d8SF?(ofFlr<<>hktcqw{qEl9#Svjd0~AS?##BI`e#b=lwDB(8qAc?snCG!jTO_<gHfE7ZGPJmDofCc{H#gSTBngKC4{h*N$xvz{88Nl2z7~+&b-FvW`n+CX#?>NIEUJql7|htB3fR<pRH%l0Pyw4103)*m9KnI5_slLVQeO>_8FhfIiY7|7d11(Nc<P<PPn2gbfua$0`a8(vWNb=5%9vZ!5Jfn76D+03^JdY3#YF-hpo){690n7M@0Z3NDDaCB7%sF@}8B2*I1@2t1soGR%b>)28PH({EXmVre`2B3$zG<M&<=-IOU*O8q-KV|$BH?Vw8AYNXRe8eV67Xyn^Cc=R!Q7nUP@$1S}URrMK6f^D5&?jD@`+xsYm+d}zWl}9%Sb2YJ}FTYG6liM6c?W#R|w@ixESkp^7^bS-(%HAxqmn&z=HzGfky8|V;^&mlWmSa-zk!5dM&K^i+sEtZp*y=CLoFRsHRyLsZlSRtUDHg#iSm#6^Usp&T#z`1kxxfpjycH7V3{ykk;Q^n3Y6jIfDVn?`{m(<E;7Q{L!rfTJ@PQwYJYqg4hxDwuMnAGcn!ff!V*h@@SDFieQxTBDS;0FE7sa;9A%Eq-W_(u7ET%SYb&?(Xtami)$0+yteYa|V%)pX(l^OYL2#VmYY`I*FEM|!!BaUFsXfhtX)(KQGE&H+F5^Dhyy306bt&;#ltw>LgjfYld95Nsa*u3fp5OmKFj394kCFP5b`3UO*(nmofld`x3;MzoAnbxL@dGf=zpEJe#$i$-6DD@AZrr3N>dfEDbFCYzj4`Cz}b5^1Z(+n&3p0%Q()W?fO+b%)AKbyhZbmg>(58JO3Ql^|aQ_bQ$IrqA*1O}6B9!n=%88&EUMjf{l4Xh!qTkDYmAL6l#2$ic+;)RsWc-r^I)f<#J7jukWvoQtkH5UGC$SQd+JkBV9faGv_ekV<#TKB#?NH1Rl9CQ4u3zOs}3Rie(EmV4DM*AleM?_ztcIS45zk>T7?}@HRFby_a=~T<h46TaHCO_EWI57OgUMA!!LgDw@(-{r{1ai5ewccbnzZ$<ReGc{D1OUP9${*n^1|2M;PUvYJrX2wz`-aQ~p++0Rwo>(z8qyDx8za`a=^mVunyFE!mn-_p5(*|YN0M@ek2MC|5u;GP7u$$yYx5vsCDkH!P9W{ml)~o{Ho2)0#mbcQ-Q6o-KvXL)JO<to&Z>4mEZ<s#5mh8bvI92t+m+wmy8154_MM0G*9^TFAB1!vtOFoMHF9riK$-B)Z=7-7O-HEA1mz~3gLR3|{Mj_jCv6M}Z5;8O2qxDM2&(VU0G%qo^CDNcKcY7?>PZ^sjP1CoY2)GZ+me(1&o;75T&FXJqr}sk>)CIm2y`-tf!&TTOVBC6wZYg4o*@R)yG)?3LTKcqPD|<MEyfLm74x2;@N0$DB#+4um2+99a6%DY9B<ru$`KPf=-VKaLv=~|Q2TJwH1r*(#rC@WGrjeZ*h|6(=Ovi}D^PVuCfperrr3QC4dIRW`IYGX{m@Jhn+mgp_y<*6_jHh-32bh51Mi9UCMz1$6QfZAtKi_GiSAxAZbW?>0~;TVmw}_-UX8Wp=z11(KODk=GeK|42r2*udxZu}F*>^xi$z+^?s8xK`AA_@w@R0&Fj<JfJ+VPwM!Wu4#F^?xL`BZ-9KJm+vhG<7hQwC2qxuTYg0xS*2z=zRjWuG#M+l#7F3w~A`E=>e=B6s%m^Vm6BrQgDH$<4H+(yYseep9E8re1`r!%)hU*2csUz792R>d<13MSF8kp|#PnYYj|8hD}*YS<2%?BerlA|bLzRyGyv?e@fq(l^|+n};6=PqYIP*==&^nJc$cB*d$z0+`tVvis@I){nI-;5x>hLL`t(>F6M11aBoBb}(_IbQoMiN!{g@cJ?ThcyN~ik2IH7=fm(V+fV=h_ata&g3`PG84;-~na36)x5gPoOcG*LxBs8+XY8OrGPHdrj#lj8`|*N46Da|`y-}h7W|wPor|meAIT7jwyyoC-8HB*5V4a6&_ug_0vBGUN+5MK374*=Q(n6!$y+S?ZNP9ickry|y%+8DN8c|gBV}sr!B0T(zIgf3sj_;tgI49<!(3*l}Hmo-U={Cl=GedB^Uef26&7@IYThx*Az*6zgflhDH_!we2Oy{BnEtKk|30<dw0?v5$z*$n;O1L#z{d+3hzMqJ+s3mPAC|HHK+@d(r9kLVVDe=tLlro%YPhjQpd^z)f1I~|R^%P{lG>gJABH7m(R8(vcs~Xmw-Cb81?IJ%hRU6MI$r75fLLz804oOUT?H(8kkyx{Ot=Z_|U%#Ylx1Tcd%Ba_*oNCu0cyBUrE9oVo$97Gw1K>sOLt}70U@CaiRAPbNV^d%P3DM<R_y4lVO0J{Vpg-8>g0p)bT#aK}OLW<w@6dPEt-Jlh2bLb1UD2pAwQEz&Bx^ws7oZ$XFFBM{zTjnG3nf=rVR(qa4=aF7pIhU(YMTpLZ2p+A^=!HxB*>%6xB<a9Iq;ndHO30S=__^JHv&V3FV*z*Ie;ka=7Ev=nUG9WXK5E$RkKICwKXS_4e|65Jq+R?L>9H1AE_dI+s0xgr6DhMqZ<RFB1oB7tp_QppA6EI_M>VsZw(~DX9g>S9&_p`e!<bzpXx7jFMq?VC&MHeYbsNK<V}q_?xS-v+QQ7A;t!A<7vnDkRXO!KKRYYq63`T8_y!<hxF(#pJ;m`Qn#-LF?!3%Ae;GJP15A#%z+j8bXwpu-+6Y$mI0<@=j|v`OS5#h6Ms_qVBM7~d$fO(RSWHA!W@`8mmsm;zl?1<jgS{|1#MRS8ej*rv*-9#skIgB|RRD97(00&~Op8T7cChl~npkE}h`E)lx-$R|P6ipP;&SX^4Y1wq{XNVnBdoO&^eZe4x}v;{$(8yYQHsTEDY_%jxYwl_2K-MlFP9zP<m!KS*~pE1W-5ET10|sAV)MMn=}=c(d>zD0T6klxDL^jcwn%lrypMhau4&>LqZC<q@kC8zKkw}wMZe1Zr8#9%NA5y3Sa6{3C&axA5P>KmvU^Wm2E66&TP92sux8CXL;zR5qO^~l!&=3nF*sZJlvYk<)~!2kc)j@8woPj%%6{Y|v^arz2z`I9FEK3{2Iv8ZqVxJ*8g38Eu&i_CrACBKCt|wY8x%jS+1caI00yn5fEgF>EhqXi$pP-&;V363(+-4wOCX9tllH#Yzaj)9O<gRld=NBTk)WOiZviNB0K|K`D+oAO%0GQq^j8~!6w^@DatzK6Kj9CTF5{!{3i#6Q1~&_`xMfuxGGM}0qu}tOzJ0gZnzlOmBQ9u(=K2V+hVn_r9ixkADB<vPCQ1;O;I{s|s)OOxzZ$ef8Q<j6v8ka7Pcc;p`$1(vr6GM8vamrJ;659&`l#5;9b$gjN~mRU%dz^?{rTDBW_rC+S23Rzc*%~3{mN(B!?2`B8!Q0w^Ywi09t=O+r{Pq-ZX%Wo8+~Y+l2={}w)bVe|3(a?$&ON_vc;p6-5uy`Y62n057i8sisx3YD<de(55!lTpy`xxr+E{6bPArhwFAPyY4`NvZjFMno`-6S74z%?K!lMReR=6B(lv9etVJgkubuj{55k9>xoS5A604ju4-OqbmAdMipipB!!>J4qV7QXdw_m#tjB*%9x{qp-)@_pU;Bj~#F=<bz+!(eiweuw&a4_s5;E~;CjI5QroesdB;<D5}<=!7fPT=qc!qeaMnt31rbfYFsSdN+<w%?d)Y|T!Q&5h#Uf(bkeo!jAdFqtvyhp(`=TY@n;?!|<k*gIYO-Cs7rwI=+jLS%k{(6_P`dO^;T1P1>4T9`(j267|m6!Tu@J0(I)t$RC!MEl0U&n%dwi{;24SRfOxvE$z_XzN^-ai?fbaBap`!?`_4k+{dTLhrbqePe-iT|YxH>w6*MLnKyg0qJr#%D9np9BQa>nCfohgGaonxV)~@Z}z}W-C)3OZb$u<mT-G`=3IX_1-^s)dnQTr0*)D-5S(7j#~xYdb*;JPfLl3O?#%K9r{c;kLIg>S8Uh3wzdcxKXK)!!|0C0M7ta(nBNTfYhe$5_=`AqVOsAvhftp?)emzjAPN^QLyH0>R5<*Kid_K}V=%vrtRw4QzNn=G~k3(%K69kXuttdbo&2egi9M6@T@I-l5qUHqaH#Bd4v0aA;%qR8Rex1cK_KyB>W#Kt@G0%LsNKGjRZATErTU%;+!02Eguk7$5BEPBbDQ7a=*WiGJ{gX#Z=P52454UL)SE$(P^t_F{c~YYeZN-e+S(DVCQGK_7g-L;rwZRvR#@fK|8n0WG%1J&eA)h^f&0vjfh42skY3a;fnG#BDTsUoJ!1-YLrB}qQN*AIMkp)w(f#*N1VFG<*Xq<6t8&`72bD>io+WGV{{M^`Bka{bza)(-|wsySW-OByPJqEI&RY6rlc~ap)X;aS$Lcz!~rk{q7#rW=ddL3X~l4P^NdR&U!rWVKm2?M?6P}F-ucS0w3fxEIC;mHaehqHwbXQ-~OCajKz_H{eXFhS=HrC;gFBB~jtD#iJjiE50YWw%59T0;W-PiY_qTN&P6PgJm72(d^mFe$0PbL6!lSJzhvB{A4KKM*hZyg@4r&aCvE{)C>0HiJ5N@D5(=ZUwNpLbYoXbTo1jbsIaG=#%2_Nr46NNe$x>dR}vGeN1G*s2$N>dCYOms<LW&4_6C^+t7>Rgpci}Td}g+BtJr;yn;Vyl?g>7WUtuxOYqa)!OT9IneoK01@kNxKnD7L_=+WZ$;ZmYsZkkwV4>Zq$-a6(kIRszu*yz=fO6}UqTe!)Fzdr=IV1By>m5@|A!KEA^oP=|h6-2N8gc1hL#2sk#>e-Wh>OSyv-lEY=_|DB8!F!i(wVFZv*?20Az>`ERQIYS@pV~gC!i`0<Z(emSf;lkfO;kA`afo?JR|=hn^_#X00UAvRg4Ls9TO<qt(?41#<}{;=xm+}Md<YCG}9!kf*IloK`C87-$)_}vCQmDteQI-*OV&68D5{dvGuLdNX&~Y$tg1&6`DX^ZwC)w&jhC%b^C;A7|Pa8P+PXqU>4mEx)f$(!t>>!Y4q<hqj?50;*g|Itx^enpuUMJGzZ3H6>ZLdmeHt>$8mYCXEU@L%v>?3^)*Ri{<nqKB|XTgOBg+3&l(o<Zw^U7U6~L~V04HXM5)OA^;GUxRGpzB!jdG6Y6pn&pV&&n1|7g10gGQrpt98&Bk17Dmtx+-DDr`h)G$4y3?14HPmK!xIgHn31u}o6AhM|S>D_cOU}8U9BdNJtf0~8I4gw%_8E~0Z5wn8%VXL3)>IVW2Gqgj-@;Au-{VnZjKA~ko+M%jXR+C#F*Djij5^vw+;D!YD{^WG)Ii`m_*#7XgK@$V`;OaW_bvb1fqJxAEC2Zlcz@5UUxvm^jB2fOf2cPjDN>K2}&EGasMUww^u)IgKG4)fScy$^{jp{nIoa9{a%@g<|Q8R3B>&s{RaWxqMc-AYL9@dZ`v9h)~tYXw5u;f#{>9br&T-6$Dvj`}xmP_6wqE_L#)d*&ZJ8ylw90Wqin<3S1=>|ldwxBjx_)eH%@<^XSU7e5s_qT<FX#f5jJQ3)!yKt)#LnefcAT1%l4~8e@3?iuiZThR}A2!kM*l-<-L@>RkiZXaKsI^>ER}Mq<d4?qhN2(Y#ohI^PgmZd<(a!*eQ+C;^>)rPce+g^~J+{_(fC@mfX2W!Lq*;Ky*HPj#xPZ1UDN&HLAIE3Q{*|66w{HYK*$SgKxrGAiKAqmRQ|>`mC;*?ffOn&R!;lwiD^eh_g*#nbk4<fPvr<%k000004Y*qbx|kS100HH8|B?d$FLLqtvBYQl0ssI200dcD")).decode("utf-8"))   # E27 NBDF: the same E23 lab (sha ef393fbb), packed (notebook < 400 kB)
    (TMP / "e23_lab.py").write_text(LAB_SRC)
    sys.path.insert(0, str(TMP))
    import e23_lab as L
    REQ_LOG = TRACES / "requests.jsonl"                           # memory rule: the full text of EVERY request
    _stream_run = L.Stream.run

    def _traced_run(self):
        try:
            _stream_run(self)
        finally:
            if not self.err and any(m is not None and not (isinstance(m, (int, float)) and math.isfinite(m))
                                    for m in self.margins):
                self.err = "NONFINITE_LOGPROB"           # final audit H2: NaN / Inf logprobs are never data
            try:
                r = self.row()
                pr = self.body.get("prompt") or []
                r.update(port=self.port, sampling={k: v for k, v in self.body.items() if k != "prompt"}, prompt_len=len(pr),
                         prompt_sha=hashlib.sha256(json.dumps(pr).encode()).hexdigest()[:16],
                         ttft=round(self.t_first - self.t_start, 2) if self.t_first and self.t_start else None,
                         sec=round(self.t_end - self.t_start, 2) if self.t_end and self.t_start else None, t=time.time())
                with _SAVE_LOCK, open(REQ_LOG, "a") as fh:
                    fh.write(json.dumps(r) + "\n")
            except Exception:
                pass
    L.Stream.run = _traced_run
    PLE_EXPECTED = json.loads(r"""{"model-plefp8-00000.safetensors": [5200027199, "dc2e845b7edd35bda92834fba3626bf7d199e28d6aceac99fee654aade390cfd"], "model-plefp8-00001.safetensors": [5200027202, "899eaa0716e28594468a1389ee58cb065c23907f1270de3831f5ecb0a4f82d56"], "model-plefp8-00002.safetensors": [5200027198, "06fd8a11abf0419a669f89397b8d70dd6ff42d401e6b2a037c65e49704faaf71"], "model-plefp8-00003.safetensors": [5200027189, "c6aaa1fc08e84eced3c8151ac8679ed943888eba0fcef2556963693430f95bd9"], "model-plefp8-00004.safetensors": [5200027190, "d94e97c96d3ea09208614da016960f3f4b429f47a044c898a51b493f42f74ba2"], "model-plefp8-00005.safetensors": [5200027190, "03d5d4792e14a4ab55bae50bb459624b01786f818bbcfcaed1a5d0235af484c6"], "model-plefp8-00006.safetensors": [5200027190, "586cccbc12383021bc9bc02f206d9b19fbd1672373ed9a5d91e3b0ce34c2418f"], "model-plefp8-00007.safetensors": [5200027190, "c4a23bcc10f3cde6b633e82c282a9a518e3a64d433628283e1bc592c94cf3d6c"], "model-plefp8-00008.safetensors": [5200027190, "2dc8098c0d020bff277c9cf499a6b908e17836b70a5f949dfa24793371c9a87e"], "model-plefp8-00009.safetensors": [4400023163, "61de98b89bb79f386795787d7a76827a26f1e292c26edbb0a1b613da455f5a9c"]}""")                     # 10 FP8 PLE shards: [size, sha256]
    # exact Kaggle mount paths (foysalemonshanto Models, checked through the API 2026-10-31; no recursive search)
    MD = INP / "models" / "foysalemonshanto"
    PART_DIRS = {1: MD / "qwen3-8-flash-next-w4a16-merlin-p1/transformers/default/1",
                 2: MD / "qwen3-8-flash-next-w4a16-merlin-p2/transformers/default/1",
                 3: MD / "qwen3-8-flash-next-w4a16-merlin-p3/transformers/default/1",
                 4: MD / "qwen3-8-flash-next-w4a16-merlin-p4/transformers/default/1",
                 5: MD / "qwen3-8-flash-next-w4a16-merlin-p5/transformers/default/1"}
    PARTS = {}
    for k, d in PART_DIRS.items():                                  # the five checkpoint parts (Models from the kernel outputs)
        mf = d / f"q27_fn_merlin_part{k}_manifest.json"
        PARTS[k] = (d, json.loads(mf.read_text())) if mf.is_file() else None
    PLE_DIR = MD / "qwen38-flash-next-fp8ple-overlay/pytorch/albucino-fp8ple/1"      # our FP8 PLE overlay = albucino's shards
    PLE_DIR = PLE_DIR if (PLE_DIR / "model-plefp8-00000.safetensors").is_file() else None
    MERLIN_PAY = MD / "e23-merlin-env/other/v1/1/merlin_offline_tar.bin"
    MERLIN_PAY = MERLIN_PAY if MERLIN_PAY.is_file() else None
    INTEL = INP / "models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1"
    INTEL = INTEL if (INTEL / "config.json").is_file() else None
    WHEELHOUSE = INP / "datasets/foysalemonshanto/vllm-0-32-0-cu13-wheelhouse/install_wheelhouse.py"
    WHEELHOUSE = WHEELHOUSE if WHEELHOUSE.is_file() else None
    ARC = COMP / "arc-agi_evaluation_challenges.json"
    if not ARC.is_file():
        raise FileNotFoundError(f"{ARC}: attach the competition data")
    MERLIN_OK = all(PARTS.values()) and PLE_DIR is not None and MERLIN_PAY is not None
    R["inputs"] = {"parts": {k: (str(v[0]) if v else None) for k, v in PARTS.items()}, "ple_dir": str(PLE_DIR),
                   "merlin_pay": str(MERLIN_PAY), "intel": str(INTEL), "wheelhouse": str(WHEELHOUSE), "arc": str(ARC),
                   "merlin_inputs_ok": MERLIN_OK}
    ABORT = {"why": None}
    NVSMI_OK = bool(L.gpu_mem()) and not FAKE
    R["setup"]["gpus_at_start"] = L.gpu_mem()
    if not FAKE and not os.environ.get("RC2_CPU_KIT") and len(R["setup"]["gpus_at_start"] or []) != 4:   # CPU kit: no GPUs
        R["errors"].append(f"expected 4 GPUs, nvidia-smi shows {R['setup']['gpus_at_start']}")
        save()
        raise RuntimeError("GPU_PREFLIGHT_FAILED: " + repr(R["setup"]["gpus_at_start"]))
    L.start_watchdog(HARD_STOP, lambda: ABORT.update(why="HARD_STOP reached: watchdog killed live servers"))
    log("RC2 | budget", BUDGET // 60, "min |", json.dumps(R["inputs"])[:900])
    save()


In [ ]:
if not RERUN:
    # ==================================================================== CELL 2 — host facts, checkpoint merge, engines
    def run(name, cmd, timeout=3600, env=None):
        t = time.time()
        try:
            with open(LOGS / f"{name}.log", "w") as f:
                r = subprocess.run(["bash", "-c", "set -o pipefail; " + cmd], stdout=f, stderr=subprocess.STDOUT, timeout=timeout,
                                   env={**os.environ, **(env or {})})
            rc = r.returncode
        except subprocess.TimeoutExpired:
            rc = -9
        txt = (LOGS / f"{name}.log").read_text(errors="replace")
        R["setup"][name] = {"rc": rc, "sec": round(time.time() - t, 1), "tail": txt[-1500:]}
        log(name, "rc", rc, f"{time.time() - t:.0f}s")
        save()
        return rc, txt


    SHIM = TMP / "e23_libcuda"          # FlashInfer JIT links -lcuda; Kaggle has only libcuda.so.1
    SHIM.mkdir(exist_ok=True)
    for d in ("/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu", "/usr/local/cuda/compat", "/usr/lib64"):
        c = sorted(Path(d).glob("libcuda.so.1")) if Path(d).is_dir() else []
        if c and not (SHIM / "libcuda.so").exists():
            (SHIM / "libcuda.so").symlink_to(c[0])


    def host_facts():
        """A0: what the Kaggle box gives us (glibc, driver, GPUs, PCIe, peer access, cgroup memory, /dev/shm, disk)."""
        f = {}
        for k, cmd in (("glibc", "ldd --version | head -1"), ("uname", "uname -a"),
                       ("nvidia_smi", "nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total,pcie.link.gen.max,pcie.link.width.max --format=csv,noheader"),
                       ("cuda_version", "nvidia-smi | grep -o 'CUDA Version: [0-9.]*'"), ("shm", "df -h /dev/shm | tail -1"),
                       ("tmp_disk", f"df -h {TMP} | tail -1"), ("nproc", "nproc"),
                       ("cgroup_max", "cat /sys/fs/cgroup/memory.max 2>/dev/null || cat /sys/fs/cgroup/memory/memory.limit_in_bytes"),
                       ("cgroup_cur", "cat /sys/fs/cgroup/memory.current 2>/dev/null || cat /sys/fs/cgroup/memory/memory.usage_in_bytes"),
                       ("meminfo", "grep -E 'MemTotal|MemAvailable' /proc/meminfo"), ("ulimit_l", "ulimit -l"),
                       ("omp", "echo OMP_NUM_THREADS=$OMP_NUM_THREADS"), ("topo", "nvidia-smi topo -m; nvidia-smi topo -p2p r"),
                       ("pcie_now", "nvidia-smi --query-gpu=index,pcie.link.gen.current,pcie.link.width.current,power.limit --format=csv,noheader")):
            try:
                f[k] = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True, timeout=60).stdout.strip()[:600]
            except Exception as e:
                f[k] = repr(e)[:200]
        try:
            import torch
            n = torch.cuda.device_count()
            f["peer_access"] = [[torch.cuda.can_device_access_peer(i, j) if i != j else None for j in range(n)] for i in range(n)]
        except Exception as e:
            f["peer_access"] = repr(e)[:200]
        return f


    def host_mem():
        """MiB. avail_mib = max - (current - inactive_file), Merlin's own rule (RC1's guard counted page cache as used and
        skipped arm 3 with MemAvailable 55 GB); /dev/shm counts against the cgroup."""
        out = {}
        try:
            cur = int(Path("/sys/fs/cgroup/memory.current").read_text())
            mx = Path("/sys/fs/cgroup/memory.max").read_text().strip()
            st = dict(l.split() for l in Path("/sys/fs/cgroup/memory.stat").read_text().splitlines() if len(l.split()) == 2)
            out.update(cgroup_cur_mib=cur >> 20, cgroup_max_mib=None if mx == "max" else int(mx) >> 20,
                       **{k + "_mib": int(st[k]) >> 20 for k in ("anon", "file", "shmem", "inactive_file") if k in st})
            if mx != "max":
                out["avail_mib"] = (int(mx) - (cur - int(st.get("inactive_file", 0)))) >> 20
        except Exception:
            pass
        try:
            for l in open("/proc/meminfo"):
                if l.startswith(("MemAvailable", "MemTotal")):
                    out[l.split(":")[0] + "_mib"] = int(l.split()[1]) >> 10
        except Exception:
            pass
        try:
            out["dev_shm_used_mib"] = shutil.disk_usage("/dev/shm").used >> 20
        except Exception:
            pass
        return out


    TELEMETRY = OUT / "telemetry.jsonl"


    def _telemetry():
        """Host RAM + GPU memory every 10 s for the whole run (Merlin's floor ~69 GiB inside a ~173 GiB cgroup)."""
        while True:
            try:
                with open(TELEMETRY, "a") as fh:
                    fh.write(json.dumps({"t": round(time.time() - T_START, 1), "host": host_mem(), "gpu": L.gpu_mem()}) + "\n")
            except Exception:
                pass
            time.sleep(10)


    threading.Thread(target=_telemetry, daemon=True).start()


    SHM = Path(os.environ.get("RC2_SHM", "/dev/shm"))


    def shm_names():
        try:
            return {p.name for p in SHM.iterdir()}
        except Exception:
            return set()


    def shm_cleanup(before):
        """After a boot's server is dead: only /dev/shm entries that appeared during THIS boot (absent from `before`) and
        match the engine's prefixes are unlinked; everything else is recorded, never touched."""
        out = {"host_before": host_mem(), "removed": [], "kept": []}
        try:
            for p in sorted(SHM.iterdir()):
                try:
                    sz = p.stat().st_size >> 20
                except Exception:
                    sz = None
                if p.name not in before and re.match(r"(torch_|psm_|vllm|ple|nccl|sem\.mp-|cuda\.shm)", p.name, re.I):
                    try:
                        p.unlink() if not p.is_dir() else shutil.rmtree(p)
                        out["removed"].append([p.name, sz])
                    except Exception as e:
                        out["kept"].append([p.name, sz, repr(e)[:80]])
                else:
                    out["kept"].append([p.name, sz])
        except Exception as e:
            out["error"] = repr(e)[:200]
        out["host_after"] = host_mem()
        out["removed"], out["kept"] = out["removed"][:60], out["kept"][:60]
        return out


    class MServer(L.Server):
        """Merlin passes --shutdown-timeout 60 and the PLE transport unlinks its /dev/shm storages only on a graceful
        teardown: SIGTERM, wait 75 s, then E23's kill (SIGTERM + 40 s + SIGKILL of the whole session)."""
        def kill(self):
            if self.proc is not None and self.proc.poll() is None:
                try:
                    os.killpg(self.proc.pid, 15) if hasattr(os, "killpg") else self.proc.terminate()
                except Exception:
                    pass
                try:
                    self.proc.wait(timeout=75)
                except Exception:
                    pass
            super().kill()


    def boot_gate():
        """Merlin's pin leaves ~0.9 GiB free per card: every boot needs near-empty GPUs (< 150 MiB, not E23's 1024)."""
        g = L.wait_gpu_release(300, fake=bool(FAKE), gpus="0,1,2,3", free_mib=150)
        return {"ok": g, "gpus": L.gpu_mem(), "host": host_mem()}       # callers launch only on ok is True


    CKPT = TMP / "fn_merlin"
    SHA = {"thread": None, "ok": None, "bad": [], "sec": None}


    def merge_ckpt():
        """Symlink the 5 parts + 10 PLE shards into one directory; names and sizes now, sha256 in a background thread."""
        SHA.update(thread=None, ok=None, bad=[], sec=None)        # a stale result never gates a new merge
        shutil.rmtree(CKPT, ignore_errors=True)
        CKPT.mkdir(parents=True)
        expected = {}
        for k, (d, man) in PARTS.items():
            for n, (size, sha) in man["files"].items():
                expected[n] = (d / man.get("model_subdir", ".") / n, size, sha)
        for n, (size, sha) in PLE_EXPECTED.items():
            expected[n] = (PLE_DIR / n, size, sha)
        bad = []
        for n, (src, size, sha) in sorted(expected.items()):
            if not src.is_file():
                bad.append(f"missing {n}")
                continue
            if not FAKE and src.stat().st_size != size:
                bad.append(f"{n}: {src.stat().st_size} bytes, expected {size}")
            try:
                (CKPT / n).symlink_to(src)
            except OSError:
                if not FAKE:
                    raise
                shutil.copyfile(src, CKPT / n)               # local Windows harness only (no symlink privilege)
        idx = json.loads((CKPT / "model.safetensors.index.json").read_text())
        need = set(idx["weight_map"].values())
        bad += [f"index needs {n}, not merged" for n in sorted(need - set(expected))]
        cfg = json.loads((CKPT / "config.json").read_text())
        if (cfg.get("text_config") or {}).get("ple_embedding_dtype") != "float8_e4m3fn":
            bad.append("config.json: text_config.ple_embedding_dtype is not float8_e4m3fn")
        rec = {"files": len(expected), "index_files": len(need), "bad": bad[:20], "ok": not bad}

        def hash_all():
            t = time.time()
            import concurrent.futures as CF

            def one(item):
                n, (src, size, sha) = item
                if FAKE:
                    return n, True
                with open(src, "rb") as fh:
                    return n, hashlib.file_digest(fh, "sha256").hexdigest() == sha
            with CF.ThreadPoolExecutor(6) as ex:
                res = list(ex.map(one, sorted(expected.items())))
            SHA.update(ok=all(ok for _, ok in res), bad=[n for n, ok in res if not ok], sec=round(time.time() - t, 1))
            R["setup"]["sha256"] = {k: SHA[k] for k in ("ok", "bad", "sec")}
            save()
        if rec["ok"]:
            SHA["thread"] = threading.Thread(target=hash_all, daemon=True)
            SHA["thread"].start()
        return rec


    def sha_gate(timeout):
        """The first launch waits for the full sha256 check (Merlin itself never hashes shards)."""
        th = SHA["thread"]
        if th is not None:
            th.join(max(1, timeout))
        if th is None or th.is_alive():
            return False, "sha256 check not finished"
        return bool(SHA["ok"]), ("ok" if SHA["ok"] else f"sha256 mismatch: {SHA['bad'][:5]}")


    def prefetch_bg(paths, tag):
        """rev 3: warm the page cache of a cold 17-30 GB /kaggle/input checkpoint in the background (many readers)."""
        def go():
            R["setup"]["prefetch_" + tag] = L.prefetch([p for p in paths if p]) if not FAKE else {"fake": True}
            save()
        th = threading.Thread(target=go, daemon=True)
        th.start()
        return th


    MERLIN_ENG = None


    def merlin_install():
        """P2 payload (.bin = a tar Kaggle does not unpack) -> install_merlin_offline.sh (UV_OFFLINE); payload deleted after;
        a failure is cached so later Merlin steps fail fast (rev 3)."""
        global MERLIN_ENG
        if MERLIN_ENG is not None:
            if MERLIN_ENG.get("failed"):
                raise RuntimeError("Merlin install failed earlier: " + MERLIN_ENG["failed"])
            return MERLIN_ENG
        if not MERLIN_OK or MERLIN_PAY is None:
            MERLIN_ENG = {"failed": "Merlin checkpoint / env not attached or incomplete"}
            raise RuntimeError(MERLIN_ENG["failed"])
        if FAKE:
            MERLIN_ENG = {"fake": True}
            return MERLIN_ENG
        try:
            pay_root = TMP / "merlin_pay"
            shutil.rmtree(pay_root, ignore_errors=True)
            pay_root.mkdir()
            t = time.time()
            with tarfile.open(MERLIN_PAY) as tf:
                tf.extractall(pay_root, filter="data")
            R["setup"]["merlin_extract_sec"] = round(time.time() - t, 1)
            pay = pay_root / "merlin_offline"
            shutil.rmtree(TMP / "merlin_off", ignore_errors=True)
            rc, txt = run("install_merlin", f"bash {pay}/install_merlin_offline.sh {pay} {TMP}/merlin_off", timeout=5400)
            kv = dict(re.findall(r"^(MERLIN_[A-Z]+)=(.*)$", txt, re.M))
            shutil.rmtree(pay_root, ignore_errors=True)
            if rc or "INSTALL OK" not in txt:
                raise RuntimeError("Merlin offline install failed (logs/install_merlin.log)")
            sys.path.insert(0, kv["MERLIN_TREE"] + "/tools/merlin")
            import launch as ML
            site = subprocess.run([kv["MERLIN_VENV"] + "/bin/python", "-c", "import sysconfig;print(sysconfig.get_paths()['purelib'])"],
                                  capture_output=True, text=True).stdout.strip()
            MERLIN_ENG = {"kv": kv, "ML": ML, "cuda_home": site + "/nvidia/cu13"}
            return MERLIN_ENG
        except Exception as e:
            MERLIN_ENG = {"failed": f"{type(e).__name__}: {e}"[:300]}
            raise


    MERLIN_CC = {"mode": "gcc13"}       # rev 3c: Merlin README:229 / install-env.sh: FlashInfer JIT at first start needs GCC >= 13



    KV_FP8, KV_BF16 = 2479838700, 2464415744      # Merlin's own L4 pins (23,034 MiB cards; preflight.kv_bytes_for_vram, 150 / 79 blocks)
    PROFILE = "fn-quad-24g-sm89"
    XHIGH = json.dumps({"enable_thinking": True, "reasoning_effort": "xhigh"})
    HOST_PHASE = {}


    def merlin_fn_server(name, kv, port, kv_bytes):
        """Merlin's launch.build() for Flash-Next on the L4 profile: KV pin lowered for 22.49 GiB cards, the checkpoint's own
        chat template at xhigh (Merlin passes medium otherwise), Merlin's host_phase checks recorded, E23's env overrides."""
        eng = merlin_install()
        extra = ["--served-model-name", "q27", "--max-model-len", str(MAX_LEN), "--port", str(port),
                 "--default-chat-template-kwargs", XHIGH] + list(NB1_EXTRA.get(name) or [])   # E25 NB1: per-boot flags (last wins)
        if FAKE:
            return MServer(name, [sys.executable, FAKE, str(CKPT) + "-w4a16", "--tensor-parallel-size", "2", "--max-num-seqs",
                                  "8" if kv == "fp8" else "3", "--port", str(port), "--max-model-len", str(MAX_LEN),
                                  "--kv-cache-memory-bytes", str(kv_bytes)] + list(NB1_EXTRA.get(name) or []),   # E25 NB1 (fake only)
                           {**os.environ, **(NB1_ENV.get(name) or {})}, port, LOGS)
        kvv = eng["kv"]
        gcc_bin = str(Path(kvv["MERLIN_CC"]).parent) if kvv.get("MERLIN_CC") else ""
        base = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "CUDA_PATH", "CUDA_HOME", "LD_PRELOAD", "CC", "CXX")}
        base.update(GPUS="0,1,2,3", PORT=str(port), KV_BYTES=str(kv_bytes), P2P="off",
                    VENV=kvv["MERLIN_VENV"], TREE=kvv["MERLIN_TREE"], CHAT_TEMPLATE="", WARM="0", CACHE_ROOT=f"{TMP}/merlin-cache",
                    PATH=f"{kvv['MERLIN_VENV']}/bin:{gcc_bin}:{eng['cuda_home']}/bin:{os.environ.get('PATH', '')}",
                    CUDA_HOME=eng["cuda_home"])
        if kv == "fp8":
            base["SCALES"] = str(CKPT / "qsa_kv_scales_262k.json")
        base.update((globals().get("NB4T_BUILD_ENV") or {}).get(name) or {})   # E25 NB4T: launcher knobs (DRAFT_HEAD) into build()
        plan = eng["ML"].build("flash-next", PROFILE, str(CKPT), base, extra=extra, kv=kv, checkpoint="flash-next", status="untested")
        try:                                               # audits: tree products, venv, checkpoint marker, PLE dtype, toolchain,
            eng["ML"].host_phase(plan)                     # scales file: a failure stops this boot before any server starts
            HOST_PHASE[name] = "ok"
        except BaseException as e:
            HOST_PHASE[name] = f"{type(e).__name__}: {e}"[:600]
            R["setup"].setdefault("host_phase_error", {})[name] = HOST_PHASE[name]
            save()
            raise RuntimeError("MERLIN_HOST_PHASE_FAILED: " + HOST_PHASE[name])
        env = dict(plan.env)
        env.update(LD_PRELOAD=kvv["MERLIN_SHIM"], CUDA_HOME=eng["cuda_home"], VLLM_USE_RUST_FRONTEND="0", HF_HUB_OFFLINE="1",
                   TRANSFORMERS_OFFLINE="1", LIBRARY_PATH=f"{SHIM}:{os.environ.get('LIBRARY_PATH', '')}",
                   PATH=base["PATH"], VLLM_CACHE_ROOT=f"{TMP}/merlin-cache", TRITON_CACHE_DIR=f"{TMP}/merlin-triton",
                   FLASHINFER_WORKSPACE_BASE=f"{TMP}/merlin-fi", CUDA_DEVICE_ORDER="PCI_BUS_ID", VLLM_PLE_OFFLOAD_READY_TIMEOUT="3600")
        for k in ("CC", "CXX", "CUDAHOSTCXX"):
            env.pop(k, None)
        if MERLIN_CC["mode"] == "gcc13" and kvv.get("MERLIN_CC") and kvv.get("MERLIN_CXX"):
            env.update(CC=kvv["MERLIN_CC"], CXX=kvv["MERLIN_CXX"], CUDAHOSTCXX=kvv["MERLIN_CXX"])
            lib13 = Path(kvv["MERLIN_CC"]).parents[1] / "lib"
            if (lib13 / "libstdc++.so.6").exists():
                env["LD_LIBRARY_PATH"] = f"{lib13}:{env.get('LD_LIBRARY_PATH', '')}"
        try:                                               # Merlin's own scheme: PORT 8000..8355 -> 29500+d / 30000+100d
            mp, vp = eng["ML"].distributed_ports(str(port))
            env.update(MASTER_PORT=str(mp), VLLM_PORT=str(vp))
        except Exception:
            env.update(MASTER_PORT=str(29500 + port - 8000), VLLM_PORT=str(30000 + 100 * (port - 8000)))
        env.update(NB1_ENV.get(name) or {})              # E25 NB1: after build(), which strips every MERLIN_* variable
        argv = plan.argv()
        HELP[name] = L.help_check("merlin", argv[:argv.index("serve") + 1], env, argv)
        R["setup"].setdefault("merlin_plans", {})[name] = {"argv_tail": argv[-60:], "status": plan.status, "host_phase": HOST_PHASE[name],
                                                           "env_keys": {k: env[k] for k in sorted(env) if k.startswith(("VLLM_", "NCCL_", "KV_", "P2P", "SCALES", "CUDA_DEVICE_MAX_CONNECTIONS", "DRAFT_HEAD", "DFLASH_"))}}   # E25 NB4S / NB4T / E27 NBDF
        save()
        return MServer(name, argv, env, port, LOGS)


    STOCK = None


    def stock_install():
        """Fallback only: stock vLLM 0.32 venv (E18 / E23 installer) with the -lcuda shim on LIBRARY_PATH (RC1's sampler crash)."""
        global STOCK
        if STOCK is not None:
            return STOCK
        if FAKE:
            STOCK = {"vllm": [sys.executable, FAKE], "env": dict(os.environ)}
            return STOCK
        rc, txt = run("install_stock", f"{sys.executable} {WHEELHOUSE} {TMP}/vllm-venv", timeout=2400)
        kv = dict(l.split("=", 1) for l in txt.splitlines() if l.startswith(("VENV_PYTHON=", "CUDA_HOME=")))
        if rc or "CUDA_HOME" not in kv:
            raise RuntimeError("stock vLLM 0.32 install failed (logs/install_stock.log)")
        env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "CUDA_PATH", "LD_PRELOAD")}
        env.update(CUDA_HOME=kv["CUDA_HOME"], PATH=f"{TMP}/vllm-venv/bin:{kv['CUDA_HOME']}/bin:{env.get('PATH', '')}",
                   LIBRARY_PATH=f"{SHIM}:{env.get('LIBRARY_PATH', '')}", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                   VLLM_NO_USAGE_STATS="1", VLLM_CACHE_ROOT=f"{TMP}/vllm-cache", CUDA_DEVICE_ORDER="PCI_BUS_ID",
                   PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
        STOCK = {"vllm": [f"{TMP}/vllm-venv/bin/vllm", "serve"], "env": env}
        return STOCK


    def stock_fn_server(name, port):
        eng = stock_install()
        flags = [str(INTEL), "--served-model-name", "q27", "--host", "127.0.0.1", "--port", str(port),
                 "--tensor-parallel-size", "4", "--enable-expert-parallel", "--max-model-len", "32768", "--max-num-seqs", "4",
                 "--max-num-batched-tokens", "4096", "--gpu-memory-utilization", "0.88", "--disable-custom-all-reduce",
                 "--language-model-only", "--engram-config", json.dumps({"cpu_offload": True}), "--load-format", "safetensors",
                 "--dtype", "bfloat16", "--kv-cache-dtype", "fp8_e4m3", "--default-chat-template-kwargs", XHIGH]
        # diagnostic only: RC1 had a 30,208-token BF16 pool at TP4; no MTP here, so prefix caching (default ON) is unaffected
        if FAKE:
            flags = [str(INTEL) + "-w4a16", "--tensor-parallel-size", "4", "--max-num-seqs", "4", "--port", str(port), "--max-model-len", "32768"]
        return L.Server(name, list(eng["vllm"]) + flags, eng["env"], port, LOGS)
    save()


In [ ]:
if not RERUN:
    # ==================================================================== CELL 3 — prompts (Flash-Next tokenizer + its own template at xhigh)
    R["sections"]["A0"] = {}
    _t = time.time()
    R["sections"]["A0"]["host"] = host_facts()
    R["sections"]["A0"]["merge"] = merge_ckpt() if MERLIN_OK else {"ok": False, "why": "Merlin inputs missing", "inputs": R["inputs"]}
    R["sections"]["A0"]["host_mem"] = host_mem()
    log("A0 host", json.dumps(R["sections"]["A0"]["host"])[:1200])
    log("A0 merge", json.dumps(R["sections"]["A0"]["merge"])[:800])
    save()
    TOKDIR = TMP / "fn_tok"
    shutil.rmtree(TOKDIR, ignore_errors=True)
    TOKDIR.mkdir()
    # the Merlin checkpoint's tokenizer when the merge is OK; otherwise the Intel (stock fallback) checkpoint's own
    MERLIN_PROMPTS = bool(R["sections"]["A0"]["merge"].get("ok")) and (CKPT / "tokenizer.json").exists()
    _src = CKPT if MERLIN_PROMPTS else INTEL
    if _src is None or not (_src / "tokenizer.json").exists():
        raise FileNotFoundError("no tokenizer: the Merlin merge failed and the Intel fallback checkpoint is not attached")
    for _n in ("tokenizer.json", "tokenizer_config.json", "chat_template.jinja"):
        if (_src / _n).exists():
            shutil.copyfile(_src / _n, TOKDIR / _n)
    from tokenizers import Tokenizer
    from transformers import AutoTokenizer
    TOK = Tokenizer.from_file(str(TOKDIR / "tokenizer.json"))
    _HF = AutoTokenizer.from_pretrained(str(TOKDIR))
    _tpl = ((TOKDIR / "chat_template.jinja").read_text() if (TOKDIR / "chat_template.jinja").exists()
            else json.loads((TOKDIR / "tokenizer_config.json").read_text())["chat_template"])
    _tasks = json.load(open(ARC))
    _ids = sorted(_tasks)


    def _grid(g):
        return "\n".join("".join(str(v) for v in row) for row in g)


    def _task_text(k):
        t = _tasks[k]
        parts = [f"## Task {k}"]
        for i, ex in enumerate(t["train"]):
            parts.append(f"Example {i + 1} input:\n{_grid(ex['input'])}\nExample {i + 1} output:\n{_grid(ex['output'])}")
        for ex in t["test"]:
            parts.append(f"Test input:\n{_grid(ex['input'])}")
        return "\n\n".join(parts)


    _INTRO = "Find the transformation rule of each ARC task below and solve the test inputs.\n\n"
    _PH = "E13_BODY"


    def _render(effort):
        return _HF.apply_chat_template([{"role": "user", "content": _INTRO + _PH}], tokenize=False, add_generation_prompt=True,
                                       enable_thinking=True, reasoning_effort=effort, chat_template=_tpl)


    _RX, _RM = _render("xhigh"), _render("medium")
    assert _RX.count(_PH) == 1
    _head_txt, _tail_txt = _RX.split(_PH)
    _HEAD = TOK.encode(_head_txt, add_special_tokens=False).ids
    _TAIL = TOK.encode(_tail_txt, add_special_tokens=False).ids
    TPL_MSG = [{"role": "user", "content": _INTRO + _task_text(_ids[0])}]
    TPL_IDS = TOK.encode(_HF.apply_chat_template(TPL_MSG, tokenize=False, add_generation_prompt=True, enable_thinking=True,
                                                 reasoning_effort="xhigh", chat_template=_tpl), add_special_tokens=False).ids
    R["prompt_render"] = {"head_text": _head_txt[-600:], "tail_text": _tail_txt, "head_tokens": len(_HEAD), "tail_tokens": len(_TAIL),
                          "xhigh_differs_from_medium": _RX != _RM, "xhigh_sentence": "xhigh" in _head_txt,
                          "template_sha256": hashlib.sha256(_tpl.encode()).hexdigest(),
                          "tokenizer_json_sha256": hashlib.sha256((TOKDIR / "tokenizer.json").read_bytes()).hexdigest()}
    _TASK_IDS = {k: TOK.encode(_task_text(k) + "\n\n", add_special_tokens=False).ids for k in _ids}


    def prompt(band, variant):
        body, i = [], (variant * 7) % len(_ids)
        need = band - len(_HEAD) - len(_TAIL)
        while len(body) < need:
            body += _TASK_IDS[_ids[i % len(_ids)]]
            i += 1
        return _HEAD + body[:need] + _TAIL


    TOK_SHA = "0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3"   # = the 27B Merlin tokenizer.json
    TPL_SHA = "c3cf9e34abf4f9e36c2d72165aa9c132d3e2a725b6c2586aaa3a8af9d7a81041"   # = the 27B FP8 chat_template.jinja
    R["prompt_render"]["source"] = "merlin" if MERLIN_PROMPTS else "intel_fallback"
    R["prompt_render"]["hashes_ok"] = (R["prompt_render"]["tokenizer_json_sha256"] == TOK_SHA and
                                       R["prompt_render"]["template_sha256"] == TPL_SHA)
    if MERLIN_PROMPTS:                    # the pinned hashes belong to the Merlin checkpoint only (Intel's are recorded)
        assert R["prompt_render"]["hashes_ok"], "Flash-Next tokenizer / template differ from the pinned Merlin checkpoint"
    assert R["prompt_render"]["xhigh_sentence"] and R["prompt_render"]["xhigh_differs_from_medium"], "xhigh render missing"
    BANDS = {"2k": 2048, "8k": 8192, "40k": 40000, "90k": 90000, "116k": 116000}
    NSTREAM = 8
    PROMPTS = {b: [prompt(n, v) for v in range(NSTREAM)] for b, n in BANDS.items()}
    MAX_NEW = {b: MAX_LEN - n - 512 for b, n in BANDS.items()}
    ORACLE_P = [PROMPTS["2k"][0], PROMPTS["2k"][1], PROMPTS["8k"][2]]
    LONG_P = [PROMPTS["40k"][3]]
    TF_P = {"40k": PROMPTS["40k"][5]}
    R["prompts"] = {b: {"n": len(p), "len": len(p[0])} for b, p in PROMPTS.items()}
    log("prompts", json.dumps(R["prompts"]), "| render", json.dumps({k: v for k, v in R["prompt_render"].items() if "text" not in k}))
    save()


In [ ]:
if not RERUN:
    # ==================================================================== CELL 4 — the step runner (E23, verbatim)
    HELP = HELP if "HELP" in globals() else {}
    WIN = float(os.environ.get("E23_WIN_SEC") or 90.0)   # 120 s FULL windows
    EST_SCALE = float(os.environ.get("E23_EST_SCALE") or 1.0) if FAKE else 1.0   # local harness only: walk the whole order
    SM_TF = MODE == "SMOKE"                                            # SMOKE teacher-forced: the 40k row only
    ROWS = {}            # oracle greedy rows by key (same weights)
    TF = {}              # teacher-forced rows by key
    NTOK = 128 if MODE == "FULL" else 64
    LIVE = {"rec": None}                 # rev 3c: windows land in the step record before every save (a hard kill keeps them)
    COMPILE_ERR = re.compile(r"ninja: build stopped|Error building extension|error: command '.*(nvcc|g\+\+|gcc|c\+\+)' failed|"
                             r"GLIBCXX_[0-9.]+' not found|unsupported GNU version|fatal error: ")
    OOM_ERR = re.compile(r"OutOfMemoryError|CUDA out of memory|out of memory")
    XID = re.compile(r"NVRM: Xid|\bXid \d+|CUDA error: an illegal memory access|illegal memory access was encountered|"
                     r"unspecified launch failure|CUDA error: misaligned address")       # rev 3e: device faults, not any CUDA error


    def keep_after_error(rec, e):
        """E25 NB1 R2: an exception after the measurements (post-processing) keeps the whole record + postprocess_error /
        postprocess_tb. A clean boot (status None / NOT_RUN / DONE / TF_OOM, correctness CLEAN, quality PASS) keeps its status
        (None / NOT_RUN -> DONE), so its dependent boots still run; any other record -> POSTPROCESS_ERROR (status_before kept).
        R2 audit N1-3: postprocess_ok False; whether the dependents run is decided by core_measurements_complete (core_check)."""
        rec["postprocess_ok"] = False
        rec["postprocess_error"] = f"{type(e).__name__}: {e}"[:800]
        rec["postprocess_tb"] = traceback.format_exc()[-2500:]
        if (rec.get("status") in (None, "NOT_RUN", "DONE", "TF_OOM") and rec.get("correctness") == "CLEAN"
                and rec.get("quality_gate") == "PASS"):
            if rec.get("status") in (None, "NOT_RUN"):
                rec["status"] = "DONE"
        else:
            rec["status_before_postprocess"] = rec.get("status")
            rec["status"] = "POSTPROCESS_ERROR"
        return rec


    def step(sec, name, est_min, fn):
        """One measured step: skipped when the budget cannot hold it, recorded on any exception, never PASS by default."""
        rec = R["sections"].setdefault(sec, {}).setdefault(name, {"status": "NOT_RUN"})
        R["order"].append(f"{sec}/{name}")
        if ABORT["why"]:
            rec.update(status="SKIPPED_ABORT", why=ABORT["why"])
            log("---", sec, name, "SKIPPED_ABORT")
            save()
            return rec
        if left() < est_min * EST_SCALE * 60 + 300:
            rec.update(status="SKIPPED_BUDGET", left_min=round(left() / 60, 1), est_min=est_min)
            log("---", sec, name, "SKIPPED_BUDGET", f"(est {est_min} min, left {left() / 60:.0f} min)")
            save()
            return rec
        LIVE["rec"] = rec
        LIVE["out"] = None                       # E25 NB1 R2: the body's record once it returned (set by watched() / run_boot())
        t = time.time()
        log(">>>", sec, name, f"(est {est_min} min, left {left() / 60:.0f} min)")
        out = None                               # E25 NB1 R2
        try:
            out = fn() or {}
            rec.update(out)
            if rec.get("status") in (None, "NOT_RUN"):
                rec["status"] = "DONE"
        except Exception as e:
            part = out if isinstance(out, dict) and out else LIVE.get("out")   # E25 NB1 R2: never discard what the body produced
            if isinstance(part, dict) and part:                               # E25 NB1 R2
                rec.update(part)                                              # E25 NB1 R2
                keep_after_error(rec, e)                                      # E25 NB1 R2: clean -> DONE + postprocess_error
            else:                                                             # E25 NB1 R2
                rec.update(status="ERROR", error=f"{type(e).__name__}: {e}"[:800], tb=traceback.format_exc()[-2500:])
        rec["sec"] = round(time.time() - t, 1)
        LIVE["rec"] = None
        LIVE["out"] = None                       # E25 NB1 R2
        if ABORT["why"] and rec["status"] in ("DONE", "SERVER_DIED", "LAUNCH_FAILED", "BODY_ERROR"):
            rec["status"] = "PARTIAL_HARD_STOP"
        log("<<<", sec, name, rec.get("status"), f"{rec['sec']:.0f}s")
        save()
        return rec


    def with_servers(servers, timeout, body):
        """Start, wait ready, record facts + memory, run body(servers), always kill; then the GPU-release gate on these
        servers' GPUs only; abort the lab if they stay held."""
        out = {"servers": {}}
        timeout = max(60, min(timeout, left() - 120))
        try:
            for s in servers:
                s.start()
            for s in servers:
                ok, why = s.wait_ready(timeout)
                out["servers"][s.name] = {"ready": ok, "why": why, **s.facts(), "argv_tail": s.argv[-44:], "gpus": s.gpus,
                                          "help_check": HELP.get(s.name)}
                if not ok:
                    out["status"] = "LAUNCH_FAILED"
                    out["servers"][s.name]["log_tail"] = s.log()[-4000:]
                    _xl = [l for l in (s.log() or "").splitlines() if XID.search(l)][:8]   # rev 3e/3f: start-up / capture faults
                    out["servers"][s.name].update(xid=bool(_xl), xid_lines=_xl)
                    return out
            mem0 = L.gpu_mem()
            for s in servers:
                out["servers"][s.name].update(gpu_mem_after_ready=mem0, host_rss_mb=s.rss_mb())
            try:                             # rev 3: an exception in the body keeps everything measured so far
                out.update(body(servers) or {})
            except Exception as e:
                out.update(status="BODY_ERROR", body_error=f"{type(e).__name__}: {e}"[:800], body_tb=traceback.format_exc()[-2000:])
            for s in servers:
                out["servers"][s.name].update(s.facts(), gpu_mem_end=L.gpu_mem(), host_rss_mb_end=s.rss_mb(),
                                              alive_after_body=s.alive(), rc=None if s.alive() else s.proc.returncode,
                                              xid_lines=[l for l in (s.log() or "").splitlines() if XID.search(l)][:8])
                out["servers"][s.name]["xid"] = bool(out["servers"][s.name]["xid_lines"])
                if not s.alive():
                    out["status"] = "SERVER_DIED"
                    out["servers"][s.name]["log_tail"] = s.log()[-4000:]
            return out
        finally:
            for s in servers:
                s.kill()
            gpus = ",".join(sorted({g for s in servers for g in s.gpus.split(",")}))
            rel = L.wait_gpu_release(180, fake=bool(FAKE), gpus=gpus)
            if rel is False:                 # rev 3c: one more kill + 300 s before the lab aborts
                for s in servers:
                    s.kill()
                rel = L.wait_gpu_release(300, fake=bool(FAKE), gpus=gpus)
            if rel is None and NVSMI_OK:     # nvidia-smi worked at start: a failure now is not "released"
                rel = False
            out["gpu_released"] = rel
            if rel is False:
                ABORT["why"] = f"GPUs {gpus} still held after {[s.name for s in servers]}"
                out["status"] = "FAIL_GPU_RELEASE"


    def oracle(port, key=None, refs=()):
        """Greedy rows (2k, 2k, 8k) + one 40k row; saved under `key` (a target-only oracle) or compared with ROWS[ref]."""
        rows = L.greedy(port, ORACLE_P + LONG_P, n_tok=NTOK, timeout=max(120, min(900, left() / 6)))
        clean = all(not r.get("error") and r.get("tokens") and not L.degenerate(r.get("text")) for r in rows)
        if key and clean:            # rev 3d: an errored / empty / garbage reference is never stored (consumers: NO_ORACLE)
            ROWS[key] = rows
        elif key:
            R["errors"].append(f"oracle {key}: reference not stored (error / empty / garbage row)")
        res = {"garbage": [L.degenerate(r["text"]) for r in rows], "errors": [r["error"] for r in rows],
               "text_head": [r["text"][:120] for r in rows]}
        if not FAKE:                 # rev 3h: chat-path render vs the production xhigh render (recorded per arm)
            res["template_probe"] = L.template_probe(port, TPL_MSG, TPL_IDS)
        for ref in refs:
            res["vs_" + ref] = L.compare(ROWS.get(ref), rows) if ROWS.get(ref) else {"status": "INCONCLUSIVE_NO_ORACLE"}
        return res


    def tforced(port, key=None, ref=None):
        """Teacher-forced 40k / 90k prompt logprobs (last 256 positions); saved under `key` or compared with TF[ref].
        rev 3c: on stock vLLM only on its own short launch (prompt logprobs = chunk x 248k-vocab logits: OOM at 0.90)."""
        rows = {b: L.teacher_forced(port, ids) for b, ids in TF_P.items() if not (SM_TF and b != "40k")}
        if key and L.tf_complete(rows):                                 # rev 3e: complete references only
            TF[key] = rows
        elif key:
            R["errors"].append(f"teacher-forced {key}: reference not stored (incomplete band)")
        if ref:
            return {b: L.compare_tf((TF.get(ref) or {}).get(b), rows[b]) for b in rows}
        return {b: {"n": (rows[b] or {}).get("n"), "error": (rows[b] or {}).get("error")} for b in rows}


    def levels(port, band, lv, server=None, **kw):
        """rev 3c: the guard includes admission (prefill) time; a band stops once KV-bound (the fit is recorded); a dead
        server stops the band; every window is written into the live step record before save()."""
        out, fit = {}, None
        for n in lv:
            k = f"{band}@{n}"
            need = WIN + 90 + min(3600, n * BANDS[band] / 1500)
            if ABORT["why"] or left() < need + 120:
                out[k] = {"status": "SKIPPED_BUDGET", "need_sec": round(need)}
            elif fit is not None:
                out[k] = {"status": "SKIPPED_KV_BOUND", "streams_fit": fit}
            elif server is not None and not server.alive():
                out[k] = {"status": "SKIPPED_SERVER_DEAD"}
            else:
                try:                         # rev 3: one failing level never loses the others
                    out[k] = L.steady(port, PROMPTS[band], n, secs=WIN, max_new=MAX_NEW[band], **kw)
                except Exception as e:
                    out[k] = {"status": "ERROR", "error": f"{type(e).__name__}: {e}"[:300]}
                rm = out[k].get("running_max")
                if out[k].get("status") == "CAPACITY_LIMITED" or (rm is not None and 0 < rm < n and
                                                                  str(out[k].get("status")).startswith("INCONCLUSIVE")):
                    fit = rm
            if LIVE["rec"] is not None:
                LIVE["rec"].setdefault("live_windows", {})[k] = out[k]
            save()
        return out
    save()


In [ ]:
if not RERUN:
    # ==================================================================== CELL 5 — the arms
    LADDER = [1, 2, 4, 5]                    # 40k: 8 streams would need 8 x (10 + 7) blocks of 4096 > 149 FP8 blocks (preempts)
    TEMP = 0.6                               # production sampling (Merlin's --override-generation-config is T 1.0)
    E23_TF_SCALE = {"tp4_vs_tp1_layout": 0.0772, "nvfp4_vs_bf16_kv": 0.3787}   # E23 SMOKE rev3g mean |d logprob|, for scale
    PREFILL = {"tok_s": 700.0, "measured": False}
    OOM_STEP = 268435456                     # Merlin's note: lower KV_BYTES in 0.25 GiB steps if the boot OOMs
    # Pre-run capacity (Merlin's rule: blocks per request = ceil(L / 4096) + 7; FP8 149 usable blocks, BF16 ~78)
    R["capacity_plan"] = {
        "fp8": {"pool_blocks": 149, "40k@5": 5 * (math.ceil(130560 / 4096) + 7), "8k@8": 8 * (math.ceil(32000 / 4096) + 7),
                "90k@4": 4 * (math.ceil(131072 / 4096) + 7), "116k@3": 3 * (math.ceil(131072 / 4096) + 7),
                "116k@4": 4 * (math.ceil(131072 / 4096) + 7),
                "note": "blocks if every stream grew to max length; the ladder windows are 90 s, so real use is far lower; "
                        "90k@4 and 116k@4 are capacity-limit probes"},
        "bf16": {"pool_blocks": 78, "40k@3": 3 * (math.ceil(130560 / 4096) + 7)},
        "long_rows": "1 natural xhigh ARC row (max 120,000 new) + 1 forced 60,000-token row, temperature 0.6, run CONCURRENTLY "
                     "(concurrency 2): not a single-stream latency number",
    }
    R["correctness_meaning"] = ("Both Merlin boots run MTP K3: CLEAN = BF16_MTP3 vs FP8_MTP3 parity plus probes / sampled / greedy "
                                "evidence; it is NOT target-only (MTP off) correctness. A production GO needs an MTP-off oracle and "
                                "a forced-rejection test.")
    CLEAN = ("EXACT", "NEAR_TIE")


    def _windows(o):
        """Every steady window of a step record (levels dicts + steady_multi replicas)."""
        out = []
        for v in (o.get("steady") or {}).values():
            if isinstance(v, dict):
                out.append(v)
        for k in ("40k", "90k"):
            for r in ((o.get(k) or {}).get("replicas") or []) if isinstance(o.get(k), dict) else []:
                if isinstance(r, dict):
                    out.append(r)
        return out


    def oracle_gate(o, need=(), extra=()):
        """None when clean; otherwise the NO_GO / INCONCLUSIVE status. `need` = refs that must compare clean; `extra` =
        further comparison dicts (replica greedy, 90k greedy) that must compare clean too. rev 3e (audits B-O1, C-F2):
        garbage under load in any window and failed arithmetic probes (real servers; the fake has no chat) are NO_GO too."""
        orc = o.get("oracle") or {}
        if any(orc.get("errors") or []):
            return "NO_GO_ORACLE_REQUEST_ERROR"
        if any(orc.get("garbage") or []):
            return "NO_GO_ORACLE_GARBAGE"
        if any(w.get("status") == "FAIL_GARBAGE_UNDER_LOAD" for w in _windows(o)):
            return "NO_GO_GARBAGE_UNDER_LOAD"
        if not FAKE and isinstance(o.get("probes"), dict) and o["probes"].get("ok") is False:
            return "NO_GO_SHORT_PROBE"
        sts = [(orc.get("vs_" + k) or {}).get("status") for k in need] + [(x or {}).get("status") for x in extra]
        if any(s == "FAIL_HIGH_MARGIN" for s in sts):
            return "NO_GO_ORACLE"
        if any(s not in CLEAN for s in sts):
            return "INCONCLUSIVE_NO_ORACLE" if any(s in (None, "INCONCLUSIVE_NO_ORACLE") for s in sts) else "INCONCLUSIVE_ORACLE"
        return None


    def gated(o, need=(), extra=()):
        g = oracle_gate(o, need, extra)
        o["correctness"] = g or "CLEAN"
        if g and o.get("status") in (None, "DONE"):
            o["status"] = g
        return o




    def _finite(x):
        return isinstance(x, (int, float)) and not isinstance(x, bool) and math.isfinite(x)


    _compare_e23, _compare_tf_e23 = L.compare, L.compare_tf


    def _compare_rc2(ref, cand):
        """Final audit H2: a non-finite margin on either side makes the comparison INCONCLUSIVE before any token logic
        (E23's compare read a NaN reference margin as NEAR_TIE)."""
        for rows in (ref or [], cand or []):
            for r in rows:
                if any(m is not None and not _finite(m) for m in (r.get("margins") or [])):
                    return {"status": "INCONCLUSIVE_NONFINITE"}
        return _compare_e23(ref, cand)


    def _compare_tf_rc2(ref, cand):
        """Final audit H2: a non-finite teacher-forced logprob is never MEASURED."""
        for side in (ref, cand):
            for p in ((side or {}).get("positions") or []):
                if p and p[0] is not None and not _finite(p[0]):
                    return {"status": "INCONCLUSIVE_NONFINITE"}
        return _compare_tf_e23(ref, cand)


    L.compare, L.compare_tf = _compare_rc2, _compare_tf_rc2


    def short_gate_failed(o):
        """Final audit H1: a failed probe (real servers; the fake has no chat) or a failed / garbage sampled request stops
        the boot's body at once; gated() then gives the NO_GO verdict."""
        p, s = o.get("probes") or {}, o.get("sampled") or {}
        return (p.get("ok") is not True and not FAKE) or s.get("ok") is not True or bool(s.get("garbage"))


    def oracle_ok(orc):
        return bool(orc) and orc.get("complete") is True and not any(orc.get("errors") or []) and not any(orc.get("garbage") or [])


    def _windows(o):
        """Every load window of a record: the ladder AND the long census (audit: census_long was not gated)."""
        out = []
        for key in ("steady", "census_long"):
            for v in (o.get(key) or {}).values():
                if isinstance(v, dict):
                    out.append(v)
        return out


    def evidence_gate(o, expected_rows, require_sampled=True):
        """Fail-closed evidence (three pre-GPU audits): probes, the sampled path and a COMPLETE oracle are mandatory;
        a missing piece is never CLEAN. expected_rows None = no oracle in this arm (diagnostic arms)."""
        pr = o.get("probes")
        if not isinstance(pr, dict):
            return "INCONCLUSIVE_PROBES_MISSING"
        if pr.get("ok") is not True and not FAKE:          # the fake server has no chat endpoint (E23 rule)
            return "NO_GO_SHORT_PROBE"
        if require_sampled:
            sm = o.get("sampled")
            if not isinstance(sm, dict):
                return "INCONCLUSIVE_NO_SAMPLED_TEST"
            if sm.get("ok") is not True or sm.get("garbage"):
                return "NO_GO_SAMPLED_PATH"
        if expected_rows is None:
            return None
        orc = o.get("oracle")
        if not isinstance(orc, dict):
            return "INCONCLUSIVE_NO_ORACLE"
        if any(orc.get("errors") or []):
            return "NO_GO_ORACLE_REQUEST_ERROR"
        if any(orc.get("garbage") or []):
            return "NO_GO_ORACLE_GARBAGE"
        if orc.get("row_count") != expected_rows or orc.get("complete") is not True:
            return "INCONCLUSIVE_ORACLE_INCOMPLETE"
        return None


    _gated_e23 = gated


    def gated(o, need=(), extra=(), expected_rows=4, require_sampled=True):
        g = evidence_gate(o, expected_rows, require_sampled)
        if g:
            o["correctness"] = g
            if o.get("status") in (None, "DONE"):
                o["status"] = g
            return o
        return _gated_e23(o, need, extra)


    def oracle(port, key=None, refs=(), prompts=None):
        """E23's oracle plus completeness: every expected row present, no error, exactly NTOK tokens (greedy rows run with
        ignore_eos, so a short row is a defect); a reference is stored only when complete and garbage-free."""
        prompts = list(prompts) if prompts is not None else ORACLE_P + LONG_P
        rows = L.greedy(port, prompts, n_tok=NTOK, timeout=max(120, min(900, left() / 6)))
        garbage = [L.degenerate(r.get("text")) for r in rows]
        complete = len(rows) == len(prompts) and all(not r.get("error") and len(r.get("tokens") or []) == NTOK for r in rows)
        if key and complete and not any(garbage):
            ROWS[key] = rows
        elif key:
            R["errors"].append(f"oracle {key}: reference not stored (incomplete / error / garbage)")
        res = {"garbage": garbage, "errors": [r.get("error") for r in rows], "text_head": [(r.get("text") or "")[:120] for r in rows],
               "row_count": len(rows), "expected_rows": len(prompts), "tokens": [len(r.get("tokens") or []) for r in rows],
               "complete": complete}
        if not FAKE:
            res["template_probe"] = L.template_probe(port, TPL_MSG, TPL_IDS)
        for ref in refs:
            res["vs_" + ref] = L.compare(ROWS.get(ref), rows) if ROWS.get(ref) else {"status": "INCONCLUSIVE_NO_ORACLE"}
        return res


    def kv_census(s):
        log_t = s.log() or ""
        lines = log_t.splitlines()
        kv = re.findall(r"GPU KV cache size: ([\d,]+) tokens", log_t)
        conc = re.findall(r"Maximum concurrency for ([\d,]+) tokens per request: ([\d.]+)x", log_t)
        pick = lambda rx, n=24: [l[:240] for l in lines if re.search(rx, l, re.I)][:n]
        return {"kv_tokens": [int(x.replace(",", "")) for x in kv][:2], "max_concurrency": conc[:2],
                "scale_lines": pick(r"K/V scale|kv scale|scaling factor|DEFAULT 1\.0|static K/V"),
                "engine_lines": pick(r"WNA16 MoE backend|MarlinExperts|PLE offload|engram|cudagraph_mode|CUDAGraphMode|PIECEWISE"),
                "startup_lines": pick(r"Loading weights took|Model loading took|torch\.compile takes|torch\.compile took|"
                                      r"Graph capturing finished|init engine|PLE.*ready|Available KV cache memory"),
                "oom_lines": [l[:240] for l in lines if OOM_ERR.search(l)][:6], "host_mem": host_mem()}


    def sampled_check(s):
        """RC1 died at its first sampled request (FlashInfer sampling JIT: ld cannot find -lcuda): one short sampled request
        right after ready, then the engine must still be alive."""
        g = L.Stream(s.port, PROMPTS["2k"][1], 64, temperature=TEMP, seed=5)
        g.start()
        g.join(900)
        if g.is_alive():
            g.stop()
            g.join(10)
        r = g.row()
        return {"tokens": len(r["tokens"]), "error": r["error"], "finish": r["finish_reason"], "alive_after": s.alive(),
                "garbage": L.degenerate(r["text"]), "ok": len(r["tokens"]) == 64 and not r["error"] and s.alive()}


    def prefill_probe(port):
        """Time to first token of one cold 40k prompt (1 new token): the measured prefill rate the ladder admission uses."""
        ids = PROMPTS["40k"][7]
        g = L.Stream(port, ids, 1, temperature=0.0)
        g.start()
        g.join(1800)
        if g.is_alive():
            g.stop()
            g.join(10)
        sec = (g.t_first or g.t_end or time.perf_counter()) - (g.t_start or time.perf_counter())
        ok = not g.err and g.t_first is not None
        if ok:
            PREFILL.update(tok_s=round(len(ids) / max(1e-3, sec), 1), measured=True)
        return {"prompt": len(ids), "ttft_sec": round(sec, 1), "tok_s": round(len(ids) / max(1e-3, sec), 1), "error": g.err, "ok": ok}


    def window_garbage(ids, win=2000, min_len=200):
        """Final audit H3: the first garbage 2,000-token window of a row (None if every window is clean). A whole-text check
        would read 100k tokens of healthy reasoning as LOW_VARIETY (unique words < 8 %), so windows are checked one by one."""
        for i in range(0, len(ids), win):
            seg = ids[i:i + win]
            if len(seg) < min_len:
                break
            v = L.degenerate(TOK.decode(seg))
            if v:
                return {"start_token": i, "verdict": v}
        return None


    def long_quality(name, r):
        if not isinstance(r, dict):
            return "MISSING_LONG"
        if r.get("error"):
            return "FAIL_LONG_REQUEST"
        if r.get("window_garbage"):
            return "FAIL_LONG_GARBAGE"
        if name == "forced60k" and not r.get("reached_60k"):
            return "INCOMPLETE_LONG"
        if name == "natural" and r.get("cut_by_cap"):
            return "INCOMPLETE_LONG"
        return "PASS_LONG"


    def long_gate(o):
        """None when the record has no long rows or both rows PASS_LONG; otherwise the verdict for the top-level status.
        Priority: request error > natural garbage > forced garbage (synthetic past its first EOS) > incomplete."""
        lo = o.get("long")
        if not isinstance(lo, dict):
            return None
        q = {n: long_quality(n, lo.get(n)) for n in ("natural", "forced60k")}
        lo["long_quality"] = q
        if "MISSING_LONG" in q.values():
            return "INCONCLUSIVE_LONG_MISSING"
        if "FAIL_LONG_REQUEST" in q.values():
            return "NO_GO_LONG_REQUEST_ERROR"
        if q["natural"] == "FAIL_LONG_GARBAGE":
            return "NO_GO_LONG_GARBAGE"
        if q["forced60k"] == "FAIL_LONG_GARBAGE":
            return "NO_GO_FORCED60K_GARBAGE"
        if "INCOMPLETE_LONG" in q.values():
            return "INCONCLUSIVE_LONG_INCOMPLETE"
        return None


    def long_rows(port, secs_cap=3000):
        """One natural xhigh ARC request (its own stop, up to 120,000 new tokens) and one forced 60,000-token row
        (ignore_eos: 'synthetic' past its first EOS), CONCURRENTLY (concurrency 2). Full text in traces/ + requests.jsonl.
        A row cut by the time cap is incomplete, not a correctness failure."""
        cap = max(60, min(secs_cap, left() - 900))
        jobs = {"natural": L.Stream(port, TPL_IDS, min(120000, MAX_LEN - len(TPL_IDS) - 512), temperature=TEMP, seed=11,
                                    ignore_eos=False),
                "forced60k": L.Stream(port, PROMPTS["8k"][0], 60000, temperature=TEMP, seed=12, ignore_eos=True)}
        t = time.time()
        for g in jobs.values():
            g.start()
        for g in jobs.values():
            g.join(max(1, cap - (time.time() - t)))
        for g in jobs.values():
            if g.is_alive():
                g.stop()
                g.join(10)
        out = {"concurrency": 2, "cap_sec": round(cap)}
        for name, g in jobs.items():
            row = g.row()
            (TRACES / f"long_{port}_{name}.txt").write_text(row["text"])
            ids = row["tokens"]
            n_tok = len(ids)
            sec = (g.t_end or time.perf_counter()) - (g.t_start or time.perf_counter())
            sl = {}
            for k, end in (("to_8k", 8000), ("to_32k", 32000), ("to_60k", 60000), ("to_100k", 100000)):
                sl[k] = L.degenerate(TOK.decode(ids[end - 2000:end])) if n_tok >= end else "NOT_REACHED"
            out[name] = {"tokens": n_tok, "sec": round(sec, 1), "tok_s": round(n_tok / max(1e-6, sec), 1), "error": row["error"],
                         "finish": row["finish_reason"], "garbage": sl, "window_garbage": window_garbage(ids),
                         "think_closed": "</think>" in row["text"], "cut_by_cap": row["finish_reason"] is None,
                         "tail": row["text"][-400:]}
        out["forced60k"]["reached_60k"] = out["forced60k"]["tokens"] >= 60000
        out["forced60k"]["label"] = "synthetic past the first EOS (ignore_eos)"
        try:
            m = {k: v for k, v in L.prom(port).items() if k.startswith(("vllm:spec_decode", "vllm:num_preemptions", "vllm:kv_cache_usage"))}
        except Exception as e:
            m = {"error": repr(e)[:200]}
        acc, drf = m.get(L.ACC), m.get(L.DRF)
        out["metrics_end"] = m
        out["mtp_accept_per_draft"] = round(acc / drf, 3) if acc and drf else None
        return out


    def tf_last(s, key=None, ref=None, aa=False):
        """Teacher-forced 40k LAST in a boot (chunk x 248k-vocab prompt logits can OOM the last PP rank); free memory first.
        How a server death here is classified is decided from its log in merlin_boot (TF_OOM vs TF_SERVER_DIED)."""
        o = {"gpu_before": L.gpu_mem()}
        if key:
            o["ref"] = tforced(s.port, key=key)
            if aa and s.alive():
                o["aa"] = tforced(s.port, ref=key)              # A/A: the same boot, the same prompt (the noise floor)
                TF["aa"] = o["aa"]
        if ref:
            o["vs_" + ref] = tforced(s.port, ref=ref)
            o["aa_" + ref] = TF.get("aa")
            o["e23_scale"] = E23_TF_SCALE
        o["alive_after"] = s.alive()
        return o


    def merlin_boot(name, kv, port, timeout, body, need=(), expected_rows=4, require_sampled=True):
        """Boot gate (ok must be True), one Merlin boot, cleanup of the /dev/shm entries this boot created; a boot OOM
        retries once 0.25 GiB lower. A server death during TF is TF_OOM only with an OOM line and no device fault."""
        kvb = NB1_KV.get(name) or (KV_FP8 if kv == "fp8" else KV_BF16)   # E25 NB1: the B3 pin step
        tries = []
        while True:
            gate = boot_gate()
            if gate["ok"] is not True:
                return {"status": "GPU_NOT_EMPTY", "boot_gate": gate, "tries": tries}
            shm0 = shm_names()
            o = with_servers([merlin_fn_server(name, kv, port, kvb)], timeout, body)
            o.update(kv_bytes=kvb, boot_gate=gate, shm_cleanup=shm_cleanup(shm0))
            srv = (o.get("servers") or {}).get(name) or {}
            oom = o.get("status") == "LAUNCH_FAILED" and bool(OOM_ERR.search(srv.get("log_tail") or ""))
            tries.append({"kv_bytes": kvb, "status": o.get("status"), "oom": oom})
            if not oom or len(tries) > 1 or left() < timeout + 900:
                break
            kvb -= OOM_STEP
            log("boot OOM:", name, "retry with KV_BYTES", kvb)
        o["tries"] = tries
        if o.get("tf_died") and o.get("status") == "SERVER_DIED":
            srv = (o.get("servers") or {}).get(name) or {}
            if OOM_ERR.search(srv.get("log_tail") or "") and not srv.get("xid"):
                o["status"] = "TF_OOM"
                o["status_note"] = "the server died of OOM in the teacher-forced call (run last); everything before it is valid"
            else:
                o["status"] = "TF_SERVER_DIED"
                o["status_note"] = "the server died in the teacher-forced call without an OOM line (or with a device fault)"
        g = gated(o, need=need, expected_rows=expected_rows, require_sampled=require_sampled)
        if g.get("status") == "TF_OOM" and g.get("correctness") != "CLEAN":
            g["status"] = g["correctness"]
        lg = long_gate(g)                                  # final audit H3: the long rows have their own verdict
        g["long_verdict"] = lg or ("PASS_LONG" if isinstance(g.get("long"), dict) else None)
        if lg and g.get("status") in ("DONE", "TF_OOM"):
            g["status"] = lg
        return g


    def arm_bf16():
        def body(ss):
            s = ss[0]
            p = s.port
            o = {"census": kv_census(s), "probes": L.chat_probes(p), "sampled": sampled_check(s)}
            if short_gate_failed(o):
                o.update(early_stop="SHORT_CORRECTNESS_GATE_FAILED", census_end=kv_census(s))
                return o
            o["prefill"] = prefill_probe(p)
            o["oracle"] = oracle(p, key="bf16")
            o["census_end"] = kv_census(s)
            if not oracle_ok(o["oracle"]):
                o["early_stop"] = "ORACLE_INCOMPLETE"          # no reference: the TF A/A would measure nothing usable
                return o
            o["tf"] = tf_last(s, key="bf16", aa=True)
            o["tf_died"] = not o["tf"]["alive_after"]
            return o
        ok, why = sha_gate(min(1500, left() - 900))
        R["setup"]["sha_gate"] = {"ok": ok, "why": why}
        if not ok:
            return {"status": "SKIPPED_SHA_GATE", "why": why}
        return merlin_boot("merlin_bf16", "bf16", 8000, 45 * 60, body)


    def fp8_gate(o):
        vs = (o["oracle"].get("vs_bf16") or {}).get("status")
        if (o.get("sampled") or {}).get("ok") is not True:
            return "FAIL_SAMPLED"
        if (o.get("probes") or {}).get("ok") is not True and not FAKE:
            return "FAIL_PROBES"
        if any(o["oracle"].get("errors") or []) or any(o["oracle"].get("garbage") or []) or vs == "FAIL_HIGH_MARGIN":
            return "FAIL"
        if o["oracle"].get("complete") is True and vs in CLEAN:
            return "PASS"
        return "INCONCLUSIVE"


    def arm_fp8():
        def body(ss):
            s = ss[0]
            p = s.port
            o = {"census": kv_census(s), "probes": L.chat_probes(p), "sampled": sampled_check(s)}
            if short_gate_failed(o):
                o.update(early_stop="SHORT_CORRECTNESS_GATE_FAILED", census_end=kv_census(s),
                         fp8_gate="FAIL_SAMPLED" if (o["sampled"] or {}).get("ok") is not True else "FAIL_PROBES")
                return o
            if not PREFILL["measured"]:
                o["prefill"] = prefill_probe(p)
            o["oracle"] = oracle(p, refs=("bf16",))
            o["fp8_gate"] = fp8_gate(o)
            if o["fp8_gate"] == "PASS":                      # load tests only on a PASS (audits: INCONCLUSIVE never runs them)
                kw = {"server": s, "temperature": TEMP, "prefill_tok_s": PREFILL["tok_s"]}
                o["steady"] = levels(p, "40k", LADDER, **kw)
                o["steady"].update(levels(p, "8k", [8], **kw))
                o["census_long"] = {**levels(p, "90k", [3, 4], **kw), **levels(p, "116k", [3, 4], **kw)}
                o["long"] = long_rows(p)
            else:
                o["load_tests_skipped"] = f"FP8 comparison gate: {o['fp8_gate']}"
                o["status"] = "NO_GO_FP8_KV" if o["fp8_gate"].startswith("FAIL") else "INCONCLUSIVE_FP8_KV"
            o["census_end"] = kv_census(s)
            if o["fp8_gate"] == "PASS" or os.environ.get("RC2_TF_ON_FP8_FAIL") == "1":   # default: fail fast
                o["tf"] = tf_last(s, ref="bf16")
                o["tf_died"] = not o["tf"]["alive_after"]
            return o
        return merlin_boot("merlin_fp8", "fp8", 8001, 40 * 60, body, need=("bf16",))


    def arm_bf16_load():
        """Only when FP8 fails: the ladder, the census and the long rows on BF16 KV (seqs 3). DIAGNOSTIC_ONLY: this arm
        has no oracle of its own (the A1 boot is the BF16 reference)."""
        def body(ss):
            s = ss[0]
            p = s.port
            o = {"census": kv_census(s), "probes": L.chat_probes(p), "sampled": sampled_check(s)}
            if short_gate_failed(o):
                o["early_stop"] = "SHORT_CORRECTNESS_GATE_FAILED"
            else:
                kw = {"server": s, "temperature": TEMP, "prefill_tok_s": PREFILL["tok_s"]}
                o["steady"] = levels(p, "40k", [1, 2, 3], **kw)
                o["census_long"] = levels(p, "116k", [1, 2], **kw)
                o["long"] = long_rows(p)
            o["census_end"] = kv_census(s)
            o["certification"] = "DIAGNOSTIC_ONLY"
            return o
        return merlin_boot("merlin_bf16_load", "bf16", 8002, 30 * 60, body, expected_rows=None)


    def arm_stock():
        """Fallback when Merlin cannot serve: stock vLLM 0.32 + Intel W4A16 (RC1's correct engine), -lcuda shim, FP8 KV."""
        if INTEL is None or WHEELHOUSE is None:
            return {"status": "FALLBACK_UNAVAILABLE", "why": f"intel={INTEL} wheelhouse={WHEELHOUSE}"}

        def body(ss):
            s = ss[0]
            p = s.port
            o = {"census": kv_census(s), "probes": L.chat_probes(p), "sampled": sampled_check(s)}
            if short_gate_failed(o):
                o["early_stop"] = "SHORT_CORRECTNESS_GATE_FAILED"
                return o
            o["oracle"] = oracle(p, key="stock", prompts=ORACLE_P)          # 2k / 2k / 8k (32k max length: no 40k row)
            if oracle_ok(o["oracle"]):
                o["steady"] = levels(p, "8k", [1, 2, 3], server=s, temperature=TEMP)
            return o
        gate = boot_gate()
        if gate["ok"] is not True:
            return {"status": "GPU_NOT_EMPTY", "boot_gate": gate}
        shm0 = shm_names()
        o = gated(with_servers([stock_fn_server("stock_fn", 8003)], 40 * 60, body), expected_rows=len(ORACLE_P))
        o.update(boot_gate=gate, shm_cleanup=shm_cleanup(shm0))
        return o


    save()


In [ ]:
if not RERUN:
    # ==================================================================== CELL 6 — E25 NB1: measurement tools (NB1 R2 verbatim; E25 NB4S dropped the unused capacity / prefix / routing / PP2-trace functions: commit_mode, commit_ab, commit_counter_delta, pool_blocks_from_usage, reset_cache, _hits, alloc_sequence, reset_succeeded, fit_alloc, kv_accounting, n_fit, admission_test, residency_bound, residency_capacity, _f_valid, resolve_F, f_from_log, residency_probes, block_layout, nb1_complete, nb1_tf_req, lp_compare, fp8_self_check, nb1_ladder, _prefix_counts, _reuse, prefix_tests, _merge, _clip, _inter, microbatch_costs, pp2_trace, merge_routing, pp2_profile)
    import gzip
    import statistics
    from collections import Counter
    NB1 = R.setdefault("nb1", {})
    SCORECARD = OUT / "lever_scorecard.json"
    TF_RAW = OUT / "tf_raw"
    TF_RAW.mkdir(exist_ok=True)
    HOSTMEM = OUT / "hostmem"
    HOSTMEM.mkdir(exist_ok=True)
    BLOCK = 4096
    PIN_STEPS_GIB = (1.0, 0.75, 0.5)             # B3 pin ladder (addendum 5): the largest step predicted safe first, then down
    PIN_FREE_MIN_MIB = 512                       # a step must leave >= 0.5 GiB free per rank after warm-up + a 90k prefill
    KV_STEP = 512 << 20                          # the smallest step (+0.5 GiB per card)
    RAM_NEED_MIB = (69 + 4) * 1024               # Merlin's ~69 GiB resident floor + 4 GiB margin, on its own rule
    BOOT_READY_SEC = 30 * 60                     # readiness gate per boot
    SEQS_HI = 16
    CAPTURE_HI = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 40, 48, 64]   # Merlin registry.capture_sizes_for(16, K=3): seqs x (1 + K)
    COMPILE_HI = json.dumps({"cudagraph_mode": "FULL_AND_PIECEWISE", "cudagraph_capture_sizes": CAPTURE_HI})
    PROF_DIR = TMP / "nb1_profile"
    PROFILER_CFG = json.dumps({"profiler": "torch", "torch_profiler_dir": str(PROF_DIR), "torch_profiler_with_stack": False,
                               "ignore_frontend": True})      # idle unless /start_profile (annotations only while running)
    EXT_FLAG = ["--worker-extension-cls", "nb1_worker_ext.NB1WorkerExt"]
    DIAG_ENV = {"VLLM_SERVER_DEV_MODE": "1"}     # /collective_rpc (read-only extension) + /reset_prefix_cache; 127.0.0.1 only
    NCCL_DIAG = {"NCCL_DEBUG": "INFO", "NCCL_DEBUG_SUBSYS": "INIT,NET"}   # B1 only: the real NCCL transport per group
    COMMIT_GRAPH_ENV = {"VLLM_RECOVERSSM_COMMIT_GRAPH": "graph"}         # B2 boot: commit graphs captured; flipped to eager at once
    RC2_FP8 = json.loads(r"""{"argv_tail": ["127.0.0.1", "--port", "8001", "--tensor-parallel-size", "2", "--pipeline-parallel-size", "2", "--enable-expert-parallel", "--disable-custom-all-reduce", "--speculative-config", "{\"method\":\"mtp\",\"num_speculative_tokens\":3,\"draft_sample_method\":\"probabilistic\"}", "--max-model-len", "262144", "--max-num-seqs", "8", "--max-num-batched-tokens", "1024", "--gpu-memory-utilization", "0.96", "--kv-cache-memory-bytes", "2479838700", "--kv-cache-dtype", "fp8_e4m3", "--limit-mm-per-prompt", "{\"image\":42,\"video\":0}", "--mm-processor-cache-gb", "0", "--mm-processor-kwargs", "{\"size\":{\"longest_edge\":8388608,\"shortest_edge\":65536}}", "--compilation-config", "{\"cudagraph_mode\":\"FULL_AND_PIECEWISE\",\"cudagraph_capture_sizes\":[1,2,3,4,6,8,12,16,24,32]}", "--enable-prefix-caching", "--mamba-cache-mode", "align", "--enable-prompt-tokens-details", "--no-enable-flashinfer-autotune", "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder", "--reasoning-parser", "qwen3", "--override-generation-config", "{\"temperature\":1.0,\"top_p\":0.95,\"top_k\":20,\"min_p\":0.0,\"presence_penalty\":0.0,\"repetition_penalty\":1.0}", "--default-chat-template-kwargs", "{\"enable_thinking\": true, \"reasoning_effort\": \"medium\"}", "--shutdown-timeout", "60", "--block-size", "4096", "--use-replayssm", "--prefix-match-unit", "64", "--served-model-name", "q27", "--max-model-len", "131072", "--port", "8001", "--default-chat-template-kwargs", "{\"enable_thinking\": true, \"reasoning_effort\": \"xhigh\"}"], "env": {"KV_BYTES": "2479838700", "NCCL_P2P_DISABLE": "1", "NCCL_VERSION": "2.25.1-1", "P2P": "off", "SCALES": "/tmp/fn_merlin/qsa_kv_scales_262k.json", "VLLM_CACHE_ROOT": "/tmp/merlin-cache", "VLLM_E2_GUARD_MODE": "warn", "VLLM_E2_PLE_EARLY_REQUEST": "1", "VLLM_E2_PLE_FAST_IDS": "1", "VLLM_E2_QSA_LOGITS_GRANULE_MB": "256", "VLLM_HOST_EMBED_TABLE": "1", "VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS": "0", "VLLM_PLE_OFFLOAD_HOME_DEVICE": "0", "VLLM_PLE_OFFLOAD_READY_TIMEOUT": "3600", "VLLM_PORT": "30100", "VLLM_PP_LAYER_PARTITION": "25,23", "VLLM_QSA_KV_SCALES": "/tmp/fn_merlin/qsa_kv_scales_262k.json", "VLLM_SKIP_P2P_CHECK": "1", "VLLM_SPARSE_INDEXER_MAX_LOGITS_MB": "256", "VLLM_USE_BREAKABLE_CUDAGRAPH": "0", "VLLM_USE_RUST_FRONTEND": "0", "VLLM_USE_V2_MODEL_RUNNER": "1"}, "kv_bytes": 2479838700, "kv_tokens": 513940, "max_concurrency": 3.92, "pool_blocks": 149, "F_rule": 6, "F_engine": 6, "seqs": 8, "gpu_mem_after_ready": [[0, 20903, 23034], [1, 20903, 23034], [2, 20863, 23034], [3, 21389, 23034]], "gpu_mem_end": [[0, 21427, 23034], [1, 21427, 23034], [2, 21387, 23034], [3, 21401, 23034]], "windows": {"40k@1": {"status": "STEADY", "tok_s": 70.0, "mal": 3.086, "step_ms": 44.1, "running": [1.0, 1.0]}, "40k@2": {"status": "STEADY", "tok_s": 107.4, "mal": 3.251, "step_ms": 30.3, "running": [2.0, 2.0]}, "40k@4": {"status": "STEADY", "tok_s": 161.0, "mal": 3.017, "step_ms": 37.5, "running": [4.0, 4.0]}, "40k@5": {"status": "STEADY", "tok_s": 190.5, "mal": 3.073, "step_ms": 40.3, "running": [5.0, 5.0]}, "8k@8": {"status": "STEADY", "tok_s": 261.8, "mal": 3.154, "step_ms": 48.2, "running": [8.0, 8.0]}, "90k@3": {"status": "INCONCLUSIVE_PREFILL_IN_WINDOW", "tok_s": 48.7, "mal": 2.93, "step_ms": 60.1, "running": [1.0, 3.0]}, "90k@4": {"status": "STEADY", "tok_s": 159.8, "mal": 3.022, "step_ms": 37.8, "running": [4.0, 4.0]}, "116k@3": {"status": "STEADY", "tok_s": 140.5, "mal": 3.223, "step_ms": 34.4, "running": [3.0, 3.0]}, "116k@4": {"status": "STEADY", "tok_s": 160.3, "mal": 3.016, "step_ms": 37.6, "running": [4.0, 4.0]}}, "tf_40k_vs_bf16": {"status": "MEASURED", "n": 256, "mean_abs_dlogprob": 1.8764, "p99_abs_dlogprob": 5.464, "max_abs_dlogprob": 7.15, "rank1_agreement": 0.3672, "forced_token_rank1_both_or_neither": 0.457}, "ready_sec": 999.1, "bf16": {"kv_tokens": 269042, "max_concurrency": 2.05, "pool_blocks": 78, "seqs": 3, "gpu_mem_after_ready": [[0, 20905, 23034], [1, 20905, 23034], [2, 20693, 23034], [3, 20693, 23034]], "gpu_mem_end": [[0, 21425, 23034], [1, 21425, 23034], [2, 21215, 23034], [3, 21215, 23034]], "ready_sec": 411.1}, "rank_load_gib": {"2": 16.87, "1": 17.33, "0": 17.33, "3": 16.87}, "rank_graph_gib": {"0": 0.33, "1": 0.33, "2": 0.43, "3": 0.43}}""")      # RC2's real boots: argv tail, env keys, pool, memory, windows (rc2_results.json)
    RC2_ORACLE = json.loads(r"""{"bf16": [{"text": "We need solve ARC task. Need infer transformation from examples. User gives Task 0934a4d8 with Example 1 input/output, Example 2 input/output, Example 3 input only? Need solve test inputs? They provided Example 3 input: 194499276 maybe incomplete? Need output? Need find transformation rule and solve test inputs. We need analyze.\n\nWe have ARC tasks represented as strings of digits? Example 1 input is 31 rows? Let's count. Example 1 input lines:\n1 3533665414", "tokens": [1596, 1144, 11290, 67964, 3274, 13, 14235, 22846, 17454, 494, 9989, 13, 2570, 6486, 5261, 220, 15, 24, 18, 19, 64, 19, 67, 23, 440, 12996, 220, 16, 1879, 45447, 11, 12996, 220, 17, 1879, 45447, 11, 12996, 220, 18, 1879, 1132, 30, 14235, 11290, 1228, 10804, 30, 2302, 3766, 12996, 220, 18, 1879, 25, 220, 16, 24, 19, 19, 24, 24, 17, 22, 21, 6970, 31084, 30, 14235, 2468, 30, 14235, 1423, 17454, 5728, 321, 11290, 1228, 10804, 13, 1165, 1144, 22903, 13, 271, 1596, 599, 67964, 8806, 14807, 430, 8796, 314, 17955, 30, 12996, 220, 16, 1879, 369, 220, 18, 16, 6761, 30, 6558, 579, 1698, 13, 12996, 220, 16, 1879, 4965, 25, 198, 16, 220, 18, 20, 18, 18, 21, 21, 20, 19, 16, 19], "margins": [11.375, 10.75, 0.25, 3.5, 0.125, 1.75, 2.375, 0.625, 4.375, 0.125, 4.125, 1.375, 0.125, 1.0, 1.25, 4.875, 8.6875, 10.375, 10.375, 9.6875, 8.25, 10.1875, 14.0, 10.4375, 0.125, 0.875, 1.25, 8.375, 3.875, 1.0, 3.75, 7.375, 3.75, 12.0, 2.625, 8.375, 6.75, 2.5, 8.75, 10.5, 6.625, 0.125, 0.375, 0.5, 0.75, 8.25, 4.125, 1.0, 1.0, 0.125, 0.25, 4.625, 6.375, 9.25, 2.875, 0.375, 4.125, 4.25, 10.9375, 11.5625, 10.625, 10.5625, 9.5625, 11.5, 10.6875, 0.5, 2.25, 4.875, 2.125, 1.25, 0.125, 0.5, 0.625, 0.625, 4.25, 1.125, 2.875, 3.125, 3.125, 4.125, 0.75, 1.375, 1.75, 0.75, 1.0, 4.25, 1.5, 0.5, 1.125, 0.625, 4.625, 1.625, 0.25, 4.5, 1.0, 0.0, 1.25, 8.5, 5.0, 1.125, 2.75, 3.375, 0.625, 0.125, 1.125, 1.875, 7.25, 1.875, 2.125, 0.25, 0.875, 6.25, 1.875, 1.625, 2.625, 2.875, 2.125, 2.375, 8.125, 10.9375, 10.0625, 9.0625, 9.875, 9.375, 11.25, 11.375, 11.875, 12.9375], "error": null, "prompt_sha": "8df9ed2c640ef5bd", "prompt_len": 2048, "finish_reason": "length"}, {"text": "We need solve ARC tasks. Need infer transformation rules from examples and apply to test inputs. We have two tasks: 1818057f and 195c6913. Need output transformed test grids. Need analyze carefully.\n\nWe need produce final answer likely grids. Need infer rules.\n\nTask 1818057f:\nExamples: inputs/outputs with colors 2,4,8. Output changes some 4s to 8? Let's inspect.\n\nExample1 input 10x12? rows:\n1 22224", "tokens": [1596, 1144, 11290, 67964, 8806, 13, 14235, 22846, 17454, 5426, 494, 9989, 321, 3668, 310, 1228, 10804, 13, 1165, 599, 1330, 8806, 25, 220, 16, 23, 16, 23, 15, 20, 22, 69, 321, 220, 16, 24, 20, 66, 21, 24, 16, 18, 13, 14235, 2468, 22773, 1228, 54455, 13, 14235, 22903, 15060, 13, 271, 1596, 1144, 7936, 1534, 4087, 4222, 54455, 13, 14235, 22846, 5426, 13, 271, 6065, 220, 16, 23, 16, 23, 15, 20, 22, 69, 25, 198, 39044, 25, 10804, 14, 39652, 440, 7736, 220, 17, 11, 19, 11, 23, 13, 8984, 4203, 1010, 220, 19, 82, 310, 220, 23, 30, 6558, 579, 23331, 13, 271, 12929, 16, 1879, 220, 16, 15, 87, 16, 17, 30, 6761, 25, 198, 16, 220, 17, 17, 17, 17, 19], "margins": [11.875, 11.375, 0.75, 3.375, 2.625, 3.0, 3.25, 0.5, 3.125, 0.25, 0.625, 4.75, 1.125, 3.25, 2.0, 5.5, 0.875, 3.75, 0.375, 0.5, 1.375, 6.5, 1.125, 2.25, 12.625, 12.0625, 11.3125, 12.9375, 12.4375, 8.875, 10.8125, 10.75, 1.75, 7.375, 13.5625, 13.875, 11.6875, 11.25, 10.75, 12.375, 12.375, 10.75, 2.25, 2.25, 1.625, 0.875, 1.0, 0.375, 0.875, 1.375, 0.125, 1.875, 6.5, 1.5, 1.0, 1.375, 1.0, 4.75, 4.625, 1.75, 1.0, 1.625, 0.875, 0.625, 0.25, 3.5, 1.0, 1.25, 4.625, 11.625, 5.75, 11.125, 12.3125, 10.8125, 10.9375, 11.0, 11.8125, 0.125, 0.0, 2.625, 0.375, 0.25, 0.875, 7.875, 0.75, 0.875, 1.875, 8.125, 0.75, 7.375, 2.875, 3.875, 2.125, 0.25, 2.0, 3.25, 2.625, 0.625, 2.625, 2.125, 10.75, 10.625, 0.75, 1.75, 7.5, 2.75, 6.375, 0.25, 1.75, 1.375, 0.75, 3.25, 8.375, 4.625, 3.75, 4.375, 2.625, 1.5, 0.25, 0.5, 5.375, 1.375, 0.5, 9.75, 4.625, 12.625, 11.4375, 8.375], "error": null, "prompt_sha": "8750743e9dcadca9", "prompt_len": 2048, "finish_reason": "length"}, {"text": "We need solve ARC tasks. Need find transformation rule for each task and solve test inputs. We have multiple tasks: 247ef758, 269e22fb, 271d71e2, 28a6681f, 291dc1e1. Need output transformed grids for test inputs. Need infer rules from examples. Need be careful. We need produce final answer likely grids. Need analyze each task.\n\nWe need parse tasks. There are two test inputs for 247ef758? It says Test input: ...", "tokens": [1596, 1144, 11290, 67964, 8806, 13, 14235, 1423, 17454, 5728, 364, 1754, 3274, 321, 11290, 1228, 10804, 13, 1165, 599, 5081, 8806, 25, 220, 17, 19, 22, 797, 22, 20, 23, 11, 220, 17, 21, 24, 68, 17, 17, 10481, 11, 220, 17, 22, 16, 67, 22, 16, 68, 17, 11, 220, 17, 23, 64, 21, 21, 23, 16, 69, 11, 220, 17, 24, 16, 7385, 16, 68, 16, 13, 14235, 2468, 22773, 54455, 364, 1228, 10804, 13, 14235, 22846, 5426, 494, 9989, 13, 14235, 381, 16097, 13, 1165, 1144, 7936, 1534, 4087, 4222, 54455, 13, 14235, 22903, 1754, 3274, 13, 271, 1596, 1144, 4563, 8806, 13, 2532, 513, 1330, 1228, 10804, 364, 220, 17, 19, 22, 797, 22, 20, 23, 30, 1049, 2640, 3284, 1879, 25, 2423], "margins": [11.75, 9.875, 1.125, 1.25, 3.875, 2.0, 2.875, 0.375, 5.25, 0.875, 1.5, 5.625, 0.875, 3.0, 2.875, 3.125, 7.75, 6.75, 0.5, 0.5, 1.5, 5.625, 1.375, 3.0, 11.75, 13.375, 12.625, 11.3125, 12.0, 13.5, 12.625, 0.625, 8.4375, 11.625, 11.0, 12.5625, 10.5625, 11.0, 10.375, 8.375, 7.375, 9.75, 11.5625, 9.125, 10.0, 9.75, 9.4375, 10.3125, 10.4375, 9.0, 8.1875, 9.4375, 12.625, 9.75, 11.0, 11.3125, 10.9375, 9.625, 9.5625, 10.0, 8.1875, 7.625, 12.375, 11.125, 11.375, 10.625, 10.25, 10.375, 10.0, 2.5, 3.0, 2.125, 0.125, 0.125, 1.875, 0.125, 4.125, 3.125, 2.0, 0.75, 3.0, 0.75, 8.75, 4.125, 0.125, 1.25, 3.875, 2.75, 0.125, 0.25, 0.625, 3.875, 1.625, 1.875, 0.75, 1.0, 0.375, 2.375, 2.125, 0.5, 0.5, 2.5, 1.875, 0.125, 0.75, 0.375, 3.625, 0.125, 1.375, 0.0, 4.125, 7.375, 2.5, 0.375, 12.75, 6.75, 12.375, 4.625, 8.75, 12.1875, 10.25, 2.0, 0.625, 0.25, 1.125, 7.625, 1.75, 2.125], "error": null, "prompt_sha": "fc63e297d382cd8f", "prompt_len": 8192, "finish_reason": "length"}, {"text": "We need solve ARC tasks. Need find transformation rule for each task and solve test inputs. We need output likely transformed grids. Need analyze many tasks. Need be careful. We have multiple tasks with examples and test inputs. Need infer rules.\n\nWe need produce final answer with transformed test inputs for each task. There are tasks: 2c181942, 2d0172a1 (two test inputs?), 31f7f899, 332f06d7, 35ab12c3, 36a08778", "tokens": [1596, 1144, 11290, 67964, 8806, 13, 14235, 1423, 17454, 5728, 364, 1754, 3274, 321, 11290, 1228, 10804, 13, 1165, 1144, 2468, 4222, 22773, 54455, 13, 14235, 22903, 1599, 8806, 13, 14235, 381, 16097, 13, 1165, 599, 5081, 8806, 440, 9989, 321, 1228, 10804, 13, 14235, 22846, 5426, 13, 271, 1596, 1144, 7936, 1534, 4087, 440, 22773, 1228, 10804, 364, 1754, 3274, 13, 2532, 513, 8806, 25, 220, 17, 66, 16, 23, 16, 24, 19, 17, 11, 220, 17, 67, 15, 16, 22, 17, 64, 16, 318, 19186, 1228, 10804, 66926, 220, 18, 16, 69, 22, 69, 23, 24, 24, 11, 220, 18, 18, 17, 69, 15, 21, 67, 22, 11, 220, 18, 20, 365, 16, 17, 66, 18, 11, 220, 18, 21, 64, 15, 23, 22, 22, 23], "margins": [10.125, 11.5, 1.125, 0.25, 3.5, 2.125, 2.625, 0.5, 5.25, 1.0, 1.75, 5.75, 0.375, 2.375, 3.625, 4.25, 6.625, 7.375, 0.25, 0.125, 1.25, 0.5, 0.25, 2.0, 0.5, 1.625, 1.0, 1.0, 4.375, 2.875, 0.75, 2.125, 4.125, 4.0, 0.375, 0.5, 1.625, 3.875, 0.375, 2.125, 2.125, 2.125, 2.875, 1.625, 2.875, 1.125, 0.75, 1.375, 0.125, 1.625, 2.25, 1.0, 2.875, 2.5, 3.0, 0.125, 1.125, 0.75, 1.25, 2.375, 1.75, 1.625, 0.625, 3.0, 0.125, 4.5, 0.5, 11.625, 10.625, 12.0625, 13.0625, 9.5625, 10.9375, 10.375, 10.875, 1.0, 6.5, 9.8125, 10.25, 11.9375, 13.5, 12.3125, 11.625, 11.25, 11.25, 0.375, 2.5, 4.125, 7.25, 0.625, 5.375, 9.75, 9.625, 12.4375, 11.25, 10.4375, 9.0, 11.4375, 10.8125, 3.25, 8.375, 11.875, 7.375, 11.5, 11.0, 10.5625, 8.8125, 10.875, 10.375, 5.625, 10.125, 13.125, 8.625, 7.375, 11.1875, 10.5, 10.25, 10.9375, 5.25, 9.75, 10.1875, 7.25, 11.4375, 12.0, 8.9375, 11.0625, 9.5, 10.3125], "error": null, "prompt_sha": "49b3958bb234d4fd", "prompt_len": 40000, "finish_reason": "length"}], "fp8": [{"text": "We need solve ARC task. Need infer transformation from examples. User gives Task 0934a4d8. Example 1 input 30x30? output 9x4? Example 2 input 30x30? output 4x5? Example 3 input: \"194499276\" only one row? Maybe truncated? Need find transformation rule and solve test inputs. But only one test input? It says Example 3 input: 194499276. Maybe that's the test input? Need output. Need infer from", "tokens": [1596, 1144, 11290, 67964, 3274, 13, 14235, 22846, 17454, 494, 9989, 13, 2570, 6486, 5261, 220, 15, 24, 18, 19, 64, 19, 67, 23, 13, 12996, 220, 16, 1879, 220, 18, 15, 87, 18, 15, 30, 2468, 220, 24, 87, 19, 30, 12996, 220, 17, 1879, 220, 18, 15, 87, 18, 15, 30, 2468, 220, 19, 87, 20, 30, 12996, 220, 18, 1879, 25, 328, 16, 24, 19, 19, 24, 24, 17, 22, 21, 1, 1132, 799, 2713, 30, 10380, 57739, 30, 14235, 1423, 17454, 5728, 321, 11290, 1228, 10804, 13, 1921, 1132, 799, 1228, 1879, 30, 1049, 2640, 12996, 220, 18, 1879, 25, 220, 16, 24, 19, 19, 24, 24, 17, 22, 21, 13, 10380, 421, 579, 279, 1228, 1879, 30, 14235, 2468, 13, 14235, 22846, 494], "margins": [11.5, 11.25, 0.25, 3.625, 0.375, 2.25, 2.25, 0.875, 4.125, 0.25, 4.25, 1.25, 0.125, 1.0, 1.75, 4.625, 9.0, 9.9375, 10.375, 9.5, 8.875, 10.6875, 14.0, 10.625, 0.125, 0.75, 0.875, 7.5, 4.5, 1.25, 3.5, 0.75, 0.0, 1.75, 4.0, 2.625, 1.25, 9.375, 5.5, 4.25, 2.0, 0.25, 2.625, 0.875, 8.875, 3.75, 4.375, 5.75, 5.5, 4.25, 3.5, 5.5, 0.0, 8.0, 7.25, 7.375, 4.625, 5.875, 1.5, 3.75, 4.875, 9.125, 6.75, 0.25, 0.625, 8.25, 8.25, 12.3125, 11.5, 10.3125, 11.0, 10.1875, 11.625, 9.875, 4.375, 2.25, 0.375, 1.0, 4.875, 0.0, 0.25, 5.75, 2.125, 0.0, 0.25, 2.375, 2.875, 6.75, 9.125, 5.125, 1.875, 0.75, 2.375, 0.0, 0.875, 1.875, 2.125, 0.75, 2.25, 1.625, 9.1875, 7.125, 9.125, 5.625, 1.125, 7.9375, 7.0625, 10.375, 11.375, 10.8125, 10.5625, 11.1875, 11.5, 11.625, 1.0, 1.625, 1.375, 2.625, 0.5, 2.375, 1.75, 2.625, 2.0, 2.625, 0.125, 1.5, 0.25, 1.5], "error": null, "prompt_sha": "8df9ed2c640ef5bd", "prompt_len": 2048, "finish_reason": "length"}, {"text": "We need solve ARC tasks. Need infer transformation rules from examples and apply to test inputs. We have two tasks: 1818057f and 195c6913. Need output transformed test grids. Need analyze carefully.\n\nWe need produce final answer likely grids. Need infer rules.\n\nTask 1818057f examples:\nInput/output grids with colors 2,4,8. Output changes some 4s to 8? Let's inspect.\n\nExample1 10x12? Input rows:\n1 222244", "tokens": [1596, 1144, 11290, 67964, 8806, 13, 14235, 22846, 17454, 5426, 494, 9989, 321, 3668, 310, 1228, 10804, 13, 1165, 599, 1330, 8806, 25, 220, 16, 23, 16, 23, 15, 20, 22, 69, 321, 220, 16, 24, 20, 66, 21, 24, 16, 18, 13, 14235, 2468, 22773, 1228, 54455, 13, 14235, 22903, 15060, 13, 271, 1596, 1144, 7936, 1534, 4087, 4222, 54455, 13, 14235, 22846, 5426, 13, 271, 6065, 220, 16, 23, 16, 23, 15, 20, 22, 69, 9989, 25, 198, 2425, 45447, 54455, 440, 7736, 220, 17, 11, 19, 11, 23, 13, 8984, 4203, 1010, 220, 19, 82, 310, 220, 23, 30, 6558, 579, 23331, 13, 271, 12929, 16, 220, 16, 15, 87, 16, 17, 30, 5396, 6761, 25, 198, 16, 220, 17, 17, 17, 17, 19, 19], "margins": [11.875, 11.75, 0.5, 3.125, 2.625, 3.0, 3.125, 0.5, 3.0, 0.5, 0.625, 4.875, 1.0, 3.25, 2.0, 5.625, 0.625, 3.875, 0.375, 0.75, 1.375, 6.75, 1.375, 2.125, 12.8125, 11.9375, 11.6875, 13.1875, 12.0625, 8.4375, 10.3125, 10.5, 1.625, 7.25, 13.5625, 12.625, 10.5625, 9.375, 12.0, 12.6875, 11.3125, 11.0625, 2.0, 2.375, 1.625, 0.875, 1.0, 0.125, 0.5, 1.25, 0.125, 1.875, 6.625, 1.625, 1.5, 1.5, 1.0, 4.75, 4.625, 2.0, 0.875, 1.5, 1.0, 0.625, 0.5, 3.375, 1.0, 1.125, 4.5, 12.375, 5.625, 11.125, 12.625, 10.0625, 9.875, 10.125, 11.625, 0.125, 1.375, 0.5, 1.0, 0.75, 0.375, 1.25, 1.0, 1.75, 7.75, 0.75, 7.75, 2.375, 3.625, 1.625, 0.25, 2.0, 3.5, 2.75, 1.125, 2.25, 2.375, 10.8125, 10.75, 0.75, 1.875, 7.5, 2.75, 5.25, 0.25, 2.125, 1.25, 0.0, 7.125, 3.625, 5.5, 4.5, 2.0, 1.5, 0.75, 0.875, 0.0, 6.375, 1.0, 0.75, 10.0, 4.75, 12.625, 11.5625, 8.375, 11.4375], "error": null, "prompt_sha": "8750743e9dcadca9", "prompt_len": 2048, "finish_reason": "length"}, {"text": "We need solve ARC tasks. Need find transformation rule for each task and solve test inputs. We have multiple tasks: 247ef758, 269e22fb, 271d71e2, 28a6681f, 291dc1e1. Need output transformed test inputs. Need infer rules from examples. Need be careful. We can analyze each.\n\nWe need produce final answer likely grids for each test input. Need determine exact transformations.\n\nLet's parse tasks.\n\nTask 247ef758: examples show", "tokens": [1596, 1144, 11290, 67964, 8806, 13, 14235, 1423, 17454, 5728, 364, 1754, 3274, 321, 11290, 1228, 10804, 13, 1165, 599, 5081, 8806, 25, 220, 17, 19, 22, 797, 22, 20, 23, 11, 220, 17, 21, 24, 68, 17, 17, 10481, 11, 220, 17, 22, 16, 67, 22, 16, 68, 17, 11, 220, 17, 23, 64, 21, 21, 23, 16, 69, 11, 220, 17, 24, 16, 7385, 16, 68, 16, 13, 14235, 2468, 22773, 1228, 10804, 13, 14235, 22846, 5426, 494, 9989, 13, 14235, 381, 16097, 13, 1165, 628, 22903, 1754, 13, 271, 1596, 1144, 7936, 1534, 4087, 4222, 54455, 364, 1754, 1228, 1879, 13, 14235, 7995, 4581, 51192, 13, 271, 9764, 579, 4563, 8806, 13, 271, 6065, 220, 17, 19, 22, 797, 22, 20, 23, 25, 9989, 1420], "margins": [11.875, 9.75, 1.25, 1.5, 3.625, 2.125, 3.0, 0.125, 5.25, 0.875, 1.625, 5.5, 0.875, 3.125, 2.875, 2.875, 8.5, 7.125, 0.375, 0.375, 1.875, 5.375, 1.625, 3.0, 12.125, 13.3125, 12.9375, 11.8125, 11.8125, 13.5, 12.5, 0.375, 8.875, 11.6875, 10.9375, 12.25, 11.0625, 10.6875, 10.375, 8.3125, 7.375, 10.4375, 11.1875, 9.0, 9.8125, 10.125, 10.25, 10.8125, 10.125, 9.3125, 8.0625, 9.5, 12.4375, 10.4375, 11.0, 11.1875, 11.1875, 9.6875, 10.375, 9.0, 7.75, 7.625, 12.875, 10.8125, 11.375, 9.625, 10.625, 11.0625, 9.875, 2.375, 2.875, 2.125, 0.125, 0.0, 0.875, 0.75, 2.0, 0.625, 3.125, 0.875, 8.375, 3.875, 0.125, 1.25, 4.125, 2.875, 0.375, 0.125, 0.5, 0.125, 0.875, 1.375, 2.5, 2.25, 0.75, 4.0, 1.75, 2.0, 0.875, 0.625, 1.125, 3.375, 4.25, 3.75, 0.875, 0.125, 0.5, 0.875, 6.25, 1.5, 3.125, 10.25, 2.625, 2.25, 3.25, 3.625, 3.625, 6.75, 10.375, 7.625, 12.5, 11.75, 11.25, 11.875, 13.4375, 1.5, 0.0, 0.25], "error": null, "prompt_sha": "fc63e297d382cd8f", "prompt_len": 8192, "finish_reason": "length"}, {"text": "We need solve ARC tasks. Need find transformation rule for each task and solve test inputs. We have multiple tasks with examples and test inputs. Need output transformed grids likely. Need infer rules. This is complex. Need analyze each task carefully. Need produce final answer with outputs for each test input. Need be accurate.\n\nWe have tasks:\n2c181942 (one test input)\n2d0172a1 (two test inputs? It lists Test input twice)\n31f7f899 (one test)\n332f06d7 (one test)", "tokens": [1596, 1144, 11290, 67964, 8806, 13, 14235, 1423, 17454, 5728, 364, 1754, 3274, 321, 11290, 1228, 10804, 13, 1165, 599, 5081, 8806, 440, 9989, 321, 1228, 10804, 13, 14235, 2468, 22773, 54455, 4222, 13, 14235, 22846, 5426, 13, 1061, 369, 6150, 13, 14235, 22903, 1754, 3274, 15060, 13, 14235, 7936, 1534, 4087, 440, 15796, 364, 1754, 1228, 1879, 13, 14235, 381, 12995, 13, 271, 1596, 599, 8806, 25, 198, 17, 66, 16, 23, 16, 24, 19, 17, 318, 588, 1228, 1879, 8, 198, 17, 67, 15, 16, 22, 17, 64, 16, 318, 19186, 1228, 10804, 30, 1049, 11140, 3284, 1879, 10598, 8, 198, 18, 16, 69, 22, 69, 23, 24, 24, 318, 588, 1228, 8, 198, 18, 18, 17, 69, 15, 21, 67, 22, 318, 588, 1228, 8], "margins": [10.0, 11.25, 1.0, 0.625, 3.375, 2.5, 2.875, 0.375, 5.125, 0.875, 1.75, 5.875, 0.375, 2.875, 3.625, 4.125, 6.375, 7.5, 0.25, 0.0, 0.25, 6.25, 0.25, 3.75, 2.0, 2.625, 3.125, 2.875, 4.125, 2.75, 0.5, 1.375, 0.625, 0.5, 1.875, 0.125, 1.375, 0.25, 0.0, 9.625, 0.125, 1.375, 1.0, 0.625, 3.125, 4.125, 0.25, 1.875, 0.125, 0.875, 3.875, 1.5, 2.75, 0.0, 2.5, 0.875, 3.0, 4.75, 1.25, 1.0, 0.125, 1.75, 10.625, 1.875, 2.625, 0.125, 3.0, 3.5, 1.125, 0.25, 10.5, 13.375, 13.3125, 12.0625, 10.375, 9.625, 11.0, 0.125, 1.625, 7.5, 1.0, 3.5, 7.75, 7.0, 10.375, 13.0625, 13.1875, 12.0, 11.1875, 11.875, 11.1875, 7.125, 5.75, 5.25, 8.5, 0.625, 0.75, 0.75, 1.375, 7.375, 0.75, 0.0, 4.375, 8.9375, 9.875, 12.8125, 11.8125, 10.6875, 10.1875, 10.5625, 12.125, 4.625, 5.25, 1.125, 0.0, 10.875, 11.6875, 6.875, 13.0, 11.125, 12.0, 10.0625, 11.375, 10.5, 9.25, 9.0, 2.625, 5.25], "error": null, "prompt_sha": "49b3958bb234d4fd", "prompt_len": 40000, "finish_reason": "length"}]}""")   # RC2's recorded greedy oracle rows (traces/requests.jsonl), BF16 + FP8
    GEN_RATE = 45.0                              # tok/s per stream used only to project window growth (RC2: 33 at 8, 70 at 1)
    PP = 2                                       # pipeline stages of the profile: one engine step = one micro-batch (C/2 requests)
    TF_BANDS = ("2k", "8k")                      # FP8 self-consistency only (< 2 min); the BF16 / FP8 isolation is NB0's
    TF_P = {b: PROMPTS[b][5] for b in TF_BANDS}  # RC2's TF prompt (variant 5)
    RETENTION_KNOBS = {   # Merlin source (ad42e37), recorded, not changed
        "MAX_GRACE_PINS": "16 finished turns keep their cached prefix referenced (kv_cache_manager.py:735, class constant)",
        "MAX_PINNED_WAITERS": "16 waiting requests' hits pinned until admission (kv_cache_manager.py:940, class constant)",
        "enable_grace_pins": "enable_caching and partial hash hits (prefix-match-unit 64), kv_cache_manager.py:234",
        "VLLM_E2_SCHED_MISS_DEFER_S": "120 s default (merlin/envs.py:40): a cold miss may wait for protected waiters",
        "VLLM_E2_SCHED_COHERENT_YIELD": "on by default (merlin/envs.py:41): need-sized, session-coherent yields",
        "reset": "/reset_prefix_cache (VLLM_SERVER_DEV_MODE=1) drops grace and waiter pins when they alone block the reset "
                 "(kv_cache_manager.py:1107-1113); it fails while requests run"}
    for _b, _n in (("8k", 34), ("40k", 16)):     # distinct prompts (RC2 has 8 per band)
        PROMPTS[_b] += [prompt(BANDS[_b], v) for v in range(len(PROMPTS[_b]), _n)]
    BANDS["60k"] = 60000                         # residency probe band (addendum 7); RC2's PROMPTS bands are unchanged
    PROMPTS["60k"] = [prompt(BANDS["60k"], v) for v in range(8)]
    MAX_NEW["60k"] = MAX_LEN - BANDS["60k"] - 512
    RESIDENCY_BANDS = ("8k", "40k", "60k", "90k", "116k")
    V_CENSUS = {"8k": list(range(24, 32)), "40k": list(range(9, 13))}   # never used by a steady window in the same boot
    V_PREFIX = {"8k": 32, "40k": 8}
    V_ABORT = 33
    SUFFIX_TXT = ("\n</think>\n\nI have a first rule.<|im_end|>\n<|im_start|>user\nCheck the rule on every training pair, "
                  "then give the final test grid." + _tail_txt)
    SUFFIX_IDS = TOK.encode(SUFFIX_TXT, add_special_tokens=False).ids
    R["prompts"] = {b: {"n": len(p), "len": len(p[0])} for b, p in PROMPTS.items()}


    def psha(ids):
        return hashlib.sha256(json.dumps(list(ids)).encode()).hexdigest()


    R["nb1_prompt_sha256"] = {b: psha(TF_P[b]) for b in TF_BANDS}
    _osha = [psha(p)[:16] for p in ORACLE_P + LONG_P]          # RC2's own prompt_sha formula (16 hex)
    NB1["rc2_oracle_prompts_match"] = {k: [r["prompt_sha"] for r in v] == _osha for k, v in RC2_ORACLE.items()}
    for _k, _v in RC2_ORACLE.items():                          # RC2's recorded rows become references only for identical prompts
        if NB1["rc2_oracle_prompts_match"][_k] and not FAKE:    # (the fake server's text differs: it builds its own reference)
            ROWS["rc2_" + _k] = [{"text": r["text"], "tokens": r["tokens"], "margins": r["margins"], "error": r["error"]} for r in _v]
    NB1["retention_knobs"] = RETENTION_KNOBS
    NB1["sampling"] = ("every request passes its own sampling (greedy: temperature 0; load: T 0.6, top_p 0.95, top_k 20; "
                       "Merlin's --override-generation-config is T 1.0)")
    R["capacity_plan"] = {
        "rule": "blocks per running request = ceil(L / 4096) + F, F = 6 (3 mamba groups x 2; RC2 analysis: 149 / 3.92 = 38 = 32 + 6, "
                "live transitions 6.0 / 6.1); a finished session keeps floor(L / 4096) + 3 blocks (up to 16) until re-admitted or yielded",
        "rc2_pool_blocks": {"fp8": 149, "bf16": 78},
        "worst_case_F6": {"8k@8": 8 * (math.ceil((8192 + 8000) / 4096) + 6), "40k@5": 5 * (math.ceil((40000 + 8000) / 4096) + 6),
                          "8k x 8 hold (cold sequence)": 8 * (math.ceil(9000 / 4096) + 6), "40k x 4 hold": 4 * (math.ceil(41000 / 4096) + 6),
                          "90k x 5 admission (max_new 3000)": 5 * (math.ceil(93000 / 4096) + 6), "pool": 149},
        "retention": "every capacity test starts from /reset_prefix_cache (retained sessions released); windows record the idle "
                     "(retained) blocks before them, so running = used - retained",
        "highest_levels": "computed at run time from the measured pool and F, never above max_num_seqs read from the server log",
        "tf": "B1 only: 2k + 8k FP8 self-consistency (teacher-forced A/A + greedy cached-repeat), < 2 min",
        "c3_note": "at BLOCK_SIZE 4096 the attention page is larger than the mamba page (RC2 log: 'Padding mamba page size by "
                   "21.01%'), so bf16 SSM state cannot change the pool's block count; B2 records the smaller minimum block it allows"}

    # ---- trace classifier fix (A7): QSA / Marlin MoE / top-k before the generic patterns; GDN '(anonymous namespace)' is
    # already handled by the lab's rev 3h _cat
    L.CATS = [("nccl_sendrecv", ("sendrecv", "send_recv", "nccldevkernel_send", "nccldevkernel_recv")), L.CATS[0],
              ("recoverssm_commit", ("recoverssm", "commit_", "boundary_fold", "final_fold", "conv_compaction")),
              ("qsa", ("qsa",)), ("moe_marlin", ("marlin_moe", "fused_marlin_moe", "moe_wna16")),
              ("topk", ("persistent_topk", "cooperative_topk"))] + list(L.CATS[1:])
    L.COMPUTE_CATS = tuple(L.COMPUTE_CATS) + ("nccl_sendrecv", "recoverssm_commit", "qsa", "moe_marlin", "topk")

    # ---- RC2 long-row gate, stricter (audit 2): a natural row needs finish == stop and a closed think; forced rows
    # (ignore_eos) are stability tests only. NB1 runs no long rows; the reused gate is fixed for any later reuse.
    _long_quality_rc2, _long_gate_rc2 = long_quality, long_gate


    def long_quality(name, r):
        q = _long_quality_rc2(name, r)
        if q == "PASS_LONG" and name == "natural":
            if r.get("finish") != "stop":
                return "INCOMPLETE_NATURAL_FINISH"
            if r.get("think_closed") is not True:
                return "INCOMPLETE_THINKING"
        return q


    def long_gate(o):
        g = _long_gate_rc2(o)
        lo = o.get("long") if isinstance(o.get("long"), dict) else None
        if lo and isinstance(lo.get("forced60k"), dict):
            lo["forced60k"]["counts_as"] = "stability test only (ignore_eos)"
        if g is None and lo and any(v in ("INCOMPLETE_NATURAL_FINISH", "INCOMPLETE_THINKING") for v in (lo.get("long_quality") or {}).values()):
            return "INCONCLUSIVE_LONG_INCOMPLETE"
        return g


    # ---- the #60490 backport + the worker extension (written before B1)
    P60490 = "# vLLM PR #60490 (issue #60489) rebased onto Merlin v3.0.0 (commit ad42e37feb7611c4ffd24017bf20a7e2dc94c6e5,\n# tree 07c16a79d077f1e06c19d0eea84efd64f0b63540 = the tree the Kaggle payload installs, install_merlin.log).\n# Same code as upstream; only the import context differs (Merlin's import block). One marker comment added.\n# pre-patch sha256  a23d5f90507ad5185f8b8e8d6ba13ae0cb53b2506264fdfbbf6a60b1171dc653\n# post-patch sha256 88fe0a8f4dd2ef90fc6aefb5771b5b48e40c0b1d575039283c52aeca19b8bd6b\n--- a/vllm/models/qwen4_exp/nvidia/qsa.py\n+++ b/vllm/models/qwen4_exp/nvidia/qsa.py\n@@ -18,8 +18,9 @@\n from torch import nn\n from transformers import Qwen4ExpTextConfig\n \n+import vllm.envs as envs\n from vllm.compilation.breakable_cudagraph import eager_break_during_capture\n-from vllm.config import VllmConfig\n+from vllm.config import VllmConfig, get_current_vllm_config_or_none\n from vllm.config.cache import CacheConfig, CacheDType\n from vllm.distributed import get_tensor_model_parallel_world_size\n from vllm.forward_context import get_forward_context\n@@ -45,6 +46,7 @@\n )\n from vllm.platforms import current_platform\n from vllm.platforms.interface import DeviceCapability\n+from vllm.utils.math_utils import cdiv\n from vllm.utils.torch_utils import (\n     LayerNameType,\n     _encode_layer_name,\n@@ -75,6 +77,7 @@\n from . import model\n from .indexer_qsa import QSAIndexer\n from .ops.qsa import qsa_e4m3_uses_native_cast\n+from .ops.qsa_indexer import _TOPK_WORKSPACE_BYTES\n \n logger = init_logger(__name__)\n \n@@ -782,6 +785,31 @@\n         if isinstance(metadata, list):\n             metadata = metadata[0]\n         if not isinstance(metadata, dict):\n+            # E25 NB1: backport of vLLM PR #60490 (issue #60489).\n+            # Profiling/dummy run: the indexer allocates its selection scratch\n+            # on demand, so touch its peak here for the memory profiler (as\n+            # sparse_attn_indexer does): the prefill logits budget or the\n+            # worst-case decode logits [decode tokens, compressed columns],\n+            # plus the top-k workspace.\n+            logits_bytes = envs.VLLM_SPARSE_INDEXER_MAX_LOGITS_MB * 1024 * 1024\n+            cfg = get_current_vllm_config_or_none()\n+            if cfg is not None:\n+                spec = cfg.speculative_config\n+                num_spec = spec.num_speculative_tokens if spec is not None else 0\n+                decode_tokens = min(\n+                    cfg.scheduler_config.max_num_seqs * (num_spec + 1),\n+                    cfg.scheduler_config.max_num_batched_tokens,\n+                )\n+                block_size = cfg.cache_config.block_size\n+                columns = cdiv(cfg.model_config.max_model_len, block_size) * cdiv(\n+                    block_size, self.indexer.compress_ratio\n+                )\n+                logits_bytes = max(logits_bytes, decode_tokens * columns * 4)\n+            _ = torch.empty(\n+                logits_bytes + _TOPK_WORKSPACE_BYTES,\n+                dtype=torch.uint8,\n+                device=output.device,\n+            )\n             output.zero_()\n             return\n         main_metadata = cast(FlashAttentionMetadata, metadata[self.layer_name])\n"
    P60490_REL = "vllm/models/qwen4_exp/nvidia/qsa.py"
    P60490_PRE, P60490_POST = "a23d5f90507ad5185f8b8e8d6ba13ae0cb53b2506264fdfbbf6a60b1171dc653", "88fe0a8f4dd2ef90fc6aefb5771b5b48e40c0b1d575039283c52aeca19b8bd6b"
    P60490_MARKER = "# E25 NB1: backport of vLLM PR #60490 (issue #60489)."
    NB1_EXT_SRC = "\"\"\"E25 NB1 worker extension (read-only): reached through POST /collective_rpc {\"method\": \"nb1_cache_info\"} with\nVLLM_SERVER_DEV_MODE=1 and --worker-extension-cls nb1_worker_ext.NB1WorkerExt. Reports the ACTUAL cache tensors bound\nto every attention / Mamba layer (dtype, shape, bytes), the KV cache specs (padded vs unpadded Mamba page bytes), the\nphysical block count and this rank's free memory. It never raises: every part returns its error text instead.\"\"\"\nimport json\nimport math\n\n\ndef _t(x):\n    try:\n        import torch\n        if isinstance(x, torch.Tensor):\n            return {\"dtype\": str(x.dtype).replace(\"torch.\", \"\"), \"shape\": list(x.shape), \"bytes\": int(x.numel() * x.element_size())}\n    except Exception:\n        pass\n    return None\n\n\ndef _flat(kc):\n    if _t(kc):\n        return [kc]\n    if isinstance(kc, (list, tuple)):\n        if kc and isinstance(kc[0], (list, tuple)) and not _t(kc[0]):\n            kc = kc[0]                              # one entry per virtual engine\n        return [x for x in kc if _t(x)]\n    return []\n\n\n_RT = {\"on\": False, \"installed\": False, \"rows\": [], \"passes\": 0, \"calls\": 0, \"max_steps\": 200, \"bytes\": 0, \"errors\": [],\n       \"first\": None, \"skipped_capturing\": 0, \"skipped_prefill\": 0}\n_RT_MAX_BYTES = 16 << 20\n_RT_DECODE_MAX_ROWS = 128          # a decode / verify micro-batch: requests x (1 + K) tokens; prefill chunks are larger\n\n\ndef _rt_layer(name):\n    import re as _r\n    m = _r.search(r\"layers\\.(\\d+)\", str(name))\n    return int(m.group(1)) if m else -1\n\n\ndef _rt_wrap(orig):\n    def select_experts(self, hidden_states, router_logits, *a, **k):\n        out = orig(self, hidden_states, router_logits, *a, **k)\n        if _RT[\"on\"] and _RT[\"passes\"] < 2 * _RT[\"max_steps\"] and _RT[\"bytes\"] < _RT_MAX_BYTES:\n            try:\n                import torch\n                if torch.cuda.is_current_stream_capturing():\n                    _RT[\"skipped_capturing\"] += 1\n                else:\n                    ids = out[1]\n                    lay = _rt_layer(getattr(self, \"nb1_layer\", None))\n                    if _RT[\"first\"] is None:\n                        _RT[\"first\"] = lay\n                    n = int(ids.shape[0])\n                    if n > _RT_DECODE_MAX_ROWS:\n                        _RT[\"skipped_prefill\"] += 1\n                    else:\n                        if lay == _RT[\"first\"]:\n                            _RT[\"passes\"] += 1     # one pass of a micro-batch through this stage's first MoE layer\n                        keep = ids.to(torch.int16).cpu().numpy().reshape(n, -1)\n                        _RT[\"rows\"].append((lay, max(0, _RT[\"passes\"] - 1), n, keep))\n                        _RT[\"bytes\"] += keep.nbytes\n                        _RT[\"calls\"] += 1\n            except Exception as e:\n                if len(_RT[\"errors\"]) < 5:\n                    _RT[\"errors\"].append(repr(e)[:160])\n        return out\n    return select_experts\n\n\nclass NB1WorkerExt:\n    def nb1_routing(self, action, arg=\"\"):\n        \"\"\"Router top-k expert ids, ONLY between 'start' and 'stop' (the profiling window): FusedMoERouter.select_experts is\n        wrapped once at class level; every decode / verify-sized call (<= 128 token rows) that runs in Python (not during\n        a CUDA-graph capture) keeps its ids (int16) with the layer index and the pass index (counted at this stage's first\n        MoE layer; NB1 maps pass -> step = pass // 2, mb_index = pass % 2 under PP2); <= max_steps steps, <= 16 MiB per\n        rank. FULL-graph replays run no Python, so graph-replayed decode steps are NOT recorded (vLLM's\n        --enable-return-routed-experts writes the ids from inside the graph; it changes the forward graph and memory, so it\n        is not used on a measured boot). 'stop' writes the rank's part file <arg>_rank<r>.npz.\"\"\"\n        out = {\"action\": action}\n        try:\n            import torch.distributed as dist\n            rank = dist.get_rank() if dist.is_initialized() else 0\n            out[\"rank\"] = rank\n            if action == \"start\":\n                from vllm.model_executor.layers.fused_moe.router.fused_moe_router import FusedMoERouter\n                if not _RT[\"installed\"]:\n                    FusedMoERouter.select_experts = _rt_wrap(FusedMoERouter.select_experts)\n                    _RT[\"installed\"] = True\n                    model = getattr(getattr(self, \"model_runner\", None), \"model\", None)\n                    named = 0\n                    if model is not None:\n                        for name, mod in model.named_modules():\n                            for attr in (\"router\", \"_router\"):\n                                r = getattr(mod, attr, None)\n                                if isinstance(r, FusedMoERouter):\n                                    r.nb1_layer = name\n                                    named += 1\n                    out[\"routers_named\"] = named\n                _RT.update(on=True, rows=[], passes=0, calls=0, bytes=0, errors=[], first=None, skipped_capturing=0,\n                           skipped_prefill=0, max_steps=int(arg or 200))\n                return out\n            _RT[\"on\"] = False\n            out.update(passes=_RT[\"passes\"], calls=_RT[\"calls\"], bytes=_RT[\"bytes\"], errors=_RT[\"errors\"],\n                       skipped_capturing=_RT[\"skipped_capturing\"], skipped_prefill=_RT[\"skipped_prefill\"])\n            if action == \"stop\" and arg and _RT[\"rows\"]:\n                import numpy as np\n                rows = _RT[\"rows\"]\n                offs, o_ = [], 0\n                for r in rows:\n                    offs.append(o_)\n                    o_ += r[2]\n                path = f\"{arg}_rank{rank}.npz\"\n                np.savez_compressed(path, ids=np.concatenate([r[3] for r in rows], axis=0), layer=np.array([r[0] for r in rows]),\n                                    pass_index=np.array([r[1] for r in rows]), mb_size=np.array([r[2] for r in rows]),\n                                    offset=np.array(offs), rank=np.array([rank]))\n                out[\"path\"] = path\n            _RT[\"rows\"] = []\n        except Exception as e:\n            out[\"error\"] = repr(e)[:300]\n        return out\n\n    def nb1_cache_info(self):\n        out = {}\n        try:\n            import torch.distributed as dist\n            out[\"rank\"] = dist.get_rank() if dist.is_initialized() else None\n        except Exception as e:\n            out[\"rank_error\"] = repr(e)[:120]\n        try:\n            ctx = self.vllm_config.compilation_config.static_forward_context\n            sig = {}\n            for name, mod in ctx.items():\n                ts = [_t(x) for x in _flat(getattr(mod, \"kv_cache\", None))]\n                if not ts:\n                    continue\n                key = type(mod).__name__ + \"|\" + json.dumps([(x[\"dtype\"], x[\"shape\"]) for x in ts])\n                g = sig.setdefault(key, {\"module\": type(mod).__name__, \"tensors\": ts, \"layers\": 0, \"example\": name})\n                g[\"layers\"] += 1\n            out[\"cache_tensors\"] = list(sig.values())[:40]\n        except Exception as e:\n            out[\"cache_tensors_error\"] = repr(e)[:300]\n        try:\n            agg = {}\n            for name, sp in self.get_kv_cache_spec().items():\n                d = {\"type\": type(sp).__name__}\n                for k in (\"block_size\", \"page_size_bytes\", \"page_size_padded\", \"dtype\", \"num_kv_heads\", \"head_size\", \"mamba_type\"):\n                    try:\n                        v = getattr(sp, k, None)\n                        d[k] = v if isinstance(v, (int, float, str, bool, type(None))) else str(v)\n                    except Exception as e:\n                        d[k] = \"ERR \" + repr(e)[:60]\n                shapes, dtypes = getattr(sp, \"shapes\", None), getattr(sp, \"dtypes\", None)\n                if shapes and dtypes:\n                    try:\n                        d[\"state_dtypes\"] = [str(x).replace(\"torch.\", \"\") for x in dtypes]\n                        d[\"state_shapes\"] = [list(s) for s in shapes]\n                        d[\"unpadded_page_bytes\"] = int(sum(math.prod(s) * x.itemsize for s, x in zip(shapes, dtypes)))\n                    except Exception as e:\n                        d[\"unpadded_error\"] = repr(e)[:80]\n                key = json.dumps(d, sort_keys=True, default=str)\n                a = agg.setdefault(key, dict(d, layers=0, example=name))\n                a[\"layers\"] += 1\n            out[\"kv_cache_specs\"] = list(agg.values())[:40]\n        except Exception as e:\n            out[\"kv_cache_specs_error\"] = repr(e)[:300]\n        try:\n            out[\"num_gpu_blocks\"] = getattr(self.cache_config, \"num_gpu_blocks\", None)\n            out[\"mamba_page_size_padded\"] = getattr(self.cache_config, \"mamba_page_size_padded\", None)\n            out[\"mamba_ssm_cache_dtype\"] = str(getattr(self.cache_config, \"mamba_ssm_cache_dtype\", None))\n            out[\"cache_block_size\"] = getattr(self.cache_config, \"block_size\", None)\n        except Exception as e:\n            out[\"cache_config_error\"] = repr(e)[:200]\n        try:\n            for obj in (getattr(self, \"model_runner\", None), self):\n                cfg = getattr(obj, \"kv_cache_config\", None)\n                if cfg is not None:\n                    out[\"kv_cache_config\"] = {\n                        \"num_blocks\": getattr(cfg, \"num_blocks\", None),\n                        \"groups\": [{\"spec\": type(g.kv_cache_spec).__name__, \"layers\": len(g.layer_names),\n                                    \"block_size\": getattr(g.kv_cache_spec, \"block_size\", None),\n                                    \"page_size_bytes\": getattr(g.kv_cache_spec, \"page_size_bytes\", None)}\n                                   for g in getattr(cfg, \"kv_cache_groups\", [])][:16],\n                        \"tensors\": [{\"size\": getattr(t, \"size\", None), \"shared_by\": len(getattr(t, \"shared_by\", None) or [])}\n                                    for t in getattr(cfg, \"kv_cache_tensors\", [])][:24]}\n                    break\n        except Exception as e:\n            out[\"kv_cache_config_error\"] = repr(e)[:300]\n        try:\n            import torch\n            fr, tot = torch.cuda.mem_get_info()\n            out[\"mem\"] = {\"free_mib\": fr >> 20, \"total_mib\": tot >> 20, \"allocated_mib\": torch.cuda.memory_allocated() >> 20,\n                          \"reserved_mib\": torch.cuda.memory_reserved() >> 20, \"max_allocated_mib\": torch.cuda.max_memory_allocated() >> 20}\n        except Exception as e:\n            out[\"mem_error\"] = repr(e)[:200]\n        return out\n\n\n    def nb4s_env(self, keys=\"\"):\n        \"\"\"E25 NB4S (read-only): this worker's os.environ for the comma-separated `keys` (default CUDA_DEVICE_MAX_CONNECTIONS),\n        all NCCL_* / CUDA_* variables, pid, rank and device. CUDA reads CUDA_DEVICE_MAX_CONNECTIONS at context creation, so\n        the process env (inherited at spawn, before CUDA init) is what is in effect. Never raises.\"\"\"\n        import os as _os\n        out = {\"pid\": _os.getpid()}\n        try:\n            ks = [k for k in str(keys or \"CUDA_DEVICE_MAX_CONNECTIONS\").split(\",\") if k]\n            out[\"env\"] = {k: _os.environ[k] for k in ks if k in _os.environ}\n            out[\"nccl_cuda_env\"] = {k: v for k, v in sorted(_os.environ.items()) if k.startswith((\"NCCL_\", \"CUDA_\"))}\n        except Exception as e:\n            out[\"error\"] = repr(e)[:200]\n        try:\n            import torch.distributed as dist\n            out[\"rank\"] = dist.get_rank() if dist.is_initialized() else None\n        except Exception as e:\n            out[\"rank_error\"] = repr(e)[:120]\n        try:\n            import torch\n            out[\"device\"] = torch.cuda.current_device() if torch.cuda.is_available() else None\n        except Exception as e:\n            out[\"device_error\"] = repr(e)[:120]\n        return out\n"

    def apply_unified(text, patch):
        """Strict unified-diff applier (no fuzz, no offset): every context / removed line must match at its stated line,
        else ValueError. Lines before the first hunk (comments, ---/+++ headers) are ignored."""
        src = text.splitlines(True)
        hunks, cur = [], None
        for ln in patch.splitlines(True):
            if ln.startswith("@@"):
                m = re.match(r"@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@", ln)
                if not m:
                    raise ValueError("bad hunk header: " + ln[:80])
                cur = [int(m.group(1)), int(m.group(2) or 1), []]
                hunks.append(cur)
            elif cur is not None and ln[:1] in (" ", "-", "+"):
                cur[2].append(ln)
            elif cur is not None and ln.startswith("\\"):
                raise ValueError("'No newline at end of file' hunks are not supported")
        if not hunks:
            raise ValueError("patch has no hunks")
        out, pos = [], 0
        for start, n_old, body in hunks:
            i = start - 1
            if i < pos:
                raise ValueError(f"overlapping hunk at line {start}")
            out += src[pos:i]
            old_seen = 0
            for ln in body:
                tag, s = ln[0], ln[1:]
                if tag in " -":
                    if i >= len(src) or src[i] != s:
                        raise ValueError(f"context mismatch at line {i + 1}: {s[:60]!r}")
                    i += 1
                    old_seen += 1
                if tag in " +":
                    out.append(s)
            if old_seen != n_old:
                raise ValueError(f"hunk at line {start}: {old_seen} old lines, header says {n_old}")
            pos = i
        return "".join(out + src[pos:])


    def apply_60490():
        """After merlin_install, before B1: patch the installed tree's qsa.py (pre-sha must be Merlin ad42e37's), verify the
        post-sha, the marker, a py_compile with the Merlin venv and that `vllm` resolves to the tree; write the read-only
        worker extension next to the tree's vllm/ (on the server's PYTHONPATH). Any miss raises: no boot is started."""
        if FAKE:
            tree = os.environ.get("NB1_FAKE_TREE")
            if not tree:
                raise RuntimeError("PATCH_60490_NO_TREE: NB1_FAKE_TREE is not set (local harness)")
            tree, py = Path(tree), sys.executable
        else:
            eng = merlin_install()
            tree, py = Path(eng["kv"]["MERLIN_TREE"]), eng["kv"]["MERLIN_VENV"] + "/bin/python"
        f = tree / P60490_REL
        raw = f.read_bytes()
        h0 = hashlib.sha256(raw).hexdigest()
        rec = {"file": str(f), "sha_before": h0, "pre_expected": P60490_PRE, "post_expected": P60490_POST}
        if h0 == P60490_POST:
            rec["applied"] = "already_applied"
        elif h0 != P60490_PRE:
            raise RuntimeError(f"PATCH_60490_PRE_SHA_MISMATCH: {h0} (expected Merlin ad42e37 {P60490_PRE})")
        else:
            new = apply_unified(raw.decode("utf-8"), P60490).encode("utf-8")
            if hashlib.sha256(new).hexdigest() != P60490_POST:
                raise RuntimeError("PATCH_60490_POST_SHA_MISMATCH: " + hashlib.sha256(new).hexdigest())
            tmp = f.with_name(f.name + ".nb1tmp")
            tmp.write_bytes(new)
            os.replace(tmp, f)
            rec["applied"] = "applied"
        for pyc in (f.parent / "__pycache__").glob("qsa.*.pyc"):
            pyc.unlink()
        data = f.read_bytes()
        rec.update(sha_after=hashlib.sha256(data).hexdigest(), marker=P60490_MARKER.encode() in data)
        ext = tree / "nb1_worker_ext.py"
        ext.write_text(NB1_EXT_SRC)
        rec["worker_ext"] = {"file": str(ext), "sha256": hashlib.sha256(ext.read_bytes()).hexdigest()}
        cp = subprocess.run([py, "-c", "import py_compile, sys\nfor f in sys.argv[1:]: py_compile.compile(f, doraise=True)", str(f), str(ext)],
                            capture_output=True, text=True, timeout=300)
        rec["py_compile"] = {"rc": cp.returncode, "err": cp.stderr[-400:]}
        origin_ok = True
        if not FAKE:
            envp = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}
            envp["PYTHONPATH"] = str(tree)            # what Merlin's build() exports for the server
            cp2 = subprocess.run([py, "-c", "import importlib.util as u; print(u.find_spec('vllm').origin); print(u.find_spec('nb1_worker_ext').origin)"],
                                 capture_output=True, text=True, timeout=300, env=envp)
            rec["vllm_origin"] = cp2.stdout.strip()[-400:]
            origin_ok = all(l.startswith(str(tree)) for l in cp2.stdout.strip().splitlines()[-2:]) and len(cp2.stdout.strip().splitlines()) >= 2
        if rec["sha_after"] != P60490_POST or not rec["marker"] or cp.returncode or not origin_ok:
            raise RuntimeError("PATCH_60490_VERIFY_FAILED: " + json.dumps({k: rec.get(k) for k in ("sha_after", "marker", "py_compile", "vllm_origin")})[:600])
        rec["ok"] = True
        rec["forward_test"] = "B1's greedy oracle vs RC2's recorded FP8 rows (unpatched RC2 FP8; B1 adds text-only): B1 oracle.vs_rc2_fp8"
        return rec


    # ---- host memory: previous engine gone, our leaked shm removed, page cache dropped (measured), Merlin's rule
    def cgroup_snapshot(tag):
        """memory.current / stat / events / pressure (first 12 kB each) to hostmem/<tag>.txt; key numbers returned."""
        rec, txt = {"tag": tag}, []
        for n in ("memory.current", "memory.stat", "memory.events", "memory.pressure"):
            try:
                s = (Path("/sys/fs/cgroup") / n).read_text()[:12000]
            except Exception:
                continue
            txt.append(f"--- {n} ---\n{s}")
            if n == "memory.current":
                rec["current_mib"] = int(s.strip()) >> 20
            elif n == "memory.events":
                rec["events"] = {a: int(b) for a, b in (l.split() for l in s.splitlines() if len(l.split()) == 2)}
            elif n == "memory.pressure":
                rec["pressure"] = s.strip().splitlines()[:2]
        try:
            (HOSTMEM / f"{tag}.txt").write_text("\n".join(txt))
        except Exception:
            pass
        rec["host_mem"] = host_mem()
        return rec


    ENGINE_RX = re.compile(r"vllm|PleOffload|ple_offload|EngineCore|merlin_off/venv", re.I)


    def engine_procs():
        """Processes of an engine (vLLM server, workers, PLE offload) other than this notebook; None off Linux."""
        if not Path("/proc/self/stat").exists():
            return None
        me, out = os.getpid(), []
        for d in Path("/proc").glob("[0-9]*"):
            try:
                pid = int(d.name)
                if pid == me:
                    continue
                cmd = (d / "cmdline").read_bytes().replace(b"\0", b" ").decode(errors="replace")
                if ENGINE_RX.search(cmd) and "py_compile" not in cmd:
                    out.append([pid, cmd[:160]])
            except Exception:
                pass
        return out


    def clear_engine_procs():
        left0 = engine_procs()
        rec = {"found": left0}
        if left0:
            for pid, _ in left0:
                try:
                    os.kill(pid, 9)
                except Exception:
                    pass
            time.sleep(10)
            rec["after_kill"] = engine_procs()
        rec["ok"] = not (rec.get("after_kill") if left0 else left0)
        return rec


    def drop_page_cache():
        """posix_fadvise(DONTNEED) on every checkpoint file (merged dir symlinks resolved + the Model mounts); only after the
        previous server is dead (never on the pinned PLE pages of a live server). A measured attempt, not an assumed fix."""
        if not hasattr(os, "posix_fadvise"):
            return {"available": False}
        t, seen, n, b, err = time.time(), set(), 0, 0, 0
        roots = [CKPT] + [v[0] for v in PARTS.values() if v] + ([PLE_DIR] if PLE_DIR else [])
        for root in roots:
            for dp, _, fns in os.walk(str(root)):
                for x in fns:
                    p = os.path.realpath(os.path.join(dp, x))
                    if p in seen:
                        continue
                    seen.add(p)
                    try:
                        fd = os.open(p, os.O_RDONLY)
                        try:
                            os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
                            b += os.fstat(fd).st_size
                            n += 1
                        finally:
                            os.close(fd)
                    except OSError:
                        err += 1
        return {"available": True, "files": n, "gib": round(b / 2 ** 30, 1), "errors": err, "sec": round(time.time() - t, 1)}


    def host_ram_gate(tag):
        before = cgroup_snapshot(tag + "_before_drop")
        drops, tries = [drop_page_cache()], 0
        after = host_mem()
        while True:
            av = after.get("avail_mib", after.get("MemAvailable_mib"))
            if av is None:
                ok = bool(FAKE)               # local harness only; on Kaggle an unreadable host is never "enough"
                why = "no cgroup / meminfo reading" + (" (local harness: skipped)" if FAKE else ": HOST_RAM_GATE fails closed")
                break
            if av >= RAM_NEED_MIB:
                ok, why = True, f"avail {av} MiB >= {RAM_NEED_MIB}"
                break
            if FAKE:                          # local / CPU-kit harness only (a 30 GiB CPU box): recorded, never enforced
                ok, why = True, f"avail {av} MiB < {RAM_NEED_MIB} MiB (local harness: recorded, not enforced)"
                break
            tries += 1
            if tries > 3:
                ok, why = False, f"HOST_RAM_GATE: avail {av} MiB < {RAM_NEED_MIB} MiB after {tries} page-cache drops"
                break
            time.sleep(20)
            drops.append(drop_page_cache())
            after = host_mem()
        return {"ok": ok, "why": why, "need_mib": RAM_NEED_MIB, "before": before, "after_drop": cgroup_snapshot(tag + "_after_drop"),
                "drops": drops}


    _boot_gate_rc2, _shm_cleanup_rc2 = boot_gate, shm_cleanup
    BOOT = {"deadline": None, "fired": None, "tag": "boot"}


    def boot_gate():
        """RC2's GPU release gate (< 150 MiB per card), the previous engine's processes gone, then the host-RAM gate."""
        g = _boot_gate_rc2()
        if g.get("ok") is not True:
            return g
        g["engine_procs"] = clear_engine_procs()
        if not g["engine_procs"]["ok"]:
            g["ok"], g["why"] = False, "ENGINE_PROCS_LEFT: " + json.dumps(g["engine_procs"].get("after_kill"))[:300]
            return g
        h = host_ram_gate(BOOT["tag"])
        g["host_ram"] = h
        if not h["ok"]:
            g["ok"], g["why"] = False, h["why"]
        return g


    def shm_cleanup(before):
        """RC2's /dev/shm cleanup (this boot's torch_/psm_/vllm/ple/nccl/sem.mp-/cuda.shm entries) + the loky semaphores the
        RC2 log reported leaked at shutdown (sem.loky-*), only when created during this boot."""
        out = _shm_cleanup_rc2(before)
        out["removed_loky"] = []
        try:
            for p in sorted(SHM.iterdir()):
                if p.name not in before and re.match(r"sem\.loky-", p.name):
                    try:
                        p.unlink()
                        out["removed_loky"].append(p.name)
                    except Exception:
                        pass
        except Exception:
            pass
        return out


    class LoadMonitor(threading.Thread):
        """Every 10 s from launch until ready: per process of the server's session, major faults and /proc/<pid>/io
        (rchar, read_bytes) + the cgroup current; a slow worker (RC2: 873 s vs 125 s) becomes diagnosable."""
        def __init__(self, name):
            super().__init__(daemon=True)
            self.sname, self.stop_ev, self.path, self.last = name, threading.Event(), OUT / f"load_monitor_{name}.jsonl", None

        def sample(self, sid):
            rows = []
            for d in Path("/proc").glob("[0-9]*"):
                try:
                    st = (d / "stat").read_text()
                    rp = st.rindex(")")
                    f = st[rp + 2:].split()
                    if int(f[3]) != sid:
                        continue
                    io = dict(l.split(": ", 1) for l in (d / "io").read_text().splitlines() if ": " in l)
                    rows.append([int(d.name), st[st.index("(") + 1:rp][:24], int(f[9]), int(io.get("rchar", 0)) >> 20,
                                 int(io.get("read_bytes", 0)) >> 20])
                except Exception:
                    pass
            return rows

        def run(self):
            while not self.stop_ev.is_set():
                srv = next((x for x in list(L.LIVE) if x.name == self.sname), None)
                if srv is not None and srv.proc is not None and Path("/proc/self/stat").exists():
                    rec = {"t": round(time.time() - T_START, 1), "procs[pid,comm,majflt,rchar_mib,read_mib]": self.sample(srv.proc.pid),
                           "cgroup_mib": (host_mem() or {}).get("cgroup_cur_mib")}
                    self.last = rec
                    try:
                        with open(self.path, "a") as fh:
                            fh.write(json.dumps(rec) + "\n")
                    except Exception:
                        pass
                    if srv.ready_sec is not None:
                        break
                self.stop_ev.wait(10)


    # ---- per-boot watchdog
    def boot_left():
        d = BOOT["deadline"]
        return min(left(), (d - time.time()) if d else float("inf"))


    def due(sec):
        return boot_left() > sec and not ABORT["why"]


    def watched(body, cap_sec, name):
        """The boot body gets `cap_sec`; every step checks due(); at cap + 300 s the server is killed (hard backstop)."""
        def b(ss):
            BOOT.update(deadline=time.time() + cap_sec, fired=None)

            def fire():
                BOOT["fired"] = name
                for x in ss:
                    try:
                        x.kill()
                    except Exception:
                        pass
            tm = threading.Timer(cap_sec + 300, fire)
            tm.daemon = True
            tm.start()
            try:
                o = body(ss)
                LIVE["out"] = o if isinstance(o, dict) else None   # R2: step() keeps it if anything after the body raises
            finally:
                tm.cancel()
                BOOT["deadline"] = None
            if BOOT["fired"]:
                o["boot_watchdog"] = f"FIRED after {cap_sec + 300:.0f} s: server killed"
            return o
        return b


    # ---- server facts read from the log and from the workers
    def _pick(t, rx, n=6):
        return [l.strip()[:300] for l in t.splitlines() if re.search(rx, l)][:n]


    def server_caps(t, argv=()):
        na = next((l for l in t.splitlines() if "non-default args:" in l), "")
        caps = {"source": "log" if na else "argv"}
        for k in ("max_num_seqs", "max_num_batched_tokens", "block_size", "max_model_len", "kv_cache_memory_bytes", "num_speculative_tokens"):
            m = re.search(rf"'{k}': (\d+)", na)
            caps[k] = int(m.group(1)) if m else None
        for k in ("kv_cache_dtype", "mamba_ssm_cache_dtype", "mamba_cache_mode", "worker_extension_cls"):
            m = re.search(rf"'{k}': '([^']+)'", na)
            caps[k] = m.group(1) if m else None
        m = re.search(r"'language_model_only': (True|False)", na)
        caps["language_model_only"] = m.group(1) == "True" if m else False
        m = re.search(r"'cudagraph_capture_sizes': \[([\d, ]+)\]", na)
        caps["capture_sizes"] = [int(x) for x in m.group(1).split(",")] if m else None
        m = re.findall(r"Setting attention block size to (\d+) tokens", t)
        if m:
            caps["block_size"] = int(m[-1])
        if not na:                                     # fallback (the E23 fake has no args line): the argv, last wins
            a = [str(x) for x in argv]
            for flag, key in (("--max-num-seqs", "max_num_seqs"), ("--block-size", "block_size")):
                idx = [i for i, x in enumerate(a) if x == flag]
                if idx and idx[-1] + 1 < len(a):
                    caps[key] = int(a[idx[-1] + 1])
        caps["block_size"] = caps.get("block_size") or BLOCK
        sizes = caps.get("capture_sizes") or []
        k = caps.get("num_speculative_tokens") or 3
        caps["captures_cover_seqs_x_1pk"] = bool(sizes) and caps.get("max_num_seqs") is not None and max(sizes) >= caps["max_num_seqs"] * (1 + k)
        return caps


    def pool_facts(t):
        f = {}
        m = re.findall(r"GPU KV cache size: ([\d,]+) tokens", t)
        f["kv_tokens"] = int(m[-1].replace(",", "")) if m else None
        m = re.findall(r"Maximum concurrency for ([\d,]+) tokens per request: ([\d.]+)x", t)
        f["max_concurrency"] = float(m[-1][1]) if m else None
        f["max_concurrency_tokens"] = int(m[-1][0].replace(",", "")) if m else None     # R2: M of the line (F_log)
        m = re.findall(r"Padding mamba page size by ([\d.]+)%", t)
        f["mamba_pad_pct"] = float(m[-1]) if m else None
        f["min_block_for_mamba"] = math.ceil(BLOCK / (1 + f["mamba_pad_pct"] / 100)) if m else None
        f["kv_group_sizes"] = (re.findall(r"kv cache group sizes \(([^)]*)\)", t) or [None])[-1]
        f["side_cache_layout"] = _pick(t, r"Side-cache layout", 2)
        f["initial_free_reserved_gib"] = [[float(a), float(b)] for a, b in re.findall(r"Initial free memory ([\d.]+) GiB, reserved ([\d.]+) GiB", t)][:4]
        f["model_load_gib_sec"] = [[float(a), float(b)] for a, b in re.findall(r"Model loading took ([\d.]+) GiB memory and ([\d.]+) seconds", t)][:4]
        f["graph_capture_sec_gib"] = [[int(a), float(b)] for a, b in re.findall(r"Graph capturing finished in (\d+) secs, took ([\d.]+) GiB", t)][:4]
        f["rank_load_gib"] = {int(p) * 2 + int(q): float(g) for p, q, g in re.findall(
            r"\(Worker_PP(\d)_TP(\d)\S*(?: pid=\d+)?\) INFO [^\n]*?Model loading took ([\d.]+) GiB memory", t)}
        f["rank_load_sec"] = {int(p) * 2 + int(q): float(g) for p, q, g in re.findall(
            r"\(Worker_PP(\d)_TP(\d)\S*(?: pid=\d+)?\) INFO [^\n]*?Model loading took [\d.]+ GiB memory and ([\d.]+) seconds", t)}
        f["rank_graph_gib"] = {int(p) * 2 + int(q): float(g) for p, q, g in re.findall(
            r"\(Worker_PP(\d)_TP(\d)\S*(?: pid=\d+)?\) INFO [^\n]*?Graph capturing finished in \d+ secs, took ([\d.]+) GiB", t)}
        f["ssm_dtype_lines"] = _pick(t, r"mamba_ssm_dtype|mamba-ssm-cache-dtype")
        f["gdn_fallback_lines"] = _pick(t, r"Falling back to the Triton GDN spec decode path")
        f["share_workspace_lines"] = _pick(t, r"MERLIN_SHARE_DRAFT_WORKSPACE")
        f["mm_warmup_lines"] = _pick(t, r"Multi-modal warmup completed")
        f["host_ram_lines"] = _pick(t, r"host memory available now|Cgroup memory preflight|Skipping auto-prefetch", 6)
        f["sched_warn_lines"] = _pick(t, r"max_num_scheduled_tokens", 2)
        f["oom_lines"] = _pick(t, r"OutOfMemoryError|CUDA out of memory", 4)
        f["static_scale_lines"] = _pick(t, r"applied static K/V scales", 8)
        f["default_scale_lines"] = _pick(t, r"DEFAULT 1\.0|no static calibration", 4)    # audit 2: the MTP layer's QSA FP8 KV
        f["ext_lines"] = _pick(t, r"Injected .*NB1WorkerExt|nb1_worker_ext", 2)
        f["shm_leak_lines"] = _pick(t, r"leaked (semaphore|shared_memory) objects", 4)
        return f


    def graph_facts(t):
        caps = sorted({(a, int(c)) for a, b, c in re.findall(r"Capturing CUDA graphs \((\w+)\):\s*100%\|[^|]*\|\s*(\d+)/(\d+)", t)})
        return {"captured": caps, "resolved_modes": _pick(t, r"resolved cudagraph modes before capture", 8),
                "draft_identity": _pick(t, r"e2-identity\[.*role=draft", 4), "recoverssm_commit": _pick(t, r"\[recoverssm-commit\]"),
                "recoverssm_path": _pick(t, r"RecoverSSM path taken", 4), "jit": jit_kernels(t)}


    def jit_kernels(t):
        return re.findall(r"JIT compilation during inference: (\w+)", t)


    def nccl_transport(t):
        via = Counter(re.findall(r"NCCL INFO Channel \d+/\d+ ?: ?\d+\[\d+\] -> \d+\[\d+\] (?:\[\w+\] )?via (\S+)", t))
        return {"channel_via": dict(via), "net": sorted(set(re.findall(r"NCCL INFO (Using network \w+|NET/\w+ : Using [^\n]{0,60})", t)))[:6],
                "comm_inits": len(re.findall(r"NCCL INFO comm 0x[0-9a-f]+ rank \d+ nranks \d+", t)),
                "pynccl_groups": _pick(t, r"all-reduce backends .* for group", 6),
                "debug_on": "NCCL INFO" in t}


    def rpc_cache_info(port):
        """The read-only worker extension over /collective_rpc (dev mode): actual cache tensor dtypes, specs, free memory."""
        body = {"method": "nb1_cache_info", "timeout": 120}
        try:
            r = json.loads(L.U.urlopen(L.U.Request(f"http://127.0.0.1:{port}/collective_rpc", data=json.dumps(body).encode(),
                                                   headers={"Content-Type": "application/json"}), timeout=180).read())
            res = r.get("results") or []
        except Exception as e:
            return {"error": repr(e)[:300]}
        out = {"ranks": res}
        dts = sorted({x["dtype"] for w in res if isinstance(w, dict) for g in (w.get("cache_tensors") or [])
                      if "mamba" in g.get("module", "").lower() or "gdn" in g.get("module", "").lower() or "linear" in g.get("module", "").lower()
                      for x in g.get("tensors") or []})
        out["recurrent_tensor_dtypes"] = dts
        out["num_gpu_blocks"] = sorted({w.get("num_gpu_blocks") for w in res if isinstance(w, dict)} - {None})
        out["free_mib_per_rank"] = [((w or {}).get("mem") or {}).get("free_mib") for w in res if isinstance(w, dict)]
        pads = []
        for w in res:
            for s in ((w or {}).get("kv_cache_specs") or []) if isinstance(w, dict) else []:
                if s.get("unpadded_page_bytes") and s.get("page_size_bytes"):
                    pads.append({"rank": w.get("rank"), "type": s.get("type"), "layers": s.get("layers"), "state_dtypes": s.get("state_dtypes"),
                                 "padded": s["page_size_bytes"], "unpadded": s["unpadded_page_bytes"],
                                 "pad_pct": round(100 * (s["page_size_bytes"] - s["unpadded_page_bytes"]) / s["unpadded_page_bytes"], 2)})
        out["mamba_padding"] = pads[:16]
        return out


    def rpc(port, method, *args, timeout=120):
        """POST /collective_rpc (dev mode): a worker method by name with string args; the per-rank results or the error."""
        body = {"method": method, "args": [str(a) for a in args], "timeout": timeout}
        try:
            r = json.loads(L.U.urlopen(L.U.Request(f"http://127.0.0.1:{port}/collective_rpc", data=json.dumps(body).encode(),
                                                   headers={"Content-Type": "application/json"}), timeout=timeout + 60).read())
            return {"results": r.get("results")}
        except Exception as e:
            return {"error": repr(e)[:300]}








    def scales_dump():
        """The static QSA K/V scales file (all values) + which layers it covers; the MTP layer is not in it (default 1.0)."""
        p = CKPT / "qsa_kv_scales_262k.json"
        try:
            d = json.loads(p.read_text())
        except Exception as e:
            return {"error": repr(e)[:200]}
        lay = sorted(int(m.group(1)) for k in d for m in [re.search(r"layers\.(\d+)\.", k)] if m)
        ks = [v.get("k_scale") for v in d.values() if isinstance(v, dict)]
        vs = [v.get("v_scale") for v in d.values() if isinstance(v, dict)]
        return {"file": str(p), "sha256": hashlib.sha256(p.read_bytes()).hexdigest(), "values": d, "layers": lay, "n": len(d),
                "k_scale_min_max": [min(ks), max(ks)] if ks else None, "v_scale_min_max": [min(vs), max(vs)] if vs else None,
                "mtp_layer_in_file": any("mtp" in k for k in d)}


    def free_per_card(rec, name):
        """MiB free per card = total - the highest used reading of this boot (after ready, end, every window peak)."""
        srv = ((rec or {}).get("servers") or {}).get(name) or {}
        rows = [r for r in (srv.get("gpu_mem_after_ready"), srv.get("gpu_mem_end")) if r]
        if not rows:
            return None
        tot = {i: t for row in rows for i, _, t in row}
        used = {i: max(u for row in rows for j, u, _ in row if j == i) for i in tot}
        st = (rec or {}).get("steady")             # R2: nb1_ladder's steady holds "_plan" (list), "_fit" (dict), "_seqs_cap" (int)
        ws = list(st.values()) if isinstance(st, dict) else []
        for w in ws + [(rec or {}).get("profile")]:
            if not isinstance(w, dict):            # R2: only window dicts carry gpu_mem_peak_mib
                continue
            pk = w.get("gpu_mem_peak_mib")
            for i, u in enumerate(pk if isinstance(pk, list) else []):
                if i in used and isinstance(u, (int, float)):
                    used[i] = max(used[i], u)
        return [tot[i] - used[i] for i in sorted(tot)]


    def free_after_ready(rec, name):
        srv = ((rec or {}).get("servers") or {}).get(name) or {}
        row = srv.get("gpu_mem_after_ready")
        return [t - u for _, u, t in row] if row else None


    # ---- KV accounting: pool blocks from usage denominators; cold-cache cumulative admission (physical snapshots)
    def kvu(port):
        return L.prom(port).get(L.KVU)
























    F_MAX_BLOCKS = 40                                # R2: a per-request fixed-block F above 40 blocks is not physical here
    FIT_MIN_SAMPLES = 3                              # R2 audit N1-6: a fit is trusted only with >= 3 all-resident snapshots,
    FIT_MAX_RESID_BLOCKS = 2.0                       # a max |residual| <= 2 blocks
    FIT_LOG_TOL_BLOCKS = 1                           # and |ceil(fit F) - log F| <= 1 block
    F_LOG_SOURCE = "server max-concurrency line (this boot)"












    def profiler_tools():
        """ncu / nsys availability (DRAM bytes per stage need a server LAUNCHED under one of them: a separate boot)."""
        out = {}
        cands = ["ncu", "nsys", "/usr/local/cuda/bin/ncu", "/usr/local/cuda/bin/nsys"]
        try:
            eng = MERLIN_ENG if isinstance(MERLIN_ENG, dict) else {}
            if eng.get("cuda_home"):
                cands += [eng["cuda_home"] + "/bin/ncu", eng["cuda_home"] + "/bin/nsys"]
        except Exception:
            pass
        for c in cands:
            path = shutil.which(c) if "/" not in c else (c if os.path.exists(c) else None)
            if path:
                try:
                    v = subprocess.run([path, "--version"], capture_output=True, text=True, timeout=60).stdout.strip()[:200]
                except Exception as e:
                    v = repr(e)[:100]
                out[c] = {"path": path, "version": v}
        return {"found": out, "dram_bytes": "not measured in NB1: ncu / nsys cannot attach to a running vLLM server; DRAM bytes per "
                "stage need a boot launched under ncu (dram__bytes_read.sum on the Marlin MoE + dense kernels) or nsys: NB3 / NB4 item",
                "torch_profiler_window": "the PP2 trace window runs after the ladder, on its own streams; no benchmark window overlaps it"}


    # ---- requests with usage (cached tokens), greedy logprobs and teacher-forced rows with the cache counters








    # ---- JIT warm-up (excluded from every metric) and per-position acceptance
    def warmup(port, n, secs=None):
        """Multi-stream 2k decode, ramped 1 -> n streams, excluded from every metric: compiles the kernels RC2 saw compiled
        during inference (_fill_num_accepted_kernel, _causal_conv1d_update_kernel, _gdn_recoverssm_verify_kernel,
        _ple_conv_kernel, _ple_conv_writeback_kernel) before any measured window."""
        secs = secs or (45.0 if not FAKE else 4.0)
        srv = next((x for x in list(L.LIVE) if x.port == port), None)
        j0 = jit_kernels(srv.log()) if srv else []
        ss, t0 = [], time.time()
        try:
            for i in range(n):
                g = L.Stream(port, PROMPTS["2k"][i % len(PROMPTS["2k"])], 4096, temperature=TEMP, seed=900 + i, ignore_eos=True)
                g.start()
                ss.append(g)
                time.sleep(min(3.0, secs / max(1, 2 * n)))
            while time.time() - t0 < secs:
                time.sleep(1)
        finally:
            for g in ss:
                g.stop()
            for g in ss:
                g.join(10)
        L.wait_idle(port, 60)
        j1 = jit_kernels(srv.log()) if srv else []
        return {"streams": n, "sec": round(time.time() - t0, 1), "jit_before": j0, "jit_compiled_in_warmup": j1[len(j0):],
                "errors": sorted({g.err for g in ss if g.err})[:3]}


    def accept_pos(port):
        try:
            txt = L.U.urlopen(f"http://127.0.0.1:{port}/metrics", timeout=15).read().decode("utf-8", "replace")
        except Exception:
            return None
        pos = {}
        for m in re.finditer(r"^vllm:spec_decode_num_accepted_tokens_per_pos_total\{([^}]*)\}\s+(\S+)", txt, re.M):
            p = re.search(r'position="(\d+)"', m.group(1))
            if p:
                pos[int(p.group(1))] = pos.get(int(p.group(1)), 0.0) + float(m.group(2))
        return {"pos": pos, "drafts": L.prom(port).get(L.DRF, 0.0)}


    def _prefill_done(r, n):
        tail = ((r.get("admission") or {}).get("tail") or [])
        t = tail[-1] if tail else None
        return bool(t) and t[1] == n and t[2] == 0 and t[3] >= n


    def _req_lines():
        try:
            with open(REQ_LOG) as fh:
                return sum(1 for _ in fh)
        except Exception:
            return 0


    def _ttfts(port, start_line):
        out = []
        try:
            with open(REQ_LOG) as fh:
                for i, l in enumerate(fh):
                    if i >= start_line:
                        r = json.loads(l)
                        if r.get("port") == port and r.get("ttft") is not None:
                            out.append(r["ttft"])
        except Exception:
            pass
        return out


    def _counters(port):
        try:
            m = L.prom(port)
            return m.get(L.PROMPT, 0.0), m.get("vllm:prefix_cache_hits_total", 0.0)
        except Exception:
            return None, None


    def window(port, s, band, n, max_new=None, secs=None):
        """One steady window, timed only after every stream finished admission + prefill (the lab's settle needs all running
        streams to have emitted and be gaining, then an explicit check of the last admission sample; Merlin's cost-aware
        admission may hold a cold miss up to 120 s, inside the lab's >= 420 s admission limit); a window with prefill inside is
        retried once. Recorded: tick_ms (one engine step = one micro-batch of C/2 requests under PP2) AND cycle_ms (= PP x
        tick, the per-request advance time), per-request TTFT, uncached prefill tok/s (prompt minus prefix-hit tokens),
        retained (idle) blocks before the window, per-position acceptance."""
        retained_before = None
        if L.wait_idle(port, 60):
            retained_before = kvu(port)
        def run():
            if max_new is None:                 # RC2 / E23 levels(): MAX_NEW[band], WIN, the budget / KV-bound guards
                return levels(port, band, [n], server=s, temperature=TEMP, prefill_tok_s=PREFILL["tok_s"]).get(f"{band}@{n}") or {}
            return L.steady(port, PROMPTS[band], n, secs=secs or WIN, max_new=max_new, temperature=TEMP, prefill_tok_s=PREFILL["tok_s"])
        a0, c0, l0 = accept_pos(port), _counters(port), _req_lines()
        r = run()
        if r.get("status") == "INCONCLUSIVE_PREFILL_IN_WINDOW" and due(WIN + 120 + n * BANDS[band] / max(500.0, PREFILL["tok_s"])):
            c0, l0 = _counters(port), _req_lines()
            r = dict(run(), retried_after="INCONCLUSIVE_PREFILL_IN_WINDOW")
        a1, c1 = accept_pos(port), _counters(port)
        r["prefill_done_before_window"] = _prefill_done(r, n)
        r["running_eq_level"] = r.get("running_min") == r.get("running_max") == n
        if r.get("status") == "STEADY" and not (r["running_eq_level"] and r["prefill_done_before_window"]):
            r["status"] = "INCONCLUSIVE_RUNNING_NE_LEVEL" if not r["running_eq_level"] else "INCONCLUSIVE_PREFILL_NOT_DONE"
        if r.get("step_ms"):
            r["tick_ms"] = r["step_ms"]
            r["cycle_ms"] = round(PP * r["step_ms"], 1)
            r["requests_per_tick"] = round(n / PP, 2)
            r["microbatches_per_s"] = round(1000.0 / r["step_ms"], 2)
            r["wall_clock_tok_s"] = r.get("tok_s")
            r["tick_note"] = ("tick_ms = step_ms = one PP tick (one completed micro-batch of C/2 requests); cycle_ms = 2 ticks = the "
                              "per-request advance time; microbatches_per_s = completed micro-batches per second; wall_clock_tok_s = "
                              "generated tokens / window seconds")
        adm = (r.get("admission") or {}).get("sec")
        if None not in (c0[0], c1[0]) and adm:
            unc = (c1[0] - c0[0]) - (c1[1] - c0[1])
            r["prefill_uncached_tokens"] = int(unc)
            r["prefill_tok_s_uncached"] = round(unc / adm, 1)
            r["prefill_note"] = "admission.prefill_tok_s counts cached prompt tokens; prefill_tok_s_uncached does not (incl. a retry)"
        r["ttft_per_request"] = _ttfts(port, l0)
        if retained_before is not None:
            r["retained_usage_before"] = retained_before
        if a0 and a1 and a1["drafts"] > a0["drafts"]:
            r["accept_rate_per_position_incl_admission"] = {p: round((a1["pos"].get(p, 0) - a0["pos"].get(p, 0)) / (a1["drafts"] - a0["drafts"]), 3)
                                                            for p in sorted(a1["pos"])}
        return r




    # ---- prefix continuation (2-turn replay) and after an abort






    # ---- PP2 trace window: every rank's trace, stage overlap, idle, verify rows per micro-batch
    SCOPE_ROWS = re.compile(r"^execute_context_(\d+)\((\d+)\)_generation_(\d+)\((\d+)\)$")












    CAPTURE_SIZES = None
    ROUTING_FILE = OUT / "nb1_routing_ids.npz"   # NB3 reads it (dataset foysalemonshanto/e25-nb1-routing-ids after a run)






    def quality_gate(o, ref):
        """RC2's fp8_gate with the reference as a parameter (B1 vs RC2's recorded FP8 rows, B2 / B3 vs B1)."""
        vs = ((o.get("oracle") or {}).get("vs_" + ref) or {}).get("status")
        if (o.get("sampled") or {}).get("ok") is not True:
            return "FAIL_SAMPLED"
        if (o.get("probes") or {}).get("ok") is not True and not FAKE:
            return "FAIL_PROBES"
        orc = o.get("oracle") or {}
        if any(orc.get("errors") or []) or any(orc.get("garbage") or []) or vs == "FAIL_HIGH_MARGIN":
            return "FAIL"
        if orc.get("complete") is True and vs in CLEAN:
            return "PASS"
        return "INCONCLUSIVE"


    def flag_map(argv):
        m, a, i = {}, [str(x) for x in argv], 0
        while i < len(a):
            if a[i].startswith("--"):
                k, v = a[i], True
                if i + 1 < len(a) and not a[i + 1].startswith("--"):
                    v = a[i + 1]
                    i += 1
                m[k] = v
            i += 1
        return m


    def config_diff(argv, env, kv_bytes):
        """Effective flags (last occurrence wins) and env vs RC2's real FP8 boot."""
        ref, new = flag_map(RC2_FP8["argv_tail"]), flag_map(argv)
        skip = {"--port", "--served-model-name", "--host"}
        flags = {k: [ref.get(k), new.get(k)] for k in sorted(set(ref) | set(new)) if k not in skip and ref.get(k) != new.get(k)}
        keys = set(RC2_FP8["env"]) | {k for k in env if k.startswith(("VLLM_", "NCCL_", "KV_", "P2P", "SCALES", "MERLIN_"))}
        envd = {k: [RC2_FP8["env"].get(k), env.get(k)] for k in sorted(keys)
                if k not in ("VLLM_PORT", "MASTER_PORT") and RC2_FP8["env"].get(k) != env.get(k)}
        return {"flags": flags, "env": envd, "kv_bytes": [RC2_FP8["kv_bytes"], kv_bytes]}


    def boot_facts(s, o, name):
        t = s.log() or ""
        o["caps"] = server_caps(t, s.argv)
        o["pool"] = pool_facts(t)
        o["graphs"] = graph_facts(t)
        o["nccl"] = nccl_transport(t)
        o["workers"] = rpc_cache_info(s.port)
        o["host_after_ready"] = cgroup_snapshot(name + "_after_ready")
        env = dict((R["setup"].get("merlin_plans") or {}).get(name, {}).get("env_keys") or {})
        env.update(NB1_ENV.get(name) or {})
        o["config_diff"] = config_diff(s.argv, env, NB1_KV.get(name))
        return o
    save()


In [ ]:
if not RERUN:
    # ==================================================================== CELL 7 — E27 NBDF: arms Q > D4 > D3 > D7 > D4b (DFlash2 vs MTP K3), long bands (NEW code)
    # The modules and the CELL 7 body are lzma + base85, each sha256-checked here (the readable body = nbdf_code_export.py).
    import importlib, base64, lzma, gzip, types, inspect
    def _unx(b):
        return lzma.decompress(base64.b85decode(b)).decode("utf-8")
    NBDF_LIB_SHA = "b014775162fb55401826fa8c498f7009b5d47ad15cc5cb365ab23e5ed37b1929"                 # nbdf_lib.py (new), nb4t_lib.py (NB4T verbatim), nb4_timeline.py + nb4_ppack.py
    NBDF_LIB_SRC = _unx("{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;y4wcfL#C)Xbna~aS1SFih*j!rfewm_Y?31yz*c2z=V$}+l^xG-uzl5b((ZPWK??z?)iv8fk&mx@s6&hE6En58^Xhm$q9Z~pcI1s&~p8O$jyn*9A+JebnsO&rQ@S7jgsWF3~^5JZNbU>`mZz3s<a-ldzd#G_VhylqdARG<qGj)OPQuQdNY4B|H$E(&<M3}-b(pqENbpwx)JOU6Ekwv`M-XFkGe%LONl4IhqJj@T?*@x<<LYU&Rek!hPN3by=WYlQ$Jdhg&>l1B1I!4uY{KZ8Votjv;N|0C3dvIm)2xH$SdYj5FG*9{1I0|VwwNKu9>tby4QP$%D9DjjGj%D{6_sa&lJnf)@N0i$9g@VvyU9iBomm;Y>W{6E_}^V<iU000l7lhCV3taF3t5S$M9_7^5D`=-T=}E0ax$-&+IQ=50EoD(ikP=^Fr%Jr+W4TpHao)eLkI5$R$x&Z2c4@4I5oc_fd0O98?Iz6UC6kfQ|xhzRplKc=s4JucSxW${=rbtPGxgAaWaPo{jt?i)G_0FZOnqu;_duYeIe-{zfI~*%n`!N)YUzHOT<Ivh9%WHX}{IdWZc^inB8kUov){!%ebcXY%aF2npH6szx9unJErRf0P=h0cOImF(?UR9M0)iS5G`4P<&PH`}(3iP>QDKdGjYMG;2z%*LD4x0K37d^llxHNB{riCGrsEg;%JrbT|RN3_n-Ou@f#>c|G8CBC_+<j@i<C+<$X1V^<n;<+nJ(!Pt{pem$f%!{LrXqTysrM;kvH)+oE!bRtcUf@U_zeV}j@@@aJDL!(rRFvwc0n+H`}u&w5&cQ!Qb<;`ntUK*Sh?wlhccX3es3)dZ=)rW}ZYl%l9l-W1ToTM}2`^+rtKiLh(t~q8ryAg&3q;Q-ikulo%_ku}%rANlE%~U!x=v<|e$PSHYuT~1dYuP9t*WBZE9bNRsx-4+=c26<&aJfJAkSlHYQ9!JyKx7TuYn(@Iu2v2VY60jjeVMd?PtgP{YYYS<3Y#_7m>p8(=Te8qU6($;np=}oI40M}@g2sx(W)UZr#}&o){r*M<<EznqCE;v{x>)4NX3A%|A!mqf<dJ{ymjE;Yx)YA;xQPtQS5L9mqP`jT>Z6P1NJE7t$O~5swAwG_~xN`G_yj1wv3qe3i=a!2ZaCvRgZ2fO2^=~2kmlk(La|s04mOWxpR?JLUj;ycp-Bp+Xn7G`Idqleq1neP!Cp)r*anxU}7!+?s?_E{N3HMW$gME2qRg6Gg<ONmuQJm_9m)ut!Pl3+aOPCq915~m&^=3FX>Y$K(~DpQ%9ATxM>P~kkOUFoDv9Gt*>d__gOBr1J@<sZ9+Au>qan&X2^Mh3Vlz(3g$;>J>>NIw&I>-6Rv)nWrMY1FK1L~m>sUYe*^%G8DVOEgknF!B4zEzF0@*Bc8BTGNp!Vf_HxT>FTK8UgdZx<XMO{QE|mvw`=fiUAj2kjZmDf?a1|lh$fU2H=!a4m{qb;o6tIsWSfvw18%xHfl5PCU+vBGvILUtGgF)w~eHw)jA?Le>3IZpG_r8;P^Uakq_8glVqF=9kx3@=Z_4-05#PL<=QX;7l)&i!WBzG_t{caN83)4yV<}B}3_V|GWjgbyaWg{|LipFd~JgMVwQ`TSCe>$<ZU8yH2S>EhER4;bb^h#!tVzEvo@*5GL3M^m`jWyRZIm~aG|7;@PB3fu*@U*WeVCjmi79bNUU}uWp9nx@UPek@7_^tp)rf4s1X*Um-Av4nj%HeA`(}5Uji$wpS`}0H3>}hj3+UguQ>Fu)1+``yBnn8^`kH!Kdh^Of)e>*Yg-VB~&`J^xns#3TV>c@2tqyS<$dTy|mYSAFeCHvrGs#5*i<h;6nV#wruOZS_*2{#mP3rQEFP>O83-l>6#zXl9%h(V;#ZBU~6*%#b}ho!5c(iu+`+N&nspUrxKm?RC_(q?^!>KqJVvH6Rg$DPYU0(7Y}d(JfSA(@5}DZwRUQz$+nxOGp+>TRiTORtZ2uxE&49m!ab1m@cz`rzpmnZkt7(@_68XsY8JPa-lQ!`A;)2G$7TR(-*!l1xY|ve0V3<7$>XqfGrLp#e@2+Yuc(Pe0Azx!v2Pfzd1@QVTXX6!?^OPo?)_+38VAR)LsBAzgBz46>{@+>KJcdtFts6O&zmmReIl=4La&G$7B=BO-lSN&j#Z^gt`<;oM9a!H6MmQnX-`yHvxn5K<A7LWFv8t5e&m)WbDB94(qEt4Gi(@9oqhs=by0h@Sg?l!e2#He_C>#e2!fjYK5DUB7gQH!52OYlg*GA<#wbL`YV_t848@3C@~+;Dv4&H>-*>i<qUc&Uj9+_IHHhVR!eR^!W{YC%Z>>U~4xN=V}#o-GUHHstcUXdEK2_9OVi}huiG}Q=n`bG|D1S_}CohKT4TBZ5r-VjmrEvc~5iAjg*xQH?4<HtwIH*2kX?~Z^`^n6jCN#eGY~~4C&KMr3)4t1W*iuX!ZVDN6U)=ew);+KHU`}nk$a9Bqwu|%?(Bn9DS%xc8I04%gPl(S@5-Eyp%+G`ku1=tEM~XMJ=Y2>ZT%sIdrO3u{+yo;HZGV0?kQ8XhDa;MPTeXh(o@dF84uT)tGJa)}cbfPsblfJVfb$NH?<>uEP-GH-KDueq-}AOpCW6Q7r`iH*1fy`zI&Ay4$k?bh&yvh%VCTNrE+Y!0ztE4ybs4eOv7A=h3tXrua23PDy)}gR1_u4jgcbR4awi4$$Pwn_;p1mC!ocC?HK(!;gN$&H(pdY%-v?QI)VrI-`%wdS9W7m*rHO3}Sc?F*?|;IGlxn9Quxm{<-^(FEEAdB_j)9;F3~3bTm~@*f^BRP1M}$qn?lHZ7o~z$iM7{ptwmT=r(S-N5;f;jecGK`~U?EYGX8(zb09U^5&!lwd=DPM+WXWy~4$0sa3)_7&l<aI2#;#7ttwsu{TmRyMk&s3Yfkr=Cdt3M`R#SIEA$ZB_JklhZ~W1xsC-jziHPUrM4HZR7eH!o|S$!fOL9s3NkG%XFQ}~V^phk9&&qtW+|}w-?6mGQYhXftW1PL^60L<_aQdeow=&k#!t;9sNwHgzRDT(S__Ag55>GkE9{^YrqL!uE|?lMa2p7eP%TrywF?39a_fXLUn%_HuyGa2)2N+FCsK6qJ@qzdR<V*}=B9IeOD{7Jsiaeo-mq&Kx{R+ReE+Q;CRQ1>dn(&%`X`2KGC6b$#GV{&UDIDQy%w&+rwfwPlRKDH-8MheR6M2URODlcw-8_w-EhZ!O=nBRed5LtXn9k^4XuS_Y!m6KrRMK=7<u-d6yaWRoUa(xZpmUix31)2BA|XHmpOK^!#hu{{9D34Sd{Xqg8o4F$a~YX@IYcz6;hp$?^y|itB*bBe}1LBF6M;t#hm26kCdx@lW=IwHHgdMpDg|MYE7gD0ZFH6yfHTEGQvSDn|eE-*RY8hG|d6RP;LYylI6jT@G4YJuk9Up+Nb7R@HqnW3jnMBXOl(F|3or{QTh(>YtNy07P}ZW|GRSG>66jJk2Zpq3ph28?X1!GJ{JE$3}%%-$y$ynm2$hE9E@MwbWjkl0K@ohbu$ZLnJdQ(z}?XJc<l==#ntu^7mfN#0LyFK7;M#!<u{rs<cJb&*iVb2?iY9R7S2-dG<)B@ZRz3ce)5k-jHZmFM%2?6<TW<CxHLGEC^_LXO1N1uG0$HOlbM}?^&V+kV0p95mg9Pnd&7;s7*cfP8)Wm}?Q|AYZ(_%1%92t{1zvjwLw+f&fwz40X&*or)KtY;!Y4V3$v#^%rclQ!LmqeoNza7iw<<jGMyX9$xztP{+}$>1xWKkbsIa{9?f}wLpmo?KgJ*n&5>yp5X}QQs!sk0I?d}@wdXhxr_x?hd@F1oOS?RY7S-Mui@FeVd$;?IB>cV~!CJ!0+Cg~cDXnoY9{Kc81Y2De)>n0KwL;(cjc<_8D=_4wS?PagWS05sx87=X*P1`;R%y%8Z9d$JfM-_k>&bBLTOgZI^g0;^dY0yP(juj$Mb(h6ozl$y1G~+&QoIm$>7a2>EnupJFVtAR}8#O;)*Lk`YIkV*?&v=?7EhFeNiJmqM&kJu#*Mq7Ubwkkw9iA6w`XfH(Q_ei$L%HDG+B!K2V?Gq2_380f>nqj-d|<H#9o=Onaab-vAhX>dH<}q}>i4KLK9t~w3L#o#qB$uxZVH`o{WGe#B`YI{vo^g<DzjBNIalYXB6DuL{0D|(HR}gqrf$VPKR|I%Y}}m=<qH`u49dm~#37BRrlMI1&CXR^C^I+x`i+c1@%HFjr5RX75$%-8qDul%dG4HXIMOF<^=a$tgiT9<d~YUq;g_V~u>L;tRaE}6hM(4w2@!Wd!aXVH)nPt3ExGdDFm!$?K$E$DUk4po18`A59Zd>enB=`X(rvY2Un50&m&L*c(QJrtvjSi>{E_@+b4%6=OJc556o?@a?YY)C@XbT>*hX<%4GgphQHb{hr3+ec0DQDU@M4MOou1mHAyMR~UL!pp7ZnHeVAvgX+hRoRUJz@*awi`pjW=e{;VL<zu}69|`~ye7ClfzdIstCO-cRE-{-E)sy@Ksq4)C>5HS95X8ktXjz`f_fAh6=eI4!2p3(|K`oj8~+39sEfuDy)COavt-qXZJw{(l_UCS`947PPjaO;cO9$RTMCVwJ{VHkL>8T)o><T?+$=c#0zOrUu#^)_0vT->Nj!%L{X=Z>t|@8O)CzYBoBruFC!-MjcaAHD_Tb?P4~C_S(ZuIVwaRuD{;i?fcSm5D;?2E1+SD()p1FUNACzwQ)Fb0I;tBd0us`2nO1~gBHC{@jvTz``v`Wk~zYNd6l9`dqpeOzA3-lrRCXc63HG;ouwx01*fQp13|WvT8+wq?I<KUCxsIX))pDok`+jIR{5{~&hf}ppmgTcD#*-@i7jeXE|t)!5@AU1aHL`zC8fg5-SgqzVj{yre*Q2q@&?N+_`_M;F$wb?IyYw6hqDVpNlI;7m9`_F^`snfwBWPmdr+VxOY#j0M*z#p=OOURM9}$5^(afjdgd^_R0X>R)rp#&A4nq4<;Z%gINsysB@rPuppFo3Np*0%j;P#3?5gDFQ`oR8^m?As{gXiB%FrK*`Nexf)82O>3lh}Wy#AeD@HH$zQ_;2Mu^lJQQ+lSXfzFcV_6J6TEj<H_^+pvC8Z*+g7F5}n2<}BSx!*?31jhDc%F#$BdZ5v%2t;ROovIw>y;1hh*S<zx)BST<+TX(pV0EZ;l~bs)?bo`SCd1O6UOKEBGS&Gr&)I8@$)$7FF94^j<rXEpgfbAX+=<m4PSUcp35X1ZcSfPp+`!2xebg-M@Ft!MHn^k|>We9xK_NL<4I)g0i!!+oFiheS&AAxOnB?f<qOI|;NO7(TcplCy-~W4j#utPC7#bH#e0JzXzIIxixph3Q$Gq}l&5fb_Y?@cZs7aejfVj2W<EGT)+x)>gNS(-Uo~&HLIgjA112Lmc=h+1g#rAM9BR?-4;R3-o_&pg}1jl_N9s>mjdIgAj7k~Nm6;ky|VBX^nHeiq>XsXvq9D|#v;2es}cu1KY=N6kvKu9G^C2KDIeZD*Vw*w7iR4Sx4ftB}Tb94&ajp6VpPDBoO?$yBHfEIE=&xzNvGf9g}A443^?%;$098;%=nQtKy!?zbTTU-ac5c_8^E-UGIoS}-d)14oR!CVdo?MJ>X7f_kjm-|1={OZ*gPyih#BeW(weG3)JeSdO_>b$kOF4Voff;lczIbIYlkw5%j><+%!ao+De{us3Xy9vCQn`Df^eqHp*p@%MW3|@LHF{{u0+vy=0U(0pn0c^<*Ez+X`R>VwtknO;5jM$1WOLlPVJ}!Dl;25>uZynNxV4<k?{zc(XGY#zGS#O*FORDmXxCGXGc)ncwiH6;7&=GVR{@Wnj{gt5FE_QRR?FIPKD*TE>FLy24k=$iY8c`i4{r$e;EdYq>LWpWs5f%XUx?;9hOI7Eupnf1~&w_;=HH}WCb%QHjUBbn~ZevpKQ#5<4d;hrtNrLK|U=5y0{TCHNN1G$Rk7V#L)FW91#JF8?z7dMki6>29+0lA6m_5(M>cAr{w_COE8>=r}IaCa)r&&zvCf!BwIbC7yRf~kPFOpQ9Bckhy6Sbc6viyADbc;V%;kdnuJU0EQB%@VpzVFb;2#4+oZnfI#`Zl;Ks%?X|l}qupy0kbIii(A;F8*x23D5TATN=>I&V-1I(n5-n1eJ@`=7YwlM`pK|!O(8`(uqEj$ppHy#6QDdcru{E7^j<kvIlm={wo2k|HfCMn_M!<M5WEWR%H$<qF&74*A`ZV7alb4tVt&AYeFi;-fwzCEa*I?A1y@oNy(h3v7^hJmdTRagvXz9w}@Wf+$doGm_ZPji;Bp+gohQd1~1wvy39U|dI}SLLt}v8;2N`^q5M-<(mMTG01a(Mn%DIqe`=+>Bt~{_Tx90@o|AgqfGFm~0(|Iyc4p>#gCcrcRb{*+|6j>6`W(0ugFo9!F_C0lAipFY2ZHYAkl{53grsq~pv9PZ(T?Wqx?4t%&&0_lH@rl>iAl*|5ubNUI!jIy>jdZCzt)McDHsIlB`WE2m0~$=H~<>qTtkwocLpA&-ogtAe)aYEeYN57??x$m*;w{PbBaM}k<s;CM>8~K1gq%Dt!n71008#7<syctKT)v>5%;TVf%^I!#L*ooo!xK&j5m)=Sb1I{OM}iUpXrAyEE-mt2tfJV4qFDBV}gQT{*>!Z*{+n0kZ4k_UL79PrCnUUe}W{Dx#nSF?;$&|<bD|wh+wZpRT(*|LG9ow-MDL}5`K8k<2nH&8ud<)^8<t}asnmg@#c{F-U}x&%mvGsRv~&JttAd`$KfPUjIKg16h?FBv(K&`V6BuH8An?my<^7uFA=cSK-dTmEI(Jf-j~Ghfbv74rTL{DySfQYuV_tIn&yK$MI5@VCx7x<NAQsrSqW_52bJ`k3S10E96PV#Wj0tYcl{--|Kbt|bj_ICtA(JJ&|m)Bz^bv9HB>;{d0?HH6XWn|_HhtIwOK#;;=<X<7whxU8XRu!SKAQt1Wcb1a_0khJ-xDniT>$%K&z-dqgA)i%uhFxn3wsyX>c~^w9_fjSl~ldc0GVDuZ|ri*EU0npiq<sLiWAf_9-sjLqBSs#RU+=fLrkUw)>Pd5~uJl4ct!XVe|~F*rYXgRU_BSKK|Fou4|klXkt2}+hes2J&mJnYxu|5*UALJ+pj<XZiaFba3>sCE5oN_Nr-v}iS|!Fiw!$$?Zw=kJ`Y!*$egD0cmWsn-+6`7<|bunxJ<Tbd_G{0{0U=<L8k&P(-q-&wT9&mBnP<a>^nqvw9<hRu?zk_FQ0vkEj)@KHNdNT4tp%XmFJ<PipRSZ?h5T^tOgHfl=U|7+(Y;Di__j-7W>fc?DJE7AQyRxm@KQaTFaTRpLbOHP_CWSvwNm_P92%}SQp!ok0Icl)7iKAf-j}!Njw$RJTSfuj9BdGj!zO*c3TGKdA7W4=2Vt+47v7^u^E1XRgIMs6<ms>vC{@hV+6c6F<+_`nfz57_Y5BKy3F!)+)^pI{r#VrvNT0Ql-cmRMvJz*b3a!$K)ZdiH{J?-M8r||QJvmAjV-Ep@pTI6ZPO`XosO`af;O-Z5w?;rgK3sDyc@yl%zEcbx@ysyg<LR=&NUyGwn)=!KO<TDlUEsW<4;=8L1ZeHJ62}rqtjuL;sn@@S&f)`HG7a2FW&|g&S}oGXlHag)5H=u<r+fLn&`i4uiT&hk{v!gIEUST9{MU~p+%X8E-Uez)J*Sz17H2faN?T)x#v_eNbO*h5eRS`p;uU~g*f4&An?lYfZ<a#*55lK>qrs>{}t4dZ}?A5l5&uuSGZ_7V9!{m;uJ;~j)rwxX8X10WEu)k3G*w^Zh6set_3-!t4u-dn0N)1)D5AYR?Hyb7y#yy{^X!tS2N#ebZhf-zZsZ*uOsx;SEzBK7I><npMSZZ_9e^mVmS)sEQX-^(=m`#S`A$m?Iv;^U!w)iFB)x29%unE<5>tJ49HwAjfzjEG$`nQ;BxLf1(*>~i#QXfK48u^JkQnyjNc1tAb>XP8NyAr_9lfaTfvbOK8*uj+S65wHEpxIJg4dI5qJpuNukXY?kIO^>kr=vFP}`6S0bPm)uL(eq1iS0KjK1U(D)gWZCKkS9QYf+_a&u{aOioGTFha0RR*-X#O7gwh-qdPIpOO-1T&VRMq~ep49O`cNW`BD^qvNi9Y!~iX@2+1&fuEi+x3fb72mam>v$#xv>FSc%KgKuNP{LdWb->K1Dm(OFk?HD1k;!WdHx%(<~#TlVHU+m!rF9cn9m|H>*k6Pebt>QQKz9PFMJDP9*NQJCB69-qtJrXZjv}D(Mv0GOzcYYu;XsUnah&U)D%{7fT|WWN!Nm@5A=S?`t3tINBwv5Eb^5qHE#NTKViNNga1|d0>YxnyT8<;&%?gyrtqTO4zTuL)HmH<NI%k+dvT8`hPcW9&9H%{%<Z5TLYK_s2c`p-;2|IX7dpvA^Hg{4nB`=FJceY<2b3U>)XmK4)_jbH1Q3Ky4wmzG_aKp3=B59SE`X6KZE<lp$_heeq=9N#ORRiOKG3yqZ`F$JiXvlDjwb{c*@3_=C%slDm2sTwT$OSfIMO00Nw9<M4%(z4pAqz|_ks8a8i*8DWrJ$<Gj-f3vGX0`P0+eFh2hh-5?l~vD69skP{F(AdyE7K?fcq5@V#<j>dzEf>aoq8JkS7%EfPinb-^z%2VJWix!4-s{SgANJ()gO^X7n0lL#&q%YLv{kuIoNQL-57e45AlPVRsDD{A600_TXhqy|k*<^?zKr9c*oK{$a^O?3<U#sIL>eM?r`|45B?zKf9L&M{2uU{QZ^IESR<$Pn*EfV{cLQK>VMj~nz$8NAE9pjPwGfhHP|{O#34#(Sfx#eoEO?tf=GN5iQrqfO+^%gBgnw-7!tWx(@c;`U*bN9=hbrUTu>o}(jJ)Ye5>A&RtZiXiwFQ`(}XN~A9^Fq->)Hm2$UIj`TtxC^9T-oF4~HZA9vziQd$6)#Y?DW|5eS5MbEz?VWTdb12ZP1l4xNC9wWwQt>1vadnE=eG+N$nhojdi36=A~vx{eXQA<h-*153j)x=)~r4tbyY|l&I_i-RqP28dB!Rsk>jsC|A6h;{W+Tk`rNrs(%pxdS9}CjC*>2^O52mtw3S{g9U@X22TFxF*_Xxd%r<{oFbWmcc`CwH{I?zYbTiQM+g6G87#2}+OPXkw(fx^2N$6?8GNO?Y6(73qsnRnxNI{R-kr<(WUBQWCe(RuA#If8az+$iZ^G!3C`)lxP{zJ1I+E*F?Nv7Rgv7W>hDN_=@b5$qhTs8w2<=!D~pB@W@_#9m;ISl;rMBC9-!3-1P)Tcc{{DPuBZ@19`w~8NiwHHdFKc_)&3oDIHWy%+#Z(3o^U;0C}B~gF>y7Os!ooF0NYihHoU0!N#n>U=@jpv$-C{_-s`9x-5luvyVRJ=%30E^h^a};oCNKWPQnn*tJIWPnN3|e36eOHP)h+KLKFwnIH{%gOTbeCw1SM`yU2XL_2b{)Hw(7{;adYSZ994YK`(+ZwNV~&=$qP{H$2B-5Zl@Wfap-@=ZJE2hBGa=M6kiZ*wf74&u(i4YIQm|Dy37i#Y=8q-M-#_O37rYbpY0wIk#1hsOPEoRk`#m*DI;VBamo`Q?fc?CRx2df?D_0BWSoPP3FCx`9bgG;afuNpwaqFhDn=H1x5pT<u?@p{<9tfS+Q6?qXc>o4^nzu35d>US6lJJ!CmEG}4#v<L`lG-x)2lNBnruTTRZRe52{^<hJk@0=4*UtSSWken$O?XaA#}+C2aSvu2^7LZv!|c&k;s%~d6&<frG;qG=AtF9I@Kvf6m?qBPh;~H<$p<o~qABLJs$Sa&<<%x(Pudm-V~<fmo%@vQZy+AYq4_Jj%t@WSN6e2Y5pn^w%M#9@2?C(r<Oz|XtDI&Q<3)^JMgedN!RfmdO3~})MF7i2MIM~$PHv!`zI0d%ai9dF=Bg0%>8GAj$yS6^_RxBX+7+dnOO@_K-p87u1{)Vs%k9%(@0w$GDihbl2L7fbKdhblCgpS&r47zlGEH_!`MGi+zX{`Rs&;_Bpg<Zy{g?q+?m$xYiI#4!JTVY~^;o!orUQQln4s2;4}^q5IywswLJZ@(Y<Qr6Uw~!KC%I7XWO^AT%nx6X%Gl;5{%gb;y@C^wKRfAfrHy`z=~zDV*{)3XVda;fTv;zg9EZcIHT$~`_F;NVDX`HNlpP?sLd04ehfe76)TW%R0^{P_Vv<k&!!@oxXh9H`s8){!JTh&59^ia)uhad)!d=rnUA?X>aOj|2T_(jg{_=)IrU^_BD2<8M1V3C}>CNt7W5QtA87a#yO`w+-_7GgB58-B|M%Ae29`4#sG^--?2L{pU*8IEp5Emwi<JrUs)j|HP7mc%8@{)SKJ%4S~k+_FjU$H$OJh05j;yn6kKZt4e7iVm*53%~xW_ob*;#jmNq7n@lf53961fvy01##_UB&f(Kde1b6V)UbK)7x38X<etLa>y)%VDbMdrJtgWj_@nJ_y`b>svcd2A4?upT`NR_;b8a01+$X7fZEMO5t8))t5!%+_Osa7_-;_p8UPy%MJEj{LeMn<>**#0Vxs5Bmsz~K2S8r96>>;Cn%#@lVM_H{%#UlCyKJL_$%q}$pfB_48&QZi+6dA!G_%KaUU|2c<sd&5)w-fizL4z#vloRVsHGy&5RpW{Es$2Thq4)J85H?N79GV%RY>LIh+=8Stw5A5Fgob>4~Ob%j7vb*ND&B;cG4Y_6oHAPbXo)80&gx2BU?^vJ0Q^U1VTocsh=E<PZlFQP<_5iT`LG(`@SSD@WHTf?`)Ni9i$4jh(6JXr}9UrthuquNN$16tNcYwhDnno&px?Ra#I=nj2QTq7e}PsCY#3TD8hRwMU)rMj3Z8vEPy)NPRXwHuA`RU{MFM#_G*B1FAZ+8>g28g@E<qGA}`!wK@|gGjQRfH;(5clZEu-S?INHVx{F4Ue=s+{kEASCz4U<oQ!|5ECL{Qtsq~Z?Vu#XUJo+tmLKK&R>L+^$+>y2*2vm=U#BCojy*(tq^8iq>{AjL}x0o!j*Zq034n_njhmDAeFU2dRf?EArf{qEF-y5NA*^hKP#EH=J1H`gbZv3_N&r{b!uYill$Xn{I$kUfZ6L-5Tl417SCi=1J4M=*_<&GQ4fr>#{_^7;KQ@;3D^I-PQf-5|Ys<OaVj~hq8`Jne&%=_G#OIxcB95)~Hkq4ji5ms8-Jb@<3aa$`Hx82vH*63Yt1r4AOxe16b#3*#kB5|H*$5h?Wk)xTrB*h3}JPHHaaU{6EMQQLp+AASru>&VS!F<@(-hW{J{DrA5FdVm6ze<=%v+)Jg3%(QSS{gWma0Rq6lcV6J^9<qRBIDuN6Sv+7S&2s0Jcu)uHc$Q3_m(VEWqGO7)Lp<{=NioOFU$g(Q!hKYJ@}3=Zn@*`I+H30!-unv*sb41ZukoD{R{Sw4jnw#YVtA}lJ~$B)C5#2@ufEu)+VnegYZwHFK@jX{feWG^D8AZ!##~W@zbIlyB|PjXO34E-=Q!z2rDn2@UmGuZ1QAp;g$t*l2Scyx$-XN<I7UlZUx1NXg{{!**eRGyU0I|y6uR7725H2JP`oFbTFuJJXHko$Kh0M!x+HN3*rIqM)Qt6QtsB&*&wK#T+Te1pX02WOYbRN2^VKhs!}e8?^dqPbgQdgcoaj)*1mrXqn5PC0+`RwproNGc?Zp<6Td6_GDOmT@fs2NApx#-=FPPab~w>@XAax9@4m-<bpbQIy){0-BL_%I|F}g(sZ6UdIb0+;%~ulr(r6jwNdip@sf*1`^UsUOWoA<T2L27(Zl+*gFTv`ACySZYI@5}i!*rQlpEyen79o2E`7lB_AR-3Gt?~n593QT5{r77ISA=Sdk~xX0La@t!jP<(MLNyInO+*2*MQTR^u@|x2$!!EqsFors){|^~MTs`=Y4m>*_jrpLdxBBUlfXV5bfqfXZ6<~iufo0I(H)>@qeOytr8I%#(nr;OnBE4o^GLsKSowGpg6?8gs~WfX(+2j*E`ZY01&q6Ww$iRnC_nXXKb*o#Dk2m>RD1tUmk)Y$dT$!3)d73AIShMdWv4z1+QO?<qO{vA&%@qDMG$H8sl!}FKYHw1b+~Yii(m3I5jK>Zxdg9p0?>eXGs*J(<ztGQMk>LBz5}7H#UNEcsL$^{FeIS#&Cs3lX8RcC4(g321QQC6u^(b~O}00Tu~pq9p*-Q5=EZ02KD?+vd`iz3aK3614}p+uG%k_!{Zr*WpYOv78$2Nz9dFTLO0;uIJUOX|G5p9cm0}-66VI88{5^&=O4=t4w!=0A!;dvZb+`L*<!>Zks8JWL@m#27DBvu$;;CycEtnhws|jEevG-HLh(1jRwlwu49_Y%algXP$Mlj|;rucTA2<A<ouM$=jFc|-kj{yTrHFZ+%&#f$|h@0RF##U?bhm)YgnyFvz>))(d=)QPe{-<FYcPouf7gZLbXy%^B_f=~&Cdb99-&$0xvL@P{OVbtWj%@qs@P_kMyY-6M`g^mYTyL{J_`Kkek@K~nZnEq(Pj^ts(So6!7``H%HbxnSJY1De&h^qvSsHg=Tfm#;GUVRDH??fl#*YIV2Hdw7$vFVn?V;h-mERWTBYs^?91woT!`KDjt&IhA|0)X2UYu(RZF*L2qvAhfPq1?UCt=CJ**5R+e+W|3VkZ$G%{C8gw!zAqT0k)6=+U$w<?MMZ5ANu9T2Fc|_G&P!gnW%O`hQhHZC#LxZQ5VrTU10NZ*A;k@TQwL6eLcw&2|N-0QX!x>pBK#iGkNtU+XdrM4|EQLhTLXfR6_?A!f5-^W}(pwMDgx>q}ScN*QwjgBAl*!%;Li0EfI0;AFz<Rvi5!OCRX&{mAqkLF|ApuoIs{8`LlC`5asSHyf~gO2d4r_zJG6$tOFyP@g;rM2Rg~+}H04@?n65w=|)rv~8i+bipM{Z~cJ<u?&>j*XGpzrz^4VLJD_oNS46Ipt`h_-k2~%Zk}~v0_+jJux5e!h8o&1{#b>-Y`NLGS?fNh=LwF&fmF7kcR$4Oj~YzP8>9h|h$!C0f=w~UYH?1kL0LKygfQ1%W3X9BO>i6U>P`XOsMbb&N=P)$z8<Ruq)ZKBYhrEdA##pjU<%2u1f7L_5>@U&um0)Uq@{xrO<SzOfh~)~qkx)Y_bd~8z&4pI-&u(<w{T3RHIC5t>-h?%BE;g9H5bw%6bN?+=Lg(ffE&)CMTq(o%B|vhJ7hh79ep1M)GG=SAIOrm;VOtj8xb31sX$hPajAA%g+ESxaI2{2($oy$aYP%h)VY>6!tf+V9^lWWV|rDF5rb+<qnC}e)BDfKA%E)5qfMwVnh`XtyP8UJx6qj;o+lSdkOrXNKgmW#O4=sIk+Lb)a=aps@C1F4XHKk13UEC1T0F>N21UKi2}G*olxc;tK~0+qTPNA(!2hUvdd3@2la?+&JB~w2{U(c*X}R?zFXPo$uX6#9;RPP4b`MSM5tp#ztH|^FVT&mi*zIBLH9s?>-unHMq)_mN?zGpR)^xo|MrGnlrR}^T4olzhz+%e7UfPL9Mh9Cxbd|<-Kpo@ADSaCQp@9>T2Xl!PIx6D9b#K2{2$V3{ZIX}XGzL9#hQ;GvCa!LTXGAKzE{(bmi_a3eE5~T}lz3>VwOJ0*ytc77D2$|jF3xy1KX^D@Yb}vhM}|htPau*+;ZiyN!tvsOHGU{_P;>dxL_vsins1r8!Kd}lG?eJ{bM;y4=d#e4)RrV@z|1k@8TG6B%!+$5c4%Zh8h<KqRV=hHIQrMtVku}Be3Q*WGgnW7cK)T<AL;c%qL}7$#)`RIAgzMvH&Eu157J*ysCVwmis1)qpzw}ucCF6d@b4mU`(gUL3sX`EzG--sA;Bo*OCZAb*9=^9n7fVI8!PTkg?kVp6_s5>qJY0;c&Gy$9TtiGtqOhtqslvcN8@iPhk6PJM}!1XX6x$m7*M+5N@&!x4klwhE;<u(X(&xOWS!TD*5E7vb1d{7T7#nTBEeSB$IIExu`HdJ5~a5Lb&qYaH6>%MO}qTg+cq$Q%*}=65U95*%AGtEo42Mfat8+6XNPpk$5YH?GvC3D6ocBt1M=&9zy&g({~1fmvO1t@6&Y$KAdfou(B!(daC8qqZ1zJji8q;p3`&`;<Xgn+L38>;LtfZ{HrOv=X^9=OBW#L2-#@ZVMqFkv5^^RytlIIR`e0o*hB4t<CEqTgie;-i-1kqBi&o1t=Pr6{nt3lCdRtynL?e6ozsq_+ZX6LZ{!RdyG%~&6uBBkTGM5)R?$zNhN=oR&{z+)+LZQ$T=>P}iokQ_{517R^#z4iuh5vxZIn^LO!kHu_%F_m%f=^?41|jaEFn1O^lQPw)e*U^Itv*QM?m8S?RCE-3zxnbBPk56=Fh5@i<`Dp9M;DDsp$ep*c@BDfUmE+huym(tGvXu!5_#sKsFew$b)lk!+X$h=ctBZz1aP}^U+Put<fe>xC<xbVm5_I8z$LYe4_b>R3lk4c)%AnK*}7IC<+Cd2l!Nf;d({uFQ^DU(3^zXJ5JG%)=yOqyG9OCCR7y#gMzbTgo6CN5X=}4wc~l%WNbPfrCHVM?c89iG)~vXDPA$|2to5Pj$jwz~T!f}PcCT8&p}<WWYIjMr)zQKn-Vkg?`~s0A^)@6PtE|H+$Z(r<GBaf*Ht#m5_F=CyX+lhbRtx5aQvye~-e`!AdLHQU*VI-a1Z_N1xwL|M_}uiy-6zQiA;QH+;1}h@A**cPHEGaM-%(pi-XP$B9f=EkIK}?t;`aFR6A^Qrw{rhPpd(KVuEo5#fu_fKQngBNSWfqsiWf!R8|jIeEJ9(GsU=3h6<BE7DpXDkiD4pD|L=XRG%2``yUw3ry(Zl=upWAIc*7WIzv-*+BQe+o$8CEhfECB18=z9VLqmK)24FaC88!C`JAMXkJ1)tDMQ3`8@#$24E(H*vzAIUJ4S#Z<b)<Rq0_62~=Ubs<qL3XEgqzt<1wd)cq38ysDZt|i!)Ya4L5BZ1C*%v+EELea4hKB$$rXAcHKOr4dgJD0SKariI)H~XV4|)_d7WN3Ld?;RaX^^@{AO{6@?)wY|J>8}D(KF$6qh*HToHZQUC)99c)QP1GbvYENT?2L;L~Y%DQNY-Q&D%2=NQUEjQ8ua5ohW`NFQ;~tC%5Q7;1Ms$57%@4+uBEr)<{>0<a5xBFmY(o>`HG&|3nq`RZ4A>()@j=t>KHQTVStS3A&e;j#*a(!tSdUqU3+z7f050rtf|{`rjWT?32sUp77~?@hFXKNd=gJdU3$BZA{~P{}EZG{!a?JPlNhorKP(a5EX;DJ`fUk)EfZb@Jd~%xe3!Gh?Hz2J6I;pO|dsa}}((NL;wC>qra#jCB?mOged?5?dH6SueqL?k@p{-Tg{Unz$cmBLE-W(HdzHpkCIQ@lOq3-1atP^*GEpfv$Xulk7T-O17CQRujONO_A{d>gI3ahQ2D+w)y@&GM|<IV$e{lA(Om{bxX3z_6P)-4|*VgxTnoQ3aKz$p(=jK4cr0N6kVI%Ak^ZCN>-dWu$%}10T;J7YMR*3T2D->k2RN+<EWo0fB}-^`_@lpAU`fmXp>v(Y|>+sBzX}?=eZH1mx=yiD`&i^NWgkIn1bQq|FuTrtH+?$@vQIsLsTVbL$YblQXtVVJlM;Vbn1=}Fm(?MS-b&ug4M4r+(4D{(x~$$?oz<w=Uk1i3>|jS!H!*S>|t&qTsd61(Qu}fx_L-06K)TjtYcd&tEa!V+TXN9?}AfZZyzMq8ujx3A&GVO!NDe9F+@1hKG3wvqsK}r=_R(G-J#8TjU51}??Y9gh4P9%on+(5p7YK!aAT?K@Dj$0%S>wn)09f+5sDx6DF-a42o&`2!ARff>+uTRCyJXNg|23~>x_Aw9EX}zJd@ez{s<qw;(V+U(_T(Tsvzrbx`qI;2G&PNTISkhfEcWK>G;j2r3#@ydIle+_xUj$V)ONR(qwk(>uu=ZuJHG0b3kspvYF&7E>%#nf_`#`F_W(MegDethY47!Thhh%2UTO@-uHF)@6rQ~L*k>%fZGj=3iYqMfzGuUMCQM;EOQ$rZL`nAC%9Uk9_)Q^XIqwW0qt~Z=k0IF1uKfXkg&f@4n({De{2a9O_OOx_C?5ira)=9eIhL~SDm7VwC9;kVV4?RAxtY<4jT673u+CN0<$6>v#YxTZdVAhFCD2Rl@&o0V*?M-S_*qbSR&VQ=W$|zz^NA|syrG<Hb?erd5>sH4upJnfTBOE@8^;D+X2RSAbCNwn6gt6>qn(l!g9(;dA@+Op}gE5t|oBBj?WBlh0CX4`tmBMw*ZxsRDE_*ui&jW&ozabSFMiv;p2r7Q5rarguh%}&jvyVlJ1vZb-{n(1!(sLsWxnG019bnG>`D?L*?^+MS;3+eo*!NJzbWR!=;R;484}B%}B5m=I%;&KunWO48!{oyLFbw6vhY=!!&jMew)u{&rr8e_k-jiDDOQ<6i5gWO~9JhB3`m6O`zRFM*Yf#Pv5ukGU|x)N_fZbXn^aYpdI;SA9Hv8y5BCr_n;2RuD;b}GatV2)$HK2b?lEn{}*>bg0F)2_+0%w(SQ8xqjYBHsuKuYMq^qBsO+h@#xS)QiNYUtY_Dr)(#0Y|@0@c+saD|+)gv#SQ+3?YHojZg_g(-+!$*0&TxT&79K)}taw}4{I1ncuQH{x0e>H=dXat#%_u$sW%@NGr(}3Wv#wg1KsCn5ckc3<nCxJgWITJRRe`_2&GET76a4(!-rEau)RX>CODuoRcA*EDFF$p)qKB>vAxizNo)R2K>YF{2Vk_aI?#d364V|ck&ua0(&C$z_m01Kq@m$iPcPdM}yNM)oqgTm8`;ZsRe-Z5Wc<S1j4{zESc|DG)*DLXX;u5i0CRQv=Abe&!mf9&w*+L8UMCxvml%%h0n#f2wY=jH{23Ld_EK7yW2>OnT@t%sau;fkqep6Y7AUd-vVUq4<=j6D@Wtcn&JitCi!OeJ<xVm2=xcjIOSC&qpr)>4k<Q@g^4qXp!dny*4tERu`I+k@Vn>P!1!e_?d)RP$Wqaj{5H%C0H=$UJ^QMvsivC%4x*gm{X#<S1VusNhAK*&U#?<b34(LzIb2mE82$%@N4qogQY;2r_~MeoZqDvPQ|TFR}&T^pF!;IA@*CfqWzZYL>*B_eN)`g4^al0`9=Y#QV?joTD|vbe9R;#xcHwZQn{HORGlT=Gq3_Z}0DbAAXFrsT(s_1TW+*TfDos@Z`&HBT|-ecr5oXci5$)=GVXyZ9SZ_x2_?6DtDKrawQND)=O8a3hbQMA6tZQ3kkr1jp@>%{a^eDxDl4L6k^ZVJuSMyi&b|Xb2{<HTlzkjEq!07hP>W<lv{&D+&5e=uD4Vxr|#vB)E}|FZsdUYw}NOcMzR7Y6vT<v&fMU(zPrFL9%0Z-_%v?U51ro+q=G-@xisL8n*CNogS?kWD@YpGpaEwvD#qOzqKJ>>!VO~mnS~HR&YmktA|Xb>)bgaMAu5Ev7EH6H$QBT7*}N#sk9ptG?H$QlVv8F^4rSjj_xA}k(<Z~x*@XcWPBXO8tH+9K{zMmC>gT9%xRt>wCm#!UY)!vBh|kX#Sx-LfzKA@+CQExx`v@R#*HpvRvTMcH@)E!#U}Pa)X`|%8F$b|5nlX33MbW_$C7~718a2^xl(zw7dOhR+9C>0T9SdNX-g=W<!vlg3AIC52rKZ*|XEPm;cpL_vPZsQM>vTE@;QN1yS%Qc_Gs(EzoMyJ9hC<KdrJ7a@BR8gt_a1OcWO-IOQ`XN=93_Fdvc+NAE+vmeh?JemhxvgYnwtslBNp(+sVzi$kZ3$=nokESOL7XHP&d=r^CDxJeiq|La=POKpU%!bc<!CU2QpHMRtwwLz}LOR;#innaqrT~*LC{Q#TNlgZh_alKTfQ$q^J>>asZn3XszRlWzxBu05&QGtf6>6n4!A!)|!J35zC<QJ%2-doi;EaS>I$i`;8yp3tqu0G`BeO<^K(FKTT+%ua%)J?oPEjhTAr)AKgP$b9H@ZDUrPSZMVzZPd9dl8J*UdO*P)py2oV2zSYJ2m*w^>o70sqq#Y8}Kc8oaT?1@m>t%;(J3NuOY?mhU=fyrKDn>PJHP@soX@Ks4l*{}`Kmo5&E7iV8%b^GVQ9q>CAt1#b51v#>)c>r<ZXW^(0CO4f-JV}@2ia#>KcrmmuK;y(F`mhh7qKpsxUEwWJGQiEx4tuwbSxQ9aN3^s9DB?~z9EJEj>v1np846Ds8Ni~-E4s!SsSfz8xS~UWY1C)TvuF44(5Gd-rqJ7iHCJI(;S}j^Sz03Z##hIp(<0<RTuf>xk3HR1WJu856B^@@%i-iQtKZ~(%JWz@wFLfN+Qzm@9Pw?_z90#bnv3dEw=ShB>Y}5&dwq+XfO&4=-Xig_RfWa35p;MY42GR&w27T1C7O`Lw;TZP#McY<M#maZ1RfO&NXS;LhX3jzXYdaTcp8K=^xl7g#uX*|I*#cAc;=(#sF=@D#j+OnIGHZsapi=%g&^crFCvD>*dE>uRVGlIP2C714xjo3dz3t-IP%6JIoGs1<U>4tLE4yE<nX4!7Pg`WGUcOy@p8--8;3|Zxwpf(nG=$=b+}!!_Z{H+PUDno8Ah?W<`$=o9d>#%bQ$d)6r!m^yLQ*=upaQxR*D^^vZrx(j<%QH4i1Cf(@Q0qw`~T7#AQIHE2s^G^Ro~k*q6^=t@|HF}+2qi;(E3AJ6p{2&*`^uv%KE84kZFBg5b5kpvX=hmBl7RuSO(M6zRhB)*gSjr5c7RxvGC;6mC0?#(-Nd_F<SUmewf(59Lr%XEVz8e;Wq&KD&NIDMFU3zUv-P!sSefzqaq*Gx*GSAAf`hWEc?5~`HeEVA&m5nL7NAjAqwl#8@A@`pp?2@D4q-}o%*>6Gig3`cJd&tuU8Ci|+AdTx-h3DMqX=%LBy_%3)uD<#)>#IA`iEXhOyg@ki}OCcv`%dyi?!V=5e@4WvqKJ_i)Jr^MEi_IFlBKz#mJt1Qiz%EPWo$f)Yl0SYBDOyuvIk1HEEUF;05=LJsgUHwSaq(|}>e=Sd34r&;j(+FyplJY(x*i|cc3h%6^*Crr+=$a0UI_`cq|Pw3ZU}FSs?P3(yeh2J<fm)2U}o%!Ed49U>hvr&1HjPY804N3tLB-^4#C~oA?>r_*Wm_ashosOpG~2EU^$kpSHOXRlO2T*6H;&ctsw)HY5~}x894xnrz}Z{EPdf&V2+(J`R=5t#;UK<h6ObWA#Bq5u|X<AYt7m;EA#`|&BXFzEBXR%U9`xgaug#zS3N(wFcZ73cLGS08^9=k0HkP*h+xj+0HapaEpc?Y{5ZL4vbS0GtcWGabX2Oq@>@UA0mRJl){aj2fZ7*!rGT(Z2)M9wBwQoZVJ2e6wQLZsxWKRY!To->E=HGgJViCnT>cA+Eh_G5G?$fAl8vWXha@xppS<<R(Fxo{$sRTFfN?U#)4$Svq8sjqqkge;0&-lXn5bmZ8&tmVOnG^BYf4jG*YTCKBd}Tam%Fr2kR+p!6+km+N^s$XTZhGBHm^sxMt;^OJ*#3*0O1QHdfOo3nDvjR-tx?3`cpttbqH9yh>-B~E|Y=XsytA%*nLi-iV+4<b^D72E^@L8xu?#*#|rbBDe6kQp0tAHOUp5cvvE69XpES3bm4$(c7N#mQptltmBylc=Z}TH);=<Fr8@j|OTdP_hBPRGRz2ob3|h&J^CurlZMaaAVP2_M4Oo(TnA4y`V??cI0*1sK_Z?)Yc=rkYBXE+HYH}dXbAlq_(}nwV-$h{2HR<D^@B7eH0c)l?z>xzY-}B$w082BkV4T`1fM6L)8%;#o3eYRKw?*Fu?jtyp0(#+Y0``7cS~W%(izpr<&sp2j7fCXI%C^KHqfgle7Q@u4^7+oU;?s`4lsazlIazJFZ5SPP;NnJl-2bIJT9yYgGE;{y+=%MBLNxC=%p~)l9WB6Fp`nb~uI|442avIy-w&;LJzWx(S7um$e4=bx!Aa$f_powA#sHEr<uycT?%xmFsdGWYRPinzLl7coO7gvlq3nNVv*Ras76j9OWPwPG3*X4AY}tJe;(dD_*&5|uOmfC*K;xE5k=9(%X7DTa8FP1<|E-MvSO4zz2@cQvW}(m-ux#juebev9RK&HL`C^@-G;5xHs{40G(^Kli5a*Rza<lt{GF!%W7u%nFOQ=Fl2vL-Mlk5Baq-?-7h+pTri<seHBA^i}Z=IWSLOpFldLu)#VzRP`T=vJ~yq#+jo1HU?Eb2hXw756B#nRXRD&2b-Ha;`;IUooVXd#bh3T{FxF!ykNk#Qp&?>%dPe;@evY9L_-r(wxD0#WSyq|d&1N~LbVNX9m00{6#&P85?`oyD!<;SDyVEmDRQI>b{FO}#%uy6InbT;{EgyffvOx?ggBGjIlpqXFO*|DNUiWdZdF%|_j@mYcityY)CiW=sKlPw`zBDW=X1MU0XtH3FV%l%;jfga1$Fi1j6h>L?2L$1^umvmrZ}h8${jAfw);?OfN-M}7D@x@diNL@vKTUDUy}vzQwf3YUX2dxH*JF7owEQWs$C^xWZwv*#v!1s~P)Z2Aej6f0}KkehjDMWEq7wO@$(Ic*1$D0Q2)#>?Z^$GLjkC$BVD3M<Q*(E){y)bjHQ<a7A#Z7Q^?zp2gC!whhtc@k3y4dmm#>6Jh35(;tbjH6_6qmyi!iS*A1lnA?6ymq+b{2}cN?I_K&LZk7byiJ_((Colmk~PX3C?chjG35DHULq$PXPUthcPfzXtSal)W%$CB7h1;klb|a%;V&<bL&D<No)^8(8t4=4ykIF{<o`*it~2%^F@2Ld!%b}4j=X#DBC_e-)_-Jmex412YU6oBA#PFgiFmvig#tokO$#kKD{5Vu2{j&=@!8ymoEK@wd7j0qe`l^|V-*Tl<b9_5m==$aSmmB|MNgRgSAt(Nv9<aV+0uYwpN#)YMI~u68CS|BSfX92JQa#kA6a54f4q?HhGy@T8&CV0RUGZ`P|BZm*Q3LvRWC=ur|ABhkt1~dl@5T2Q!o_gT9w>R|BuA$N2M{?Eb3YpZq3mrNEZUBCgvT&i4e=JyvuY8VZ^n(JO^LYrzY^6t$Kf)3A~;hN>+0ikzxVEECaS$-TiKbg!F{_D1mfbAFnkZ%->!+Bui(zWE_-aS9^>%pmQ87yM53S1l%c0AbI_kCUbF7e`n9IwB*1g^hz84`7N0hKSA2YMmu5Q=CpSfcaFQ9D!kh%=h&fWA+h$?X8#+$AIV&+j*Ti>8Ards<q@ENTb{wI<1pC6HOub&HIJnu4bR=yeMHrMkJqt$=0Z(gKw73EV^s@297JQ_Ce5pr{Yr&+(^9s>^(r?<oQ!L9s-Wt9C^nKz`WjS5@<`6sp=gm6Y^$dtdOP%#@8tm|3EaU{`wM=r*z}_bOkIs;jGU!_7pD%5I4X-gG{!vRRy6alXufWQ!I|HJ)O+PkrX|Ua(7A$s{r<#g(7ovF*>Y2y-?qor7R1Wa3W!K+F8Og8r(QmS>bcAL*J9^rs!(G9<~Z(M_tC)mx~Jw7b2ucqQ%rZf$Wsj9(de#?eg3y{HzA#47pAal@Ss5WTV3P*wnb{~3$vC{ct=!fW;aIkkO*?jXxy9vI7{)E;W-oPx?@cWabLOEi<7~*vT4D0D6OrSlJU<dlmvKDeYmU_Xj?2gt(i;wVzK{E>tq#I6-;|%j^jBYhZsC=P#*5p#cOMCSyze;hlrYXV~afVLfQPlfNz41E%6Y&GU613j(ExQTUzJ-U>qaf;l2wZ9|(5h5-1Xbv9&muc@o8ez(G_bZqE$CT{)1LmVWoI5n8BRg0T)FQgSZeCf{}VQlV%a!FFZ(P9?S1RN*m>>+IS0R3}_yn)E5T;c$<So#&Bn7?JT}tXwoWk$0<~AeRrXDN!_w!*mLk{wb?u8C}#+uNTZWT{2!8h3$G<#oqWev04ksw%pOaP6f1C!=S3xH<&6VvQc$ez4lT*5(;r|QS4?zks7lZS2{$W($a5|n2(tGlnBs!^$Ji9my?f6AoVIM6(Vge36Tlox#xqcIA&yH%^G45Qp60(<s@_q10YWBRLX>W+*_Vh5+QcZ4LoNhazSh$4;q>d5fXYk5Te%L({0elHq|pT=}Jx}=faryC1s3JmKDq$r<B~m+NU#a@>09gmTM~PKeh^Gl4>kOn~cvFG)YRm0S#El5Pp4f(MI^CtjL1(lj1m@KgOffn0c7PPC|W3K&vpEH;;2<DFN%^C*U)cvCk_5(wXwp6{hYfPnIhhb6806ZTGG0QIOH&%QEXk!6)^%g|qpB$QvG5vQNqi2uSMv>J?=D<83bf-h7JTViz3<Ss21!&ZrEkUpC7b%!;r81>p9?(Ng9=09AIGF`MR>gC9LFL(3W<=c@ELPhGIrGstG<-J}-iO7{7bK?886N=Tx;ZB&v)F2)w}mfD3+$s4wQ;WOrq)KNxiD{i!dC>$-A*z;M=P<7ki5@kTl5|tMzfOBw!LTzOi1hm&XQXVpQJ++#7CgFAXgLNixq>+I3VG(Em+1zEm*c*>IKf!+B`PXb{=B2&kSUYU+c4@Mz@?jxml--r&@e83Y3}FI1(v40<>}_8{&biI_FHV*e6jzt1Lx{$*>LGoQEcTco+(=U38NzsciQ*Ku<gK}a3K#897(Tz=HwBf7DeHvTaq+T!0~X=>zZ^(yo;KK_LamY++@B*XR&r>dy-yBO9Mv6!@H{p_i&iA7UvMF@io>Sr9c7*1OPl=a!WYtdhj&}$^s?(lAgZ!u)R(l@BFbaum&+__?R}jhe3Dki6%TM1-^`5Ql9mJX<$XmcyYJAHV9_&^7_P{1oRzup(9b1vSLFy$|2L1J_3j$iGSrf-6ksn9&faz<%ZYqkNa!x&+TWHM@yC}1#Cc?OxH?HL#BxAyiuy?nr=iVBC7o1LBaFtVm2HDz7#)LfGEyza$_Wb%>h<Z$FS&tddpG<qUq(jYpiEs?(Q<SBO(*#{Bx?@D7x&d3uJ3KBLuAN=>rJM=64Oj``K@lx2ABDwnJ?1t?Ncw(NetVv3-n1_J&%r-jf5DEIzgP<f|GXDvg3v!dV@a=J*zGD{>5D*Tpuh!!ZuKgvD>hyqbq43vm*4+T}_(+VB~0N7bVUrf<Tu66q-+QzGJ#@l6c=7tcd)mp!NW83se_xqEC(U_y}!ZdCtLU%;(a_i)Nslm{z(ja@qyFOzvjdc7ig{XGV;0_6@jS-uK&<`lPO$#=Ctuq+KrnWk!CsjQ14E-GO?l!t`ztyixc7M0%VV=#MqhRaKw?-rKF2zl%r!)<vAX8Bv`BQI<@QBw$KGcu+;I?acED>W!dU!sQuLj@F|#JnOXuaF`rHKCVNMkeSDnh6$GVcfLC?O<6uLZ)Z}|K;~5@fcne+BN@hv*2dwXVE-RoD_KI*V{Of))v!&I#>H{6k2XI@rJ^HIcPrM7|HNoMpkR0sm*8(l6v>C;1XAHY6VjrWI$#=ap5KMycb<PU&e2V?msT$PlB)s`gN8!Er|`>vpNPA1u23My33tz@9Oiy;Ogb=xKWQtUBAmjg^Wj~+DHC5ekVsi@Kr<#&ipdwtuQ-a9`6BO3Wr$y?)$>HK#H{6(a}oi_k|>UFfYU~NrTi}xjnTA=4D4bhMEJ^${z)lNVGxXyKmnOgqqG;^_wv>`t@5cab{%x}{}71W0TO(k%lis#=)_!@I|^Se6<;w1`4}~0wQ=YJ??&`up)<X+U9Hy8eOlhjeXyGrW+EL_LW&!_`r$j+75cGLj`Dy6Wi`-nK4Bukbon(c$pK9jANkN_-L9l7wYY1j%XVvp4vBT2&~%`*85EUaO!AxdV!tg!sckn#w{)aIpqbQ4AuF;jRQj->T&0L`b?TGHkJW48W^v&ZbSh_PsoP6B%PwaW{30p&B*cBAQihS$L5QdmztfAht9;7R{9~7)2Rt(|wq<~;eYJd;MPI60byDesH?>krqrj#ZR5eX;<gqu!_(3*G{~fv=T%evIC&7J<Wi2G5y!FXieJEmV99im{A;skfF4JrD9~D+;E<By4k({_SMis0KUPyQ`R3s5q!`&xO`UV#7yaAA<cE=78UbH8-ok-|!-bI2JAn04Sa<N*r^r+6v3F0nNtSEe^2%cLx-bc5S_2c?cl{yTUmN)iJgy3{oA|DHB;*^BT;(hG$nV&L+ZH&`xJS4f#La?(az3ip#TgV^xUHY+hPv{z3i*+1_Z*;rn<=h3zM80mznQo9okaV}F?6-;QMak^2;U1w+5Yshy3~d9!$~kkyr@<L2`7WRY^}n80GOaF^4}m-^QO6=Kfso<U$hZZCZj(o#!$}_L=9XZ&oqW0Y8x&lC+{w;mqUzMm1`ka^EIXoKMM*|9jQ8vb7w$(q(+ngq3aOD=Y$U!z2|pVl>{6-T^TV&$sdH$q<Uw17(ot<ag7PgA>RAv>P>}X6BqA4;o3&=va_amMvarAoIQ`ZdzJqI05obpNc*6>A<WO?(Zg=S&9TDQ%oV!t*wPcn467Sh>UovlIBW}&(-SRYIWk{qUaoxA4V%o@(OW-tD<~WGtf)hr1o5=`mr(rzG7ih)HB%0Mb?T*9u3nsi>CBEN}<<*Cm1@;l{Xr7nE0QRFrR&0OaqWuCp^r-6U8{+nNi=ea8`4}zEP7hTSw(caC)=}<MD2~v|(O&PDM-ser;nfV>>p(+Kw0e!Z$@!jOEnkAe3r6kJ;}pfUQG-*PC&QxY!Aw&TIXDw1a+Y{h_Z`KLw_{^EX1+w(soib!=I^bMFwW%;xJ|Hg$F;31-soyUM4Wg4){i_{GYlDAm|#|;>K}^bg0+F-1_hd@*u`$kwEITKByeB%v%7dg(YT0bfB?rPfQiok?xsJ8am(FWI=iE`e#7;|0f^~eN>t<eBE_q`H6Wk<cZ2yRp;lDhA<+XYX}m}R>wLG0ZB3;@K+-bn=*@Zv;WvIy1-RSgDzsz*LOl5ew^BSn-LelajoB?`-Ag9$jMdTEO5`A;I3&5@Ue>{N1nZd#YyCEh7=>^2PJ7j>Az(aga#S_>rcAz3t;(;gm~NAc6>&Q0q!CH~?_`J4w)GjS<I@e8)(u+ye@z4L9}s7F9ePMCWdMlGL}o;ge=L8N7hn+U@_j8uFqkG3dG|cBN-1*w_SRLFtp48r03Beiz~hQhYB-Y47C9ag_~k00ysm+w?~py!vk}1uCg@C!kC_)y8+Ojzzk;1+@*$#%iT1!v1Gx;)H}((+3cPiVx{?@X+O4Q0PFrBC)ZkRD`#<$jCIVby<&Cs?HC=T35jMK`2U`V6^Id*83J$t1AoQvc(LRM25hP{u2V}L6AgL0(RK|r)M5eWmi1lN+EhOy2WI-QYQ9|q`rH@`!7(_5N9R1;)uTA%y<~XYU+y(mrUS@EG0$kJ+4MC`avmMtF*Dz?Qs6^IYX|YYA;*sYMlP`E}6_PIv&1-;k-XNQaLsDOir7gf1Hs;Xz_TYcm1{2(Q<FSU2A<%F_JXWnaSraYqfrZfccqMg=be2JjH<g-d;#wzVQ}cX~n3a5C@m+%@z{fxnjo614?_~Ytf;QoS80?_A)Vkg5!~bpo3e5WuY|6hvm=O=RUf`j0hr{>*w{nW$;AwvsBLuM9aXkSH$f%*a?j5qu>R2%*y9AZ*u&Fi_YV!EKcz}@`U~2RzNoaGOp<%SFYXa4y$Qg-VNUjmCXh<wZ<WHrx>XY=c)pU1bx>^oRZ*Tz?7&Spgfve<11hX_{-6^|H<cH_#9JR(y+W5!|b`2Ywf@@R&%$<nlQcKuTHNnA}N8a8nesI0%bqmnq{nx!8i{(Crpc-0=9XB8jH7cxi;Yl>50DT^~nBTQzl}E9ypG>|J7VH!CfXY+~cZJG)i4PM5R)z$vZ8QM=yF8ql4lE+RnRt}@9xP74k*hxKxEmu(Cn}Z)z`W`VKsHr(3q-f1RlMi*aVc#h?qEjc)oeJ75zi1)X_}IJ+aeh0g|#h>f|IUGl^qfODSRttxR@K}*}3}riI1x15|Cep;U9lT8KZeVv<QIF^F;<P)-!H9MMHO08^h03EBWzGXi18Y<O^;vI7zRp7uVH!;+`)@+S>KT2~5b>JEEoe8keYo<8=*%nWpZyaZT+MT0LDMAM?$p&5Fj@{IibM2?ntSO+L^kuZDw)L|c}RLsoS@D?6t*To#^tZS<iwwT*<KP7a^!<{Vf%RIR~tIlr<6y4d6qtY(WSvwO8jqax8*v3&&CaE$?aM6xCRP}9Vdc00V*6Ea0dWjLvfBr<ge^%1Z5_zcp4)uE!NM7;mn(DxtB0v{TxS^3K7?;J2aeO~f0H_G2c0YFBDD#E^Lugq%BK3QB%N^oVn#G)240|F598KVkILF8<-$XFmTZdpi`w|GVaTQ^z3$3vmzmt4YZ*A_?7p<*6%`8Nr}GhRRraN%cKjBDfJnD7BeX=PtI1P!Hto9f{duaxFbyVpAfr_&r?7_apAivH&l(aNn^gZ}vkK6P+^>2|31rwv^hWH3gztGK;vkW|o8V7+e%lrO~{Xwd$11XA?*<VY9ClY;ah3<77Vk`z+v_%xnWL-?<}LtJ0RU$Xh}2;&`v>7aM7gplI{gvfA_wr+{3IGY2Od9?B&|8RD~L29j9@bwaEiv01kvaOCLhHWq*Rd6yyQUVnN0I-$twu}y>BigZsYfJ!=sus$}l?`fcG`yS2OQit9E^)txnd=#xjBw@~`c_4erT>Zm9R5E+9%=l8(K^8NPnAQ%HC)CdmLY|<;Iv*dCvuMl{gZKLhI0{|f~0tGL{~!WjwE33WFt#U-gMTmJi)XcM9%XS{(hY<iH0&T@?Oi#rNd%<&lp@P4(?|tTfK9~8Ww2nbV69I7^suHL2M~tUU_)sTR9Osux)h?5Yqn9TPdEs-D&<K)0TIjUKB?uV+zn8f|~Oi(#vq*N$umRqz70UwdTj$sm78n{&{BN>$>$L^1ZcA`5t33^DMdUGgV)Y)efb;8O$d3gXKS*Y*bs^QnI%+1;+z%jF)Hie5Al=y(Xm&iGP}eXUYAJJnB!|Eru(VV9FVJCt>v_|DyDklbpoNy!)eNYi$G{Emm(2;eA&&k%0*!ShYI<!jR}8Fvc%(<1Q)+%$zGC70K3G@F+>64Sp_gNQ-=#7Hy27<!XCnY3UoX?wY({R>6g$fX{<)spVImdi@z;de&M&A;RI@)&DM%8RJ;^!*TB8(a}7w{73b^C1H0E?Be|9Zf<{^x<kIleuc#~bASDHSNMmFrLuPLxvz!Y{WqZt&EN9o!WrDg!Vq+l$7TbmPDEuS`;b36kl9?ZLoy{n?)hYb`|4L&8($#3O`j#1lYXws6WHRNH;z~U$m37xtL!GwUTNmuA!xbrz)MLs5<#9l%ER^>TiWudOjga;vR3kq`E11Kx-#?1oxf8xJ_Ak%DATVs4s+rOl*wM_Ig2whk_Sev3i-2QVFb)zOGEPfr12j1A$K5bzy{~K_y>9yCye(}f$qdRO9rq~#pZV{p#BkUYBGjAblt2T{9MI`VQy#Mp8OJ5Q8O*6#xU+orl|lTZ`T!C&aS=I%TCbBh3Q|T+%PL)aT@LHOgJE!5@JHIWo*q_qg2^g2>YenzE+E7v+pRZZlvnJC~FVpi)O5m9p=!K?-m7maVQ*IEo|9e;JCP!d+l?fe<MCl7|kXt_b+gdtKjPa_1b;Q3J0aN*p3oW+#1>7a4%T6dg5jf>%@*!u~j<r4t3}q047h;*h<O58z!I~NdE?hnY-cJ1!gR6u_BUystc~nZj5%E#L6~^wS2?tSX%e4I#pf&ejusBI|{@(DLcK*G%a<3M?<34B4d9Vn=T*4zXo8pM_{BA^ur5vHj+my6MePJgv$7~KTh+Q3v%b4THDUXaE@d=+=l<FiY)gMLBpet*98hu$2p4^ybNPLw1Yafg<!G)QU0!S6QP4z%^Do;qlIimhvZ5UQ6=hPRwS;sF>rtt*K9WLaOg-n*2P84*7@*svMa;J_hr$FWzBU68V0k&?>uLy)H0>*DGrke7(H$s+O3Xu4E@H`Aydc_*KQ?(wKvkvxmfoNoW=BhmVkSK(xkyH(8J1Ns4$c2@Eo?!6re03!c6=TxptskIhA~%=ayC%tg4t7MpD6T6+|fV4RF6Y2ch(Qubrt!>GOsM&4)>Gl-P&M&#roBb{1DMj7QlLvio2@`ED6)_iQ4XC1s&-00-+CqzEy*3KXC5n_e%tU=ffc&%vD^u@!g{gXIM9OiT_h34-EUlTfAIDv}6`oQg6K{YkoCAdc8m4O^OOlJ1$AXz?GAtfgUFWlK7L@*@wh^x6uMgpA;C;A$v5)8`uxW<*1q;MN^459*VQcwymaLF_wvMTYJALT{tpSEc&Dxuz;|`?G4B;Ie`D!g68XXJr<MQ|B2J!{u|up~T7B9>Nc0I86HGH(p?V4qcx%EAs|tMA^_tS9=LlK<^vk>X!Dva;1_TG!j{B-||=u;K$vvgOUg_g><FWo9d{rgM6)_LPBQue*Sp;-3AU}V16HLL@Gz9<d7zFzpFMp1c)F&C%Rn&EeC4CD(dJ?bkP6n>2Vt!oovYHh*B=FbVc0p&4gv5bbm>^ntiHKz_IR4GkkWJwTIJ1L+iph)+}@{?jO7igFf29Ls@r-7{%ww*>HI7Q+~0P<eV%5KcxLBqiQrXBg_lr+ay@2m6b>`<URaBmR<ff8hZ%Rv4xyF2hPdHUA%6^yf7Y4gqtX9eQri??M_n7mmS=<#^q92E%yWi6J3YLUN_jg!VV}MZ%V_Z_Hm_)>N`3RpOrcEuL|lZ8{?l+_uMVrn3rxC6=X6)p-vk&-ms#mre~DqiU7*il>pP}=BOWXmkEiG6lih}uW0q^MyZ-=@7K@`TE232?FNF%?<axTAx_$|CqxO(iLKQ0SOISB7V*-)0=_^)pA6uN%@g`MXUwtIN^Sd*iqSX4krmih#qN6b=E8?zp!pXJj0cs<M&%f<j?!`;g;i(r`AUVia(TFy$JN#>Sz9DK)RIgVpz1K8K=npA6*MBBpIKQRL-<T+vc=v=xNblZyF+Kn@AqS3%8g*qk>$jUGULh<IvH?pEHr3O?;c!M*u7+vk%6Kf1Js&YP`ZvNHecAG86|h6`|t5DSqdI>Q&v{FKTL*+$fIAxnJvZ%Hi--`9c)b7#hKt9ib@0iD(tGDyc$)q*G7-XkTtv>wS^91+*tiM%7-cBOQ6TEba)S|U+s$|h1#x+^|7y_0YhXd)t-3;re4m4D?!-$9zl#?^0ezmVGzJ*U&a**LFd<9#xEGDDd%`j&yt};SHT!&k@LoYY?+&Fx3>9j3(5RoD+*E<2e2U!$$Orp(J<bk<GUH1J=3UVccYH>oLm_rI*&Uj{LQa@;%MDV>&Kzu)xAO1u4W(9R6lemw?zf63{g)zR~(!}p0dkBOXe#O0*NXdkf^(OId6LgDis~|=N_N=pR47fkU&Jr>%CwZn!JXeuakjRT-<-oaD`na(VO4>GIoq&85`kUO4YJrf&^w`$`oG1l;hmgErv}b6{#b%g46U$m!dUsHhm4PYB$o=Z_cWK&&E`}Ge)1fKfxl-?`&l9{&`=a^VaiFc`7HY0O-3XtT56kNULO>Avv0zo1YT=0Bhf$r@&JI^y83h=N*#EHxPpY*!S)kTR!5j4#d7o(A^1I>>QI>xfSYvt>aV!tZ>5)vaCpAF*r(nlOKGZIL~;m+sC<t*7M5Xf#uccdn2HBbKADi$>;%ARft&ZSwS$=1goxPuJb1dUpyp1wO%yN5u^{<C>1~Wj#$x={t7X{bie$0Hg*2+2B~_k5_35?-hy_HSq#D)vs`9111{zJ8vhNnRX*}d5nkMXfP{iGY=BcLZ&c%`!qOis#z9&1($gHzC$iU2-6jChn`Eeq@Uo@h_SWNBLZ04fkI~vIw&K4JQr~|l<e$Xz2-uP-G~Zr#SqrxWsq;I_=5}KNSu3}4*@1)5V;ejO&1esbBm@dN2#YYB=jlcY0|D_`PHaqvq@V+<--jX;@dt4MDCVZYsuShw*;sWhuF>7?2nkDQ3gd6Z-@NT_5rk9DOu>VIO;%FZDKtE0ay$N`e(64eaW71e^khYpXTf(HCp=S2tC-;_<llpiTHED>PZDZMds{Or?_6}2f!;-H2#Mty$;ll+$h^M&q)hj{(h3P7oJ-{3BAMoj660+ag3zdjq0^G9Brjs^rm{aiC#5&=@82%08lR%gPp}U{$?}o2=UC)gD`vBuD)RBDR^Hm$Dqk%a!$=av2@PM-$RDKHnhPbe>Ga7uD*wt?IgREZN6Qc`5QXA%f+9V~T>7T3Vg0Et=OW>r)WBqi*Xt$myxR#F@|VCJb>@sbT9UQ%jvH}CHC?tE<_CebvckNs=51^!$Wj$}XC2D(@*wFV6LSyMqb#SP;dB$$Qn+g+AConH*Z%*YS<|3+d6llC!E)nYxTP49W!of6%M$iZDoD5MpQ*cNZkgIMpgBR(@2svLeacc`Z~#>9MI%4j=YGpr(HwELtg<Pe;_4gzRS=7?6+;&ja3cfCqsb$z;|1oSP26JDGr1#EAEl<-Q=t@PEJ<G*5mG5LR3QwEm&)9BNRr1d&}P&(yk7OfYnOCDKW~GtF52Y0NLn)zhb|@u0d;vN$l{i^^bORb`H!YH?ZB7KU4iPCUvI{0=(hoF=#qEkfzAqK*IQ~&Qo6P&gP)Pv9-gAUt@*?L77pfZ`2%j#R+T{evb{<B!I0%i?x*0JQHhPisoC+cyCdM)xrl%mO|eJ$E|`KxVP=C?cYvpd*Iz<XnS|$&{R~A`1wvtBtSe*CHt3JXB3<T<%OEeZG^BGsv_7Dh#$`_*&^e1Pqn+7Ju5kI~BZ+9*(hAN)GFwyA7qJV`Q$%-RwWw>SQA+D<Qq7GG-3Kdry4s8hX&^1k*I=KsXH8OFVRn)@C95j*xn+9;)`93jnnLZ1q+#diU2v?_RRTr^&n)_TJ%)9w%ccGire?7S`vrBlz2D+S9nB4cjQ6??1{@QBDF=K-k2>1q=KDIN27u{}@E~nsuD`>1f5v)(<SFs7Ycd=9$qm@-raKR>JG^KGAsSmQx3pdJ6l}uzL`lL0I?qUX%rT#|^HyQ^`Hc_}%^T(m`A@`DMeuz*sPSIfIl*S)tBJ{+8x<%S*k{dU5(!v9#XMfHy`wu6A{Z5)SlTm{3NDv0XQ`Li5{pD5(sc|Tn!>YDmxZ#?6bVYE=3BC3u@8fLdhudlE19fxdGO%**JxAq6KX|j32)3zdQ?L$(R!tHMHW74iJ+pdU1#HZY?EJCbW>TM*)*In9}c~#dAC#YMD&I^)A3!;<I*WSfbt(LK8e&WQ{R6~uLy|J3DKb@ub^Lc+Y5oiPNXW;8CdZA3n~~6kqxy7#w9*1TA9pLHX00c6WXSOqR7nqUyFS!@#w6FqOdXwYQYLSaI`T|e!PC7$Hh7Dw2DKCT_ftMjk^oz-dJ&sjyr%Xk|&)oBcDhK@+cXVJ2R$f5HVfupvIA5n|769cV*c9#NK|BW|V|vmSv(Ew5L{@=H}r4=nscxvUWb1#-P>ZEL@9`uZPiv#U8CuTv6RkKHK-Td_ZTM9JCG8EkKx4a#}z)_A1yIuk8BMfEK#gMX9^ayJvJs+U9=;UpxtFZK@nHRRA>V%nt*QSNw$P^KdLD@`{z>$D;L2Y6trUNH4Dy8)}{^3#eZj6HIiHiMm?sD%oW)HEEHWXcksRD56MJrq}V;SwPC4CkdbA!u<7c$Ia5JOYjK)k-NdWx{3f<)J4i0jcuZ`+!4mYudQs;(mI6mJqx$*569iI@BuX_Zw}>+x<KI(W5Z`pVGas=WJ*zhVOy=x#@SWRA)JdLX@T`_abmlq`VQA0>DSJvOQcno3=P6SShtB$Dn)@!k`ohe^UHTj-PlJMuF8l$7G`7cQqqZ)&d4GL2&}LPe&(~wxhzHlWH#G>8j)FyDTjm@T%<_;Lhr@TQMq`N?(V+@M;sw06BMsEs4}?NPb9ZiJi}T*W6Y!P(;f9^r6~fy`iPe}IYqr&ykE=;BJ$bY1KrMFOTXi<rfil~(!b<^LdsJkSwzsLdc^NX67!%y1Az_-sq(_@MgvY0CnS9hJ3!t}7^;1&ZR3xgeEh3BWj{~&Dr%$zOVfK7spI5yIStj;-6)0>{;m)8(c<AT^$9J&^T+|37g7=>&E3wHz8u3s8>l6#?w>-@qLda)w6zW)Zng2Ek}u=e@l1Ry5eqIKw9t(yL}x21q|HtyAq<t?z4C-)%&jRT2j>YINxz`zD0tQ!90r$9A<eJ%@6t+M*1^6`YuO8#*U6@}H+WC|<&D2D^B}eGAL2)<9K8hVUW4;akq3$XWPeAx#;!a+j94)tKz=i<Sj<?B(#-AKaf)kpz{rDfZpT6N0UVzqARCptnkr>FY<Ygf$V+r_jYk{7{(qRJN;`Z#-}W^lb7eL5{}W5jaD|;lt#LRek9+UH5IAqCTm3|=Jz%L9@6zb1l&H10*V0P->rjXo<yJm~@IN3;#Dxx0BIr8ug84;Fz7v9qyT9Tvqxb1b^rD*+=y|=9l<<_(#8ZHY2TGX_Da;U2(`QM7|4T*jkj;1`GBU$yu-Fw*sX-zs6E$}?ruYTxo<m<!eNe|(7P)wW6PP*K=6M&JLr+(UuO)_s$nGY`;u}Hj=qouXz4@n5IXN2qyQ2_b&rlHJB7sxR=8B6P+wyY|j6MeUf{Cg9v|K2g)eSF`rrSaQhI}rFgwo0ZsgYFUYEEVmR|j?e=_ciS-EkAZ#ikc;F|>USFoPb1_W0ADN63Q~`WpcH<wqThPH8cX2$VYCFEvtRcDWnT?sV5{P(hG=w~<#l-vOhjw?^UK$mG}}vp9IX?}PI3x+a&~TM#h2M&zv_#2x?gT+rotjZ-k5-1?skE+ZuFrun*%sc2333%3vH>eCQFiINwg1FG<;z~k)U0TNf69JaZ0B{p<;(j1^<9Q+?jyf>XT-HbgO2Q+^aa(Y+=dFuUOCsYT6O2FL71d{x%PGQmaZK>l+uEHe=rY8>U*3_Q2Mv3dhe_Q7^;fh@ypGz`hY~iN&ME0bTxM)3Jna#X)-(JzPtjRNVMaI!9arM@po^9+W&#?g^l53kuA~w6ZH8qR>3x0$NZQR)Z9fpmy9%IgvchA*(UuxZXL!Jg=B8-T7E?RMFn$|yP_7pX|5D|n_q<G1?V31*1Cw9=FLKMMbPC$dVsizd=M3Oe848e_FWDNTSSBH`vulp4ue&pa<r6UH9@Q1i6d)+EoA|j56J1^<;4cez-MNzl+M2FElWS4daD{qQir~cR+;Bp*2EHDqi@t(rTiL9)h9JTYBn1VdE87+~ig(FNr{2~CZpEuGV{>i#=w<-_P&lKUXaA_++s9h+7+OQ?KUQ4zopH|J8`8I3aML-J_`s%_Prt@^Q!=Ao3A36n2(d!utpt^O|4P|PE4x8hQPM^h9;ER?~?PWlAy>)!*BAb}_+U^Nuio;h6r;V8Rw1@{!oEq*8*T}5`q5&7DC`M4JInAC0$Dk{8r$PucPztL;ciYzW+XXbC^!2^zjMR8}cuI<zK_rQ2|1#|X;k<3`w}Cis(=oBE`Ahe83El8!h~+AWpRVU`Z!E%Kv-9F!c1zq*GS0;v7>U>HogSG9-_=aNCKF7Em`k0EGey8~qlOoW0a$6^pUHP%sNK-ZN#-8C^J&X?j!2Nu)s0N}?yB-M>bshOu|R^$3d7>nwo;Ndq<U@ak4R*3<F#b?4-v&-Zg+X_O1oH48affmvqawxbnqtw7WkdGL*U6_Tm8A8h9d`b|836s=utv-904KZhCaKVla)DVbY^v~LZG`Ln<W~%wi;4ec58OKrp~|Fy9(u%ke=(1j)9&(ux&jg>BDqWL0wCOJ&N0ly52jUq0#r|dl{}LL59a>X~pI&wR##F<OzD}`_WGAUWD+04$RC_RpbouaL+U1c%`ICng_rpRRZlgIIcAq^r2rPC%p8>1*Aju1V;!{T=+9N&s|{~jKp{E;)aZRo;R@RflBb{hTr!v?qG9cu&&)U$gwb*|5VYO>o5FIcaGKP%E%1G4IreQ^|zGB^mWN^v)*r)jk3SLQs!p$2;v7}mlE0!fJ<k_n3Xbq{!4+T4;l5UE-evik^nHI_&0KlD}m6KZBJ^fRe81(+Lzz)9hMHHEp}O%z$8vn>(uzkqu8)Qt%<)^HG&uGGi;5K)J8wd1rLbUAOym24t;}s!!@{9RPeKp6b90TV!o8U+2VH*vWg!%UFUu$E%q!5C~6<u&8a7p4<wDe$WKAl!;<Z@mJ=i!i_1wHaXjH2j6x#dotfrDU)Cz5rhHr%Gepf_yIT878Nfm}7jW`?sLXv;f2O<Xoa)u%YSl<8EH?#iE<Dy<?Oz}m`qPpS0(CZPLTc<&!rF~&T!bGjvc5{sv%WY61o09Jo2dDwXA7CTPYyE}<)lRr@~bFLwWHyWDjT2(uTcOYEA&zF+&>O~w)N{^J0pKyC;xe&u#KNyGl*{j`O<c#bp+H8OEG4~MSBpM8+1sloSqxP6#0;qewtxV6k-+El6#F_2D49AqE1*w{$f}3!>tPt^MqXG7;60lr>_a<NVkz4dCfbufP`!^*s0cvOKI8mVx#NHuPhj~Qb<qi4gPQb(^NzjU#|q{53P;ZLU${-e875JnT!2DG0jyfPfg#O*8i?kr8B}2`2li_`px0qQf5HxkBS;Fzq%HQ8QbP9?=T-OL23Fr{mTsSb}!EfZi{4T(AXsrI3okMaR;&=vCm(C-WeO<uja;IK-OKa0ySH`tX;yEGKlB>4nSEy7+EbuW9BsDk1M?!=T>f;fTwsYHKWtI_h1S5{4amS-;d3l#A5GKxDT9*9`|-bVHOa{ROU90YCkQRS=k>jV0gn{ie?^k=Q1CeA}v9xq`<Tk_3#mOpyp~<>5mjKp-&+Qx-yb&U88QVwi-iOwz4*$Ic;#}oQPgTuqiTYz-~!o0EcCQpHU`WB*+%#emSbJl1JxZ0OLLlc?XhL)eUAUfJwMp?E#nHP<iX<7_<LcBA5+GefDs0U-d08{m3+o(0FDLEE(A3$Ljd^A0n?)r`{;)l}`a$X>kL^&%#Bw|CUJI?J>OQztHp(IKs-~b7xedbL_;*df=6?J6y!q>><qn()=qdF!8`lL;l9^y$cXtDp$d#V`;)FDokRmEylja3518e%@f@7nt!Z2(eL#x@3A>19C&=gg$PSVbJZpr%48HEr&9Igd<eszHv$PSVYirV$g}%eK_kBdHZ)i%Y&c4A?eGRvsY(ztXwkIrj)-xcX?3Cicq0Y5>I!lvBBQy@Y?Ww5^hMs9<Lgt2T(7p&I1(gMkpM?TiC({=SOdvd3#7)xx<QT-2@UpFk0>A9mo|~WNzcU70|bAo($YlXM=I7=YBmq^h%&kv1@jkM+HPbU;9b@j3XfKR+nWL;#D|c0ij)$g*eu_#mX%hVi}l{quKFT9yr~qLX1<UGNUE7HkqiVGD#c;`br^Pb6!Ya9yEh2YN3#xN3=Qv(vn_o@F>dc)RkxRhm-590LvQZg5|sB0_*MSeD^(5op8y2_|G~qeYw-A<(JFTR8n)9E1`h6cXU?k0SmT`GVxPK%ZL{mDmZxVbd>=MAtX@@ekJ`<^PTy<c0R9yE*iP5a^KxbIl;mEMlZn@{OgPrVRC*{4;$TBmglnb^>>fEUo-XDQdqPe{pp2L>FW}Xua;3BXxxGp2QZvb>e0wEKWiLrJww$Wgk@gduT-?&!<-lJXL0+gUDufi=ewaa%PrwnzJ00yPkLZWb$qDZ`3L&7IQj=J^@7Ze7!p;gwp^tom1{~)cd`hlyNc>YjTUjqt1#<20KZD`^9k_w~zeQ!i6}!=+k8GwE-crg89y=iX(-tiax=0A?X&vSW;*YD%k=O*dP6@L}3`d7BjCR2=>JQ7PSxA_WZ6?p4b}W&F{1P+W??AKnU<8(nj(vH~p_Lv5KjbAE82ro`Jg%!fp+8SJypYNxN;uVB5V?SoDsZ;3QcU|q6eW4VS?^lV4ltCPAE`#zX9X0D1MTU%scZH!udYczt?=0fn)~GRn?gKtbM64R_hZlt-(_j}y(ijZd1RFS{g<)t(5->^OM9b7C>F|R)usCJ^2Jft#TF%l|C8+jhj^n*MHI+De7bn4_%=?}`&Dyg)I5leWZCt^I=AAuY*||5QW39Y4K|QNa37EDj+^OR-|0?*NRSqwMh{~42+ylDj<c_HGmjGkbT_=8yQh~Ad$ad0B6<MlhOotp6|=jU`hUhVesLF=5YT?gq`u}wPXhHe>*(}NmjqZiZb%XuywQu6>=~(vXrT0e>xJd;ym!r*X8DSN1Rz!3K*1xPfvyVZ9O>+=ysaW5kHgo=29yo^Kgwb^uBxslEts#Y3X=&8rpz`Y*X<DIYt#>?hb(AuN8|ha=5I(eGNi|O_^<p-N@U(efw;Z1C-y@Q)C<s@4=Jp6ftFITLKATJ8~Yn8eBy1pxmg<x=5qgOxEFw=lpc|56W1#A(m1!v<rwFVhTgj8iQO`0hF`LVZ0bsbM45%|PTMDtB-#l$W@9IgxAT6fZO-hl;&2Sn4#}$3fCL5Ks4v7%%CGQC0*;?SZ5wCYRaaQ2Vo5>}or^WW)T(#%1jX1`fsg!Y;;wBbppn#%VkuszKx3Fs^zD4EJYv^niP)6Q0JPm&V`AM3AuHI$FXP$Nc?>4Ee_uP>y(^k}lkmSrL~R>lGDqqd>`qvKa|w3vDwA?bV}|>H@OwxF?I-e^G3M}1-zPJM6E<1>P<xigTLe7grg>c@q%k^)V=>9;H`F|Z^Knqbd{M>|6fLB3u)Wv*-k*hI%0iH9+Ix9zP=XzQvpYz_v;4dJ0hXmX+>oXVD7c2D6Gl7}Y@zlAkg7B>x%QHq3J*hTK5@!u7ZYke!Lf@Ms@EvJSr*^ZozRLvBCkc~()`z3ay$vgJ#2|dz;F1gv9Di9DSEJH!`N~CNd(dKy=SV&<)pDtc)$uay8yuT`69qj7n)05_@w@$wO-B>B}1G^H3sbTmPL^XwR>6X5)woMzlq`fjT8}Z*ffsnIKp{W47cb{8WWb6pP=ZcHOJg8xIIHTj1mBN$EXDh!_RNg_=ZS*m^hJKpr<$)6gw~%^;RjOttUzZNfWDFrxu#T()PK7u?cX=kZ~P?qXtTEtIx2O3DebShtERCdy@tlNet2IHTYR7Wv=Q`A|;C6^{}6_pu}4~ULjDjp8-Luf$Aob5<SDdcSgU_*S~zBBB5*Iws%!u*tC`~UgYt#gIa!6vY>Vx;$*vrN_zrbyLpC=q#L~hN|jWGVtLqzH^sutaVAmvv}3@zf@6H=Jxgd&>A@&(f?;w5ENFQUrS}dmix0;XrW3Q*djIJP7_(dl>s+(NH;LHnej7BGSrHZ}x?M=6QH}g3Qz26P*USxY0N5$`vuX|bj~sfq?bW1i#%g{HY}v`-H`XnP%QIpktx3_YASgfl#@dQIZ^>HoJ{bXq6YD|Y8>XIE^c|}D8JVaG{Mgy(!J=ae(-p%XFK3Lw?a2v3?p^m)a<EMMlGh8vBIuoJWsI#o4`jj78L&hYm`u?w`b#T2gFxfZqWmSm8UMWRa3puRyU$(?WQGCIt9uvdW3WQI_+!6zu2U9ytcjG~&fnibsX=Boltan4K1y*RQInr0k{->r$f=J%r;91r_3h0mEqhZD%s*Xquc94oiM~ophLJ$+fz+{z?pMt^G}ZZK@4UT1V7JOpfKC+2<{Bc&4)d=c|Muh9ufunwZLk5Vb~{p=dFu<JR20Qw4)x$6D;)~k<>Y}P>?jt7neC>k^A(8<ajF_lUJP~@o~SkL*_r<Z5qdYtK<dBH4|uLz7@AYOd!U2&JowM?+yD^=2{s?6+~j>wMk^JyT;~Zd|9rl#Df9FxT{L}~iK>9PtTo0sdqEQIgVFNzm4o9)u`5y%Ki#=bge8(SDGMMmrUo)Qy?g4*XCZHDEQWmuep|D}-duhLY@!mY*X&W<?O*Vgj)ZP((9_ZuUk*)mce_{2X{#;YrUll!M~!6$iG+lCGGk8z2X3!&iYKdI3x6>guCGvICz{cNuNibM%R2mHEs(WllbZ-Ao!^y|4OI2BNrF<`p0zRue7GL}j9ROrbh%DnRk7)z&o#`0woomoc=+N1rx1Vsps7ZGO5s1&ztX+tZ(Fu`r<{UXB+$Y$j18kyl}>HWS<6}~uSJp@6;HiH7x8951vnXd_(P*QocAcMlVVbvWHXrTkLxKR-Ff&5BW>TT<F`3uO}VppB$kB20g?CkVRf;fUU`^9ICFk+*8Q+#(=P0q2%r$erb2&N8>=HCUpH8D5u8UMci}3{8@0)D?S^fZt&XAb{^@_55g?b>`8MRnsR<78%wIou>{_Dytjnv*XZJ_{;{gFix~tt_=WhMfDEu87{3ZFm{ibZhi`RJ|cA5+tNLe$w2@hW&U~Y(KeS>H^_&Nik5scl}y1U*-N?BJ*Mt(nH?z}Xd^@H!9fezYPw@I>n9h+iq@>OL}54FYPocrqn>)&`ZJyCYcUzDJ~=G1%_e@@rr#Wzq5e-L;ySN7TKBqp&}M)^bR{<Gkq>c~{=*vsrVQqxurV8_rjkk+X7L(FhrS1lk$rt3Pf9`5$_O5#bleW6!D*Iz%YJtg%k3iCvp$ff%(k!YIlXrWV%RYtPPS=oz}9*O~V5dOr|npln%dXC}`ZIZV3B_289Oo=;6>@6c&1EGdI6{mdQ3VuQ6F%F>l(n#e%^S#yss1Cb;d^BpRLHAA7LDh}!FCa8!-N{!kH$J)F|1mn}mCz_wpBYVq>$MnD*<jlpR90H<LUeXnHFFeNUlt)G_UX$GuIHHihZ^LPiEY9!#iOnd9KrGWCL14YOk6TC?Tw%dJ>N81+JH736D<@~kKvMXoxg@S|7amqodwJ=>n!f~nA>5d_`ep%%crrH$oBUseN<n9ePdlIt`Iv;mm;8MR?dwzMmlffzV68I>Tn5mS!3(X68P^vBH{_K2G|=E9kiF~tqfeDu`U6Py=NZqWM267S;lr#to9_Wp`qS{*A=T9aALiw=9e5^)q<{tZ6a^KUi~ll{=3;>MzNCgnjyUJ3<e;MdPCPv8C;!Wvzn-R0^8ZBy%~~6X6ZA>Zj}6_gfO~zxSs+R5L4Y?izYKX#_in3|6Yg|VtBbZT+3l8=DK~2eS%oanWIxyk+W!yz%BvA5c;JDoORXbrW{@WuXluM2%Lh+Jhm*=ox0uPiK4ZhAQA-yfAo3E7mpapYH==(bpP)mcYBkCpueW_`h&?SEbco2WDR1jqD-K7x!3>p!oZRgjs~^(>TJC5&H6b4iR+XBj;eztqpBBTEj%c870bl*CRziN1U<DfFMwx45C>HZSopr~6bw)}?w5<=?&4P{zyT{VbF}i=o>Eh_P!Yqul<v3My^kIhD9D+yB?I$|qYx-DWpI_W_p`7Maq4hQS4~1sDX=f??b#bo-I16yC1fy){c-EO;99D7%-xZ35nS9p`n+JDJ`7YN4f`r)(e*}XFA9l;ItCbd<CokdwTD!5OI{}!55PPlhP|BkoaX}}`JQ_mp`<YBBZli|ipUzUK{~I?GylktAsJ^H*Fo_P^jdquCYTFW_cr1M0d>kF$e@U9-<@JMJm);%5SrSS(NDZ#SD6tzoRj_7^f3)g5s8-_w5i#!wyHACx{Aj#&u6E!hEp5Df?E}Hip&n=EQBh`GA@Ts<)e|@d}T5bwASnl@{#qgs%F!NQ4aSO7x%qSXZdKcPX<dSFn-;M?<CF1ZlX6R`1$}3Q<yEYu`;MduA8)5tmnN-os?f~?$~sMkV&aW@qqUr?82BG=QWC4CIXq}TBdx{h%&KBZWI*`oK)0~n<G-ru0xEg{#k36JE9)QJtMnEKXrqn2Jl3)4QS=m2WoIV-<l>fT$v>#LzRjR4ZepWwoZ%+0K)yL>o80|JcGdP?^GAL=i1f@z*G8Z0BWQ1r9bGaYst;5KWbpW;4-ba6nUl$P^9KIG_A?D1{_qmJ#+Q_h!8ImIwjQj1{M5(o~32Se&=W{iBks&LiORwvhfSdl4m#T#W}H195j{|h$?Qi0Soyf|Bqa>v%mfr{ij?0Oa@4eUhG5+fhnD(3SDf|bn$^WaO}gpu}h#S^Fh-euQ<WSZsY|A|6%v_X?MsjPKT%xNuqEaV39`M^!LPM=#W$K)35X0*E7(U2n{p1(ojBodl0^h(Ivc&U&=8r4U(hIH%=$7gELtz4F6+%&}qhlaX;4gcVQpHnSqSTDC?lpPt`x7x58cbC38fTsBU;a$09yD;og)F4^^Vxq3{J9_b<r<>(_#)<Vll$(%2oXIC!<#;PHM?5gZy*Hm%jjTEo1)dklS=Z>4O{JR+NZDUeRMe@P7QnTM;sJ=d(+9Wa3i?dqh&10XxL!`%AA`mGG}|G!C1#R+Q;dRAlfFxH4njiDZQF)FGxddl+Czc@Tn%-FT$*QH7FuYZQ&4~%sVBoP~fB>>Jxbf}#u_Tsp6Bg$+dIuY<S6WE!Mq1Fip2BB)YMaj@CNk2NV!UhTq-#!-j1cUI|Gj3kQPse@F-<kAK;u9hg>jH5q7D^JkNE%Itnhw#QH8f}i=UT%VbS4^&z+pdcMl&)sJ|>i8-vn+&eQXy;GVz;tkV)63WwBN*N4;S`Ty*~$0ltze?D`5+WnLPOW?74wB<Rw`zb{e@>xTtN=GN8G$7og6It2>uXlRW!-MI$UoCw{&>%7-l;3%#Y1+<o7!o~O{ew?7#Hp0?`QnI10#or2><`ZTB@|6~0YR8h>+Mt@rBAB|zaf?A6?uXJpzXOG>w3@FmzK?-Hvj?7_BFiC8;30M|12VvHtc3p)3LnPcXmrg`%&;Dwi_AVkxBYh4c=m}``uaOy@w|#UTk!_}ymwSQ+Z&}nVn@MY+Hr61`T)Diql#QM=J3qe2$-U^p{(d*a`wK4fDM?2OY$iqMZc!65)5}_$b3hUh%+k6zLMULS761bke4Rb_N_lrt<&b$#NSv*@AWc%E_EHd-I5|C*SFBPHbAllg7rCWld#UTIzmW!{5JW0ikZNX`8tvqa@#RHa%9`OWC_3JHKJCCUB`Y6c@rpT^<@(d-MPw&rtuoY!S?En?uCL)Ar^+87sa0D*&mxiZVg<2wtgJ`Sf+e)58|ZYjhgv=C~iBN6Oc+ATaKUd?q+DHF5!4OP&)u2iNHHs<hyGbTdUyr$h20!z?rs;W(7lBFs-!jCM@BU(r9%lV&_kgOSjmv8^JKcFpXCT#S00;{J?L4%a25VSw+RoaE`E))Mk*7Nc^O@nZpz1j2qNB0>4ytuQEZ1BMz+$#Y4O@h~AcMioJ6n*jm;@8KR1qvCVe~AY97Wwr6GUzpDH4g{v!L<d<`IHkif2P!E^}O$WD~3Bww8ejDr3NNsaz`lLmz2dCDvrR~H>O#Z@q%?m9$1L)!gvmZnQ&?;)K+^U7BvS85^PzXVDrC&1i&lrh`zp(FV)GwW-3R=4qXrACl4C;jX%ujSFb=bSV8xKsHR5=lqAFq~H-+!H6-a<;TvAH_q-G3dyt!_(v-L{lPPI`(9hXO7t_*-C>tWd^=>6LAP`G8+NrRrUfe)I(*PptPeArV5gpH>bQO&BTPdr(v1swCd`MYJ>lSh`VVL;9aq%##*OdosbYv(d{g@ruwwlBVq9Y#_y)#$hv{>SBFjm2YTtRq<X+80|)q_!u>~*W$&~;zsnGn$qk$H64cx__CONzI0W;*z=$vcO?lIxPQdaksadK8fZ8X5?1eFBm7&4%_K@%?(7SWzHumFew;2Rb>8N_&R{gCC7|73Tu_kra!cH*UtB_8$nD_I@((IsQ!A=hae-Uqi*;f$l!0`l`Ms#O_(l?`wD-6~H<Lg(-^w8N`S!7j=|p;o&Y=wW^|S0-CD=k4n*&?XPam0v8LhR==uqp8hbF#N1atq#`Bj!ks`8j~1I}H^ba$fi-}wRpv!Z8chQtC^axrsAdD^bahPHFXL=TW8QTNuM&eH9X^~ztfvKpF5fCeJ+I2yPmj8ek7wd1t%mrd&s0}FOZOL(un=DsIsM}Uz$`Ai;4-IF|bcVoZUX+|mPBSU8~F<8iytW{Wp&qHK5c+Sy7+d>u$(f~Han~dWbR|~-*Ts{`KjQ=!_$^wigpNc!*Lwgs_=5gjVwJRC9E58<QW+=i^Rj6^+=|p=ZTzCL%)ami0_??ZO9B+_B4T;Stm7A51AXOKLs-*rKE<HxQ7sG5vD~6_#c0J5LD<ZYZ)2|x+)5(R)dqP|-PhNQ}h*?7VKE>M@WV^o+gre%yw44SDstF`H0!8Em_Vo6R2Js#)F+<ooBWg_<UJJY2$=QUQM!X30oR32>`?4rhg<Yc2VI{<m2I$;9AM>%R?lNUqlffCzjI`)nR|((V+BQZ`pjXKQsE?&-UlgeM0$4K`oxWwDu6sXp<N)e0YNee5hU+@^q=vL15(xLgN9|R*=SR4aLYpXz#+4A=ET(UgE}P=E!Ptg1geg7vyihU#o&G+G$DdA61K~hk1DbG(wxP^*L-MaDiTa_=WbJ?ch<-<fkgFl|1Cg*FvzfT~4Yjss_(<vA!}HvO>T3pAdn^&HM`{SoiHSuP^dR?jq%~@`8medQ0<T_`*#xdi53nTp8f6BqF+B}hb)2@B57Wl#o&y&i|0x>3pvHWoEmsq?&rt-~`^$ce_;*8u_JZD@^rBVm^1t<#XU??5{gw*cjYK*tN-=@?g)>)Q#&kJ)xGKBM0x%P9M6DcD0-;NjHCC5@_wmi7??r#&ii;<oC0#`H**K5S*=H$sXG$}B!>?p@o-qELA48|Ay{P<As=Rk7J83LLQ!@kGOF2;f6fXVN9$vxCr_=rd`T#e`UT{P*YtjA0DhHLX{WnaGMl%SxEn>^&YC>~<Im@xgN9%5N#A{V<8xm-zv{zTfPxM&>`N3Q3Zdc)ITfvu%IlCR$U#}9&hu>3rHe!}&=jMl9Enbm6vDZ35RL;Ar>!dmY{2U#_w3>)%)dk6s&uXj{;o<d{M32)lO(nvY#FNFE5fb~f(<?tC05v;dq*zhVdD)Mn@e8Y?wdVt2ZO#56QBa%?8H98}O)1)RGhOE?TKz!7s3`}<HL`2gX5?c%fX<T=;+9gjpbZmc;&|J!MD4(>%+NEbeKmwe^4Dd4xs3sGY`8&44mo6c@rd3a1aCzj0<SFHmeZi<gT}{VD?HXe0c8g#`3G0q*wO7_7MG!ED1qq|=5P{#mVzAUkOw${tM?1mATJrP+`E=>CGg{s&2A$+UgzmU068GAfE>s!e&S!1hl;rDur-{iIseI>JuPC6=|@vJH@zY|9*1;*+icO6WS~RgalLr`c9h|ARrDx#vG*B71vDa7j0zL9zTA>YYO&tQ48%2ZwFfQROZ%kniMerm=xB~q4yin$rrxp?JY{Vg&C(=g3H1ML6DzQcg#gy6?w=#Bb;)wCkqmbk{bdyP9e_7K|0wyZt?NkVSbE@0@HFO1<5)3YC>w@2(s55Cd6Op8H98y&*1$v7*N0~(wR3l5!@Ch4qyHk(U3iu74eCJ;Pt2k&FoIa_5hmu7d+>fE^{Rbu0rAJd5d=8sfc*iJff`Yz2v$8V@_a~n2|3Su2QU&&U!H~=o<m(S9E!s}X#F>P0VO0Zs0K)HDc<I#luw2y6DB_XJJM>3AR$bVnB+VQ0_60|>XbW!i9~3&s%Z1dhs&S~(ArzLw2IUlB2=Q;dZ_mp>qYrghZ8gR<8K^P*rI@g)^zycAeoKyn@Lv_LSP~|e<}SdjPk~%Jv-2u+PQlNzy|Vyj7ecdZ+7EEW7eo5OaJB~T7oB=_>Yrf54zPecQe-k`5x;pRRE+F)m@#%8at<i?(x;$=`@{pxVQ<w0KALmTgJ}MDLI?z-YJ2}=#0FTqq>`7B)zpfJ!rK+asQ=cjX2w|uv>J(Jd<7KV$5Qfff!lhP#|yP-P3F^v@NcfGRN=6Kt;^4nDGt}lIA$xZ=635HT^tUq+Ea`W0IJ03yY5`lXEk1XD)!5uN8c&ecdNRsJ|E6!MPiQ;Qp@2ag~Bwz72)zrO%oFEQV)Y@3$=Zw~(d1Yi$)$)nV!sU4bn$raGR+s+6SZ;qARJkaW6o*mK?(f6p6(x%%Q>Xu&_2K*kywaO)W=)-_(V$i{E@m3Quf`7xYR=4OPA2yzqgU>KOzJVF`F50JmU%JS-!0y9Opl^Kd;u*8P;lL&O}--u-6peRD>wy7Mz(qVUWKQ)r=>nQuIqsZtBu6|hZjluCz7Ygq#P|<NVwkb}M<Ra@xr-cddBHWi!kdv3WuK+}nm`mE?EaSHZJk&`KuC4b#T+h$Ig`cbUg8tXfqsM;_dyeDzWsE43*SKIDp(SNncr6xQj+8KqLGf|!fjND$x)~*-@0ku9%ip~CA~^BSqwDx#%C(K7*+r5twr>_^;FJRyQDSQ|RgJeN*@W7n2){Q)Y1r6T8I+xE#`gVt3_xlf8!a|`Bvk0eR0{36e@u0$(Z_9^@rkaAyu4<u%z~j6H@Gz3iF}i+Ju~>9`xS^d#;V?O!;<TL(o=wXEOiI)!@$`x)SvC}KKf-MDs!)}m~Muo$&UaF#g>Xx)XH{o4HAH^rM0R@9CG9p@zZFy@!H_EAPHe@5ewg6Qbqidz8=67-u9qv3v6JZFf&qcrsVn!3>9?{W_9)txFAY0WaCu7)RDR^Si*4#1zfUbg3U3gpbm{wiC{I7IKQKEmT`jcHk;KbcsJG3V|xeY$(L^D>EX<-<B0zYo&L%aFWU)f#Skf3imfviPFFurCJua)Qw~9z&i|+{=mC^D&sx{;HJfIsUX6^7v0~|EOw0|;k3fpWFs8iX_tV^;bL5NN_24>5@yPw4CLvc1!wXg=<xnBTKP6`!@71qI2Da+L3uVBV@T`xdfBmBk4X6HxL}OPcI={d9GnIs%9klNGo)^@rHmlS=1zEGZD`VvfWo^IzG%b6s@XzWYY`Bhan1tsthtpG2>^Bw#hV*AX9bN@ZvzUDIR!y$%=&Lu)qvi&R8Q2xYE14j6Fx2g#ZCG?U+jd1eZ*q(If`JnnQ4v~t@*Zs7TD39V=zQr?7!(LV+|0FbwIy4-sYWI?W-&*FH{)6%bR<^T7AHWr@!~M>A4CRKHl_z%?xe#<XB`yInFQa;<_zU>Soz~u{^>Vz84csk+G?p6t!Vk*=j;-&1uK%|C!fQ!6Q=kqhnJSV(Yth7DF7SM$F498>RG}H<D$rPK@y<x$?7n7K<cQ&r?%@bl)NtVsqqielPDsZK5YxCu6uM>5D=weCYXIY`?W?oO;4u#(8Uf6-R0|3$2^I#fPT?aPgBFd!rg^jR3tE0(xNBA8>Mq?ga*h{vuKtocDa>B$}~FxfO@Py;ebfa+N86qV8Ijwp#CX69}&IY7*?sxKIN&+#^J#{p*vH^JAkLjjtJ+4D<MM{vA1zK!X|3<Gf(vqz5Dv{>zNUju6kvvW%|QzTOEzoleFA@Iztlfc?Adf4;fmQsFA5&Y42*ON=*Ve{QBf^)N-t3gYHa5NU~mMK}Nsb0;ZB!U*rKr>Rgi?<NgqjaKJCWJQq)kYEs<WvV%sU_q#kW@FxJTxoxzZIi<{i7U9$*hOrb_t;cy51R=YO4tiO@ExKEs3XJx&i|;2X-;&(#*9ZqvL7);=x$Cbwcjc|J+wH!deVBpCW*<OQXZ(<WUhei4K504xz6J`(e1L5NrVn)Ta5@3FRV@_}!%%JQsiTT0Paa%^`#74ee+<K3jPN=8j10|~$<&T7O2S4@)cQCVdY{@d{ir=?Qj08ywFPbG68Y_mCmfziH96JJHjrtKhsDAP?DET<@E|RpT@tB${vxwshi9WjV^G8eVqeLG(wA(_aO{|fp;dh|xW`Zh|MM^GX1pqZ4IjP&Oe(Ik`@hH5nzE%>vSM8Ebcy~-@c5O*v;YNYtVj6g*z^W#;xSgg{Gqj`w6K?;oa9d2;0eWtI8(p*vdY@Qm%27l3_w!F`?QMh8#4K9S$iIn6iTod_RsM6FlA0FiP5`?KrZI8*uDvb3<fICX4KdKc`JVN5`}$Vw4RiZa5HXAo><&cgj1mYVY(~Jfh>l%bSG{BCuL!BsHJ5Bhs_<CK{OmpXX!;DMYAS2fyTRjl$!6+EPMglEJOUr%{nu1Wv&W&Xs#28fd+?D_O+nyH>Ooe9L}1>Kns*s?a+~Y>-Jca=VEYMowECmXYcPp{Z^mfDyze}CB#QGmbVAUI#GYnaPu+&jtR>&lT%5zrtJ;}y#1+V-QZfbQdn#dT2Vrnw3_CbEpW4Y8A}0K*dTJ2Xp3y#k?X|-R6A`%AidMdmMjEs2BNJzgr$<X3r%;8M>S}KC{{Fby4qp;`|*K{2pS1aEGp0aGE<)@=P7;clwoui9-Da0AJxE&ukX<}u<<0L7<{7%0oLrb!shh0F{trl`LTy6!l6!ZrtY6iep7#6>MoxqipL^72&=NQ&-81m5;BGKUi7?o`mxQIREMUJm@g|$?dN}&rRi1<x~7kML{3%ML)}=7J1-`R<2SCr<c!h0nFIc29#?g*SvIzHs&HUlQ}?YSG_)cKC+wqT1nd)8!dy-xN7~YB$$6hBPoxC15Z=WNtzvEy)4y92VL$Sk6e_e#y|(?SOs)n#x$O&3YT|pF51~NF@8PhXN?T8)XGwHa`J5#3%kWLQOsr=(h#@`5tYx=t9?<<rqQ482Jz}WSLm_rMs-DeJ25M{&=~Xdbj=^zk{2-^mItHqyWicnOQ@qzma8gdqIDDe$Qo;1!?T1$Q_=stkvSbm`AYSC22btswJ$|g<j$Q9eic8y((xswBa8<wp+M=1lp%^1-{j8{$OKfMkr{F9&N{l7PpW-`Y`P~#zNj5|U_Fsd*kpymL<GS8R%z>TO+#8!4+OcTKA4j*>AVd53R%y72D(1-REKw%TkVUy(wd-n35uqPhE|`<P{=a9(dS%OzJt}vIJ(Stf+t!{h%$jRkuDy06D7wRGY~;JXzAhegyLnct>sGPM1yrku0Udt+SOS&5ksUCIwer`-z-KkyXRVCUQYT|;KTiPzo14#Wxgq_tM-LNUrqj#9vd@#|^yP;!L+N8o&1>XdY{iLMEn^!EGg4#=VJQmA7qp+Neou>X<aIMxuMznQW~C+p`RwIL3QD$@BaUYSFjU<R%1{Jctr8Gi3uN9H-Rooy<#uI75!c@IS-sF_h))|kSVNOc@rjCqGHHY2^J@m@uDRBr7y&ZR${e3Z5&yQm`YH!P5>46kakU3Sl7G0Q-;e?Bfq^QT)fZ%;l8^wWW#8WZ=oDvt$&OLce7b#dFi$KZyT+a<Opan50&sTmxSx<Mt#?Tpyz<LyMYMfvrGH?r*m`M&jR@*?r}w%KaE;UnT*-S?16)BM+P-W`<==K}glK`zZtx=UZ(H{<T*SWlBw=+)ed6lwuSd(mLfXfu3!7h-Yx!i8PpNDIExTP3yt<{uX52Y?l!n&ZI=#@lbY8`l2ffbo^fTY9E4!kaNa_YAdRPq$VYOHni}bgV72Gm9TAY{DwL~7+hPGi7FPhJ(h{PXSOC66l3t*dsEUcB$)fUesGc;<E43f#$0db97u*AX~-dlDk<s++~_=+g|SSdHcaex$S@~=>n)gT}F-9X&o`8xQR`551B;&wqOnFgK;q`0|kRpMXyejEk1YGmYr(p>^~2IDNZ;dxy{`{<rnMq~aGWxT;n($!uzI)$#v=t<^t?aBptqr(s>K1~Gab-Gvdu%*=oMkPVUKPC#>L<s+9Nn5;8TObEUR_hvo+{&B2abwhn1{AE-FDU_dZ}e^0P#gC@f*E&QWNya6)7=H(;#J5Lh?8;CCCv05J)ZbI@$)6whK62X9B^<%o2}2Sx>r6)(HrO_-Wr;lBlm|bPhPHBYn1GyZX&HVx$%Tn-%UUGdfS#?MmyupxUh(`E^JaR@pX&fbkRCwCN#3!T9$Y7%+gNiCzD87pLzz^=6OQ}eZpR)*d>wSMsQhy8BY8Y@cWh#9@T>2;oldH4LJH5(&z2GaRy>NXwuE~ny{5#W15zXEy#ov@4FFfxIj%nt)`_>^BIe|7HE+;MaNh$jaG>~CNXAFXYWooyW|^D;=?XEAN-p)SI{$l#IRwQ$_t^KuEsRJ)xyvFmH{pq;6&WimoOXEw8jr6q0^43P5Q!BY=y*4f>8?L%Jw`q$F9KSG1Kntq2Vi-W861DVYsiQcOe;w1k43@Bub*P6%-3eF~Vt9_)ByMix|+f`=1xSVF`Qc)DajE_--6%^Zge)EtEZ6N#d9j`dd6#`kz5i5gW1M4aXw32;*W7LF=5m$?)8@<r`q*T8MtW+r5;0$e7Wm&4BpwJ3Y2{(M8D^(EzK!mCSk{Biz@LmBbMh^sGW7gf3v!ANTVg;|r3w8#Hd27?HpA?`6U)8-}gvp#jkokS`0-3e=5d0&?bi)tB1STq}t|4KG9}vI@?NxIKLkd<(f9G+)vClfE=15BNLOcAFB5#~)8Ejb8SPQ!BjTD68>sUnQ8Qm>!zg+-XJs#anv7yLy#z%V{qqPlWPJPun44rsNNU?xZSBPN)B^Y{84Lj3q~(jVg!%_aI(M*7^vKDkE)WBx!ML&UuRk>gze|Tj5nDqxd{el7u(Nh;KB~sJ#;%sT5d$T2!vkJFO5O%`T7SHGWM9lNRYd#F%oW{PgikjQK$cKk;k`LZ@dr1T_uDsIq{WspOST|4z}+t&H@?Q#FhUzb2b!EVIUX$6<j@x(U_<?I)e9F6{JFn7Bxtc0Ps-;k1<Y&qD}lYJP*-ei{S;3+Ur?5qXqzgIE43YXnNuOoisax`>Qw3QM5C5X!nQ(ANN8fK4~sb?;X^E^?jbcac^Bf~97M8k=TCK4{Nh@9w3WMt8BvOp5mfGA9CGe{c!amfb3$TT_8s0X~r{onxA$?FYJ6S7#*cae!m}p%}>*fSYFnkekP#6S2!4K9|;keiNVGe$NvR*EbJ5%_BYzfEI8`8?;#(&ii)DITFM@%PC1NNLxv5MyfZgi>A2TM~R}g(e<2S>NT4Zv@k?}@$b^JgZ>4mc8nNHB8~_L>|hT}Q%ZpmBTKqK7gp$y0zHaYki`lnl%}>E+O48<KCqEeFRj{KFPw`uq?*w8=KOKg<FK6q{L{wyORtUc(4Wy}L5&uY4sZrAvvL73hOi@bO>;g|=XH7!tt2?upZ(edJ}71zu?Pg=6<@;b(5qG_m-R8S!;#;zBE@b}Wc8B6N~wc}XQ0&MZ%ZqS-VCnbq5%!Vk>&rTCvY_F3XK6*>6df{-b+Siz=u6wTl`E7_kd6G5On9|WyT_Xk@Fe*eE|{}jqcj_{+eCH8vM17%gtCd(mF4a-f^$@B+dH?h!>2>7ZR-j*-JCx2-s7+i#&?3UwqX%zHckFf9H)ixdD-bjcG;wq{9f~WhRxJ;twAG677HMYMj)M-Ar9=91yN9@!(miHN{!;hd<~p2u=kk)FxCTED=$4+$qXn+mM86p8Y9+<(v}w%}>GRASjyc_oLS=FFx+5w1P}b>%gT&te<gzeRg}S%{D=#wU(3$te(WoumC8AyE`S3+uS^4O-Kb3xP(KnFrIUI9m*v;^&8~bLk>ZKxS(8rLrM7UimXPPEG$JIW@*xyg=M|wbd*l1{bW10AM?N<HZyYxa7c*qrFxxryDShb>HzkmhQH1Mz1e<-9^54P%>hLpSA2ik2Z93BE5%B-+wxS9ySUHsG#0L?s5fZluGtoggDUS~iF;wNC*f%t5Wk5e+5ibl29P*GE>SGjC~Uq_$?`?8yUKI30TQWSI>nIH1d%x~nIuaN?^*B&ob66D^C*p%Xb#N(g-I{JrAxt|po!m4Ay8@;e3Fj7dWT@MW4Q$`u(Ajm_SX9>EleyPnwfH8S3iYO8O{s&(UfFXmzF(5!^{Q825rWlto*|NO-2pQQz+OAyt&>Juj=5`T7RR-y_kS7AilYYo>&nrvl&LRK;(K_7A2(s7Oe)ftH%GAGpWkLBA}(H3_qOh)52AGEu&fOvHa?Fv4z2a_~W?{j&c$SCHR&qBKTzMa1_@~I!CfZ=Hy7YJ;`C;CCAeCy|Mw7HWr;1jhk-H%gG>WFlQSR-&@9gzaQ51{KDNEB6+jWmmZ!sg$P_I(@IEqrBT<P3NU~KqBmma7WM{D3a4_XMh%udCZhVEq+~C`wQ&GX0yHmBfM_jSJ9BUUN*qPk6?$4@@=`78+xS%s(Vk@D@kTJ8TPt^`@|^S!u8{s2uMy^F!^Fg50%=7p1}s-0)O0#1M*|__-G*k<HruPufHKLk48g#ALuJeP>2%rG!l*fm4~1x1=k@iJ|Gg=YZ;UoU{0?bnibj6;{Y<aH7o`~OSDc-==LCF5yKjF0<7J&$7g=kf@*EjQH0CIUO4OU%e{v9X(7xzR^zyemMO)*)&MfekxLy0%E}4#(Ab25Q{Ud8Bo(;Nv_o!-wn#KZ@+oAiCI_tJ3dJ0E6@W6I5*0M0&k=jw_oa|0IxKynOwfaZz2;f0rZ%a*_i7A`t8rrJ2#El<^Jvc!Vxylh~*s+!cqMJ#FyAV##E%#QZPhByoUiAQvV5~u*IKrOevfklZ?)OBi&F;vCM87~6bEG(mR1p@r{-9|meo$|9wdIq0hku-9XTsp&JuBle;lQBXU7->X-eWKZS4&h6{zX12`P30Hv1inEY{HSnu;vrTh*X?8z1?Wd4kxc1M%sF+ktd6c9wNXII$ty@XJav7;m)HY-VgIr3LF@~o^6v>EBG4bnjc`Vg}8k)dq4VVYdzL8bvm^pJy_bq4a41Df9$Q*TuAWKWnto2TXh-icC3+RqgoZ|1wulDIJ*DSMKjaF{G#-8NR4ncf`|vD3@=Ncg1*k<Hc%x8-`3HSq^Lff53_{yOeZvXg4|Ns0@8Iq(2DjWG5S~-yVJ@#aPPm+jm*kOD{J|{-1mECbJF}m_w(G4jruUACy?kz4qzlw!a1L%Q+k|l**y|}3LRJ?sMRYV-U*J)_K~cagpmfpI35Xga;@vg9v6?ziUYOwMzJ;6+%Y~dV_Z{ZkC!QuQ0phQ;V!h43>U`iEF4nhe8h+}-~>Y1Z4VX$z<hK7c3PPa=nrG(gFE}qS||r<WVWkItA07G@q$vwn6$+l0#1Ao%@#YxrIG8EJgpy*L+819?4&42&r2ri2coIWv{OPSdII3WP9%?H$;6f*uG}FkwDGW--`4O%UKcbE0-Zja!M%bvj~>%EI`WrGG96tt0)=w2JxYP_i<@98MeBy(Jw|t>1Dh&U2V@6=M@Y9)iTBcjA8WY*_!RS7jvM`VhOr9!coJw)`<H;nPwDnS3<7(`<LI=Yew@=VE}q_QD9cL%?`L~o87}1#gVTJj4rw0pcu&ZH*Gk;HL~w$+T2;uIlj)Y}qiJc@1$tx%n=&Kk$_cUVIuO+9=4xliz((0taP#5zm>+8Qp82CwYeaykXaKypC-n41f$0Kzq^qGW=>`j2d*_f9q988we&Fz$LWTZms0uD`d~}U;L-FCqGFuD;xxSbFp%?MqI`6D{Av4XcM9AGM!RLO&wM-9=iH^~rgy05R#6mvgW(bnduxTzQZNM3@mkO}ON7wb;g%6b;DA|-}MKApIZHOHvK^Tg~Qsq*u;#9Ijm6@p{Z4-16qPsV%jweM?w57j44PAhCq2EpKff|TJYn6U=d+O%Laeq7Yy_!b>Hzu1@W~=(gdU_hU_=IC8Jve?*RAOjtP$V0B4?FY^oguB%ga4~8&v_^o{F_u%-8Gy7z1425PN@au=waz@#6-QkzM8sW1AL@<C*JIAB?O7CqiOv~MYVu%kuM4-USYvZ4LHAv(q?n*y*12cud?R+zfsnppV_u+))Hjn!_ia}qfL&;>@;J8e6zG-1IHZu0=XnAW46JDF2&q0vsE_0{gITdq?XzXQ$v%i@>Y)ejuu06hNDf2e-aE|mNG-#5NE*ibV=KsjSYYA+J5EP^f1o7Ys*ZnA3NZMAIDeWdAaJgF>sKJOCvzbw3Uvli8j)-yq&Ssd!%`LTNxFSkly|m^qpBM4iVg&xog-WRr^F?U9D#p)H6NSIg=@rs7S#Z&ED&jDh-dYz|{;-9`%76-H=?@^A$}YcgpUMDZ@p3*;Zglsi^fpvKVQNGHEtisq8`Hk{1f|`ad*8uNCe)d+F*qhjWPB@7%+zd6e(s#an)uJFn<;g2{{6=#K?hS1<!L3a5M2Os-EaZhra|ZaGmZ>LNl;S!n`Y?SJjHflPR*T>>9SMa5eYoC17MBg-{d^+6vjrPR=_N(vqs4%P=zfvwHCKF1p!XpyJ4JYd;tb|aJTR&&zGLkzHv_5NLyPv&|Nh9;4b0jSL2BxXLY*@vy8u`d9e5t{<+7`zAnlWVjk5Kqb3cgbk+_aYB3YJF&tjWhdqGYa4TxKM(1<QECo=pT<$I_qZ_K==QWHkiyo<4U|?)VHXH7H^r1x495tMbC&LVI-T>nCF3cQA>o6z2=A)nH2jwy5m0=_-ZF-gJYA}ckE4Dw5sp?>UmL29pK@q9*F?E=Mw(ow*6l<B359(;2VyO0h`Lqr&OaG;|;Geicl2`YdR1CfSu5a9)cj6SK34H%&U`oX$|Of++Sr$q7%1x%aqVExGE_$S4uNXk`+ozp;CEYlCb)^D8O@+3`oL|`yr?2e*AT!*Tt!4AML@(9&{DW2X>XYQV|lV>yGc2NI+e61gSz@3+kY(cuM}z0d`WN!SU);VE>uv<A<<4@97F0@WpwNCc27dUPY-GxwO9pBh}BmN)f)R>|R0;*@R|Nwl0a?;Y+f^o2Zalmu{K}RT&fX+#~D}PRA^|phKKjI{A)ueRFsSZuYDn#80|!{(qluED_alq7bGrecvL=to1Kcu4~?8_FQNWRX{cInpHef)aeH%J9XBuF>+uA7x#z&q(aZ0DT-W`00ErA0+#Ry$fk5OvBYQl0ssI200dcD")           # (= NB4T's), dflash_seam.py (parallel agent)
    NB4T_LIB_SHA = "11e39802758b5e9472c9d341f8a909f4d170f23d4fb8e756ae2751c914c2365e"
    NB4T_LIB_SRC = _unx("{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;Q!o7xm^GeXbna~Q3)_)cIa3Kp?M;PJ(ZR*<5La|qw_8nPfyP3YG$7PpFQdSZl_Ml^LvHSz0R>wp_-LI1fh(4;d12YR;dgxrTN8U3zVcp?Z>nKj8wErrO1{EE#9W~;hKu)Z63@`Y4Qk=S<)fwpk4YSGVWVcQ}}21jgx{<S^;drckxG9rU$>zvCi<;Hh>MWwW5=F$^h}>@)VkT8_f{NqVW6DBmeNX=-qU^3^Qtib^F2V6`*u6TD@9KWCX95dcGjZ_zmT6s+zj-8Answnt;Xf{N%KRj02e!3^Yi|xhLm!@PnC5V$oB2;lzv2_DVn=SGL3))huV9?@i|c!xIA_3mokW(`6MMhPkGMe~O3Ct)tIb(g!iI)|eW66bRge2(-?yWKM{iMbWpb=#UPc@EmJHh>B2KrZ}hxC$74{s8>+M25P++BU6w<m?SEJbF&B`r>h=F8DTo;^NwfI`o+Fkyvg)Zw1RYuSyZ{EC$-X>-zC!SLW}!8=5eY97;VRf#v{?UNlMpY@9VHA4oW?<(`ZLO@`q)wKO`$o9@%nHtQUO;68;G7V1$$|3=hhi#*oho?UevIZO6G6;_u(srE2J{?tX*SO3Ru%+4q&EnDtb2#M11diD$ocdnTfk53{UWY`R|5dLR@5AXcD4Vk*DPjwU0kngXY}ckHIYy=mu?m1~FhzTZoS_&iI0<h4^`@XlNA{{5>?g~oe2eY$TAr>%~v!c6bvE0pcy{h7kb+&*dlLU^3~;_lCsJ)ftEx(Qn$q<#;<qStuB-wtEGboo2=lfq@(nzcbw?=Pb3S{<sj;082$ERLXj$j@X?<-Avg+3pxpO1sply!LTPw+wdb`!9^1WsGW-!<uSECHn==ag1_MBu&E*gw<RGtS4N1V|hW@9pO$efJeol0fB}e;6td#e=<D(jKhV*@&!v6=|L0)@uuo$B%}s<kLX%sJzlg&mq58%ps|EL_f9OFNlyBRK91u(gc3%v4kj_XS7d-VfcyKQft_~T1|yE3$*DoVJ!BNXl&7rl4|Tab$7ReH{z}rds^~6f;&WU*w5cM*<xfosPRuDb`_KTob)F#2wmX?_Awo3=0f;G`z9~(~;7eK(M`@!wGWDkQds#|*hZ!=pLV&Ycc;?EN3&27}eBu*2=?&uXYUI3qqnqNfeJvvmcR3NgNqD(4msR-=XZVaTx2++p2re4}kFiBu`iofAre^R#`J;!G>C>Kz3$HzKWzYXAMA6y<6T<K<_Trk~Kq9uvNWy(3liJ|DNVlS}*IapVN>WjO0B%s<7q8JielS?~WNR<WsEuj+?4+>SB28flppT4MyC}$s<m<F|ybO`+B-lY8E~#tv4+^rmq%ZXIq(M-IpcWazv*T*RJ#5cX1#C)v%1xnqX!&Tbo%?RT9jBf|4p-;8a0mh_r=OXY0VzwI4|lwo!&o+*7-!^aGOFB34Xw9VKoTJ-*@}z0bPcZ-TJ|0F5-I+$d4n-0!iaWh65<aC*!9F_y&G2Cy^;W&8A09K>y=YTi@V){`)wd9S6|d=G1lp@+U#WbyH=|PvPRIDc#lRSZ_I>_vIYN(pd5@^-O!GgK<2<g&NRJO`OvH->7ck;V4Y4E80=QF%Ck{*mB%b{c<qlotS-Dx2#=}`=x(&rUGHJ5Y;uBL+Cn8P%d56lqeKX#aKR#fV8R;jWl5ox=(YG!lOVB)+!}E>{!9?q>z}-*2D2`Q8$0Jk#ohyxig@ME)Gh{TkjNHKZc^2x0YZRxoBvatS&aTe^vKG^0HJ8XNJ9LKz}?H<s1O54T?ee$dF|tANT{?Gx4D1kN~(Pjr@_QU^x=yK=Lob@_xA<7wv&idBW!)q^IW9yB#PUMYI=EIWFLAAdm{(Yf}i)mAMIry+0M}K7FDg+no6FFA9gUab@m3L?ra_lt+x3iGdLf0InC{E`txo+_nai)2MAVA&N(8{cg*f*X3(n`@TZhluSsY<g=scJqTtzq#o43xZGB30cY#z&T(7?@EUX$5Hm$j4X2~bT2Q~p$Fyc$|mMkR=kku_wIlQiT1;@yiPnHiKO%gFd+h8fe*w&Y<e3r2bU}z%a{%5zG;!lD@97g!+&B$(v3@eSmDl0B3O~mx8j^PI*RO)5<$%kO@;dkd^r!jqwvn6Htul*R}JwY4$i1ndJ@<xLgPt{z^njzA=w4!sSCd`phJR@}C`tf}260l=wFiTeBI}w(J#>R2rBoL!Tskq+%A9WxVMHa1`s!{ID8Y?LXGyl6~&|X%oB3=Zxnx^vxL7ns{!$|_$ZSZdbZD&kJ-js1mw!exY9|lGXLGt_@=&}!q=iz-Ah9uy}OalE-hO5o~hyDX>bpxg5FL9a0IJ%+o`OBu<l()r2xmBEy=Mx>-xUqKxUmg0?v`!11-b2oU<ex{rD-kZ1u&^g$OWE@#smu+k9RW@NZmnueT4tu14^V!^lnawdwOLH!6|K?;l06;J5|Lt5?tEo57LKYD0tDXE4E&RbRw%`vQG|D@fe4Js@|fkaN3Q=Mdukc(1b2^_V%XQrfbv*EVW)m!q*hnC5O`?$d9Hswcxvj(R+RP6i~<1*fmctpO(c7wx-V~B80;e~DG>N2v%0i*yQ|AVW{cv(;rjxr>p%CrJifl#1!rNyH7%3X64b{<)<00!s%FIp*1X-LuiZe1=5h+%4A&F&w9EEP--BSIfz^Ul@|T!R54CM?-DmyA{=!hLgk%$Ak%OYu-F(F%7AA)I>W-h(0Ak&Jz@KDv_QGN<(V0OP78`Q3mvU&`GX5b4%K{#ptn0<;qFZeW@(5a*&!LmjPcI%u&AoE_%y42dDP(<Nz>FPBhJi${8>Ig@p7osUxzNuA;fYk;KHyM~fP4yjp90LqY^Y>U^5X$u)sB@|LPza3?i`bzJ?~JpZ7-74fye#x<C(rirrKMZ8IJT0$ntX#<kh)1@qT89#0aaBrKF5gEhB^!Tya==WD8yn=B6F6K$xxJ@bgD--JR|joqdr;%VZ}7t)AyJ-HO|t=FO~S&ny`7RHe7P*K8ep2e!rU7c8idP-fsZ@MCP^+ZioldKud~G1Ofm3{3Od;{NZxfDv;_*A73}cEQOoN7O|T)eR79zkKOiQi~r@z-?$p%xo4FPk536xfFwfR#XG}he8AS`ztTzRqg>2K%;(?BYeC;5T<=t%zyc<ls3j1Zy-15uMD~QFJfI!e#pNwmh2A(Cf=x!ub->Hr*LhMcuSRT`lprm^%0-2R{f`efUR@hetQsljIm!Gp06)!g)x3-MJ)Av^nLc;lFEPBid|{`=OW#=Qvg=b`Rav=z3%EhZ9MRwA=yY}oImvtWa31S2;lsJ`V5%B-UP=)I3k+<7UKd#VQ&&D=XP<XcXBCZ^{+1A62GV(%@-hXP`Ef+nvj2m^l!KLO!c&}s!Z3t-1kq(?GpU6t1D}6G;TV)p>7gdT;I{7&%eTtv%X>>ddm$NB**H%9rl#ggcVLVaReBR)@k*F0<Fd>0hE-RH&Nh%Y%M3`h1w7jm&;yfL!g4lFgoj<le4XXX$`b9f$7lL5H(BHceHgwojroh25Qw&ZM$&crsB@D5Oq&QoP1{{A{+ELFRUjpIDw*|5Am=yStoT8WLz!Kjn!JnC#DE^PiEmKe8i~uk$vwyj-rv5+MJR}>la1!Z6L)rx0`Y+xq2%96!tG~QjOl9vNY&TGz~}-xM@oQCtCWlgu1~R7Dh8oo<aN21!M&DI-NojM5CM~ZM?oy1iQSJBwfP^s2v1&=(AQ+h<C(vB=6>hOaO6n%mp-zb;5D5m^imoLcbZQnbfrHNsiPml~wRMQNNR$O6aBP$2<orU@}9^|GzbFSb$GUEbXWi!2rc;i3d@qYd5D0tok0K@eo7TB9zq{KM{0pxW0HHY=4|%LBu&kIJtedc1f%2(PT82p`=&3Ov-K6CYsr+fIK`|XPGs6YrQ+n1N{tU-+Q#|Za{}fdeLasO0mvcZa}Ci)Nlk%J(13jw5{5FB7;;WupJ+_W`WTo{s?yma=>6-=vZ^WPxeBBU8wx7f*TdS6viB!U=!OMJ=2gzm1WPd4ZFBMt#Jv21Mcu-ZScN#SFKj0@pq?cz))l-h=A--cKl2b!7Y6<^(sP(#tdfSejB&sBF+>6(Mr|$IfVB*;U?HHg@}@`zcV9qRo4BI)}8^bt~g65&RQk5=JWLLf#cJ2H+>7+S;TxBf1*vkn0PKKF26MG2WBYyF{(V&<@fHnFGxncNy$7Xs(xog;5lNU$n2s}#Io<ND20QSl|~-6R{pR~dqkZL85E*)dV8P^x4v~ufEBqQI8N>%y?A7OufAy<k#@Hhz%7NCKH2a=soT(oNkF+(kObrA{E_ij;0g)IG?^i0>h|k%{6jABG*f$WTd`Kyj6cD2bX7+fM>mxjJ(4ZR@aPZD4F`=#4=7+ozH-Rld3+E4-39M8z))$H!daJ+6^3Z4T4`~Cd%jZC{U`p&R`PfTG3cZ=;sjpYdu&a#9s_5{1Fxk_r=<9g%kNd3bw__@`o$y!S#dKhqT1V7@5ZyRR@9q(53th@dLH}IjKk7t4(CwvuXT%k?k0UA`-p-Fwu{rX?C8gWF}DVQE&<9zr}<F>TPb2`hwFPq;7z`_GHqY)GkTb)IF2D(r*o=5GJIrvS6ys5M{ZfoVQv+wg~2n%`ee@xC=QfTo@__KBg7dCmU*K>r*uC8v18v<V`d06cPlwQJ2k=3jVoYib5tQk-mw&<mhVD4VmhnveQ%-&I7rmaWW*>xJ@_<3e*|h7rW}wAII}`8A|SX+EV)ISjvPYc)vgFch?uCkx}1&gW>gO{z$qsX@?0Eem_?Rwl~ewG3Lwq*i<TAV4K(z2<$4Fk&IqZM1w%Sg28ktN6FJ*6vHUg5maN?g8MGKTRMgm-KF2R7?$|h+!V^<LKDO&w4ATthr>?fI$r5J|QiYQirsx+l21`m4_|E#P7tnQ)7ydfw$6b{MyB_T-)zP5{gdJrMj(`wZ8>KQjCt}8er6YzN9jh|emQ>w<0E{FKTs1t%m}mnoMxX9Jpf$?nzN_;tTMpqu+PR`Y?YO?%CrL2YFTzBy$wnU^vch-(_R!wb`R8!-=X4cIn7y?W<7s5QsH9p=s;;>wYtXC;!1J6`hc65!27{pp?I<-*-Mww^4xmYvin0Z3aCfr5RfI!TNkj|-2;+sZpDx5WsS$r75LJF588++<4xWj79hmLneQog)AP?~STgnpGAc}4o(>x_W6xs?TcRg>(HCNmz0?(|D8ovWhaJEgRJ_mY_<W*-X>D7a-7E;Cnlc5JPcL)1|(f8b6fix_=51g>czHp7c5o^$2JS(yji9)9r_J_%QhWzwA>kriCMVhLtbr^;8DdE=J8e5I0sKY0-<Gn#(c{=hY<$}@6%L@k)+VIa93Nf5wVg%G-igkHifT+@pYKbXatZ!&ZRflYsxxb=Cb1sf4d^4x7BB|gT{5N#a*!fk(m1Xl|#2uG`X#{6#qY&m$XrZP)<^!+l{1=1<b!Kb|NcPNHNk>Zn)$$7{<(qx~d;o|GZ|*sGVeYJot%SZYzKSL#gECtWL^e^?fw<ukrcMeEb1P<9htC5M=M4+7IW`7n@Rqmv=t|j$36$s4s#{v|IWZMGn0F@R!wAT58gL=%Pf|htR)(?vrAx8SJl}fI4M9Z0oohQR=|;{H-P3&2nU6|@ftxxI=T9iOdV!smP&`H+U-V4GMqz(b{Zu*cEhNfUa$!i|!=lJzEiYKXbAmz!gSjq8xr#jPe{%zJwa7`V_&;6WSCf>$36Onddy^2dCQig(VcoL%DX>r#bAp$C1l>|T{5<ZGO_{83is9Y#Ly*siA<(BxVk7>?mG)6%3x!{zrR>+$ieA2h1$zmDoivE%W@yGh_4v>te#4)KnEj;0uvw@&+LPF13x>^dD4<`!(Jx&g|Km&Q=tObLahzpH;6h!~M0Z1LMPd+-P<{$jza@_G>-bd{z|l5PCtY0wc#$VWv|>i@MMh@}VOY-)MZ8Od96bD__JL>y-lxqQra5-|Co^DUO+pdAyr@Qchl42GSLfmXQU!IXL3!7Vz#g#V5w0C#5$<9h?O{_~f?>3GK(8a$gW}~2T>*t~8P3V4iYHtgeXuXgG?baucf>oYOgs#0Rr57&!7UZ-Zz+;i+m^_uzpFwF>@e+ICv#Yh!7zuDH_;WYXk<m4D<rov`;mk)<b8RQHu1%ej110MDYx91Sz$L#y*pg~9fD}H2>u+iJZAK0B<sD10$KfVFYi`${r+GUY`A9x)8o<vyO0$)TIN9Kmq|M-Py)o5$wf#NE7be-@akQBeS<0%2O<)G8iSrW1Zd;IRAkM$FCT9YY{dVMdIdIBffbwL0_ZupZ=e$41mQ%_jcmD)0x+B@npYI^H?}PqQ2FoP^e3IcZ)G=j*fmHgp6PB+v5bwJH1zr@)(pi$pLg^k5)tRrC2EC!_v2&_tz46_qZThpuB;4pg(iQz_eEz-f`SD;8SgY?ooE)0qb<FnG8giS`-r>|;gK5+TcMrudG4y3#iD1=Wxxcqa-B<^4XIuZPbmkc$wUYbBn2uJ;&SAeJz#SWR`e*K<9-*zQEXQXDz^BpY+_y|eahGm=Qr<(x2@g79>yE-(9T8X!WT*<TN%R<-YGrSMhl?_p#e$X46PIzumk2m48rNY3x>m=P~%6=JaEfW7E`GnQOsUM405K`d3+Rm;k$|}AshaZQhQ}GTXMjk)&EXtnDaPZ7QM)*)|>8$u0f!^I{~+)#_^xwSV!Dtncz%h`@f42r1}}P2>qEEEhAX1xu$OVt&1cxjl=W6_ypAZCagzuf#~5>iicVCDN=l8B)oc-ogDMjCyrFGZ}Ubo0+~N+4JxFyC&;!9`2Neh@eV*rrO-$|e@Kkye1t^$ZL_T^GGC5v?ix0R0h4tuu!%x2PVEQ(b%GizL)B5X?d`v;QxS}hLtoCku;2Kv*Hk-3!MoTz=9ZqaN8pKDVJTY!Wg|`ffI8@f!EUX5^7<nHUq~jx&5h%PbR|=TG>0-S;@`m3D)yKmfW2)bX*O=^T9;Wu3|N~WMU+I_s6f$8qxni7v!1L_(%wT!YUBXrhz+yK_7kfXvWydm>`+N?DM8?QN0^wMxr;ygkwjnHA6XJj>X9k~UYI&k1IiH>U&`+V`fDEv?-4?dYCb4sOz!lUTe6d(=-uy$UUOJO@*9pNw$O}w-LbuN<Q)uxlavba7-MfO+i^q&%|1uX`?H6@Ye=;oQGyg6J5G!C*-=?6K(Nz<O3G{w;mwhu!~jrg2NrlPt!4R}TYa(%DIcO;f>iTJj$w4k0OgdKzYT)C0}hWe6<^*r3I}|nx<){LY9ykD66ibxN$uZVe2j~D25k@-JO?rwF7~ICBKK|vM=X0Y!oz+{_-7of3SvJ4n=?;fVfy>zux0aGk06ldqAJBPS@^P>2cmgH*|R%q?!F0&9fqKrhQq+kTd~iE8K&lo4C|@AQ>zjW58;=r{11=eWNtTQjzk$rPuV$c%iOUQ91#*?J7IoBZ6CBj4;!&jYU1L9_XI~Uh^nCN%0#35W0&+5tFMJRR6tW!>2hOV8Unp8LR-q!W{8X6L$(-0vr%@XE^_KVqOx{Q64yR~)GEjUM*^+N7b+pW@?dK*Ra5haF`ZToe7`;_T@FZtUe}Sn%e4MSyFTs_p%$HWxYG!Qg5z_tQD)HsN|FQh1DgJgT-y2qSW(7qh#=Zvx=8=bW!sr+;Kh-iT0*ot&!DIr+6Zjr-nzGTn|SjNb611t5peo)F*YU*2TV)Zq~Q=WZ@RmG?SIX)A^&)I^AiB+9%eN@8Oc*!b-0C=G=EE3DgPlNiNzaT^F}#{^3AUPY*Rjwbe0RcRs)N6>9v#&A>$w`sV8$u0cO5z`fDpu)G2QQ>GwJn&QM0z@c3g}t`Iq7N+|)>8HML-oD@!C7<Edc_da%MI^g8~-!KEUIelOOQwt}?{Sb}NCR26;XvQ+T)R~?B1YzZ)WnKX4%QvIp`B>W=(z)+Xg36vYRh*z#IlVk!`ZN(Xr~yGB$cOK2SK-xA2*kHI=pKFqnNPa3e9Pq8g?brc7>yD2J{t)ni*Z!hbc`~0MF#Ii4wz`NoZ+4Z6NLiBPU-GX>a+=a&^A)kbiYpTYzPVu;aGx`QvU#X8VQlFaXhUkHzAe})9mcbiM`W+-4V~Za1xuF>*FRxP?vXX0xIaXoNT%6mb2Jn_!uLa28<)ek?{9`Eema)N^y5J_+a6M#Zlw&_unM0`Hrd4qTflk_|eRKY4_)ly{vr<^H-oae#It*{~VI|Un*i%<$_$YCLAZ*>9Vdc9(>ZjAei^NvwJ4(i*VPbqX0>30~$^BUB!GoSrz0AO>^lsP|Jg3fhG24j{j-1pcgec-B{0uXo78n3WFj<LNZE7LoeTBU$gyb)mxmsS$OW9P82};{2SZyOyRIph)VcCO;s;VP0-L~I|<E1D8Xji<&8;M?i-i!ym1PFqx;33@v^ADI{|C#encraX6D$aRB8+lOh}{@D?abc!YA{Y1m+2Hp!%l|*OnCOsAM&ozL&r5eQO4I9G1Mxn{a>%uak}SS77jo=z)osCI<N+$VF;qX3pNYJ<iIO{DboTF3vS=n~NAed91hR^YnSdBQ#9ygADuBzeDPPZW^qA$EU&}XYrO464A~1)IxF-{m%B2Y0kG9(SI(WRe<wkQ_Vq+%X9Y6aGjol84$7K?(5W>mZNx*uEnkuZNoAw%~pLLfPO1^orIY9gcg_`XcM+H*tE7ZkLho{OT!ThOUW90xI@ro{P6|P-q!I5TrXX|r|Yb|n`0^$<13i5{&Ef<l$XZa2=AMymVziPzpEQjjN=8f2>LTymz3GAezw!*CIt_!;-6iRxRx$NizjN<7NeMuFLD2$GWF1AY*Vr0%?u1$HG}l~u@xLjufM3x9CPNZI2CTT4-d+U(w69v0uTWz*_HL2`bC`E%Tr6>WvS<$BP_1jib{iwxYmMc1+;^xtAna@iP3!COVj*B_+e*IzULL%DzXr2k*3%>aM8h6a*QJ0)Yzy)h^#t4nT}M{iY3?&0>e)WBY*V60K#*7&1iC~3IIl<;;?glhT3xSgEYGIN+l0ihGAld|4gyx;t<L6h1cP+8}wT)akk0zsC0A;6x-oR=Te?JPYk`O**e{OIvz-;0%5;2YMKDX2KA0c)HHMzawgqx5tVVR%mt7%u!%eomtPzv`^9Ac{_B*+a<@&P?Q^BSUB0&k5eU=#U!3Q)&nko~HvDkQGU|ZQU|kQ=nGJF{^Pvu4`YmAU!Ppga;UPak$JBY-k!8$xIBi=;JJ=h^OiYaLH?@Ma`rsbnfI}I=&OI5R1y)Luc*cd!^egHUF0Ra1ftWYNI4Zuy%`})^H_A83DZ~MCX?h4-lOV;0sh+4Q@4A{m|9@q*+aI=FNz)>|XouU*YV&2IU$v|syaj<0JFva=5ylXC0_;;xWk0MU>u?n#)OpiAIRSH@dr`-FVzj~LbSnQBn3ynck_x|-n9b2g!`9;|(7}^e0;F=2wP#X=;<^#0-3f_Eavd`gjcCdO<Wj^Gr1-Wks}rUV`gCDGeOSH-5~JfYgK(+ngYs!k6(!l+Itd4rLM%(%5RU@_^dF+ZhCozul#;A>m+jLa3Ui*3Zle}v2G41OtVDt7#4{B2Y5o6};XDjN-6VNkp?@wH*<yFk?t77G3^PqMm<`FJ9W>Sa{8LefE<0{RL`^RLu-hPN>+KeF7y*VqdWvq}q_|q2b4TtU!Xvw}oYT#2`ya}(RC)l6BBOJ0{##$r@B<!jxJRNcju4#H!NVlt;o$NJS-`D#bPy2|p68!<_`HX+cnK;MBmA%*W0yRWLVB)tv*ZIzHL0V`#%i5tFYX6Pw~fp9B9^aafABh7_(R6A5E5$c@R!e3$>t%za{3udt(DcXuzltI4JQHnC2G6gu$S6P(N98i2zvu`0%?u%UjuM=fLN|FP4t#pR%9ftqAj%s#7~gc`f15U*nKrov>41c0<08;0ak|=!ixoW3a8U5p=cb<O42eGN7i!A;>x1IGS+qtBf|w=Yw|k_qk^97Kh+sVBfj(|nI|m-^2j6=|EPe1r^%VQzsGIS5w9)36dh~n46L+Pag2b1d74J1ndRHsr7QV?V1Uv5@v|mrBeg6yP2D%zngKyQ;wxb(HRPIRhzR>^FNV4LvAs_w91dr^i6eUgwBLLCkPR_dhNQUqkEFz#p{24i1JoyX@I=rnGjuygRueM>+W)Ra{w6uG-JeWXtw?jbpvgva)&w}jIKLQTnV<ArnT2F#(i5Z4(A;wD!lZ*ol0nw$EB<>truEks6`s{j_E+($Vhe3N5jVWn0EwbxcaIhcA;pdhKH?m%B&oan2l$4-!!*RC+b~d(YOj@rCDKm*1|U;qb}#x$yiIeJwQ*ZCm>c_e?V8;Nl}R5?gdI~c6huUx(@O)ZTapOj;v8J65cYbVHJT_(C*IUjM=`gHXuUar{wNp3OL%0x8>R!i){V#MZjXSp*M^&uk@z%*jXUXQ4>Q>qXB_*qucoCm%u>@yi&fIYlI070`q-nBSykG$6q0ynWpuTEb`qNhG#8WL4eyiy(cRnQkX=b4`?4sOl{Xm%RD9^<1E~XO=5v3r07QNOxqtF+ngFxW9taemmyE^AutUK=)Dl&Lz=eTq?p}{3qRyKqnMoloS<9693}N_jy6%g65DE6YKA&a*^#pV?soO;fW+X;bAH9b_7&jF&L~9bR{Che@oA@ncz?I$EO*(=WsKDy5SzfB`fDKbyj?Yy=ADOFXr?!%@aGaUhWN>QtQ4aw9-nqTrGhYZ;jU0kfC2`d7asKVn6+l3^lcNu`M;^=IE0f!$crh9Z<s2Q+Y@sKzCt}Q-Mn;qKBkaXOm+bOlVuWPD-=YIIKv&yBA3|Vbfnsy;vytH@OzCtP!!!hW>a>KkZBn40Diuz_ZzbNYExhjQR_%o;YdM{kj%4CC<@2VF6CMEK&h_tnq%e9}nYJ0o{JxUyVzsaQbV?vJn%jvbR^fPxQPS+#U@n(lTCSpM<>s!+Q8YwxJ~FdBQAI8vE*|QJY#qoynH!y3*$hPZFP^;ea*;w9GRbxsAj=N?3Dm9|STTn+GdLLRO^czLBqJqHHv1~4;4EnLHz{MtSH@!D_>R(C;+w5NVz9xaN1q>!o$x=oOhnSeOrif2i_RRe)<s%Ef}Ya>C96c0w?+wM9W$FX1Wqx=kXNE6JD)!FKD^~z?cCK!<@j){xx4o6B55)zb+8t!Kn?M%|7ZEF&EH|Ln!aBxp81qq>`<d3?VB|8Z-Rc9+xNSE9F4{n*n+XbNRwN{6hnrKYZ`Nn-VN<CY5anE(pt>xSwOIde6gvedf`@%xQu#CLeamoO4cM#fhZl^8jmri6eBEHn$Josi}S6uuCOtBp?Z={g0IfND!mJt21YYlVz;~~jC3eB_HNfq71z^+0jSwbf=~DFP`;Dn6`46>XjVCzZv|c&=3#v6WW>t@eihqcx6I>(vW;gHK>$WYg?D;6_kWe+mP*)<2sulT3XU=^KAtj;!~J$T;n)$p77`Kz*M}M9Qbkkg8x_Xc!*1uez9C9olw$iKM;1`TUdRa)#T$1a?WxWGjF@&cKbMjmaknqH@1~$7>LE7e&4_{T(ZyY%Uh_Ox=)LH6GZt0w^#qY5OhdcTOw4ckMQT15om$rp*NV5?N4|qw6FAQ>M2_#LBL{h1+_K+W9|W1pXZ4YWhnB)wgIKCy{k|5+mkpU7^e^5KaT9yT6Or_gZVCC2g%gMoL3=ZqK#UKIifm}#KhJMkT(@_3nfENV{1(M9-;bq2_8Ec8VzZ%2H2erz#mqHIF_nT|7|ab}6$sgKvS*!gRv52rHyKRK1FOm;Tn$O$pvf!c@|LE1q}c}`GC&=cX#=*YaL|~VAC><P>7bIjfi?i=r0DCR>J{2+-G@*>2mT%3G8XpEQ2;YQWI*_}hAQM0&X}Quguji%yY7v=M0`T&Ec^@D<Hq&uH3Af@%*z@W<67iZ{)n4MFngu2U5+-oM-J?RB|lAR*0;4?yGaHqY$EJdBL8TrytV1EW3HwU#|3TbgYQu{V|jSbNgyG>d~7GIR=BRf4X*B`TG%F!w%aLt?;(?B7Ee6U4+pslpm0Nf4?%@<FFqRm9vkU2x8sk#3<RnTQzOko;LZ?uC~|CTYWy-y!}nFjJE&hB;xIrVDS~yc1H1f|3AaOA9dD-IX3Q_V4)PJW1vE*9mx|+S^c5g2Hl#i2zH(@V^~p>aCFPbRD6P1hgdE~zf<Ya%z(^Q5bk53E?E^>~L5lXePK%j0AfP=_2qGx<!1QyhH<BdGoDqw$`D186T-+;bZklZ1@;Rq<Y{}Oey&r)2+0RK%z+|}fv5bi_W&9ZB%ApsA3G#}oujtYjiL-z*%Joqbn;}$6mGCw?QUmWvgNh+JPzxL~?=m;f-+2v4kQ^($tE^QRBa=a+COmn{u<~(|VM^7J@SPW;vj~DutaPV1=d>+jAWD`3R)l^^Qiuw>X}Dv>cwNWPPYyC_c29sfvGzsxv^$S2pA{CwX7zxUTY*!SFS^^^bAjO$@bA-NP7rDDpDj?P?3FUHbLy{3nj`T{)xZK~3xoa_)4GDAWC8vTmg>oJ@M2H!@Y^=c-TM7dG2V19jOyBaWmIY%$_5)m3Rhc^z0&Uspw=E0PL{lIzZP=2YQ3)Yqr?tQ)U(gRzg8{lLUWa!sw~;h6%-fq2uI@UnXSvi_0F_Ky9|71S|LlF?^ZFAjK(cjA%GXMT0Ra(p_v&#O!pv)+d|<vkf4!Cew5-c+Y98N2uFo8ZL*!_B>B74OHcWeUg%pO#z&%DX%7RkC=r6pGU0?ywGh8BU<kpYiBKcLsKq|j<8gRbxSJlEwQay!>|1o>0n%n--bH#h)cx@SnH5kzZ@RQvEjABq$CZu<i#J=7TXRNK<m`F+j!1Ts+|&9HId<cg<Mw#i0i5@1fbGzG*1mvrzFt57)yA(CmAO$pBN`{#&LlZ|q>RpGd2`TbSL2&a%{nxd4pHq&65vRY&VSut92DketfF=ZNalnYJ9o7Wd_zmN$7Lo|QTi_WHwKqclVUv2;DgR(Ms#$u6m`wV8AcXC)JO10^YoRz<ki<HOkQoBwwN(I$zQ@uP~PY!Pu4|`MK0d;J9>uO_-k{lUnc_by{Lkg2{kgx;{Sa`mo)CgD^jA&FIrmA@>HVjZoN3xP?z`&PX8sZ8y=Yy?o!tVm4tny86$<4gAKVk_f|>YoCb5tx&6(7c<g?-cIOOPzs{z9G%pp;+$|h1!d}f(-I8g28MC}1!jEBrFj5c9+E0P)bR68Y4>d7UMgtx7pqr_d%>oj-bfxY9hmN;aMtftx;}|VgPZQ)m{>xJ~|ERyNvLG7P*QRD(aaFO=dV@9yqi++!Oq?t0s-*o=Pj9flWVMD3I8%FBsP+=ey0iiSxis;9+$W*Js#s}MokR+t$pEjITYzK806Bb|7`S$Y1ZY;A|GGGKR_6!bf}W_E-gT{pE1n)d#F+>-0%7b(s8kO12k!i<^ju_RaAPylpGCuWduX$UhO>`8JdKrl9bnVP{82Ue7EvnpBs}J!KajCsivOo3!T(Q8BW}gK%-*|h7!#?!Y`V}j7NlayXuQCQ-|`P$U-`_IXTax{)577o+xLz%xYqjnJNkIEOFFI&*p-?M6X-wunED4$I&2g!yyY$l(S&)|9cAXYRP#nZ)U6&>%@{_ssDU!d3O_BUX%OwTG?S^S>&<_=nL~7@;-zi~O^w#Iz?zKZOpg>`<g3?^i=TSpoy_u*A_XkdmWvxmFKXA>seIhRPpSNAtb!sxn>BhuvBe)#6;=_)W8gd2v_1&Fjr5X2;`d&F9m2_GY`cp_X3^Uz{eEflBAP$(`(d7e*fEAU+}RrCYP&?*&rFBu{iJ_zu?n7&`&Z-J&7uIgU_1hHoaG`1=_(+fue!dxbRK0pw1**L=SrxU_aQl(MNyBi%vCHCp~=%3E<N-MFFj!xhBNRI+09aWJLvkTc%YjHF<`C<hFpAPs(CU0oUbX&d<DlW*@hW*uA*N*#Rn)*iS{jvdjs0oF#XUjW`DrO-n~*o<D_E|bSzZo`?edA!H+9a(@^15=>C4*j1t!+M1bi4bJ;FvK5PcA8wO=Ow&hTh+wLBv-!=PFcc?_rAW!I(LIIOD@DVY3Kb!TOYpJJlY=C9{`04Bmg5y$to@2(LKJdGx;ezr~k60D`0IpR$>NWTuc>G9Q??%Qgha~y{*d0gs_vm^3D&WT@CwD$~J(agAxS-BMQfoUt);xfI<Fz&%oCefg7d|f32p^>!^cI}$O;o!yowA}8vWL^d=fC$Yz%tf80xrg!Me?aT?y4l|I3xFmc_661X)<=m>=@JCbY+ISnsM2zZI(}#i*-%sD|;dlr(8YSsZyqZ*Zy`{n^W?U%4DnpJcj@-*kjjdw}Y?*sgs6;sZQZhK#0#beV;~g`vd)U3<49Gr6QupJ}Fv6cFjs=EwFL)F)cN}Tb74Mt9?w7|Euiy*8;#mvlaBSBB$IG`(h=m^8%33^kaa-E1R2<XKiZxrDKof%o;6+(@3u{Z^BNz|31&#5X1(v+pBTK+V4qQTc2sOz0rZZ#(Zg64RF)k%1LcThluGgq)1RiBG+fn8f4;sLI0V`7Bje_rnNeXuCPOfN7mdGwp1~e|4ja4yp_1rD}HRoaleRQob;E`OTHoE+58eayf*U`_g|XYQL^8n@lp6jmhV<Ypp6?Ya&xL0rC=SdGUdl~^}jX$+^Nhl<@Zfk*uQlTFNfr#N-JlzAKIf6dJCP^v`gP-N2yMP0yztS<po;u+i_S|fc<EO(FKf*^nxGr!qM6i)j6p>pP25v5)R@XqE^=!e8$@rN9o_lB|GB`6f8m>JAslOM^tSSRhGIAl^E_`Wt-Q}#B#&&YAIQsLeQJ*-%{|9`%(YhZt&Dc8e%)Jm;ABiWlK4nGR<j1D@;p974`augT5MVSR0-%S1iioorV5+1u~;qtyzarHYmZqw0(7dY1gGwS}qIcQ`!_#G*i8EPtyU=tP4jL85kQprZEhgV)^iaCTOQGy9B3AT4f1$O5c3}Vs>Cg?`JiBp%Mj|7e5Ouj8wCV3)s&m2-eUO#~!Hn9sbNR)k;MVzMuZ;Nq^ax@jB~S)FisjQa)jFMa47KbA7_IX~U4gin$cu_LCsis&Ld6wMF`qX%fBPI0QI&+T(d6tilb{cmAma!FJ1t0vi<~DswA_GvZ0W3}<KLKJR|*qM$on6)BTN;x7JdzKom*O>Kzm<@hhTZ9?<+e1$ZUrB$W309>oVjzbG;h?9j2X#?XPA-`xDJA4B`Dvi@{!Y^zg6^PoySe}KyNdNh-$L#oP;s9^boLxdDUXDfK`(TOsXN-KW)OG20sJ2e`a$#VLLV2wqVn6yG2up>Dpw7)D`D0r+fF$d0u2TO_qX*XMxlKrWiq5OT``g126Q=FptC%>z#2Q3dh`QI$ooqc$g8_zdnidbA(@<|occd5>jLHb8C1iFHH3C0vN!7~-?~T7TGV^d;08~%%_YZEc6aEVbrk^Fxe8+a?QQ8p4UXcXxdmOc;(XYTJ0QH;8B|8%=;{B8`dAXT(yDKd67O{l+RCDYM<Cvk-DZaRm4dAuASpLWNZ3&q|`wBqD%9xSn|41!}L@4jVt&KYES;X{%E1GjVeZdgdCRVJTrhR1hs4$5$=L(VGa_JySbM<wuk0aw>qw6)V%bj|~rc1+ZAVx}nyEOjDVzr3RGBMYE>b&@U!bLS1VTsAQw0`-fI5AS;@6BlZ?gY?8g(dr~Fc-4w=Ee`7a4U9F+hDp<1-f6IskoMdju*zr2Q@j&+Li3%i{K6hV+7tf8asK0k$Xa9mOV-}QRa#WcoHHLH?X`;+QO}Z4}O`sCOE>|c+nyY6}Y?Kyl5~?hoHO?PMbtrP)A+V#p(dSk&$y|5;#l|*^!DZC5>@2%$8IN=Zl2)7<*DTA4W17@%81x$YEvYbzDru>Lo|HG6LY~RGEZ+&J}p9NTU2ONio(K@qiYo*FV`JzUY)Fw#A9Ym70u-e@n>Pw7(_qT^9dzyF6Oa`vPSNMG9tRC(}3)k);K;jP0XVRn}Q~FB7lxdcFxP-&tp`<g{b+&NV*Pe~Z5yD~_RdH{_&s@>rQt7266Rxekq}yFlxsB9}=^ZFoN=!igTbmCCS7MNO`;mf&E8Bc>q_Y6uf(%kpzJ&b@%&jzH4Zjnp_9VmF0|LX_H>FhrOuT8WvRwihu*3TLP)@VBspOA;A%_y=hc?5N>zeXj{zARt6+1;5IsofsF2UTX9X^S@=MW(kIsBC2&wQL@3ySt?1c<6Udr6YKc^41JJWznOezV#AC7KeIoB;Oe=U)jd=ju6)|tJipiLlAkPG%cLcA7CRoL)TE#BLqO^{KB**dQwalHjFP`2uUT^}av(g-jAOAX?zwkWm(HL)&YH|LGxayvr}W|8RPyK`U-KNw$G~O{T0RVn=ip}jtK)l&X*USdVm8dWGs3pxh*i>i1lPf}?TCIErsj+_wJR2Yr#0PzKTTl3sUhXNjQbf`m_*C1hVMrb-C<Ge=D?KSk4i5o<{;?j-w}2|1LtF7&n~8l{yRDOlN7kxcp9vG|DOZ*Q0uBHrPxCzM)t8fM#qI^zNT_hGPsC7|3C|3WUShbc1fYA2?d)Ed24Mb?xR&cPiC22+_PI-y(K0N36881q*`kPWXBju5toqzvE3((dl57X07Oo=j1@w1a1@Zp);_)fd(EF}(D^f7*9m^vegO%Av%4d?Iu1=BVS^O`41X24$-<fdtYnVb`jzRMe$VCqRQdlk3PtcKNSgr)4&4q;z#P-4QJ#^RvGkqF-(cuQt{^&Bo5tsbZ;fGT`wnk9M}aOUonlK78-h;*Ea63lS>Coj2>H;anoHYGK%c9ow~WOX3`x{_$cVIleF|0RGCO%53ELl<gDL1~u>D$<QfOELBDC7fDoO6uZ-S3sz=yeB@=Eqr<!d0%zskBh96G<p$f=4yC5*K&oT{=d^FA#>`X_|$7$0Z&k8$3%0FIxfsUxOt@wsC5y1f~|<ZsQKABF|2DMXZ-7{5J{RnW2q7nQayu!<Cdwhj6g<`J=A!Ee(~zd;>LC`zR4F!aulfK!Yf4U1H?Hn2#nb{l0RV#kb~18%AOv(L8qF+O^18JE)RXdTNn1FCJ>h;skr6fVOubh{?Wan88ACOgQ7|05v+@LJG&OS2=<p1Dj;DN*-{f0+Vy-c;4|0RN<~>T?I|Oj*P-sjZk4IUhwoC^WSH#$waS9{zgl$FckDYwn^~6#HZ2Vn+W+6Dwtw)%F~i=3>C0^%FN1fGOcASmhNyW3|M)Zb$XMTlvC1Gc7fb)5Je!E9*eg<Vki>#o+)&EK}UK`xD7_FX1%hvQ`$V7jt%@z%DqAOhFKqMXx8{au-Ty<=lF!+(jH_58`EO4?xz)UX|AIUYL(D*YIxIZMl&M9nDeG?g<ba>e8}4`N6ySnhbaCb{t`y<?F|93SUM=`Y8lFbeZ#6joR*+=9ngjC&h+F{7c8VaT}ec=3k_t2wlUV<MIc%5yg?$wB5zk7azmdC-|qQoc<Fy-XFe1pGu%i9SW<^Cmx+EyN$ERHeZ!j!0MEqVFbn1xO^a~<+t|B@B377BUbK^UIW!M$R6`}e8_B!OxHr>CU-LnX7HryrbKY61$KXy2T!FsNco(DwUAXT6v;IOUO9s~`?B~<TwM)P8p%WkEpy+BDMVUA5&JOyn`&Cl#ztaCCit7DfR3lWs;2Q<`%s7N%{3ulm2)4rE^H{lwC5vQm?qiT46!SINVBQN{K#O8sr1_aTqiq%?aC!cI}jvP+E>7@zhcwG($owPWZ#euCP7n5iZ)`-RmpVM+Pt%nF34$!iD4l$B?rp9U?f9s?btV(Ijfn8J`u*ZK+Hq-MhLhegl}h9U2<}YA#co~+xXP$v^>S)N05klW5c9U16vbuhiZZ!Wb`gjI#ZMhV?P1)fgR!s8d+K7OW12N7tQrJR{WeH<L*h`CH{9s#s(QtJ0PiV%PfT*E%bL@$J^}=8@*qSS&bN5baM9Sh+w<tB(w6#85mC``Ih*UlxaP}%$2VwC*45uD>!ZFdqDK%F7PKe7pVjoe<~y^4{#V%^x#gt@leH4uSA8I{*o;kaErfFy8Js4o&#Rk;T*qjQBJ0w=&MM-(=4@?AobEM2!9Aq$a$1XnhJEERP5T>@Ai5|6GDwn$MSpDi1%pSvXEjXW=!}F!%>G9dk+SF1*3(?5kmXaPs`?&l>h%@SB$vjK<WP`11R;X<YCUf-AfRjlF`Oj#k{_*X^vChJ2TZZi1UM|BB4}SDtg{K*+ex<08}iB%AG*|T^|llSuYOBnV8v*Rr{SBAgz0iImsJeBU^RXVpgv*F&;s~&i95)A*TSj?ui52FT%D)Wn$jPu0KSkW}WTosmyVj9GXv;94I{4AUXKi3g~DW?Fe|rzHMs|`y>FfrpVg-0Mzy`Ji3-6nKqk-v&O)m{U=4wSbS>>MLfO9JLU1IMgPdznohiRruhQJ)mev({{aV?F@5co|3TThiDf{k{e(~;064#&ooJr9A@zYG);v5t7&!4X`tXuT>wzD0+yllT<>)cAy-$L$<RR@jr_4{t@P2W4C_7E)HrJ-;#x;P4?MsYgU%CWqyV9xn+c`Gq(L6FS05*Bh4^DnKF=B`dB2Y2@k!phH^gx3}s_QoxE2LiwoqG)_)o$DJ&qB(}AYz2~iGZ8-e&fM?lIlb1V`a*YbB_GQNG>{cR<Ne7`%SPq=m?7CY#a!w#7vyy3}|bt3*#vM&7?SDIHRl^wOms$B|R+eyeAcJv@JUNmNCqomhr5*VH0AVMdZBpgTj>zJ6@dh7T=UvPKwkG{bFMNUm7OM*TRX-v6li+|Di!0`IPZjzo^#2V|UNnZR|(T(~|UyNv~ZTSJ^>!^eLb=F7Iortm7#BL~8Ym%^1$z%O-Q2)3ynZc^5*D#<l4q+}V8?>c==}$;$;Z$5Pv)93Bi1E*r_PyTdC0n=-kex-O@qm7~(5G{#)EunE4k5|JCv)Y?VKAO&u@e<(rSVI%|bKvr3&${n!UEJjdb7Vs}Kg`oML$X-C-yQ=gEOfGB~;c`dXux};rX37^EJq2XjJ{gIv1<EQkuuhMBv;Q!`(0U5o&_a5%3)oLdd(Q`ea&!v6g2q=9ohEOu8stJ-jI@|7$3-%vthS_4ei)9KFVHOCop~h-rc6JT_of&fmIG>7!g@|X5L1kE`wuRq6Dmj3w2@nZDQIphvx!M`QQ_ki?Yt{1dDF6cjFDAjtxY01s=Bq&r3)G4b(~v)3~GN^m3sK2phFT-R$67z*aFK(=4A;~0RBFK-41vDFc{!t5Z0t$%NUlp(fPpB+Hh}qconvqWvBF<#?@8{s-=r#gN5~2^)%P$o)%-;AkfjEQJo`p2^9!u?REj*{GP|Dyp#o3@+O#>APx4e|BAS#=(%T6nnN)*de#(dj=DvNfYzf3`I}#LdvFmmISAQS*OC@r58yZQnaRu+F_8JFfAC!sw4rEKBlj(&=d5ga&TA&E3hLk-2*C&$HdJC3i$I4D)3%qbMp0`uFZ^I5J;NaJ=V7BllG^B+`;jOxquis90ZCB=4kt9>^?lR8v$F$Vp(#ICjlY#+lR`UDsMeF<ks87QKJOaPyNO~m=omZpjrQFZ&JF9Cvya5plD=je>F`B<|MT%pGzr#LDUqClD#vba6{VDN%z`Xb00^3ef?KdN=q=sOA0y#M8rRm?r953PN603?sH*$Rypy=093&Q9QetR+8IMI}|8I)&b+?TFxtGb~uj3X;D18`oOx_c+ex1J|8ohqo;P4)0^4UTIq;j4p1puq<9H!(;X|@WY^ZX@Wh+C5JH{qMD1<SUNn$=N%*(|+LOEVE~RYSvKOND4oebwP|vPEh#gTX`e&Xls7Q}+n9V={2&uko~9iu|dOG2hI>VZD!)_YJxV;UN^RTxUK()EV!!Q#pbZ3OFVA9Nn2{IgMdpo~QATT~LIqr}jI6u<ma?%#JdAk!`*D(9Pk9<iB|!vp4HWh@9~+fq3*DVtl~JrJyPEwZktM<Hil;lBP9rq&LL~`gmxc7w$)HQ@Z~G9D+9Hr)94zJ@8okH{}7hkF4Fj1HZ0v-z7cFGV-09JiN&P=aO*uyDUc-;hCt3KE0^3GhC0A76odQ5mbc(m{EBVK<h`L`>H2m<2gEY?R|<q+Iy!{!9tgH-F%|s&mLZKDe=+^`RS0#E;^RsJte~l0_+bnHn&>ZX6oI{o}XbvnA`Z3t%u3qXBqW8$YqpEgF^p2J=E7=^C1n9XS@>qxe@x8#?mrdQM;6ekCg)g5#%FI<F3*F=Acvd_+g5}5)ZC&^i{d!Cr3G3&}(vLAi85f$R7x*CtRRTAOvNVPQBp%5@Kl_hT>bIdM{V>@IdB9{)t%mg!m^nm4<1IAlDgpZG$@{0B!R%82^J-UB5-)SlZNar5}5>&mxN-V6x7)vbir77wofZ#>-m_IeC{k_YdHD1tAMFO?0)g5gcRP0o?C}pWJ|rork?K(;q3Lp(#QRg0JJliPVofIsW~@Ia80<rWYA0knjbjPkFwBliQ&sc1DtGsYI*Vkhb(3xmIlIA%*Z3Vg-05KWug3KjvUi!OcYFQ``JVfy(L2B4V?5Xz$C`e7@$`7tC~0l@@PnQ86n79Q1T-<olx{bVM_q-yh4wd+p<PD}X6Fd*rsy*iy=EwnWW8gq+E&ia2^|oi$EP3sghT98fH)T{i;ByE>cUinP7=43lG5+ol7^$7kgshG7*ZoC{<9K7&<bW&^@CV93&=2s7!ThW>-_ZVBSf0>_CR=`MSp7IueK%3Yx?(AH8fAOg?M8sA3k)uYL_ELF$HT9ebezG=1gG&2~r6uoSCpa)Jxj7{opc^X2zff&$+%`zQAT>ItDNsRD4kE(TW>1h&rYDSN?$kxOm%KMfTU2R2fd$xwpfcNk>XJzT!60<U7iBsPhr#r{T2q`GRLMG-bggLchF`+w_Iux+PYi{=_MeIw%y@AELj(rC*1}dF&NE(f5){dRTDR^*{&0#A+6G2eZ(C?$CWHZ{DTTG6R{LQ;?s+aX#e1@eMZh*P0PiCx@-V9ZdfDzoXCxqQ#5+Y4XX6PlK54Mg`tZvn`a(wdZ!fgvJOJP`2+H7FzV_qzVBy_T6I`y162^LvF;@#T$0_iS}`275(t4xXxYMM<qoo5yPGLIYrf&W&X+CZlz&kKC35CsJ<vp*gG;N-erfQY*`TtW%?f>iKx3WT&O3*<$66+@a@r2aVn_)bZ%^8?$3ZiHGE;L&Q}0iBf49n?wk*ME~kpUF+rnX|TQxfv)+bA6>}{>$6bL!*hl*(D<YfZyavlx4qoQo>CJiVULV0vdmst_WLrw|>7Fw$^%sf+2~P%$@H5dLYDvLIuAu?aOZwkFpv%m32Tt5v*pe58FVu?!GSuLK?hLY2INKR2E~3tMh_5(Ck)D+Qx-I*?+fCg$WR@P2I-ToDG7Z8<H>;q57Qi``+zcdN{Es@6|fN`;Iy^Zd^W_*Nz}fLp2Z+t4fQ+823$y=PRLjNVLGL8Gi#w$`5I0Aqe`S=DrcuDdwSnD<3>H{=ZXkCYC(4<5O|M(KcIBTFiRVMn}pjJyi=tif6?T3j4W|b6_JQukPvTH)?4B+vKC9s-UcySn!_aY^{42<@I4ehz7{>Ow`MW;rjq7pF7RNqKmF*qxnegg~UN1fj$T3*-C{MHLS`f6K59D@aHW>tP2a#iixBInGvSkm!S4O)qlCqXye(NJI41xnVSS;>*EJKI`vlFFxbvFmHgi0LSp>ZY9u#PXKF%%(;$LD#X%4VWO}{2v(9Ko`Rw81;obo`F62T~PhQpRb^@<)AX_q*&Y3XU3o5Hh<z=?xrm3@CLq|fxPKAM76sQRx{4bfMoh^Z)pCC9C;Yqin7E$6ZurmcHq;~+jhy`X*vx?xWe*v#YNpwz{YjLjs(~i9#%;pE7CYqljScvfHintQ$!HM6uQ>IqkJ1`BkAXYhTF;TLyQw6tS%`_&EV~{|mYm3)vfVb%wVWn1<EAUl-&3sdRFo7(wlDUM|rObel&u?7vrk->fRciWqVDx6RAlq5~Q9XE2ay`V`ij4;RN(uRh8_UI>T{{O*H`R5x%~f1@$Ie+iU{@t{DWX*2z$!cd6^4wI$59*N=3QRTHbhp>$s2`Jxg4K4>6pI)Vrhnc<b&7-4zjES=DWeK7KNFoJC#?7EmEo6Dd`WwLORELb4`2(O!UijcPAAiv9k>)6k>L_pL>^~k(j=g7Msi*6vV`B(5{nvZ;u>xTd8AFe;)!%zEqmMGGnBOypMpbe!yk6{s-0K!7g0?6grK{|Drz9W5Oo$=HK~BpNqlwT8}*yB^&|<7Ack29Pksq&;O~-)j(m5q-Xea?VwE|x(Oey*-r8p%4`6*zE%%)xQ&H6YhN+f)apyU=cuCOx@=`bpC{b~*V0iojV_-7r(2&|T|G9dMXF0D%w6sX%<{}0yi!CGCj)DQQO#~^sYA>7I0tSiR{IJ=PP^q6CJT`K*<&<SnQ$4-90&DUSII8V>9{%EAtfpQ5nB2N{Zu#cl{bhA(DvkU#72vHf;)x>nBe}KM*utUV@f$9FC!TI$!*f8Dr6OiON5LxJQ7KWED4OXM>`@oo9?-e-L~R!aK^-jYUXl*ON;rIllDf}Za$zJltJXxitTo<qy_xv8_rIa+9_vKqF3LC-69RH0#7}RxCoq(Bl**BwM@qTv*tNuo7C)ikhH9J_mokrdS^U3vN~=Mx~}v#v>4uZ4YqKV>k%3Go3Q+oy}*V}fPP?LJk~Q65jr8oCQ|@)$*);p8IG+GL4%t^Vl`vWVYI{tg8YR<QS_lUq_7QOgj`}k{ay+H2>SX#Yt9CVF3%OF4(}}DmdnQ+y(T|#cMGn%fR;=`R&0J0;TP0jJZ3eO+z2IhebmxOm|sz=I8rEB0f~|<#P=M^!eQW>(#{^7_ewr^_}sYV#f+`I3pnU5i22p|-*xGTAvj_)Dl;ti9XnP0caQ@8fkU_!shi94Q{OE9ZL8ZJ48mrMdxC{>mE0noaBvjzjN)DJkJHbd`4Xi5ZCH)*u~?cOmVqbH6h!;HDT+e(l5x!<1^*z~gPW)Tx>-`tTdcx*7yUwnDFUUFXtz1%zA~Hc3(_ejivAv6pPW-VBG(XW4b5N(i*6AIi-!*fQE^?t!T<TL_yRYFIpyNqf}K6(w#+dE#&``SAc$D%2xx2PdjZ`+<bKJYcG5kb1gj=d^U9|T@1X&1cZyxTHHpceHXOu!6AlKM$gQpqIgl{w2Hx=yNQbH?g(M~Tws^5Tw)342qiXnv@wQn~Sy`KaK7_&Jhm%{&lqn&K8NEV&nF178m_YuRh@<zv8UxI4oyhrJ7h$2VUP<x?T36bbN?%PWz{G5rD+V()n~?{vc|3lDl^rP^#%4w15ZeOQ9WjAXm2XLHgoM$uF~UW&hY%d)Ab_5{rq$*DbS6@Z@(&5}Rf?iR7hrqzmq*EVp9=8@3vPM+`4?FRvpEnsD$J91H?yTc$_(#B-|4+=?=L9DB+lYl87y3xlwlrJC7KOI<sS2^-78r@$ppSA$|u^@>PreYpdWiI_@|^n{T2d8oQXi#lk=r;=qOd>RyRg?ANUq_;wO04ODKQ=5yt;>RP{J2gpr+pH0z7ipu#PWptY`{#P5*#moOSlbXV9G4t>~nJfQ`~F~K3^Q-y2yV>Nq)+m@E!kBB;*RCb0PTv&^BG@$Q73GGZ?j-zBzlpuP5O(a5onUZEWav+1(`|%|q?Q6Dyt8tHraof#U%|sWTdw&LwHSaARYixQ;?Q^!@nz=b6J6n%te%8i7ml9JJnvc^2vBn^P(9vkfps%QS4b~&M45X&+Tf-2WGK~kl4l2L|ijnq_Ai>h0r=eTFw^2+eE<1nsKmA*m_B^(*Djs!aLJI$r?9XR;j6q^M_|+AtE^vdXKkGt^$1ue7*qL%oNE!p>^ax0s6totQrLZ4dC650uL|uE0KSev1L*_o6w03QnzEhH=fNY7$C1_k*uo`1|;EJVjUgQb8`S|x*KTU#D!S=57Y%YDC*$r_#y|3(wZ6Bmsq_f1P>R`LZ3gMB5<c|V_+|YLZ*{n!^d9w<GNYE-Lb5641x4sR5hdRoG@<w20jb2G}%6{HEZssCo!f!5Xm(+M5kO0SD?AHI!D6#+ZK>n-oj|vhCe5m38b#>7Oe<HQ#Ii7OS0O&?6OWd1W!~s_`Op}ctM|Th@0A~=Yyf_zKx|1eP78Fn`Y$(+U?gdFz=#R_L13ULH^MjcQr<%|vqvZ}A0TB_CaBXS;00000+#uIDUG*Vj00GsL0p0%tPiep)vBYQl0ssI200dcD")
    TL_SHA = "43a0848def257e70565285b6e172566a0480dc523c76eab100ed536a5a575f5e"
    TL_SRC = _unx("{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;L}7rW?cXgXb*@(Krx}@rwH#q5Z`i_CznSd^pXjSWOYFQmMUNN`jlIeV2877X=c8K;UqF8@YR(mmmH!Xg3EJKlBDZ#shV5nX^7d{pA4*qx|RM}h4mj5C=335>JCqM9W8sP7P<BO=MHW5`v7(2C-y-ZQXFcRJI={NrbV{hhg@C%c^MOV(XQm-)VK*q78cC2WD9zLrjlXI+hhr@q>w_9mj^UexyLaJ=$jfCb7!+zA~)W+y*yE}@cO9Qvkv6i&KyvFS}@jHzvO4eFP#)H3oFCO#_emaxdK0NM6{JrIyT3r<vz`ets9E1=&NeLQ=n$tX9-O=)^YzD>_iJV#dKcaCwtR0b5W01IY?>jeN#bAbX_8)!TJeyj+^B^l>5PE*|~fM&DCVsnDPeCZQc9K&=3p}3^3eltd$SIR=a20mM;`seceC71W=pC17CW<FLRIpS?H7um84cvklb(T;e>CeU$K=oAc~D{3P5a1&kQXbvnNiKxmt`YwMqgiB!PJ^FnHtl7cehvKv6bM&$S;bJ5(k!nL4D*MtPe`7DTi57ZNed^02{G{W1#mHFp(H6-zRrEB?11tF_QURqFM1@DywpiwR@S(FgfW$08E439N#A@<9FH4h#KigY?GwJazzGkLrFnCh)W+A3g{^O9i|w-IzwZC^B{N>k*nX!}C8*+d`cwFDq2s3YZDo-S7U`MxL82GDj{)+-I#HE~NYY9K#4B%B<bP)}~(1iJB|ZL-{1wnVyz;`|v<aZp_!8nnm2M+}k*BdCAt4imUT3X57ylwxQKpX%jZ{-$UA}@6`^^1Ngg~le#wO9tzCx7%j%&)X1({+<=RNp+3B=qoq$YtX4`NrKmYNgJG)MJx4aEy~AE3X;caK;`%-FZi_GyP%@F-81cn{xC?{uNWyg3_97)7-GZpIT}zG`2Y3Q~?K|_P;<|KiIdzz%)j^29!28>n$Q3--dj4J?0z5r~{D+E_Ft5`lG|81Q?d!Ci3xoH8`_*i5(>5@JGxjrYYW#O4Z>_F_xezSv2giD^8G)h(AS77SWKeVd+EVvGPB8t~P9)!?u?diT@|}_*&Cex!CxkZtn2ORTwaHTkKcVQmV^Qm`ps$+Zj?A7(XL+>RPW=PIJg0lJOU7qydH&KByV0jGXP36sZ-9>q)9d_Ir$oC!WrDs>+#+N9C<kKvtulk)M?_^|LpT{pUeNMR>_w6Zi6gq_ypKV5!I5B9Yt&G(X8^9DuQ7KIq-0#!etlN@e^GaGD~h*FDioJ3YLi+}5xVYk11^I)V|9f@RiNJ~G_}DV4$G{-`ogyw`MRo;a%z=kU`tqmq7POM)lu0Luun=Dt&<ZAoe)H;oj`EopZCYPm7fJisuZEcRAr=YoG6fx4Gd_13UBy0P^$*xeRzv|e%ao68!aj$YehwIW^KN)-DKB>R6I*TDCV#BpgB=*KDy;CcF*xSE5cceiXiG1ard-HC2)W<He2Xcx{lt66u-76$EH4;22yF}3#G^f)5z&abkiWW^H)0&5z`#&sGRYc!D;_J7>q*PlGfInXbCfHP6L=y;y3^~)9A6RK<RZAJlSk^s~I>f^qhLCY~u>nM~i#l`EuH_tv$<2QYI!5_F&cT({yuBiXV%q!Dp;s6Hcd$pqn=u24n{!^|G_teX{{?{K5f=D!!?}E5IW3h1|9kT$Uu0LH>-#M<rqZ{N?NywdergESZQ+b=?f3<?8#=K_lhPM}|t>HtmN5)yBy)A}IJRqT|wE`0o91aP^EN=+3^$bUn(j=HaQSP%FQ36t4$1?cT;pq^Szc3~fE>?tE3&8#Kgq1wmBeJ3>jp9?)>mF>*-VD`+E_Iy32mv}eKYDM&F86ct^tH{f+_j66dBWY|GF`H9l<quK^of75Duau-_eRHyJkMQ`v0w*J-q_H?;E9}FnqCkwkfuS-resoy^g6=<}bmV3(HcyL{7*Bk#(_TWBjLX`&O^=uNUhebuPnAU7Ir~a05pLvy-UsElZ5$$)G#hD~B$jk2B%e<JZhAw6uWNc-5sb*u#OpUCVuk7igGg!BF-y=R&oU6icunf80pl)s@JIsOxWzChP<DGspq$}vC|CzPY7ZGul`#_)5=E2IF5Hk9n-rTY5b|vQ;)b3mYzGisrf~Ybyau<mu>rl56d{?#=dwOOI`@3z3+-A2&uy3?zV}<pGIS*>!)Gf(fMMt&f^0?fm0#%kNyLS;;=RFc~-_IO=<cVV#HhWM=<~mve^cSND*i0)jLv*yl0&zA8jW*g;Xa3>P$vx7MgS^k)f&XH`Iz0io^`$c;_6B&-hLR!4ID77WuGasVLCb$3!m_k99@alm6AM&ryeaS9Hk?G2F4aFF7S80~II0G#JUm>4>=FeCr`k<FOd>-(zxIt_oThGSb(0B-V}sEX5GcOx85AL2jw`GwOSwXZoTF~Ivsr@M7rf;*kK+38AZ_d1$8GlhNFcZ$EzZgR8LonXd#{@qA^-;N-Fz3xnDWgyys0*f9dya=lraIH8O)wAaRK)_t7gO%g^!fAA2QQcJs~zSW1^$i1jn`~(*1lIhJl(3V+XJ>Yk(K!i@^2ErP^ie8sE!jrQ4)jvAxBS=@YidazoZ+1L@2|aULsMS)kSC*zKw$RY`1QnU!@CVfm%ZWIjo|Q7thy0xOvK*C%g=p1h5FTfM+qcGR(Ih8BL`L&<xgbig}<B6VY@^~53JOCGuHa;o>%^vfTN0sJ>aP%!1Fofm}@{U4UOFB)aL+>~jxNthZ`QFlc5@d*K16bDf%mW2aqs*eHtu~(%7?z~ybgX&2{<<=B8Eb#HwHV=jcyYs<`ddQmmR44_UPKKQ+5^z*=lUor>8Xkbr<a{pAOr!ytR}zo^rHb+BFp)O}4V?Nkq(!4PLagm>54DkiVkaUf9QlHeBuTeg6Dlv%3crQfcXHkrU(;;;ajNq%yYYAoK!As$RikAyKTllgaa32Oz9WaTV*@`F5B3q!$>B?e9!?4=aT5)s00S=@#hBs*$2(mU%HH8hf4DwkF%G8NK`>fa;dm}SIZLJk`>w{{Gppn;k+sKzdi)JXm6SpgW`HNR+sgO{TD^GXY_Y|XA?aKfs#!aTD3dG$g6pT^u>yB`d~2~_rpv+(*Mn0FjbZ;|7)TCh>_zB%)6SHyWLGo9La!VXLB!i=6Dd%X+_ZP&ROUyc&0NW88?&=OGB{0b90xW%SQx8yWkyGRU;md(*OHg$M!PpW)CuZ0*5~=##l2E~M2`?k&9KA<8q6Io)h1%i!7}^xl$E#C3Ldr%CC~^BZvil*S0(^C2n($2l?bexh{dK@|Mc-z8(@JiR+5}zlk2`&9fS|cj#uOF&t>gGYyr^6f+-ud;YL@@1_;UXe?0HxdN9(y^cEWkZP&tdB#?rsb2mtvmR0@FyP%<#-8Hu!V1fsmaxMt(P7Hd(pW$;m$*|!aX~MGI9_uGklDGdI%cM}lp#^<N1fsc~!Gmz;<x-UA0x&u!_j|XjGoj0S$t3ep!0bU%v)OdVLwT#Z0W3aUY~+79-sXk!Q8+<h90FFoa^+-YQL!qWZ6ms4d9UG%eV|xsIP`vPrX9_m>?~bVQ6S&IWjmxs{~CWwEU0d%pV7%-^qm(|^Q2_JW3No6(W<%jgKX{l_HyPvE_i*%aq<_|qE{3wgwCq#z%(Tm`uiD0jrhVo^kcy_an+@aYl{}VpkWSs_Y1tEq0dG0Yztb&Fz=W7E<(^iKtxWLNKU_EEMJ%B|5#0R?Tc0ZC4*VM-L}*#eX#7(JoQ%TcqbG9&7zSVw9fdiEWf2*Y<{qyWed_3o=E~`V;anL_Mo*lqdGqY#FM^=+N+UV0BANo9`cYKgK!>TT1!91cXKvFAF-?`H4rJUjoor@Mt<;BpXbp|1x+bnI`FJLy-zV|t=J>$G!0p!xm*bTk6OO%gqe02-*2>}zcpxid0UJT;h+q5jV20Su4d;MvHU~!L7d_Cw~S^aQ4X{E-p@ey9yw4R`WEScDg!-oFn%9$=vWtySD%T$ESf`UewAYr6#|a37+lz>LE5v}R#aBcUGoR-w4ZsU)&q34_dxOtY5r|X<&PeJ|AZlT5h=DFXdPz8!*=4VrROM4;{P><WiFb(w|jUI{SPoEyDD(~QQB7~-c%dwdly4p1b6{R|G&P_dHwtz$r3e#U#sQ&wMFY)?_UE|f=)c=DhX{#E;V1fKX44^f%3Nq$+8swnj#^pJq|rTNg;Zg=3s=t4WUI`8cAY3evX>j+QB_-D)iGC3MKkZm=SQ4+U&*m+Wp?h;Y%eCSJo0D*4PRQoweU3kl~uN_kaChn|3bbok!ns7f6Qj;Hw@T0AWwvvlAelBo%@;9QvW`Vqb@lUI&iOA3=sDY~WqTaJ=#`=0T;v)AFge<XkuqYTn7=dbZ%#FxqZ+vlNH(t`~8PCwh391TTqcMMU<WtrcV#uX5bE$3~-xE53?NgHuD$_`lKwP+mD&SDd>Lb4#mVp1im`TSy-0FIqWGwyG6%%vPAUNG8k-%?$HqMWsoFTh~P{LBC{kV~A=j;{g>B+=k&D{r(N6nWbAnrPX22UJ5lfe9g%3AlxHcvr{zVgsgXn?}!6;e_|XON9^-dOB-g0`9RG3O>)E4U^kCtpGLtwAQceXLQ8M)Ayle5^o%0K&MA(I=aD{l;GAz!kJ9zP$8JP@hoeY={fzd@Nc<_7(a4)(Bg~3PH)7s;3+RoTDv%%7{v6;Us*r3vYv6AgS~3Ep{05*0CINZdzyJQ>{k{$kDmIP`KyB_58Zu8;_`m>HXr-Y-ECcy|NYzdK2Iwh)yq02?Vz)I7A#zTacV53>_#ywP`^rq$gx{%iCZ`3!p^6w&{fSI7p<gBQ#E9u<TsU4uW*EKFm_bKyO#^xQM!1b3Mv)vLSfLknEf}hFUSA~;6t{Sk7%65J!3~;VV<xkXC48!xi~8xG7-7JMy+8AU90M>^!uEX@*S>BM7U`8e!>u}q-IPbVYolJ<ghc*SsLk!G5P?cWID#AYQ3GTS)%Pg)NS`Lal1(av6A)oO<c?Pw6F=;3jw(X^8$8(=s)BAE+h|l>3CR0$=+VMcwxGPJZ#K;?3)49^imfA}WU8eVD5jfMXf7mMjoX-#Q1bcHCX=Or*d?lfGVVHr=)3X^kOp2kHUT_67u{*v!OK=P+!SfP#|Cr)ZZ!t_gs%7RhFCrgNutcpm^2*Z8i5uL+#^YVse*o1gD*Iz5S^<5vlp2`ok2Y9%VUJ5!}7lpx3K<=p}Er{z)$r0D|Igxc`)SwPhC&?%1#~dhIqOQy>Gh2{swKPu#)9eAoh0u&+W0@c8OL5I7~4~sEC(*FQSgL!?4x0$l+(dn7@7aZ;dwu@>(jFq2(d%*#U27m3v`ZiW-P&;Z3ox_Si~&n`&P<3ZyoBSF6^8-$33iW`vqL6Nj0Ab-!%Dm#p>5NI#btDwc}7D9Rs??qXswCm{R+2ese_SJX&-A3wK1yAR^@y)0HK_5g_NP7xM9+u#RVqYl`sRDRKTPNrI!Lt~Ei{hpF`_RqLlW?SA0;&RhX!VbpWyIomY@k4j->2((9`CuZj90Y(EFe=H${$>pNvp9Nj)bhh6vcybxUn{^C{c$|SSAl`L!^>v8PM@k_J&y&)%(=${8!_7I&j^yg@UsR$zZ_~`qvj^~cWPDWcL+xKYxQC)bpftt`Dxc+QR9i1P&?K-LBE!?n;3-Gv?Y&v02hh2hG`OrXx6r^xW<je+6cbBwN4XF63^2??AA<s0ksq96vOKx6i!5re1r@`j8W@~T0%%=h!}Jhk|-rc6YAf~U{2JLb@sT=a)`M%LaJsya5)cAcowoyTX_1%tXp$I%r?ujTzS=K`Tdi>UQ-v_@JV1Uihp8;>BU~`9#=-2DMprQnk{PvBb|!7F#(Pc5$53L?Sz1*5}pr~QM3YV%%?RlPh=Rk;JGV6ay&j8SzisYdAE1`lC1`Q4)PdsusE%Is%?pyr!UH;3u*&%UpE)`)c5Ek^mZ)7y1UfA-tg9R8)G*1kD5twX~I}rr)g->XZN&<(%Xh6zndGuux;Wj{XACgci?s@Zu}71kJvSdba_fvd-<Bk4t;=CbO9Ao1rdNo=llFWLk^_i(fPW2so}G|e{*{z<SOkehuoV|q_j=nG2mj3CWF_qjq=Ufxbg;F&lc7<uYWChf-Jf?ypF*pMzA@o9_`1X#N^gMgTEJwZyv3UwWX};y-{V8$*_s$91eSZqbLj}a2RQ3G3tJj2IlY22yk1fg?9Pzz`mB~gydIy%qjEiLLR}cy`7jLXyDWi7`YnJlwewul*SE&z9K9;A0tboE&2=2;MJkR3z|s<@p$-V_Kg9y8)~%4oTl0teOkV<>!NzF8A7;GYoXl!O=lbbJ{OP?Ujj#$?xFxL?%m$q21Uq<d9Wz6g%iCNLKRy61-ZTFIfyD}TCU*0n^F^u@l?*xr-E7KIbMILcs0e7E2t%P^>nC@Jatm;BjYm->x_}gzRa_tgo!))t5cF-rRL_(jpDv6MQ9Y*QAA*_6%)KA9=IC88Ru*y9JZr@ep=FDI6${t^}KFCL#j;{e04QxaYtWGt{I-X2-7PUPRwAHfGvfaiz*JlF)qW#6u>2}cDkk5wv2^FdltGVuY1y$lPd`<F1W#b!Uq*zmxC!JMU5b_n!L{aY-z!t$-_B+y?H*U@%~P2*N(kI%b<mtKeW<l9WH02ye3*}^_p5R=2@?8+5Q4*b|lM($i%4k6H~))-7e?B31=d*n}ON1O;U}XxV%~qy!-~N9+AY+Zc+(Ee%&dg;nf^}#^c3Zl>v!QULE1k7&-{}{wdiK!vt8f6eFdNui!?KB}UZ$c~z-k1af3;2VGre2U3`uKjJKf-n>Xp<*vJen(jN{hlb*0Nw%CQ#04`f;drXD!fnt-Z|A9E`9psAy;vo8IxNt0MjYdFmZ!y9LFUfDow;^9iIuSk-Qbo5q{^PVg2wy~b3_z5ieCm>u8EaA!$nY2bW|qoHMHX~CrbBYNS8AW>YBr0C^#`!#U%PF^*0?7FGt|8d;2rt#Wk^Uwm_cWa+*@SkYFRgA)$Ui8hyKlH?JXyAz^>99S?k3Nw)`>ZSz*YqBQ3sE-kW;GP3v*_pq8>fM3Zr8!`Y?C_dt$7y3Y}B3X1Q({u8^DECiux{~5eC#VZ_b4E22fMb=X4j(mfg>{2#z~1^nC!nQxXGZKx+;Hh9c6+NCiQkJ=;>H+_u9#CQDv(D>xls*Y^3UY#RS(9PVVv@sxlf$PMmCibrRXD+10u0=s9~_oc65qGCe=vLr^yJhYUZoK0nWhQS6;YSW5jkj1a%V4(>_6xKYQy&C(qv5h3xO30fi_1BXcx~W(cpo7P`8c0ZUSB^<LYsnX!**KQ~8(310p_3fHxMKkz7!WLY1zCDsrm!3Ypgd7v>F?+s{c+Am3!ES|izl0AH<QK3g>_6YAcj?Wr)d(=CIxxEM*O*dX}z;gK>_9VVyqv&KB<kD^#^KK(D67_YdBjWW(7@WM(pJt#$1og0GZ}Fri0xy5TIc3br(vl%-^k_%n*gj`85*ChW!zq&!eXe_R_=fpp(Pk?ymAfoYT#FP;6#3tYRNSy0F=4UB$TV6zEce&E`gm8glDKQ^=a=-p@U-XHDh}&}I;+x?&KX;VCJ>9rLTC5Jl&b)vuc<=Ues_nL6Q8V9oN4MNW}_tpG4-PBXW7X+MDJwj6(@;Dhl+e~bKtCr<HAY%U3D0wrYOR^!dV#ICZ<k1C;j~QKTUh->C!4fVZTY=;*8v3wcE%5R5`@$94oEy{tTQ3AWAnA>B%AYnXdg{n{xSC(9!UTjOQ%tP9aTFHT=^cEDB9^c=BF3PzEO52!EP}!qLotw$RzvBbM>-$HiwLFvV=g&e!m@!pNbA6}^AWX)=ZL^nhfJ-i+`3O0Rs299Wmg(9hHO$m4E{rJ7RG?&Ns;PMXg|(X%?E9;f4?up&w1lWuN#!B;yxRV&A`?B`a1PadhigHb=IW>8%_azJvGR{_6>G0;78r>XTY4O+Bfib^RD17f9QJvy94mW;s&U3^XX>j#iW`u|;#JDDH@aGO#Y`A}LUb*h&P{O2<}KASSsS(m>k6*`2rLz~yx+kBqto)qL+PHOXhGpUC<gO!KxPLJQ9u4Yq6WIMv9@!sxMgHr@1;^7KKX~Ib|uL|*Kis5|VHelN=0|}iAUx4oEB5W-VQ{1ioq16L_)lN>Owo&Ndgg%2n^(h6wE%FccZ<-Qg@gRSpMAm1^`6PxIKLVMw+}=Hk_EWNMXbD>Owc(&N=`h@DX`mYetKRn2qTg<^r!ECnd4m&_J@1&sR6@{zL2&=0d{V8rF7jub57QbwFw)dmb0143GEWyo?2!lz0%Wl%`eYnmQbZqzRxF3KZu!}tOU;RIkGa2w?>ctkx|_4dp{?;%7$e~Sz!D5g3)@N{)AB5Ge6ic65M?ZsL({EVJ}{1HmJ{^%Iz4q^y^&xR|D&U+K;av}z+o|qn_##ZZci-~_Riv`U{`Bzh7Z}XD&_L4I+OSnUjDkp;PXE7fe2j;HxcFQx8tv6X6@D;JDIeETTL5y8El}XO(_P2$1!7aA}EdmY}akr1E0M&o>S5cL@=ehq5AD{ab=DpQJi8p2kx?JgJh4k1Jtq+rSVDxVg)BMth^!}H=}Sa_2A@~BxW{_yXTCz?a1KA^q^kSU{yth!HY;qg2s;odgE+n?UakN-0(dFoIYstSg#%-&wCI%D!MJPi;0T3-r)J4$AFYv*s*(o=Zis%7DD}PqnA6z`VY30c;E&{@Prt1*}T2%ug981rO^&L2izVU`MpUlj;^*4;JmL-&HeU4JUz^y<+ThYX}Mt=5Vqv5WSmE}oW9|zen+Ou4LMpW1%d6xK)$#Ks;DzZ!$$(bkYnaD6?c}hKclN=u5NruD*C;SW9BvgGtf%4EyWxFl?N}w9M3EQK$ws%7N^9^N|owgxTU`}S=UbrL-k`<GwSSi%=s3GE=mflM(c0U9Ae+w`{M_+uPW7uW7H3LAwwXVquuTI@3rjfeTxaq0F=TJ`Xycns+|B0c%IWUU;H4TIX?eIY8m1#UV`3I5E{uQ*dSYWUlDON1QNX)ZEOx4LGjmQUK3tm=HhCau2(~XhxTzsU`A=F*z!wiS*k1s9J@1k?{W`a3P3@bfk#q?u00Y7jP~3cIKO_fknp^gqyv^@m6_l)P<?EA48tMMXv;%heVm|`Mv^kQmhpErsZcGJT|$omt7f50B4~lq9nq^R91e#51k18u76EM9QY<4~Jg5^N^QMZ?u>h`r=cK;)<jL&7?8}r_YI87ikux<&(&#A>@ms{CJBw&I^2b7;qA^>L#hDu*xeUzZTnxH8O=V2A^7L(|JBjk3PFo4N3UIo7W@oCGi70N|W#m}OV+9!{WrdaGHhw*Io9!>rVPJykLN>tXAr~~??mYGP*RFnzTI~dMwAb^f9yN$;Tft~Zffu=&*>9@dfcpV{1gpicT-xS;j#Y|oc6BFvHk|vf!ECO7$$^2rYuP!WIuVccTpfHGGBkttYy)}nIrH4qjer82hiB3Q?J=<-SYU-JA0@Itp~qX%2)c4N5;lH{#$bkAE7sqaqDM*PEW(qzAfUF|k!oPq8H2Ey(3{F1l`L=t7Tky4kLUeD)o@dJsFxX|N!iQ!--Jw5NbtA&ijzgx=V`We26<{{+GZ|m;O}O7YhN>ijw#9xlX|se>qf*_7XH=+Xdk>o&013U{34rjb*dj;nA@BHMvpoo?;dB_Anou%+h=svwvH=r6rT`dWE&>S*^d%>p2KgO@D&u~-9D8g-)N&)d<%V@(cJ_HO>1(7c!sr+Mw@k>WueDQQ3!gp#9_4P&BvOG!P|@@LG|<iw``4XZde_O=Yig2G5wqtM2K5$tZTE>U!;wzFQ=0@a0&}@(L7?8IUbyag*Z7sewNlzS!YbuMb|g|%!NR*s8k=|_IlS<awY#GS(ruiS!WXkXuQj?C_srenaqp&O)ALKi7iH4#}Rg=1!r-j17m|vj_f@MDcb;TFL}vtEsahB9xd32*`6Mex0RM4&7!QCUg)&o&-w4X&6zYI!v)dxZ#12+GC=7ulb_yNrcnRwS^sJcszr42E$sg0zyRs4?3n>XUhQP*u_76)K>5d#YWs~f>3`SuMe^$@CJ6s~Nfl#N$rQb|xv#*?v+|><cgMr)3@R7q$xoJAYohBV&u~_07I533dcw=%gqs1}Vo<)sIASBk^t>&}S@)Gvr43-6rnFfE7jm-VS*KT2&-V1}tM9nmVF<6T$rVxbNYI9Gx2}NvUa`7Ul>{kY%2$Zwjn+G6nC8G75;GK=$8+zPc_vw&XD;{S=8y)}g~&SY0}qfRO_XB_Qee(Gt`s-YdP&0Na9)y}A`!ut&TsvqJHEY<*TH{2SR<r#+WJRXJ8BMK3q3eA{F{KegI!ZO5~i)OAd;CAbDSOG-;!kP7prMJU8hsIljhpswuc@{*4YlYy>IK~t6#v}b#RHl34a@}phLjYDP6OF`i)t2#>mgrMZRP!Wn<DF0HNCb9N*Np&7|}Y{1RLm;^k6jd@`N_VoYgb&j8I}=VU=n9bTuaa0j1MYy7I4Z3-uZ>S`hl;1pmBFXevz2X|ghyc|q$mD#~(f)FO=K9J3zBI5fo>m({<6QtzJC^F-7+~jOpM~YFvCd`juG3W3G{IxR=QDuI}AI8##H)avZ7_>jI&l$|?0iD&Z>;o(t^=0H56_7+a8v7#mbFp1HKkc<)FTY4ZdvSnRbc*G?pHDbuU!w3i5(Zx0H8ij6Gd!hC{D<0cMumxvIiNUtv2L(Fs3mZXY@_P($|MVAwo5nEKRbVgD055Er2O2`KdP-6SU>o@{V0+Y)kSK_)XYbh1waGVigmTVlR-HkPI3oXg=e4C6}8FeZ}D~dh2Og0Y-<ZP&%qej$|qW(=ftJXcl721sG|ZKsyNg(7`4#h{`f85PXN95@lZ;078CH8>RC)*GjpkI?u?X0m5;XdU-7bHb75w%ort|Yyq}bAKxbtj$YnQEdkSvMbNx_tvk0Z`mo0a(o_)t)B^pyB&pkeg!i7Ee>7D4OkR|i{A(>{`5aq-IenHoEv&gBy$$K&5MbPQh7S7#*+vB48Y?frrfm2Qe4rpbrCO0=Hu}7#NU^zB|O{Umb?8<_PkO)gLb*Clj8YdnRY>Oh}gK8z3*&m<>;3((3OirRCKoP>eoC%W#WvdyX_hO5NInelPncFTDP+Q}7IF-Ww;tZ@LcU|xz+*%!ilp)<1z-3+CVrPSF!G-#6OkhQ}#F<8;pSjbl;iMr(&p9Xt`zIl2=rykGUF5ne(o`XkzQ4!mjz!_OwGYZ0STIJ=_`<r?x7_2d0-?#r5rYN;)K=HT;t=?nf+Mj4Z@RK>+vdDA3F=C{YZ#PfMxasLmo70imSyf&VTBqSKf#RY`gED~*$g2zGX<|d-Ayc66$gqDF9S-$p|R7&7cVmO$JeJgF3iP_3LcF4)5rsK`cqA)n}p?Q6!u3-Y-_WFGuF)kuO!eX+1}Gp?=0RT`BmHm=Xo>99rp18XnoGh%))gyshX^bj--u1!EiH=;@<*l>xv(@QCXEtU$j1I<JmZq&JxXV7nWF;$Mg|W6R$%>?aq1JQ+r1W%YrB`Yb7&R6zM4iVH3)n<|o}oTEGt{oL1s#J|K{A6~Rc^eP%m2s+%{Lnv8tV0?NBvz0Ia9uWiT!_de0$#AP;Yw(nx?BicAfwdFCK;d*Y?&MJgRt`F*ypq%&9aoDAmpD#Br685-(hw@dZWJAf+-ZrQ2`=@0q@d%AI%xkgk#?;2=ggqP7bZNo{LXvk>t}DS2{ew!CU%5SO8Z;h#NzGstLv+kHLW&_u{43TnJeoUd*HmFe8Xs1$)3mxhCw^PA;21!#_!+YoIm%iSjZMbNC%p2w=y2!Pzo;UP7FgB{TP`P58{5kNhR$@A|0V9S_LJxQA&I_dN-|2iuH|^iKMpbKwFvGJP2;`bpr0^*7x#6U&1r}EE>6rrUuM3K@>+ID9{LTkhfOJXOm*su;G-T3$U<d|xw<eZwpc_VCN759+<MK+I`3*7bp7`WNBCti(q1-^i1av&G+jJOe)IY*+<+Ro!Tm7i3NrZqil~=n#<8x-Xa}@Hxpza+ASL*~AYB|#Co99>hCsos0j&^WPdnQ9xY;3*epMEk%mp%|apy%dA6B1^2~bJ#x{d7+Mrd<&sAv`_dW6s2s*4SIKZxmtu)fF#xcBro<z9E4@J>>hNK$hL5~pqbZraiGSq%y**{%;@6&w6pzz!gk&4+2rPMY3{g@yP0vWe`iz_-A<VK+2pl#zG20ph<99OLBR%K6xt=|E=4bdy`Ov}=I@N6`Y4Fcq@#vfp|ymh$QtL#uZL<WfgqbcHih@$RHpA{MztyJgCZ$O9XCUN~zakjstl_oL_TVvwINn3{;o)kU@AKKt=6j^rk%bWkyN#R=N4S93s5QAIZFv1DlhNd7X#NO|+myDgj-pDb!RBTF<HBH?_-UOgn3R?%fFlnD%C%I`;##&E&f&x=%Z;ul3g^CLgGK2>`F?E|R`!=iJRJA*oJP*1oXSr<4v^+sX;ZTZ71r=3rLL=e67b`>hYv&w|b>F>*6NCfZ=*V2uZsJ^hr<hn9qR7V^%TV=gGeE0=o6<5KH)c8iufK7%X7HBK*fPFC&tAuTj>KuMv23YaSp(=l5+3_|z>v$k)Ir{r{NFjBW{p5|w`1Q1gEh{5oyt3WIPS={Jcc<E@^qX5%EasDu>o$I-w8PUxz(#^K3XV2lC7;GryGQu`WRt_zv;8#}?*iFmil3~U8o18>6Y@S5Xv;9T^%jP}wi$}iBf<~I9lf0Dp&HRGMi?|3Usu!ngsIaSPQ`>KR`H<{*qo(1&;{Scr^3=;5%;~5o~S#lyy+FbsS!mlykpPctZS|--LhU$0Q=+-KJSE{84yoa!y`+>FKy%PTDCn!kyb${?iF#1Z31Mmay>0=cLaeCZ`8{*IxB-$VmLNi`Ezf2_Gn)&7>c=GU;Q5_?g2P!*OO{B;-VY#K5G%QJKo^OJX+T&GJ&;sK;S=;RtkkX{({|*4Vc||X_sQbPo{oAH}2}hw=F5ikLz3D54sjt8$q<;^acWaaR5H?)66-9!1vhw5Zdtl%O|be@*#?W2jQbV*{084l~1MBq%yxL&T^~gi-><PWcKU+W-Mj!w%6GcjN#0ZeYcTpTIn%uQvcwY^g=O!djj%w-x`Hwz2rLRCo@-%UOrfFJ2Nl|wjRjc6zB!BCfUkn?6@q4LPfy0rH~5ej&=veT1=Vy1Rz|Ig*xPox2N7Z*{?0jQ>6KJ*i}W1y&bcnNE$`~fp47z$~a@0gdRVFu1&@vohLgf`jpCr$au9tno1`Vkj)f|0{M@$MKzbnAwtFs#MBvf2=XJQRQMjC{LM;*86F2Z{x3yRC>p-9SE>Z8$%IIGuT=Db?<DOxYdNK#iPC@eF;`y2A)25tQ<dr&3;@E@x0hn={dlBuYFI`i1d7Np=--$|OB;5=OIyG#f$!^FXWCgfO08><OV<Ncv%OBr3D~9?K1lsuX<#`0;%G}QX6G_iRK?pQ^V91CXw1)i+bw|Xekbk0rkRk0+~H->w<~Ng$o1tsU7+|L(kZ{4i=;faOqyZg?WLOtpAc+^jfL()KhFQ0l7uI0hzhVwcr||!>E|8BUozWTjBF|4^jNP0U#Wc}rAk~%_<X^4`oo=BdyOiae)HAp*&R@tFor;|??3|uoU4ksUA1p5FCRplG!AUXG5@7&z@b)j<t>b~dz;bMk4A*$HCF%!ZH{}t%wE-aR@dOx3IOwygGnu`_@d}0%g~S;jRb$eZ`cLDFL`!C5T6%-wMkItS#B{rRz;*-DD!f6g;}N}LjHUG0)lMC-#_sRQmu$1zL($F`i1|TbeJ{?G!b`}E8Fkbw*n^bZ_oD1uiOy<$tBoit-1-RP;=^Z`Q#M(kkIM1@ocMbAA6`x6Vvo<*9fv99IWJTdp7)auCbM_;v-1J`eT()qx*&^`z@BY`AtZk4x=D)-glk22%N!jk6-IG#*idPyZ^xc7|V`pG1N|ywu%PZcpjjIR&R~oe;0;Fzc|nK^{E%}7>j}s6JXRv|3G7q;h4l)LH+zJr~ya5Q>g!@s1*z#^YN6xPz1|(4&=G*LOK3AcvWc<f<sv;;1w3H&4n=&TKwU!sgShA+f#yp*IDRW-0Ey#uPD&v=07<;q+84{E<u2Dc66|AzSWyW=*+rW{8aHz2i>4s4u&%C7u7^=&f=4jA%!7gbGD}@xKQa0N01XwXA}dT!T}Tb{Y@$~D7n2m+T^M@CpRXaFH5TqqKF@w{@3K>7qZNjCS*v@yjIoie#j*)o*E2mvb6iVPA2{hT=wGkd`gI)_;ZeLaW196_{ft9u;_=mtGgZN{rSX7!35^$X!l+$8SuO|oQ$F<B{T$ec`#AEQai!oGgF9EpQr1;3u=bY1@#)s{fiILcw~{Ieb#XQ!x3@wH5uztc-rt!oDi0bKN|uLE55uZ{#78<_StG$ZSifEfzVNSfFTUdxS94}*-X<upg^6K%oUIVh-pRORD}HM&v+IY*F^Q7LI8cDwziE$ll;U&mVmJ!J;W?9x<i~kkC-Hxf4O$~L0a^WGK*^l5`||MDgzSj35NWkYd<McF-wKmiDa1`(}5AIDdXCllyl@1o^tbz44Z((@y*R^uHdUoJCh2!%8X7CLf_(Q2fPe?E?h(491w1}>D$K-T8WZi{-b|fVZ6`OYk2bhA^fH&BOK?Z1$gHbrMS&8>s-h#9oK<Y!keEwifEvVwumIHL!!yIAJ6*P#8pp}xOO6P03h=93P$uUI8(iKgHqwDFpO|e>GaHCh>11873L{2RdFf)ciEFmhuvH<TJa8S^hT+d9Y!d$9>dwq|B&b031tX`e;kJeeM&8i?K?(piwi_RL?(cEJw<N?a&c)+&W5EejMIFce48%)7JW}$xVzQug-A?iD1g;2tO8KK*<|Q%y1z?5hS0UTS??;X5`mxgN4@k}0S?3b_Ex7KMsnDy)_rA);vTCS=VVL+imL6A{IQ8kv^$Zr9s67Y;AMUP8MlCW2~O~O#t<sNxv0O)B@rctohz8~u&voP(n$;F3=*m07oD0-8R530Sp2F!Bm{Me9K9y7phQ<Q2^$gby|eiUa=kp-0pljmg72(E_Fa*_^kYD-Jdq$2amFm>f|b}^!im};YN}j&6Sna`I0&uUp$>e*>X@30sl)b0?>O*|UX|IAY5?wA+88^koaOSRkIhIU>)ipI*+%+8y*q4?x%ZXAOAV*V-WqiMU(})`^<RB!=qd~CQ^VcNF!y;^TeF}<=C|h`wKEngtog4JM4BtyHITIyXKqA+%t5oUNsrrLqs!gllH*(j15zH1wypVu?p9XwFflR`HeDuK+6^k{$I@PlYT|FEFbF{vM0k<bvSv2kVBG-OfK}4RkY1C<g`n@sQVh(B9U+O(_-rZH-?@LxQ{c)x&C{cbIK7yBgw?kUF$@dx?oP5j_)cypjQy*hm!8*&)#%*Z^-&Ejte*wx1%DqHYBoD6Xf4DrGJji&%^L9|%ctxF4%R;8c@&Y45pgjr$jxcE%3~#5F3$x<y^xA#{LQZZJ-4`Z5K(|n%)+$hJN|*-{rSj@TBKsTd--OH0W*4klmNe&MF0$LLkrm=ai=k8s_V1dO>a%w^lVw<HPY?h@Ljf&^+G*B6*BSjx$FE7fhQ>}7rqJr>wDD+ptI6T*?h?OB-?DkfitPmGJQfySD(aX7EyUksS3E0<#a6~{b<_dwbCI4^({K#7a|PM!-2dL+Khr7DDW#~_dyejg?GC9wAZR!4j9$NH?ypx+hNoK=qUKxYYQpv*O4p>FXJ^zAGn%EOm>E0zE>w<S?%?j_4;O^hUY=9TdsqHg8}R}w^88WFE}$9XF{~Jys+}{vr2~N&oN9fsPkP*2gNDXmYTjPp{y$#O*k&y+R|X9OFlX=G%f8$1#PtX6t&$7nDY(v;|^iMX4g93&UWq2+v3{oySGp}5Gc{Ik(HDbACh*;Y5~YSQ2X|8;7X$Utd;^00MzyQk<zqLS9L>ybM*Tla?8AyiX24rX#%^3@ScFG%kx}q^@Mh<5?hv>oBU{W1vgksfwfFk4R>A1U4lHfZbzGD!fTbcw$f%$4jerH6}~JbL3{Lyo9g7}*=0#v!?KuRfPk0hdUI4~v&Xdxl+5mJJ9>j%9tC9q3D*$o09w!`#3@axPXsgM>Yy2HM^8(B7lE3_ArX+qGXDhbfISQ=W297u{w*knV$8QCMFztD^H_CR-fZf<zoEtMrPMqS=BJ`K6PkWoOzj#SH&fsA&6Bvfd)ri7^dowa+#s-eWjf_&^-L(S&Yude7@0_JHtIko0^O}<14h-4=7}wmu^^VXLhYnIxo7(x`X$cdIQ3HDW<|^&WHg^S=J^K{JgI2eHWe??K@{rL)dhv?Q&>r1P3_OuCBRUoD{KK!@WjL-_Z;E|kNKNu``_{Z489tC^%tZIpNf>s0(oyl+=j8#V0aCf6v7D?QeP#2<@^gY<4d?{j=Z%Z=eiW1_aKB1v_Iayf&TL~$v9pU1$tPnFG_szR4p8uwj4NhZr3fNDh~JiYp+~f+P1II&hy`$L{FE>Vflf-f<T{=TC#o&&jS3gY@K_P03}y$;P%hOa!$ux;k`l3X4jBywNqMB4aV{4cA21NT5?JT2dtj!@UXK&iY>u0$|a#cV~2r6L4svsBLm`sP9DxkKcAyyFJ}-2mzike6kTCcJ!k|#Rx&zUl>DD^Dk2!`>vaWdV^BN+fs!-My`CA>&t)8j>L$=PtPbADHTCtFh~5y{E6tOzYtRKjsQ$H;T_6t|GdboYvEKZ<{`IN=T^-Y@qt41RBR=?QjbITGik?31+-H{D_y?VS1IvWJ4)MZ)b^~@VL_CxcAagAd+EUTXcOP$x%~XRtE;eVr<JCU;05_#boxRd7!(3nhbUnDY76;0jBUJ~odQD?Y(57KG#5cES4X;L*FbnDAd<Z9fnp|ex^Db6qvAFA%6rX6Hx<>fNV^S=M2S!D0!*8r5GH3y3)<5ueTh$TvlYi<d7)zZ=8v=KywU|w!*PNkZHNG42^YfpZqt|!@90P*184{0^QJ!<n$_jam(x)JRiwb-RMq#7JJuU9w-Jg-LN7q+O>QIn0T#-ooZTZ|`fssGoDmbOGQgKoux*Rq(4+tiDlE#KPRMBqa3GhGLF>JkC2_V2*$>4DQ^D%@__Oea?tnQ>16Zk9SJb~u@!{J53H<VbN)D8(gWQ>hiKh8xn5Rkg_5k1jzo7N-HfWkWVxU#LDB#uo0>l4g9!N146mkOwP{p({8`~{id36p_bCVWLzo>UxUa%j#9Nrbo#WlYPqbUp^NaFP!#7<5$o8!MO81K=Q91`JI3>kV+_Vt23ZM+Au}%7{1P*by&rY4S>F_|z?NMvFT*YJSUc@xH@QG94<Yx)5~$!ALfge%Lw3Du`6@?WMnl3#-3(+yg;o<u<jr=1H&nH3<B>5=pZUC%{s4<qd{W**8ole3fQ&sIEne&a9v=Ba{goc2x%6juEO6|IuUzgA@)3zdcH)t7QLj$Q5n1<;>LFKY!}Ho{1P{U$3+Pu8KD=-g=q4-+uIBA?e8&BxaYUzz`MrJ>gmVn=pr@p#__l>?h*{9?h-$1F$tjP}iiq5EbU=Gn6StIEs9p4_Ri(4+dd?nC6GL1C%Mu;~*?cXC*<9l2Xj9ik(_9);dq{O3&COT%_9sMeCKS=3>>=@tRJIq~ZUQm;Aommn+VzL_!pmJ*=OI{&;k(XxJ=nJCpjZ53`tvlIl?RCB!JO#K|R6_~G9hp@Q%hd<3+taLMU}NBGA`X;}+dEuz3L*p-ux?ksKj+#fAfc43#mm{NjV8k@$NNA-gjq_m+(kA4dJ)hcw*hpRAZudz6cGosZd%o6K<%{0*KF@7=MX{z}H>S06gsS(pwE2z*nYzB_jdsAWVKHH+h;hCS9v+o2_!i6-PNBTK+XhdcJ260s@R^>#$qf5|S&8jd<7{Vv<Vsm<#6uL7t&Er5TGJ>cYLuei9ZX4|rXZ>^HO$UBvh0j>X1oPL7oFLQIu)^0i<ll1h$M^7CNEHYQmL=+3YBj_h=#))J*$}q3B8|{!?FVqlQNZ6=TaC+R(`XJst!5Az95nT|e0-oku+lnx*-pGPm>P_Op9sP|Uyd5kiET<SbChl^{d_W!N6){yX69BRKKhv@6<48r)~4kD&6_$UdS`g(U#g0iFBFm?7Wr6K>fPF~{M{&n61#Y?^j06^E`v*nuu2ms^HcY2jQ^^NPeStYwJI3Ic)hX8Gf~|}6uIx9;4aPtAXv9Ct_N7Hf*Yf3+Bzv<w(kkcJKky$njNRDfXHgS8B^WpvkV$F85)a~vPG4cI}s14Z;9go9i(9QYBhtrf)%abQ+sOVHir&XquS?41}qLm2%ogDQVwVRea>(ha@x@W7x20m2XQ?q&AqJDWM>uQo_J`l?Be4Wpo>P{x_%!n+V2c*ad2#Qf^DQweP1bd$b_`YhZflJ*sGG`BlDvdYXV3b2zV5E;V@XjHnQB63t@fLJUA|)b1naK;&3*Dx*3)E=USv0NLm!z><cbY%Y#*dLMuU-$Y3&y`0dto^p8lEJy~mk#GC8w&5q=wS@4PC`i6Ssi_lSaFX5iO9;R+f)O(Bv>hTFC1%<sIbX>-@5`H7KKb=eS3wQ|fSwU(lBa-WGn<BZV&?9{uo~J`|ZT7z^!g9==uu<zq1$Q1nDa5br%|5l+92wyfLu+`SzAKk=cPJ^20eb>u^%a+Qo0)LEilB4?TqBGEExJHXm!Og5S@tubToL5WJEO*ywG=?J+>}LA(EC6~GY_?tr#IXCh^LGCNOer1o2Ob{EVnjZ@!=-jRkrR!wuo{^W%M}G<8yuQn%LJZV=O^?I~QTWYPP#%6v9;&=WYiS0X4d$EZg}2%6KFGP0+<nr6+uCL!C(1(tM9LN!Ms0<!{F_XIs@tT98RjQ=s}?TLd^@kc-Ph=D>+UVG*UNc!!@$hRK8~k)b&!={Zib_tW4_dpS}fc6Qg>ypF^z%xEQ+MGuWx4nF8BY}y`yn|trz%Fh{VD7Gqy82p^;k*0vFC(E~Fv<=VX1Wx||>1MFaxTPacFKW{Up%+XSwgb)<FNQVhA|gYXpG^JGKI(NQUBY7^sh?<Y1N>nm7p%7%;RHXn^wR{tX1Igp5)Q6{ETf7qn<4b<x+^L7K_U4M5oaee@0gC~6*2Yxm0z}EE7SfAfmfR)AOk1H9KRa=mo$TI%NT=WnS@^bTgG{MGwIUw3aR%$373;Ud|~+IV_ft>M^gvOp+LyIwSpx&G%9yqRKa2RH<?WEUmtg4>q*zZ@vg&Rg{~ZIlQw!n&Pl-Wh1!WU$MC#Az@Y3=M0k0{G8NAX1Kr&LZrC;3ZZ3PPa`J3?m#D89AYuVlgUY>}J@uiIaNgaT;3G(6l~M`+{_7A8`0!@P!II|$J_ggzE_*R#XmWN&t{QVtm*G_zDDd%Owo!4l<<!dHs7QOxoRyptBOKQvxDk%zBz@-q17^Vo`%GbOVXl<+0TCVYzWcl8th_~oXTB7`jw9Rv#Ai&>n`q1k>a;n;<#BE@5LwTSNY8iPX}R{T#1%L1s4LDNFrvj^*Uo1Ok$T=P0^R(OR`HD)h_4*<+p4Fd&Es&8o+z#9_HJp9bGk^NarG1VaPa+%ex92h_siW(6tcj5AS<=)I!h(d)NT<cz#|PmlXejJNSq8s=^@M#jpckW@Dt3cn8KRj)0g#6M|74KtQ$8$nM8%N3s3wWn^i}E&@lOUd^PzqlvK^Lq-gsmVnGbycn%=j$Zl69!~!Bv1|xM7>|45ss@scb)(Bxn2@a`#EaWeCNi)RtTj#J;SwI7WDovi*!Cv^2Bj@&fh|${8D@sm&p4>*Wj&B2|#=Al4&;S4c3N?YXpLCTo00Dw|#ij!Qm5-#;vBYQl0ssI200dcD")
    PPACK_SHA = "8dadf0763d43d22966603745198ad327ed6ed36001cc60a79925ea18818f2dea"
    PPACK_SRC = _unx("{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;Gv-`@?8KCXbna~Q3)_)cEP3KbfXE9&SXJMV<=hB6!@)dJk{Re`aCPtYX_C1!aYh_9-NXG4tLHXzLaOnCEN-uVY6*S1s_8BBdD;N<8>(E`kZR(i(1>Kcfij0^X~`&9tJ{S9<6D7pZ4W{bxa-QKQA8vpyYtc<dI5OO?OMj-&)Pc7^!#mRhmRW_r&Wcg_Wrv`toa1_q5hRBg|;VNV!4%C8K`aW#{6vLMQNRHBft2DFF-0*xk<hak(>dTT9YBqDDu{`nLrJCFfvSGB^SnIhVV)?3JUXzi4oI$>`$`hS@dQYOOTpyK*=1Ug`let)V0#eYpDSYOC^=0ua}uio0ZSNAsFrol+O2@W04i^K91gRKB~}Ari2_ThU3}Mpi$l)wL&FZ6)*hIrclU3g+tKfCtCU9d2eK&A7~qYpu_{Pi?i8BYU;jYPT4Wpv;5~{gvOFINWN$UYEm?lI#@AzU-m=L=2n9cJ{aDZXwMl<-1U?-1{Ti9EWI3)g&$5y<<XzlMdbQ)Rzi<NM8ERF>iA=FeF?JifFuBR|8HfXDqMuak)@zg3R2&GJm`dtEgmlu{E<l3&c3iIXu@_(inSCzpZ8=x}yyY2eZ+yoAtuN(Uc_m58|*(TQ<W83hqwvJvjj-nW`T|a?f-15SjRhi6h%UK)C>TViKD*ZBVlG1D|N^Ndgy0@*ak_oh>{Ax)0k@f3pQB2NdGMPVrk^3SVb?dxR#|?y9;T_Cj(VrXg@R3gJy(crCGYA5S|Px~dKH0a2K|W`R?w=`b9rnVJ$r`$`3d3SPbFO?A>%*AsX*21p8;o8*(-fp;H7Paz3M4l`ZvgT3{6gu+%`^ySB{s{<A;&?38A1GZA=h1tjX@dmBUmkFbYPnO;P7a450wS70z4@-k#Ex^=a%cvO~pbBMjXyE2z#I1OL%ggL$wITdXh??Q&{n{{Uzjv|vsOA!gIznu%e$)R|EV)MO6~i;dpjL<VcrldQ!a6PCR<t+N*4j#QBDrR9l~+!cj%X!iKt>pC?t84gvodU5-v!j+=QRa02%9YefFcH-TRye><3%9L3D2(tF?Ud?Llp#Z3+vu;qt2oa(B!WNF2=7L>;TI@rEq_6enXO|-$&&OmVa#~5K5aR|4E{gY=CVvf!^potv1z+Qaw0%f_P;3(gl_LiB8{18o-uYri6a$1>M(2u1tpHIy{$3!`Qkde$v)Z-3c4ylV0NOay~6{oSgxp5i?<!d8@Pq)CUR&U6x>X5?hml{%nVuZ0a3ss5Hxs$=-<`H;OozhV#eZ#Y47^HW+I8Z_P7pMLVo+FhM47Tb|vfrq}@^{EQfNLRlTL;O6uB`o;*?M`DXKVEZMN-bQYH1KAU2vmkHCfjS5EQGb&8P||f0WiEQR_f{l=+{GF6Y8Wtdst8-%?6<^!Jma~syXtBDBsb^3qMYyG7?()fHkB<e7_h7==bnpGHK%TR%^EOX=y<I`3gZW}En8O}Tr<K-5yQWC`-=OXOMYsjM4Er@e~Z(~RlLd4RJPmGDU{AM8hG)Qn4x^mTPUqZepc)}I16387dS5m5)v9E*44U(>bBEjjr*xWu(vC?u<Rp!q)1ISm-cA3%(JDHK?Xuvyp9FfN-F!IU|4Eh<98wMEbV`5tv=(eBh=BCUKw{p?3<<feVzFK)}QF)7;WjnBMG5lIiKV+)>QwTPt*;;fK|MkN7uDJ^)7_BA>zwx+d*{MA}*3WFs((dXR(n2H{z8qya3!5GiAUeFv5^fz=5MpXo*(0BLUct&?b61SO19oNci}xRYZTm_9-4Os>7!y6yvUoidLYgHRJH1?~$oRh>PzgEBHg2Y~W>G?Yq`gIjwQwZQHM;4EBiq9>apGun@V*dGkpgjo7{}vwuM7fl)Sfk_m#l+@7_lJx<hgpD@F*<PbIk8UfIK=M;a>GuS5bV}zb;SYMoy*gag*`Bc(a+At;2D@=4+&bK}dQ8+4oDrxINqwk`NLcfhrcWV6cifpZ8W?C+h#F4;zZGnKnbt`(L?XGX&rD)k?oTxv9TZbn=MUMAC+0C?1W@p`;xdY@HR1rQ9JL;cPY{xN>tnlR4<mIk2WtPkr$7c7}v8w|BS=)awTb_R9dt4>}AbJHn7rZ$NT!T|A*8knL6;N#~7T38}ANj?(^PLT5cM^C-_TYhlQc@*Ut#Jg85obBA94C!rg+UEVX+7DquN0%B24se)7~A$btj4tIWQ^ZPo3Kcgi)*s%YmsaxEhS3-*3~Up<C4!GLDdR=i5Liw{=_2DAI*zykv!OiXMn25;UP|0lO0nHnqF<X;$acM9v^3-<;&-YUSFfSSN*R3nPKt&`)-_RK~Zo8hG|CI_X@5O#@LBgLq9X?;-3$&z#3+E>0C?{lRv8wCJDJ6*!=a}*gaz)F&=ct+`$8_)kVlxU?;a(!oC_}mpcWzyR7Lcg=G4m=1X_ftgr5CK$M=5E6$Y4cjikb2$3mP;tH4}TghN}tj*?j`c|o=Fz8fQ)a7sPHCG84VfuHhw;6_gLH=vF{@m<)_ohOfflLc$%0nVye8+pgVTCl+!-ZDbFt*(A>;N0WxNzHaMp)%Ibek#gYF)CbN5`Kh*hMJho7fva>FKKikV$uIwv2s(N_w^gY5Q|S8H}vYjLBpCbC{e8L;kCKD>SVP0Bv~XCZo4CXrDanr=0Ju3@&8{JpU8a^RQlsO8K;!Eq@h)`870BI|oJv6c_cS`UR4Vam6opTK>bX7fyQ4Cb@f^PB*34928wNHJpY3hPj!9`)QZteSwjjHUt8Se>vVyAxC1%cvem2ch!BQU*oSEb4I*+ha;xx^2?NRaVR3=V-vZx49;)9J;5@MW})HDzcG6C7!iMaK9l${(?7FSE65~-U2uW*NfTpVN9tz5QCvVVp6Ooq+(l2!50>X7c|>buTZU3p8<t!g4@StNqFkQCoD#~n|CS#tUSu<@r<}mu40^-|u2q{Olyg3oy{j3!zF*%8r{6+kO#pX$2@CFN{jhN!nnu&zT1u;sLqswmm?x}^VvxH0lb0L|S+Os26it|~D_Fr5Q|;SmVqj>}928Jsy^nutFK{gi*=Hnc0t4(IC&Me!;`^!W;JCIjI8<{;eMuA%xLFP$YoZ)GJ9N%cqpl<v{zl=SsqVC4d(+=hLu_>T6@^vJ(Xx`_PPO+~(}vdg;|dwTu%+zZ`N}KC)hDG%E<I=MiGwqea@pLmn9D_C6617|AJKmouGRH&qEAH;YL-|gK($wctJ}!K*>RLL5v<RBYsyX+=q>X6kU;?u!P*q%+q8W;RCC^x?J5iGYNomwAcY3E;gy91M&@^dGNC$xB-Fv_s<g8)-&>o{<xvYCMPhf3n);yTR}9zpG7!L7-&xc---jnqXA7rn7~t#Z4EJ@Hes&e({@VQ@k>tp6gswTY=5kN^9rH(6-vZWc1(&AedYy&d<;#%~QzUCN8Q6ijrTz9Q4gk26pm#x#?>GLyXzJH)gCt-`?P^B{2HW2JX9sN+g6`h_AdtA~!Ji5pR`o7U!c<fTYT}i-aftw8HptPzFv@SHPa_$!$2}O~3_(y_@I=Ky{>PkgQCme_@#J;-2Lfp}Fx@>A9MA{l#6)on?fpDOg5}WBws!g<MKSpfq#K&I3BT&!DxW(94<C!C?V%@G-E^C+X>g56{&60uptm$_SjUB;$6>wICtf(nv-}IBf{tIC5iHE#-sE?p4m&1>%ot;C<~yhwNPAk-to)q80f8;Y{)<VVc`t3N#ZI<r1UKe2jRM!XR&>r&;?SOqV@#qP*{z79vcx?Hcjk`9q`zLd_XuPcbX_$-mn*DMC7zf+IKb9H02ZVak+ST=>T^8HYfVzrEa8%Uk@qKGjb%fNz{rW?OX&F4yX%nK!;pC6fWu*~s4z>k!yF7&R4mPOdEu67Di`nEdT1<Yqy31B81%g|O~r$d2u_?zOv655iJ9@hUkn}wpDb1egD`LZ15_B-GfzTW=(Y)Gy@8mVCkrx{Mvr)t&k38y<@;~gm9|~0PR?vj!iH{sDC~32u+IF78R!r%S>;#e(3+CwU*2+u`TsG2v<_|A=B|CwQ$pSAKRoE+Vq0il-iXpQ26zQ;=#gE;060S}X_f62v_brzWDV7-r@xg{;YzGnS?5h>q;+s|Dv%pPI;QWc!@E|2w|K5BGDvpwhkn33?Jn_iW^OTHwlqhugm54o`$S=ydU&Gy+wCcStGJRNc{E(CcTUBmvqS{s6)#pteD|Bt{zMBSdE_uF-;OUJSa2-D!;w!nYKU(mu-a?&iWxM3Azqj!_S8|G*oBb-z%VgM4S3HbN0M5+^Ixib5>SI9CQbRnxlu<U(ttMdQ%U_<nf7~yU*8t*i=H8i8r>8(mL|+W+qI5Y7r{bF^9ttPXe6E*EoSL)=$JLy&&p_$YPNLNHuth$KtC0^MW>XmE&O9_;LS7JQ?j*Z$#=ccd1Pq@+qr7(hN4w`wjkCmhwG1`QY07ZX*Wp6Th6`v=S3P?K+3!7cV9bpDg1_Bg8Lh9+nuM{r1HyROZ<fG=3e^Y8!E%4=y?DWvimv8#_X7~@FLuFK{W>sXn0z#V2{du6++S&V2hU{^1_&1Wo!)Ryf=s3>%)4O*AiShIi~{^cynFi+5@E4UB7Whh1(XmzzkM-%`X_lDsuzZ_;mv}kc922oeW>89aF^NDSXv=XS4H;C9h&M2OGqh!b|H9f#q#}O`St11Ls=W_NbjQH8AT8d+0&f=W{?cG%Z3_2m~=&>f^T-QhDOzP<LeH@2lV^!~-xDq1!CdEt4%nj5oZN!(BJQ?6{MTI%){Lw_vYyF!IRpWzeW6N9@)0pv%pz+Y4mlr{m>x7^fNDIJ4ts<aG3Rm4`dJt?{=?&Es8HmB}ZH>vxXPK0Bzg07_re6w6`PZaIX2oCv$bvn?N;H5UOWOKya?v66;NiDg|w?9^Y;wd0YxG>by}ucNj6f7du`Ey*ga=V*~GHE~FomEJpqQ=U868i}(o&j+B~%K4qIDtd9^N%E|Pr~>8uqEgyPh?1WvBY|sz?+TlULF;dC<6@`4omLq-rbXdOrPm55?N~3U&1HR}AJJ)E7G}iDlmq6E;zIxgO=E$cfVN2iLj~pSm%a-}1lm?lIO&pwhlX;udTEoeufwn;GA`<||Jb6RI{R^bliy}0wmUTKg8LdZyyn1^DqkjHh{SbBch%vvfOJmJ6P2CUz|X|MvVO{gE5!kDcBm>Qn!nEsqqK@B=p$j3^s5@PuyWyuv^E4jhULY0Yq+nb-ndDfJ1=99yFst*UK74qLlcYLFoS{kq)<#;vx6clh`z{qhwux{AQ3V6@~pONjF6!1Un|}O%B&&h)t;zuwrJ{yw1}pzoH*SbqQ#EN8D>|KWpqikYRY93>_dfTEt7{uPx6Cubz)frKrw9oP!}cG^xTjipktwQ-ZH@w{*&@vnp$<Xo_+O716fK{?uydira+7=1EOYNGtlhi8s%H7p%dF8EwQs5^K-G|g`wh7D|#^2BdO79ZYkQ%3Hhsc#)a9cPfJbb;Zv$r2*t1=5L#jd+6HUEr7L&9`f1@`l~T6r`(e&zcVfa3lnV6s@$3aTEN~Ts=M41b6@}B^B=s`8o;nyydu`yoxLNA6huhF4xuKR%;K>04A-0WlpMjJ3`e6_^By=B4k3YX@uAZ2SA7b=v;7kto<3s-ICqzk@6%>rtySB@o9GR)fGu#InidmSndbb9Z@VTv{f4s*A0AO^7%5NAK_C^3?x~47y0K9i0?!+|fnZ>Jle5aW8Q#tGa7hDqXYb=m%{)1Y{j{kHxNaZXXZfLNEiTgowi1MiR7KfULY*XkO%^E~$s*E2eI2~7)t?#Rb`Fv~=OCZl|QZbZs=*@IJYPGUcxZf-@$$pAi#So!?^0&xU5=2HFSNqx|@(O`X$a{v^oo_UrN@pd!@XtpcO!=-Uw#e=Zm2)ACYy6vfd}#!}5InwM7B+;saw_Xp=Dn12+_|`r6B8xr*qu(=X}59?=>>oC;|*E%-*@lQj`8<`ZPCGf==WKIx8|7_R`n03A1qpx!e-dmugQZ3N#oroXE(s@nHiEkuXa|;oOoeNZV$`y=#QY86O9|+>rj?Z$LP4|S82_s*hS<W$A=JOi>uW76aZSnbK;Yze|lWgi_G>(p97=X_JkT~D5SBP$J)n+q6l1y-tm!+YV@z@c_Du8bKpx!>8BuMn=Na50{3mC+`1dkR;qdCVOv}A(WjzCPbz)T;N;BhuTbTo1s35vH@b>uCgm;#LI+c{^K&4vePGymyxikOkcZH<4dW*pAYmE`mPocL6FY@A@Wnb>z&{AFur2`o`z^Z`Y74<gdT*a~l_>Q-2=<a{aZY-`Mg`_=l^PnUAFv8g_tnjS7Sde!5>F8w7;74Xsp7BmfjSzui^XF_>7~wSxbia$DN-g(+#&S*FHi{>*~xNTT0}!|yS6y8;h|L1K4RfuhqH~Gl^zH+I}dH%hm}l%G3gRt8aO7I<b$fc@V8q7&B8*$Ucv<aYx?fM3AHvBJw*g#e2lZW*wp2Hp$dE)(oV@88tCsre8*-vhM$vuUsb<n<<r`P>u*tuqnkHA$K>N=LN<P(^SFO#KBNbz`w8`ub122&P2f=O4I^bA7=`ea$vi5hv^3*AemO@CCT+w|6QuLXv&B)Tk}W5#c!Z?r=%QOznfHB)LftHUSIK5L?Oe9UHV|QN?e~+j)t-!kg^OIAydmBJvJm>GC?H_CXpN|{lJoNAcrOI-o5Z9h=c{iMEZhx;^cZs{%ufS5r~{GsFCVK@weMMz6h+XJkCWC8)0^Iiojq@aofJ#nl(PW9giMU|!?Dr%qIRQm4#<t-0m)dCeFK3m6vaM^Dy%)8O866c%Z^3cF;a<bDzL^Bd*SUkkph+N8rI8LOB7>8HR$2y&`NT0_ejIYjES9l9LG6c-~^`*dZ$G75!OL-u$m0Ed=Wpb{Jtt{vjcYIC#I-rUdkE7Xeu556VF?98dH*pIy%;n{k2nSrqs9LH_Df52JI-$+B$aV?}Shs274^=OZGN+YR3zWBXGKP&cYumkihXppxw}w+o|vUw%S(KBJW;h2bf8ysZ^xzA{I99$qpTPbbksPF4v(x?n79DA(m=nQ3-_#kUa+r!uSR&#1F1+9F}?;EOsrh<;7-14vSFk&_jw+*Er5|dRN~b-oRY2gi*)sHM<Dxkoqs6kXNj&i;6DY6YQZ|fs9_j3Hx18tj0|urQ}Mq*Q{<rQ=47Fw56nCR|3Pv*e5%u0dsy|u*VC63zgcycI}5vTfR?8%u$?Fu~Od(^e3Sf6Z`7srI<bDbY1(eJ5eK%$-)J>T*^jAnpzz2o;bK3PXoNtt0J;7xSD66yL1->y4<w50CzQd(g@3|6rvHJecf(-cjd3%dy1n)t36a%Al*8-#BtsUM{i+~D_jIXN0wOJRs#7+fH^gE0AacxD>jqIxuza0Qfu?n4L_A~&izC+5Tb82*$TB2!z2^`PUvmYpXHB|q~ewJYMzO@$tt$ku~AtX*A-f}t*5kgFqSJj;)MhI%YDe<6GY+{k7Vm69pJ12BJh`WhWI)WI{c680)k-!aOL|r^;024u<ZefTV9IMu!_}l@|H%$doXPio19Y>@r6)@h;XhPs%T9S%w}bo-+i-nW3tlta9{yHy#h~^&A6e`E48O9&Jf{I?D~WufmOXud2n&5pe;lU!9HqX0bOAV6edskK-(mtZ{P$9Pob-WeJ5}@=dD%wGuo3je#qG}`1;@Fi<p8L8lr;FK{YUk8kgiM)qpqvnlBPpmyCR@A_O1A%+q~lZ$G)a3PX6iZb~IqJr$UNE;B%C#s}l~q<ceu4O|HYxY%3~=kTk*6ztPWeFnjq${;!4VL}t~hEvn<vW@nYPdgna0&v&*%eH{Q9Kg**ONZUy^e5MTjxrx{KG!%_=AhLV?*ezk^u&;Aj18zoIcQcN$pWCC=PRZf&O8$TTF^}aO*1J3HivKH1N5;{IFPU5(@-O)ik?Pt>{0iO!h&)GP2K~h^1{?MuDpvkFlW=5<_a41=Is>s<HUu?H3?Do9UGo-m}OP^V~J8{ay8!LX19$E#r$wF6@#ZB(Biib!pdL(leQxBgZLkQyAQ<)x5Zi=#mu8^%Eg}m1Aze$^nAuk9HcnklVZ8&%tuPUnRRw{maJq!3R}#{o#b!Jmte287>3iOSMnK*ObGR%T#-%|ougpr)Ai2wQsj0``8S#^YXt?@<wwa+WFDe{tU_bem*o7Yks7h|*ZPAnqgFGeUGCwPN<CH>CRNl1|8mLCnvc|0=#E@e*e7$LY{t!f66tzfTo_XsaR~?&mBhsJaS;Z;idYs@f?mVk=7c}!D;Ar??WxGCh!zgjW$8A;(?Vn+;>Wz>X%QosTTEi!W-sIZfO-?HJkpWKkTSF&?7uBGlCRx=URzLWMtR6R*SYnLgun!T<HVPvOm~b&h3D?HWlQ_I1ZjFM{czEGl*~*+g4c-16D6@=2s)icQIW1IyehLmK1J7h71qLuJW3M&hMun<DCe@IM-EI~3NPol0)_G<mMI2=4|W`qufT!J!?TlF9fQ$1&|G*~TJ!`Lf|Aw?qe=dzUI6x(uQa({hr;p$8NmKR3$rIUqQ0iC#2hO_iaBa$u+|li(AQ(C5EyPI&-o9uIR~AuK5_n2|B$0wv@}DH<#2a352>S}wu5gLu(vpQ!ZhM!=^r3XI_pG;aP+2HHX_Lq>h9NIRi}1%06+0gsvo-0(GS@vGVCeaix_tl?)#C|km?nZ?PgvTh?gO@a=jC(PAF;QP>!-&3o+_<^#UJbc_|^uLNEwD&gu|+bn-<)O@xkEa*66*o+)E9%w>i!lkypMo0D5TT~EJ!zwf4A>K|k|_5S$ZwrN??(SOn&9v-aXC1aTzt8&B3aY9GglL0u!PL8QTu5X?2TJt!!$@#pFZO^~s+vA=Mp4~6swAdqz!sY(bZ5gy(qx=_~ouxn|z9(%Vl*`@s*5r?r26~uc^_|f~oNJd$8#E_!40ff3%i;GjQKuk0o`8KiThTG8Nue_><rKB@n|#q>nuKM>Ds@sgKD#y`<?TAVaRX)Imh+!Sj#hTBqRm<`5umep6DfB9P)Q*~`zuthalq|V0jYYg*x*kgW31`!#d>!QIKd#@iwzPlCuI;yZwjjNA@8%0#Px5k1m`9aAu}IjF<7(;oz!4f*zl8S-H4+U&m0*tXEQ_!>qPm$dMqTo7=W&f(1sRj8a+6XyvV^<^XSX>Z~fozYr>eD562uUn~!<a(+IPDb;b<t^__Ilp$na3NL!BdOAi_{yA-g@c=}ruJl<I|XtU*eI_9+0EcC`%k=#{8pR%o$jlZ-o09wy?unItH(0TKn93`=`m^1mIx+1_WCqpR#wf7%Z$UTH@qoL!7O#ThorWDS<|EXg7cs$(~z8JR$y7~0bw><@5fs%GY6F-&-2T$shsHs(p&0;%e{{fYV047TnGKKj_cf29G`hI`>s6RPkvRa0W_Y)Gfk*9-nnpgkL^6B^t91&g1kggNmcoajQ*Qxfm1nCO#vEZyJ%Lga25{s{f_gH31tsx6ZJ_wTucX8X@j<R7`370a!s?v^uT+n)j!t4{UXs-e_R#12otI0$EJe?(@NiQwm+E^@-@YJ{N`qc`qd6(zeu6fcQ<fr-(V&HpL%oVMF7@kE$+n}F1<f-2VS284S7jwV4Z-H*DMxbu43oFja2NvAwaFA!Jb;rzkZd2|eK>T8+>L=E^qQhEv&5s)A6IO9vo|Z9YpK*d(OxG!~rAR=cx?=_lKt0UkSc2VI7|YWN_qbps;PU$5RYe}%WmpiXmXr}UZ@zBnbTV*Yc-=^ZyW#DtX?|mV17DhHi#N9qai8a=e-zn<&J#abi1PWEchw3RSg3RbJDsTJVBxYqtY}5hJz5c(3!CO6K{+M_J_72>lt0w%mJPQ|!}0WEAHM)s3W?|5t3^nU3liFxQEo@5^wpkmeWw}}%|DCkR;QVmEX2m8S(-;9<_zRx3*;o8qvA<X-K~||w_y5|c`F#iiE)8;T^8axo9&cp3|v-p$|T4S02!><SnUflw%G|Ibh*=yMewlQYoisdy`hMf#0SA5dXBLcz_-+5UEA_tWs6c!9yk~Xp_eqw0XXW6b1oKTn>CO#{xU9Q9<}A8_c8-&IpyqKo|TG#e36$g!NvVQIj^qsFJbS-_^?XR*+nGSHusG_=)6#`3wFrQ@kh_SzSRZuN-lFBtfP}51I#a5yoj$EoV;Z{$EgSp!d%R9Jfmqmww7kHSFky?Oe%hP1_wLS8gn3^ex)&<WVfPn`k4t{bjRU9>EcIy`Yx{x^=M5Uax@o%+`~d7RTaK~VHoh5?peDUG;aN&TjXDuY<)w9z#`2uo%N7a>iV5d=nCc>*1I@5;fHI2hOVT=&Pkox%s_}G!(^UTDmsbG^6+;px|t8WldXOT^Emt$J@qOpSJEZF`(P2Ls#n<9kio<*=erZ9aeg+QH(G-Rg%7B2V6O9N-=a6g1iRjffi{7f%G61>ht85tJJXGNx!d;LfJM;$6u%18143K0583k&u$4kL^`iCgLYi&%xS{u2<t}RSu)2NX0TJ(|#a<J~LihA+(us*et;#}+3p2o09g-<geS*f3nZtqzm0;O9d2%v>L1dSf(DKFCY%K4OLYahYWhM~SVl|S@3*;<+;#&xH#wy9<K?VWrX0K?{K5wWrK1DB_(Z`QC#(<7rpgjW2zaXe#pJh0F30v>)4Vj{k4$()sdnqqPcB^+Hlw91lGjO)ZH45r+$_|Ke!Qe{|;z_V&4&Z){$olt9qRD_Bld8^MMPFd6Nt~=}{I^p(2j>TS{8eBR9l?;YUMbx86f2V&mV{Z$F^Px&`a*3%Z+ay6zU@sEZbKDQjWhzWrAkDrM)bX2&>;xETZ`g#w6tZsVDJ?Ys>8sV^ey;{ihn&(F*R9cp)Wi+AiRhQgElj$l$qtxWy$w8Fc{(+@P+}jC=!z7`^Z=$4>&*nDc%IXl$33LHgBo$(Bo;THzMpolafx2deKW_!Vg|Zj0b0LFd^Gr{{5VL?nf`JL4UcH^Jy<nVCqS@Kk^8H-)(tG(gWsi-$z{0B^w=tt78r1*7YoR+2MX{n!^`4NDFo}B*VPJLf>7wUNZBVb>}2i0?veX8!ExDHDlK-44})JBUICd_<$?*tOIp)%A~b6n?v<@;+($w-P#yDi%J?u{wPPbE(fFRvcuwmRK?~D3^Lh<m7to|IuxuCv!to<xmWfV<lfBZzMn-VkK1u6HE2$QV`d*FQUwu?-4f79)H~z;lbn$%qc(N%|9c*WQY3<C#p_FE)XPS@VYnjujIY7q3K6;MqEDNP;tlJ_^C`mmJEd^pQ8_jv_&L?~WJ5zohzTX7xT0lK6roQvMGGMb(l84>`AwXlIp8>G*-V;bO+C2%vs10qcAs5J-r~r{kSDVV-}OM3HvY$(GY-Wn_X!-bypz)YXatlIPKN~)^TqmfP}yB}!|@fGnf@93I5oELNK<kt5QitTxgs--e`ivZ_pyWl@R+x2G-||tgyTSs9K9wz!rL%d?X^`O2T&I>4T;dHpxp_k79n+83acB$0RQXeT$zQh@k0m{*_L1P+$Y3%N=&fwNiE5=Szf6}K^~7e;qP2`Z>6x5hWQ#B8F>ctcb?A`k)sv=vsXk#n}%>&_id#^hV)|XzYdj}6#$V%2`Lgomt<Ie=hr6|SC&6>t+73h!Sh8bdgG;oTwyC$P&%w<4D&e`29?H$e?OGi61m&GdJk}={O%io&kTOlPVp`K)h9%sf$EE>YtszmfO`&K_ughl;D<vq#c%W|dv(u&m&Q|HQ>nw5FU6O|h-}YXx0yx5p9F|W?q~9tRQ51mp|4W1bQpSMXaNxbMc+3j-=b2Sm~bS&*x=RwOo=0)tO=-VCP(oaEWH2F`jnUKCNrn0i&vpE)!?_UmUSY|{y4}ksPSTPEGbkOj9lUzl-p6ScfRD`MxmR?n^Taxk8{^K|K&|937K9k@+)4UE0m_bI@;!Top7`TKeecH&Q&mEKOVRT(sPWuYS6!}&p0Mj4*a2xg4s#<P%0{%jZCTye*oE-%w-8#f)b)SdEGkZ(hVoC{GXrQ;VegzJRX`CoLxZ^J9xwa;xMqqSkQwuOJ19?*TQBaf#cJmZ$N_vO!5To*m;gC9F~x1J?Z$?7|8+}35;&`{)nPFhOOvv;*%h>>y})nu^3@|vxiGEZpe!(bdZX_PG`u`U^s6l&XGy|BK7>C%xzWl125~740iw-dV`HZZ(1vM(wh>9<Z1aGpz?&&D^Qmw-Z}Mw)9*&r-tZF+1Ui=XfO;OumwA)GtP>VDHIqpW*W0zQ5ArGRB`P`68=T?FA)$)Rik!tDt-50xz$bCdj7v2NYpGJF>s2V3#gWSf4$^Qv$H^xV@{eQXHw@CsnBBE|KW$lT>y!b<SkCNWTJEz$SW}!UtfACS?^@6*ozBm@W-{Dk=b5$D_%aLd-7+5mfv?@h#fuUKxHP$?XDHC0C@|wl=Zr=ZfM%4w>QyMmgka<RX?eo0P<HJ)@7+d4VfXw<>2ZKgV4%$MrE8^kRkd^-S&_>A`Z7{ddTJUQA3mK>C628+B-LZ~Y7hj!&G24`>zxY36hMe4+yfVhfY_+n-)0i>sr-iwww(3sN1Hop5o&lpwH&PBqi_7#==@Rj%NcAx>a%}dIG39)aAgq~ns>>HdY0uK55Wl8nBbTIc&Y4XEt513@EV9S0Yp~ykxWRYxAe9{fvy(y)S$s&9lI*RnzqH+GK1$<`x{~iU%o7Tixbh*t5v#nuouQ)(xs+Mqci*<@+%DMrF6WLu-w2vlM@+HECEJ*9t%zooe6o(8v~BgqQ(ZhCi~o<T6iIgPl&z4@pRq9{Bho|9wHkdFnQ)@>&aWHihS0C%=C(~Zf!*tAB5W4N8E|~0?e_X8ix;4Xd65sAki!N;@kNTE!%*7;2NCP+gS{`^2*Hx^MzABeddwew!%%P0D0s4FON9m)M#IMx8L%YD!yk8Bibo`khq#p2tKbt?}{^yJy(A{@gt*jE$CWgX%3;maM%Ew(LzBVA{5KBQ38Fe0!$#^PdKM|sNB*L`FJ7fL3+nLia}`MfGK{y;|-x>btuS0Z@EkpVQzrX0zTF_)Ke6UZ((TLYPJLHDBekz5FMvz9A^59#e83p1U#3nmBmj2a5p87s)JV#wT*YtX`f_u;x=nrPkx~&77iIp%&4={Y}RzcJ4vRW#VWrh;0k<L%Ee<zQ*jwcN~=9+iv60Hu7E=9*7kd>FRRhKwkt#@q*>6DEv2q401|bKD>FLsWk#N&i}Rv{KQqe1BEFaIdOIyG?d<Q7>^fO$)vL_tq^RQAG1@X$!n73FcPU^44~GmKw?Btis-m8eQM8Br#<wwOveAY%UTO~t9Cl1)*1`D8s*lTZDo^a3oj&jRRhfr%I-3fxrg$Z|^RoU$XkJC0DMwf#-Pg`mK#+6Jug-u&q2R?2_@>~a%@^hD1L`0!)E<{0U~EU6#q;B=8fMr5vq9w2pS;+fw+A*d)bbzlrVX35DwlKAp79QSh-J_FyqFSuB&&hup{#XQj{Tzly1*_1G%E7lkXAW)yLTmY=KtB&upp5&4UY*@a6;Qgt6sE70BV*TzU~HkN&>Y#i*WrDR+1)cxsHCD0H@hp{w5ATwuOs3CtLjqj<KH_n*SYy-B=aVU<alvB*v+TPpa9~WCPlU{R93o1@BP7YNsb=e!hh*7oZ3^R=6B+*e#uu>W<oidDq}*x;?~LA7PCg6j14*dF87jB2bSD<*Dl2!1X@P(i_PixCZc^WmzCZMw=Zcj9o!-ZK&G)G0O^i3)`~L3OBeURIi5YA4=EYem3<odyiJbXO(2zyic{pLWWOwi=rp2X}Dc;aLslN^e<J{6#_xoJY+t+)`XU#8Vpf{$E~=Gj22BiTy{6OjCQ`|mce?7-go<vP@iYc*G!W-s192q;=k^4nE_e>??m(iDbEsHm>D&A_D1a%nwjxqDf90QKe2J`I=DBVWWX=ViQF#9y~gC$e?hS!;Oy_blo#-@YGz1>kCX3D5!s}W5d(CI@_c|i7;X9OTYhBEwgdGZH9nYWK3xEov1jbLS*8VXwD|h~XK!Id0jG8~duxjqjez+9<*uv-v+~hZ)nr}lfu#-_GX#?x>8}AKrTStCT!SgZf<z@)q%vzkQp**bS=sAlo!hH+c+@@2v$b&`>2tjxu}88w3L94cVxgCKYNi`SdES@Ojb3uczbC;&Z2eo5AKQ@RMFK4(dDA!2l~UWXYk43d0^NpPY<V>*Q1RynN16{}_Y^^BlLM`RuC$D7&gjOsuf}WQ?75ANoeOyS67RY{NuFU0^2uDpTgEJ(>g+mKUDyAVyxVNa1)|~1(<kt#nAMsi`ltGaJh@wHdwrVt1Nhy<0IX{Q%%(*TIX_pV_K32Z!MbzVVSPFtSLK+Zrbk^MOj<`{pHoh97FTbWM@-<_`;A24VE)817ktUfdf@^sX}VYn5FHB7_?c4O`QH&WaHnz1=4Nesz_VS}0nIN8e~aqJbwzh^K+LdhL=9N$uQ||gb87(6CDdJi_uIE`!%JcMa^5+;y03bV_a&39NH`H_4o=jx&}YVX6n({9vmHO}&uK25qx6t~7^??S9!q4}V-ok3Z$4@L<tKU)<*aX|VA7jw6B1q^w@MtAoJBL+ZWI7(ol3H#qfbzgCfHFKXtr6N1O~Vya*;1Gm=0*mO56s$FeEi#^?eCb1C5a)!-tqf?@1ndiiX1Y7PBTh2p>M<#Hrf6MS>y)_p9Ud6@S)<X7Gkp?Huyee3}Q2eKYRDBJ_}MJQy?+;l|XfK)75REB8!V;BaTU00000A1WprmBwmp00E9#qQe3J3hnXJvBYQl0ssI200dcD")
    SEAM_SHA = "341d859be5b2cc619247ff7b645304b9473d09f07995358ead2ee02be116c60b"                    # "" = the builder had no dflash_seam.py (every D arm NOT_INTEGRATED)
    SEAM_SRC = _unx("{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;baV7qg?<IXbna~aR~z}<^`mct^RE%jBwK?GV_~~IXTx@e<)InE4i$BWQ1so@0$Up72S8|1Y<?~at4{w9!}R6eQqkHnsUOAwkJc6^_i~VL&mUg`sk>^GQ0u0`?`w6hTY$$)22m-oV9F0eh7c}e7)fNE*i&&(V`ZpIHB)tQIVR7hkE#+^k&=eTBNS6FM$$0R&F>;^A?i1cw>g{%bGrS^TDv5JHIjCK#OuzIijuAcr!K!X0%|xtrU(GrUx75?Kev4?bZh5Fzxog4tpJgen+tGWb-avl~zr?g}gc<>UHMv?OCL~NMs*ivYGU|J#6!h*Bqt4#kHje#|bO(+3uHzE%TYpsG*ie3?#^4ti)-(Gh|lS$nZWX;{hQ}dj=U+G&olKumYM%2e*}R%ZVTQVL(QX_W9Y$6?y@wvCGFQ!mHf^pYG5+$F*sG+^xo|MAsREwJ}NOQ7MNot3jgfYiW?axu07!{Aw>q<QAVjrw&AP!%Xf^<ZU5%%^jCY!gd&_u8qSG2QaheaxaEDhvp?FewgHm?nQ$6oxG_-IOwSqR!|!<Wn#eBzJ^{G!QWi$c|4%yL1(;p4)?v|5lc^|9O|BC#nrUhW~E_F$~Cm}fm&i)q`jijCg0D`=Kg#95&uT6MuRn5eYIWNvI56Jq_&?QYetSGt!e_?E_2F}1Ghq;Pt;FHT^E?UDLzVZPfg*Kz!Lv}EXuKx>au?tBLzN>N?jlps-IVt_|)ZXvE1dpk^pp*i`4G;Njh_WL4#?F?+#?P+Kh8L)s)LlpvtOGD52-ow>KyReQG;Bw{Y-Pw#reaxwd8acVa`V`YI_yvm`Y2t!OU!82qa4Bc#_i-%><CEBQyeS3yUA_qUr`-Esk^5NZxU3-dfoZ*R@{`bcQiegV!T_>E?5<1%LIO=ybLaV7jlg~9Gl&T!O-{fz92e%#uaZhzDgl&)I4nJiZE@~A`@V1_){XgYJ5&TKua(4fFE<uLc;rvC^3nkwbI^aea=*87ncS4ToX{Druv621{QLm4#dj4ZcSm+38z!r7vD^S&k_x5l)ca=6L!L+J$si39GH1Z>{<d3h0E1VKIMSWUOYtQ2BZEN<ezj=GdLhlFX*wA*NzlBO6w&(sP!SvE`aI1G;!dS~jKStEIFB@LX=h=Dp}c-^h}0ezvfgU~B5&OG=|37^Bc1oN432+a8!BnK3byYsAYc)%RiekbQG^l+Yjg6{hi9PW2tVX0_{JD|~#hCt8!QCkW)Fx{{O1pIHg$0=&m$$%=FlI{W@|JLE^IGpT2vb2av{3846jRDEUjP`}1W|JuSW|WHhxLsTc*<VMJ*`T%Uz~Ps#_+0k;6L_kpfM)p(9-6he{A*dGU>EqEpcOo|Jn5AJ&34>aCQ&Xie;BWpS!`%GO@(e455{GDf(GLoX?20VwIP9^u>i3n>ntjljj%|`J&+<A5L`ZS4Yt;svaKlsJ)VWhgt@6aObaw9T9G`CQQ7z&DCN$>ufoQPBlgp)u5$@*t#+pCaN8sCXq5sJ0Vp?##EFBV`#b;|2fy;+L%Ca?@9&j877FQDNQHxowH8<%=2Wnab-SM=;1rVb?dTL^0r_@@!sG2CE9<wp5Rs#Z7%<D81{&7Yp^?hag2HSt)d&rB2iOVRU9dS=nLT3t4D78uY`$O#WK`<pJ`X6nqK@FjL$1S)Zs?dd1|lM4(+<S)@1R-!X`d-KVU583Gwd|iEL}Y>LiLiy`$8RN(XC*TazUjAcPgORrb2qjSzgPWB6fdhG5U#4-nJKtkt5PXz5dx<zIc-V;?M^Sqj~m++mzld50%3SOwH4ENejL3t}>_$?H!9LuwYPELvTzSZu@_?%8!k7vQ^+E(<lsPKDwm02l0t41Rt32zZXf(jN_K^*is~D1lQ`xg9Maygx^zCAXP2FyOphmeAv}v4NJ}&_0n7QbK8$xU9UcHRZ-Ig;Q@WY_Yd(X`!ooQx=Zq;@mf`KG7TL3)S4n}Q}N5kEVccMk%Su5$=k@(qBb#(eI3_{r;$*10zi}dFpo%I&^PwQTe(T7LlGa3aj*G&SZ2g^^NF^bQ&TY(zO&yz<I0+~^tb?xDvZsQH4?iO;bB5k44hT?83W*_;6=Ci3Y%6V1FjFK(X|@4p?Pjx5Y3*f%+diT)e~mp?jxDfzUdJ&I<PHECFmI#RqV<;y>@;&I&FvU>#EDG6efUL(0ZyI`ncx?iJyei!WoNx#Sqhl2v6o}-+M<2NH_}V@J&~<D!r;6isK@vy@%ytv)5W<2}0F1PF6nw;2tlmQL&I@C(V^5>+=`glvgCxo9BNe;X02<d7(<A!4VR%#-w|@^@<z}cIx@_cZqb&^(+puo_>rl1{>@^=beR?@c%l>5=Zp`CQ&}bpb%b_NHtm#P|?wqw*{$XIlHW6!rK(x437Ru)*`_q)9BOiclq;b@>H7}<_-Sb4oknBT{*-4ttRX3&D=|}chjGvY!cNsogy5b*EX2t)^yhveAjcVWFlYFVJ&S=k%@4LQ|BT$zyP>h5J_>Lh@aRt3P<3<-J`VL2J|(5|IlJw_B&ILz8B(F`QD~Sl73|M_P`VwzWLA?dAXw;QR&onFKm7wJ1K@7HsP{-g1&MEW(o7;zN4cR*&96{ESuoKo%Cv<ofEh?(}F^S)><X(N?pWCWb|@n=}GAeWvpkeeiaMiq%{~meA?NyeI3y2+S-OOSTN_kjne1n0La%X)+o~VqfsS1VJ)v~#UWvsM9OmqFDKskL_%v8vTlDcYz%469)*Z~AMA1He(sGW_p69+AftsWy}C(^c>y<k-T(_8$9Z4u94pL{&|as<k{FKKDWOIF0~~Fo6z<(qy?X>}j;sL<mF;bwoly=($)2KZLjjiGaHgo|D2QD}ISkmOthC4BR4l=yC7d4NNjPQL1l};znMWWdrY2DCfg+LmxdKBho`hy>y-brpNw*lfMbNGv=1H;$g~jp;Z=2W+taRuwc>F(kPfmuEM81N?J9zJ=aF?E1y<Ao&412BWq!t_ZM_XbDR!Lz*@B>|?yu>dBUb`#K5WqUv4*Vbb$y(wB_*#5f`o-=SgFf}c#v2L>N*8O&o5t)2zC=4~gw;swS8RfwOIwP<#YWnzHj(B6dY#S2N(ad(tFBnS>X-1w4@W<OM@bg<_21u!Wung(8T~nBu~2X;g;ahoB43Q|3++gEf(|ODJ-Ds0cqy^S#Ji0>3;*I`O$rp;?C|c&1~#U<qb758(jvR1hDrk7DASImf~Boz`SjK1XtR$U))_8LYq|H6ZOF!bGLN~7*SZK;!hL)PR-pPx#W=?(s;qnCL;5AdYQmrYA00gLe<avwQ4GA2TSS8HWoaq3xM99;VTX>Hwj~o)J9<~w2H-Xk&yG&9u*>D!XXf=3yk`Llgu}yJBA|>20yVA-QYvn_pa?DfV(F3-Lgux*`R1909^`Ri%9WVx%Nxa$Oi|i{Gg)BZEb8#G>8+=m^GGr7N!{$XM-NTKMQ{Ftu)*)m*64pz!>O6Sllz4$y8D~LiQxFA<xqQnXY6ksT_Aj&8o1r4t7M}1=DFFFyu6%8M@Q23&KX9fZTU)*l73VQZgW-(w$mjRv}d)jj<}%}58sdwfr39Dt(okeKkVy`0Pn?2x0f_+BofzA?>pRF{97VA9u=zYgG5{}S;eY}2NayZYuHOr!F@H0*Vun+LaN=vaA=GCS#mAbml~v`-&Jkd$lcAN=r*^^#ydlB@=+-e-*TEn5AlI&(&7W6$RWl|oTqjye{1DwqFpG%X0pWwm-&$$BRQt5xfE+ZcvU-e<LJMA_^u1f+q#vT=r9k51#5Ad`*Za;iM<l@#KrY7n;a>ucHFgB*}2r4z!{s%n`Ma*PtPr7%gEi`>fq1cP7Okx;LDt76McXH^G-rbzZ_k>e01#P80I9Lw@Pa934uVcYc-YH3*j#bZF}|{$3$%phy<j!w3q3f%hUk&7mLV9?9djuL?`NegrQ};{lLDXbk7zMZdRxB)AcR*$?osmqswv(*!yRaK8rKNiH3|v^^9P3ON}~}gJV??)GuH~kc@Y<(m%>|($sNJbnE_7Hc4MQHa1%wqmT`>8IW#lSrR5V!Vu}7uUn>S7L79FlhPW*R{OjNg|lK!1Oj6cR_1F_ze2y`#aXX<v|Gn1pY(Vu!Bal4H65ptaVK7|cgE{iN%&PdymfmUsJ!anwqYs!55EvFG}pI59a*)_2`>kAQ_}LR?fAVyQYKyM`cuF4#4ZZzGZq%~sy?sz#~q)&6T}Gb_IVG1^PS)!Cp**9ullt{Yobz+KBY_+o<miR2*Y_0%>{*LevU>D3pBqI_cDLC=5r{7#%X{YS&S=(5|f$VMH|;o=8x`qL@gEa&L!5$ZIp-8=X6KoW}iKgl`?gK&@<B79CR6t=?h*K=4Vv*Mh9xnIA=`wS}o=oB@PgJ9Y*Oy={AS<YlFcgbQ}^zAvTSp^pAqwkZ5#s2E+uO7A%jF#biZRDg<`wCCU1<yH}CPf>6>8&RDMrQCr6a^>IJ!Qn$YX!c;M^Ztyr5XZU=#;sVCTqkCDpW!GL;HtpiU#)=cUP7-x@ssM$;!00|U`?h>ABU&r@9#qr+9jN=GwKihM5*q`au=lMYBQcD0Yg8xO6xLd;vH{J&fpyh>O_KEN6lYbj-&L~cJx1m;fcU$L0Uj&*y6n{;l|fO+k-s;@jsA+U!$8X~2Y<i8B=U;rD$2`=(G%-9>J5W<xkeGM7j3OK_S9BX&qEosV~Uog9hKgIvGG{OpL?#jD22zp>7410@xGv?0VT9BAe)zd2;ci^&_O6{Ds*)P0Sp}je(qWUe3HeF(treTp?S$}+xSoNJ$pN9=_ms-j=ejK_{!$!uGB&5UiG}C<4&i}uav)fYiQd!EMKu=>*gA8uUkdQ^Pe2?5p5U>_@leI<+tXXdTOBw{*-|Z^Q)CO3)J*medv(VJF(nV4|YNpW=*8)Y({s2c7F7;T?Su@!DE?*0k4`HkaIKVL3<B|5O(W_;dZ^PhLg6V)+zf&T%CvQ$VFUBl<v_HPrt{XT4?ra=kRKayuYU9lYvNV1sKz6ho*CJjHv$~#vphO#2<yidq3*lCE_2}*u``~XuO?S1MRkNS*ar{jU>X3E_de-C4#ouqXS6p*8Dt<8Q5#<!QkQLnLA3-qSH+;2g)1&G?5$HtR+1o@`1jFP`n2(l0Qc>4Q#Waxrf4|bh*B@1xbIMv$FhDe5h4)Yx)~83b8$ePFF>zayF`5+ls8f{5<>1vB6I^dFRcm8EQVJB0pTIKthz1Dl{X?D8xwS2ePnao>uoTD$7>{B9!h?#8xIX0NKs7pY4tq&YGnhe8itnIpz3N6v(KcsVP_v?TMIyefuf=Ps<D`Iz*IP?fY4WV}$sarI^Yw{jl4{0Wm*zVE+6{knw^S1_1YEzSMd>@nH&r0KZFswsR`IOYtdp3*Of@##RETOCFn{iO?X%{FL(K$gj@|-f_vEtbhN%O|}uPnnp&!;mS>Z82!Q%P#^rXY@+j2Hggj2gRzJU`^bkp_g%p@#+%6YF^)9Ea8ca_9&u`Q5ZvGlIDue$d**{tdc~CG$p9OwSsq%=0lcR*(vhBQB@O95jU`Nrcm=F~Snx^C#`|tp*M#@2{&`G;;jJga?fAIdwm-_t7eg+Ovcs!gG6Lzzxrdfz*}rA+mk54A<XD8k1H2*WRU{C-#IdlgSkT8eV;?-r#a6y^GLp`Fnlgyq?lWG@DNK<*wW0q+eAaU(AYxOWHE@RKvqOUd61NAmD5nlN!AA;UMRz<&U5lu$@8#`j5#!}{W#EeB0w$D%UXy}Al!p!yn=tKu0z>Um+6Lc_+M5J43%m1nhKI?9p&G|aI5Fn%{-Sdc`w53mUoNqI>Q>OmCP;>q2x*Kg5(fC^#O7+=;E9N~<Icg1^o`TPW+@N4jXnZwJpzlQyz&l{%jjT{ZM9Q9qIT6Ho7t4T-YcA-<P*0$Id&ZS1t3f3Et^*ato|n<4_plY@XMORE}Y(dEBpGhV?bRHt^#_ZX1<nzQ{b?yBn+r}okQ@r1^DJ<m5%K(KI`%O>TTbwjgEK&lJA=cTqT7!{?_F^ce*zYPdFg}(ZO!aDa1<lkF@W>HGxn}lb$U3+<{{Af4Q$TX9xv07p{H@wdge|H*ka~q->*U1j4FgjmUFY3im6^F)A3+!fQmazQ7Yx$hYoDT~DJi{=6CbIB>-4lEZXqQJ*3RUL(}O8UJN>@s~@iGjPhQL}4V*=$3WRn^)_Da3#gCVz>(-I#h1{h}0hke7I$m8Dh<$xp^i5k0(aXN%CFIz0NF%382Jky;<=V)zY>_9R`S*(s-Rd7Ftto7`#)$d>=#WnU<^~Nd+y%KP(qOhGw4_$c9+4c?Az?5v1B}YeIVgQtBq;KrM%l9^0{a?-}ibGrAE|1@y7wxh>uYevd&`{B7+8*3eys%O%1iL;(`(6=tpy=L5@lBdU;AthL8tPSSC;89$4K)qhZ(+Pa;+XJ#jVU^$en#)PgF-6JdK_UMLV(ByLYhfoylNwEjMwJ36%QexlPk0;^vB?n%l){Ds}H4V9Oz#kroW25@XOcS|Sl>h#ZtUmnK7T*~YNd+wt=JW2IRUtI>SNg%k8gnc5KePJv3!_scl!~K+6(OP6qiRx9p6|EK&20xSf#<JubZf$2LO4|nTtHAA6zEaQxv{&Odvu8?b_)2y-_$G}?|}8-=#3tBB<dt9+d@40p8=DVrUOeb)w194J4q{i>!uVfl(!GKO>(`n```YX_cUl9u6Fh7_^BhIHw`@40l)<F=V0P3%h^vp#IHc?>>0prQt;1AjxQa7kFT}SKOLBN2G7B!dYdh9O5Gbs%D%ozt=kDtohzSu8Q>c4Ri3104h-%VT6hEzVJvjFP$Sw2G}_tSF-3#Dy2#aJ8M%e3JxUwx<6Hi8shxH=aFHSt1IZLKGW?Z~pU|i%$qvr@(p6XI`-rNMrN@tpWV~+^>d0SP6qonx+Z{Uutb!HcHkHvIX>=q$6%Le=W7!uN<v?-u_RRI+MaCt*nUkmfoCgOOk)&heqcaA$R%zvp2P3m<dyZKB##*k_C*yj3qI{!5BdIiKnXY+~nt2AFLu{9xu|ztyc~u_I`?-af_^3j7HveFR2JtpTLl4X!<~;uxN`L9?n$umSZ>^*0<LalGW_^;!gU;K(2z`LD=rWFtXX&9LSL~6O?xaVOus<&alt1J@1J|?=g#XZ!xZUI-(35+E-W6nS%{={t4zFnoOq1IJd8LMY>Cm<kv>Bz)ocP4Y^>&Adejyf`X0o(Hv1+CBp|aG{#oDhFm)EDgCxL0-YihT@|B=Nbu<ooNby(Tl7NhXbj6_6%FnA11`8O_DdMY!K{7ag9FapJvYIQJYPA0rr;;+J?XioBLE>XZ3_bIX;@18}nx9~9%XcZsSWZ%nhVG~Sun6pfzcqQKHtsF>67LzY5w}9Lo+(zCrh@_*GQuehB+k2<0&|>EJ-!Iwl#vgx9pO(IU2dVJ6XvdTjxjn({JVx-_d^{-}v&nbQ`foL~v$Ewv0s0!>z>B^rlrX=sGbbQWf8L?WIRQ_R1!)NwqO)ITC^R8P%$aeb$>C8M!P&44hrt#qJ9_Fqggs6fLijzrenO9)3l^@8Jfk&RfFqwYDdJgIaN3Dc+a{|6>#YjnlFL?D7)T<^@C+V@7~a}x$lY57o)lMQuP!$-*!j1h!z#pm)C7fytUOwr+6_?vOtZjliDFE{vGEW0boa!;25%)p&io&nfsTqC)GP5tjjZ<#pQyLFF2H@3s0nwVS6hdorwV&@6yfgO2i~gp6vFb$B^wuQf_P!aQ<%-R7-aEsPfa#bj`bi%<+Bj$c(HLXIm9O4#J2oJtQA=-lNL@^{;Jxv&eJHZ{}k+!1#O`S3v)J}H4_CNHUd)e)>MQh^f*=Z{cp~NJdQ?zyAPPa5VddIu22Q98vt**L7;uLL7nBgOLUv6+)&lk?0oN~x<JFfW{xlxtTVv&wsj{t>6DPvDN@63=$6~8B&lmagJUcT$Mt4Cap~5KS!vzwaVa-kdYB;?ba}`p@ag#@)~c*p4<xz}b*;}kUjr?it%~+Q6<%3;EU&twVl2kY1%O4bhSRIb$@$&L$Qto$rgSKKW-8dCs82xjWFUh5gpUn$a2~^)yYW2owUQza_(skAe;}pmc_`qbpAMk2v0tYFDkjTCDhZg&^*vH|SOS#hI&h=|5n(Rd5_l?>ql5J01CmJ8s|X{QZlW5!VL`#2#w&Gk)_L(sf?P7FQ)v9cBjcp}Da{Ef!XxAk#nS?4ai7oR(u;(=wl}BbLw;4F3H2mh(i<ub(}!~)g-SKU*e^CB%g?2KmYU(O(&AJ;@B7Lu+Tbsa`_=*lQ^9>EfG10Z2HL)7LFinVNK^?*^LjExP`QZa+G0NlQEVOCQ3~C23efAl-qFuCK4&W)U3vUJMc`#4*2tx-;GGI7&ZicbEcb;WY9;1jc~HGb*ePn^TU_e6jPc^hl4|OP$tP2d*Etf)^kTn&8(pSyd?DpK2s=|OrYc>?c6V!zVTSGi{);Y6%-2xbCi0DP_zWoAhQw0}EF}Elid<{Mo<6}Ge!uG<T>=(YGmsu9>y0yT)5piztQMY>B5MRfuvOA`2HQ>*4RmKG<U5LTXS{|#5>$5j)!uB*TO7ZTACm4{x#A<C$v{PI`@rfU`*PL^`*o4hWVcxt+<zHxsX;zxT!(Ug01sHk_tl9`N3Lp{JKj$dT6w-LoRe=@H3R=J9vyg+!~^l-co=B`LVG%So{0kJ^t{;Rj8vF@9}}t3-Rfm@V`zO7llWYx<YaaS5hq1QWd$L4itH(c;6@kpI<t|-(}^*nbOmt5_$V0C_jd<F7X0`}xLzoX5RWD@WV%=2OBp115PwCyYndj<9lmvG=dt}CcFkoWfeh}6dD|;*wfcr*k!wP$#(Nx>Abk?5_2(9Rd*&$0x!>vF%4Z$0j>ivhfk8t%1|k$9QXEE?+q&N}LqGw!i8B9Oi=T{|bju#QQ{$u2r&BnKYaO;k&U+Ea*kOBr-*8AP<*z9il}EqQ70d*fn+2v;PPwJnnSG?MWU^#fmR;`^FExv6ujC0>C?Gr+4tUNH6IxM}G@jci{KDjv_Kr~>%)r(F#T<%}VnsJl#xu)6%fXih4`?K>+H!)Y*4&riH6-GgdmqurpfAI36iuH0LAFGHrz~F>*J-OfyA^{YQtWPlcXC~&oI(zSR_&0UlQP}c)^frvhyb;801@{JV<WqrcNnnuwYfH^2q+`SAW^5Jloa*c6r6R&kpb@NF&QK<4OTdqNgK2m_PfXn4cwIcd}?8Y^UEC04_VBD_It1oD3dZ?&CW&D?M>OH`U-1$t>ZaDI2oW7s<++R(kE!!<o%WXzN>npS(<l%%kY2rxt!4wR5=v=B%bOmBJMz{O9J(v<F}KeOt-j~fZ9c}n)>p>Rx%K;C1IG*N78}NqiiYLG&46P1+Wo_^-N616lQOkSvJYlT9O4LrtU=3Eq8sEW@HQ~jx<{xOe0y;$1L6ht!YeTM5?voxSh_vCO^6XPmT-7s#2ee12{Xn829c72QtL<;1JrXkjKr57+vz1IPRNkpY6nmsoqSb4AL`~mk5l|b+V4FD<(*p8=gOmtYr^xbuku7YD*b{U!=sK2A&UzG(x1@%o|m;g2T~a=J<O(y|`!hNj(E{)S&QwdQOMT7eA%QG2Loo_=3z&of0<120%{W^b4D5_LvH7?8=kB+CdF-RZX@)maxWvl@#FwAUEF!STVIu>`S$zlNB`scARazssS-MAH%!en<M;bi(BQ>(snXVi3_)nJYU3oF&-UchA6!_p6y1p(yQ1YyGMVd3#3y;232equ#5!64g&ViH|a`iM;1Vn2b1jR3f=ut#dfA~_9%V^pwrO|w~*s(m)dYY_a=@?UBS%5l1~f$RJ~$b!=&u-libFq76CC|6bKPF*Q=K`I8l=9;|9HbT#%jG<8fosL-h{L@9y4ds0SEfVbvzdoN<K%xRK=~XkrPwyfp7=@myS_{*Oi8n@^Q=cqgsiW#Qc_qP99K1fjHT+aFTCw1&l~o7aI<k+$pwV6`}AMcu0ubmC2U3ae>XMooJ$62%IpvB^ElL+7u`S!dAkprZ%R^Z62mg~u+tQ1?4jj?7D|{VCoxY3$J**2?vs<u0VyoEMAI*j#3QthGaBT|<l(@^YdN93|)}jdap@$`&yunWwedOFdM&Vgz+oPP}hS+aA$TKtuq#zgw$I@Uhy6s5_Xz&q!gs7IGtTEUiB3!Ag+-{jDFI@d##CcL(Fc0IgNHUJ?)V8@{w^ds$~&kKP|K2Aj$wEvyc|g|OEF1%b9k!9=_Mh>LI(0rw`cC|{ovZC5#k>@N&JCtLTzI$tRf==7u|?1DzM!bRFa+ZRmSvXT*-zHy%fJ%85lCN;BMw6d^sqOtw>6)Q_Jw=5~ZyKH}T%FLs_b0C>EkOw~JD=&v+NKI@2L*>WE%=sGS#mxv+!%)#iA;)E`u$^Iu7yZtPwO=POarhM{NFU8P(D;15T`L9gZ*S0s5@K5mo|ME#S?U4h4G=s3@2089XC$C04~v3`p4W{M*`mNMOG626mng`nYJRD;!b-*?MgPkht3dMaWOMXcfsTN>v?M1B?Nk?uZ3E;fqG;&nSp`HEks9O36mp01*LueDO`Xu!?&1AFIyGgPLGD=`p8rkh+3@Ozfbs~84`1Uu81(5=FFe$kHNRrEm%-AJ+Zm?1KDMYy62{bUZOeI4HD4BbgTZIp_6MNN=Z~ucga$I6yVn#WsF$2)21Ub-99(Ng>G%!;RoG7i5X&#Knz2^$ud7+73nT2`T8XcY^oQPE;F0xYK!34Zok;A`lR6PsQUtkE)5R<gIWP}G!!A~*@X4##r2ippTxk2|8K5=k%>(Nq`6@H_IraojV3aSfl>EmDNd!G!X<cNZ#A^l5K*V^@IEHEAfq~n5R_(FN@GQjY*XFe^7tmY1;nQc|rDZ8)H0r5C!Bl9g<?nDRJIkm0qr#}61(XMMGXbp#!wg&z(GfDB<1g%;nu#!V<G%qlPb}cj7NRE9mYqoUqX|2@kK0@eL;N};4NWq~OPtBfFFaokvp{&z8r0=J$f0Px&fPs%y+YsJERD;GFtqNT<;$Z%L@U*aK`!*AC?nOVjW0VZ`NSVePLQ1TCr-(x`jj)f`CTZt>8iYf)|i?lirrH$FyXt^iCiq(rcO&*Lh1%-B%813_7vDsYA#>qqZ1CwYl=Gzw1*&WtSLXQ=^4@?u61L$^w==@kEK2{3&X9{a3?5<QfqgBh`X&dlP-Pk(`O~b=X_m6Z9_OJ#UcbW)B2ss$6<UW=@Y&w7;z>yyU&up?(Z)3B2Wua<=`k{91}0m?Apce9>~zSzxCv)Vh;Qbp5kf>h^kDXX5J4koN9rl8ubM{iL!xCP!{!YK~IqJtH2VJ_5E+K`l$cBB(5t0mjwC)eSFIN_Y|ZxK)@sb#eJxs%)y}|mhevf2Nv{YzoifZE&e(Bek9fddzYpr0KNF5B*5u2LuYtzvdK}IfGWAVaRna6S%0#G)7gend+Qgn&St2Ww3sT2b;)1S!aBG73DQwz73V@65vn%0tu4tFdjZTEw0@%T0@Nt&oasmHsIZ&Ae}JT^bOpZ)NST1}n4b>I^9ktcb?9qNXRa3BORc4htO26f3g47N0C2!yYUD3Phm&pCv)O2NoJQ})RVj$PzB2*rwSsoE9dA(CpWNFUWKy6=-`wbpV1uB2Jb^xA$%x2~L^C=n<%+d-anPsFSXVaoc0tl{-hSRvzd4hC`4ot97t9hIca=y;Df!(anIM_}8AxugEJXy4l+8ir2X?zO7GVyqkNVx6+v_7Aipkk)P3S3RVkRaGtZU`&#2tP!Xe!JQJ3UJ7Gl=4en6u_2_KIi@uSE>({W<H?hl9M#b?(W)K5`E182S_Ou9bxDFlVXjg2IrrvEfOtZzM_fA%pC0S2$T{@_N&tP=sIKBj|KhVVtd*J~O8?VN%v^(e8qPsMI{Mz%YN##VKae3&bvlX}R$i51OZmM4p_Z{O=B!c=vcMYE~6tcGw*(wvgotQwq^K=|`mfECit73VtoK4T9g^lec&jD$s?cq04>JSOcD$N7i8QOXQ*!MY&a^{+|I11#TsVkyYwNT4699!TYua&R&o)_*`a4qCA4IWy)a-Q)(4#fRcepX0P`29rPGC7cDw!^$y4XMc{eod%mbs(YlYOUE&@V$927e$>3^&F<ZY^q&l#uG8y&!z$*JgrHpFJv=-3on`}+oRYJ<{Qp$KFJ+g><QmKw9FahD$s3%r_3B-Xh^C{NvXr##v2)Gnz>oA}JrwiX5LQqjmIk$Hh(Bu6jJEw7K-%X~i>23PkekyjkVH>x?MaW{eU~IG>5b?}1J-&yKPjwF2@<iH?8t*gicwwzpUy=ybVxX><CaSfOhLi_2&BX^C&Tr8HES9Qwg$G|AK&Ey2?_mO-|DcWS)~aELwid_dY;T_)-97n;sVNYs@1}0&De6hIBZ0Qyrgk()Sv=K4rUcU##gL?uEV-J$Y6%ud^o{P^C#W~r`u8kURFPdDLv)X&Mid=m)sl?Rk2O8J06F~vM5IU{$9X5`I_2(~P%ly@I()aR<BAhrzXIB*jvpKR7nSx@QMMJ!S&W1x^Wxa&_lkN7j9cJu1|K}!7y0MzGMJbq837hh1|ZPxzdl<cX+D`&Nl+x!n$k~FD$B3H3~eW%DzC8XtStn$F-T(1!jU~qVQY2_%qg431sP9Zzw;f99~#R2a?76NEBkZlsJoo|O+<1Zc8O0l3;8aL_nO#3`Ew0o0~x$$MXoHXWT;-B<_MAf1W%?RpoQfKRunHT2E#e(X{kdKnCT6;_vlSBqV7-@pnmX%Qln5w{appJ+1PV1r-f!dY+2MD=LW*klA2!WwnYS*=g?_o&=5Gp2$^6QL`2?wb0>A{=%dEHDrJ*Pb2J-cjN~7C#a)CoDQVw1+O}kyu*ZZ80G^`o%GO-$3H88Y_q8wASoI*J{4+|ACsE*JEUwmXd*K~iGCb(GOc$-4&~~-22}6I++3&Y6Sd%}rla5fRU8W&5#aS}Wh13nJsdJ7gVy_nPB%Q@0*r-?R+=#i7C=f(^p8k>`=7IoTW^HthCQ5|2NQ(-2eI}e0>aC6l7qL6TtU2Np)2^ZeYlZ9`!^d>C5*SgTS|4{U@VnP@Pk2`<wSSeN{XL@?awoKq^Uu8N0ZDU#;k7I`9(c*Fnt$G>WUDXa1g=ODneJiB01YS0-JUObnP08ip*umgZyY5lL*ayG3RNN;dfEZtYGYLsL?Dr{x^;VTm&^2{!i21*5hVV+TpZK<TxcjHZO8VYy%#;b27;&iXj<sWYRIi6z*%+xj&`S;-$3{UJyg4qfB=?~6Uj+CXQf)OxLnTu_$oHq4YnZ9q~I#;?*eEx;q2e_9N_PcECfuz_y{BN8wm-l*MboqqW&vj2Qy9{cDqFGc3xLJ|9v}Wa{xyyy1cE7VI1+g>QgiCgQqh1rwp~V{P7&KSn;AUH{v{`{<~lswkg@%rTSrp<tYnrY@;`Szk3`@#5XU*uyrzUlUbbZk>ifmJs%XEsLd%Q@hl1^+$Q${L2TIkIk@=ce^FEtS!=gnpYxs&)ORM<2*sPA;Z)3I-YM=8Xh&KmgOf}rlAlb~7Ca<Ux4zK-s>uW=Yf!=ja%#@7s5C#@TQ3gxMJ&(G*Otg%^&~(i$yX{L=~R%`v(BanPi2(861{(OE5XzQ_D`bPy`u~8g3~jdjFzBqBzFc{cW1KWO|Ga#JgqI91AQ*kNcH$54(WI2oF*y;sgAm8x}D(1X2OhLUsS{Iz@sSJ4xYuPAv_LtoFVooq&X&=aheOLuNRj!Mq(;TcdKruUTe-|$vcAd7QrP;qHG5L54{L$Q?Grd@2=amKO3Gn)~%buuUhi_JPjw^naz6;KlaPbj2$cR5LRALc8SNnQ1tzL_Nrn18Q|4Vg)paSrQ$@Ax!(LzDw}Hr8S9E&)fn)oiY<?FTz4bJOymoDG4R!7$=hY6h4rAuMev5K<+Z3$8yJ#dg)eatE1S+?gRo_?<I(6&a+z3i(-DG)#EVn!So51g#UK#k!7!>+Gq@qr<pTbGvI7EZ*3bvu|A#D!YoO3+6qN0ze4!4Z1<>E83yexpyF6IFzhm`KlnNmUNkuQvf(&Xq>r9oF6dPJTxlKI-_+}bZt8}U4He?}^fiwv;a+;sS@8H0Wny&wDEY_3YWitD}@lFjj9zm<Hbw~3hXzBHdPsfbPT5TY7rF}d%q)_kDA^y5AA})4BssYyrrzBk@YERnCpu$X;_D5h7^S_kT*GQhy-iY9l^_KyO_kh!Ia0c`+mmV07TzqHJZ1*;9%h&!wV|nF<J@nDuY67j`T@F-hL1f=Rw{F@^|AY)3dAn)8Iek?MQw)=~=Pa&^uL}IIe)s^ME?!yl+hI5s6&Uw8X%(4dX#8CGLdgtT#F%59v;eYev5Ei?#wV*x@1>LpExi+*)*c$T)kH<6R6FO;wG#N0lsZ$e4IRK%T?GJ4&M|vu5MuCXei*7q7ETI$g2VCYef~5ZyhVHz4I$sjSlWvYNkJqx1osccxC3rV_aszbgn+ZTY9Ow#ug?k)c23{-lohdb&)fMbT)Zibye$qfNRrcdj}?$~po^A=;IIj2&c%Thsmy*r)`!grbJwGV09n)av{Zd5jWNDS`spF8G=Bp|XxtX9qDhyGi1K0?0@g6hfc>f`irX-3#ut4unLprfLRn64N-R~$JSTz{vS$1`7x8x#TqtgShiwRuU_TvkaP2jkg9I*#dAed+rYpgZEYIqXBAYx(i8xm${9bs7Nu(9%(;tROXU4-M7|*6-u_Z;w3{;8cXmW-s5#)13o90zXrk|A}LPTUBZOr=M-h~x5qAUnS1BoC$5|k#<cwK749j=4;r@Q?Mvb<MZI#Nh(y;iB7aLg+$!xMG<r#HV%R+rGj+a`;rFP0Ei5;5+|nB1oy>j}X%l)qj*=ngHft_N_{?FXZQKHGgzkFndAOHiFqXXp#SW1gM%I?>9ZCclZzsf8R0Q{XuBfR9Z<I{)#I&5*ZiOD>P>-H+K<;#rGg5rBQkC@MQfvW6FeX_kCszbh*bR6R^Kh%ABoHA!+}SXsi0L_MDut=}Z#i3}pSSDee$9uO67t`CAZsr9^71hp8-C%}Uj7t=tj<p?T}oDk%lQBq?Co<y&>Gu~bK`91#t6nkMI|NZHUvP^nbAb;_d!%t+kdU1UDJGh+j)I-e*At%0><f7DnheYB{!!~eVf`X-kC!104oLiNL(QRk_-?|0uEJmLe^265?9ekAx&QJ#HNa4vm@Yif`m<aNqNnHAO>8J8PJf0a+hNf-kZsgE@x{^;m5Bm^R&#oZ2C%v`gD@$?Wb0#c#Iux^u8ns{SktMim>4^3Bi^vu7wB=jmh~A9sfFtRq8C50+^;2?i?(4Ql`b<CW5<mmI($7N^XbygZpHAX^dqh$G8x@EYr!Sq-?dNoREZ8RiQtihSc#_gr&;ftT_!4kw%)n`LvKB^?unN^ksw{_5DKf>b7^sWbW&NxCk!N1EeITC=pn&5@94)#<3BRY62ET9gG3%a%pwCBM3_$~{YB1fGh3A+lF9hY(AzvW<Di{BMU0OmWk6#Ml9%MtCvcp^I<=~_t_KkIrNzyfbd_d3-G_i<~N#R8p?f*7n4Ov#g%se{bYT|!MHO1wG1jQK7%eAGzw@L&zMme-kZy{FDM;Z~+E;TqJ;T>4HhF|oAyu8>f-GL7jO+anwg@TsEE#UGX*`@VKC^;rhOu*H3qp6EVapt$bI0d}bqb)y4R$wD^m+xX#0Pv;VLv&b_G#(y=BOxbPz)StU8(3imQY3Gpn-k59p*7g;@`1Vr*4iG`PSb|Tk)rYYeKseQ{^ioM@02a88xHE>T0DmNt}6y->j2wV&<?bu!*VD(V&oLVr>5qUO%}Ja%YY}_E*>lFnk)}l?zKiw%n?O$9NcHep)qJPD@oX~QJ?3`?_=CGc>CPIe+8$ic0Yhyu@-TJJezLhwB(B=3jL_evKw<J4mbSzCtSf9W74>~ocfwds^r;lXu&k=OmJp;b21fz95v~IUg`E)o%Wen1?{vSd$kZ#Y8sq?BDM~KQ$T`R1IDw7!qeICXo&m^iq=FiUF8XT1N51F7@uy|rlb8|;Hw>o%>IMSRVy@JIm;3LJds_sXCqPx<!2%Q>xbl=T7EI*`E#L~TKNDkWBY`dq~Dk*n`|a59iIY3a7*Da?-#YIzbKBL?9w-J6f4`o7gbR|d~aS$lei}QVlqeJwf_)Jh`a~lz?tNo2P3SoPe1qV{Xy1EaRBQm(}sd%wB7hOFiS%8kc3FRBd@^@Jsy&(GX;ZOGGL5*6Hnq+j2J2+39SqGR7_Q7&Jp-rl$|j5hn66Q<%-(}9yOq`%tfkQHgAdUx6(TsuadxdTqI+4;xJEMcDyI6PF7?6xOOZ>pgUY0!W_Ff@@!#%OZUqgHKT2Z8<EJ-UWNfuUEOV{wB|hhkT)_|REBKL_5MCDXm?+FUb_b8W+MlGvGn1LtZBF1Cxtq4MZLV(MToVw>fYWXVIO|3lfvL)FmqW>Q1+F)iTPFfo{o3A-0il2#7Ph)UqL_Zs8*X|e+%(=gozj5OZUynxxxztRChs)l?<hNH>#!MZ?4knDVP#O^yFPXYhGy8K)a(dO(mtn20-me*BS~&3WAHyZlFP%_5!%z5BvJc-1vneWCf-EH1o6p!%1YxWo;df?}#>3D)6E$rJ8c7+n3mEPV@e(^tzN(1yuf1g;Y*f&76UrZ%5<krfAnFfm$*@x3dt0AIG?bB3E&fR%5HsyIP%{?3!UfP3|v&;bqV1WoHD-*zu*1SPHU)EAg0{{sdVg&|kb|;(|~c{P-tN*Bb~8?&&c$lW^6d3@<Z@q?;HB`A1GfFywg3{@7=vEcwVs%f?@T9a!ShHFD7^9YIpM_u)ewav=e79WNAsoMgbMhw*Um_j}qkZkn(Er=!USrwwR`4F9V)T0A9=CZXsrFHbN%X8NlPa_ZL&hr32Ho3W&wHe+l}F9gDicZ51GE-PLJsOV)6L?(3b{p=BL=~Nz-S8BLz;1g!T>TSv|vv@GjPB5=apYKzy4A|e>^i&}}B*!2l7?Qf|_N>A3Bg%TU3Lk?83r?(G*umU-xTYKFikp&ml(}isG$!~K;sc@#)R>S=a+M?hf>JekC>KlIFoE)?S#k5gkS#ErvS-HlHaV!je<3x>Fw%ZUb%DLOEo`2R1mZ(WiIfTT(#Po4I|Ipq8LB(nkl`%-o`N48VkvA91TnV(t8nFB2))U}C}zN|{wZ}WOYBM^fBpGn3d;F;FyRF{oS)E<B@SOJp`VrVG^GU-^wzp!6TL=O%-;(JWg@+6^}6%yyW>l*0CW+no^&F--l{*@m3uL9<fs4B(?BEJK(|c1Uc$0^uK<p`QhZ<pp|m)7WW;vS_x{}>awq>`*h$*vTB;h3^!zh@aWES4l2K-tR&!GgVyBp(OeCk)vFr$4xGmO)u3$^T*`^UvK7B3cElRRse4I!)rf`Aiw7Sb66DAM?$<8C66*2F@ZF5qgYYFbq+!^U*Ik~J<CYN(u<SW^@csp>#{tCTE^d0u22Se<0W$RFApnF@2et=oc%|aoOhVzwUiyEH(T^sNpB#W$vl&byG)D&nF4D?HfgZDomoy~dHc-oD@4?gpVAiGOm5(xlrZ$tz$Q{GaT`^SmEfCdvjaK;9joJZn3SlnI8#lCQ}*8pqo+@7d0XIwc&f?{lS-|RvqziqB-MS6@8<?3%Nk>%ycbMBbm_2`LYuAarnPvB;;=o}7FRf6e%IGp6K<1oA&bFl_U#|0kfdxRfzb2MM=SC^T)Z0Zr)j`=~Fp9r_CW|+3_;Lg_0m9<2PC+i6W+F10^%sW=myG_752qIrdpKyva&`vhZzOdd8l@~RT`9>4z|0Be$a0i9)=f%l-H2FUpw50bOWFHeKy&L)$$TeA1GhINWSowomHZ@-5Df%AYm8~Jn2$_vxpt7Nh>`QD?goZx*$h&$yZ5Et}2=#*cm@N66Yyg{zJ+D}eN^#>1{HYoSdMs1qCDV=vbCfO@8&N~&z3AVbz`ki~>eFH>#I+aP{fTkL1n{>Wh9(woA*gc?tTUioV+ES}6u?L_GK|G7Q6G{jd5_&}a>sqW4PK5=v`?`-kYw)~I^+l2$he~tyU=9sE2K2Th>=m2!&EDZBs-!w+3?W3o$Do=P1p!I-8Q+u2lnVj&_{ekwou!AX(wgltfX81-N|qBM*l`~ZM*$!>@9CR-Qb3bP>1ER18I0We)U(YMZ|IoHk3nW9`3}hCHu;r{YsjpXcpJ+(99SFJ~TR6>IhYqfdDk*53z&x>O*}^DOnKrTD6TG+IAK;>6agihUPy&mJspBF;DgET8%o|dimz1Jo};EDS|a4172GHPJSAGGd%t42B!vLX&wRn-bZ)3Qmt=Lz8yY%7_Jt8^mKI70h>Eh=MsE9WNiypbZT*b?smw%I*M*c9vbBq{sdh=$?Q+RY1cBGCRh!gdrb2slo=0~lL6V;E8Smw4&u<XxR%YG<eqT_zgrFcqs&0<OjkI3hh~?b@PKjRdrx0fiz^a806&2U6DO5mnC#Ca*fQ9M!sF=zJeFxkJ`#oCEi@APl{@`ySf@R6_UsmJVASNbRNl`6gU#=hS~yqFgrsBdGN1x|eT;#+KeuZDuMf#6ZlJ%+syEVBO(V?gyMN_i(}(M=p(iT~zKTPB(dB-<C9QwpUqaj)rN>W5>W6fAQCi&lfBX%<?vv%&cdVtPyGtwY>YLyFa%C3pW?p7-@+j+5601S6hHc%4Z#Rf<Ab+P3U-yFXls1O~J{Ln&J-Tl5f!)~$a@`ypX%O&`RIY<Tqr<$W4zX^Uzj=jzZ5<6}!<nZ2{_DGF1eLekgo3O9MLhS<i=C55Rk&dG=%41}P+p!af*uQ9ZZ>|JR81YDk)uh-%RwoCDLo0Wy8D9U+g8Wej_xFcBruEhanz$EbVrUu2SjIu_3=1&_r$!AXa!+clR|R}D_kn@PM@^6G|0sxlXh9p=fvD(B(%$TbJhOU3xY$N2NL>d=<=mvT57sKK7A*zVDRT<WCa%+Nf?cQLbIsyj}+#0p|uDQ4b~pYzO;gm360tCgMx1ELg0g6EJpg0jZ{KZIlI?KPf0io78HZUvTR8jN#d^E{We3-ZXWcZc=W{*l*!c^w!yiQp+L>RO6*ZDpcd@DZ;r?v=&=8mP4z#D-Dr!B^W>vvmM66Uil=8K1Q{L2WRtDEHl>%OHf0^wqC`Em#DAimds>R(h*wF0c*B_@B2de$rP<j{16sZjf#%PBkuGC?j}*`pL78mMGek{xWK*dVpt6#Weo~(QeUg{{ZF%M%(&}b-%BD6VnE>r&ioK8a8OewrBj8ycChqz0uFJc~az2C&3ITPKj60qZ3<svPR_oNxchEv$y)qFUHxQ3O1sxeRDqiu&H%rGA?BXn|qc|lHD=u(SK|4}&ix%Y>TKhWjSAHD5KP|_HX~g|E4#E3+Ho2LQ#X&C@0O{uns$TF6z}QV!9YV5gbLgImFrMUa<qi1^=#wF;4MNkJ{@=eF?sag~ZU!CUD=Xfuf*IF<d*E6Cz>k8lsO+i$IXzgRMxoYte~3kU?`k>G#LlZ7F&nah4kA-1@Prlo*9lt`!m+d^Lyte(=Yq|gfH);s&TpYIq5sZ?$s^r_@@WzpqRjL#L=_-4&t}*;*sp=rpCwLMAV7yre2YcRiBO}w`fF!sK!IC~F(0-7<LRcyfRDqQD&MPAT0-#ih>*~eOa5hxczM)iRS)H#cGk#DSGHw)p+x@5-S%WElZn*2;O|;x)<6RYb#{vH)%RP>`NLV6g~{+rCAyBF0H%|Eh(kBj^U;DzbanmYHVe^j<?(~0&16?GZ_3SJ*2+;9;E8-M?SM-Lk%;*HjgpB#x&bP6*0I*S1-ejp5sNdGcw=-4KYwB0FX`d%wTKIh)^j|CoQ7q_cK*O)!agDj0UJ>6P^%XAV)cxcI2DQuviX6pF+<U9Y4Sq@W^8x2Y7&@nI@vl(bgHOErfGD|5rmh<DCrNrBki{)Xp~|Hu>5(T*>sjG6<+J%ORkc_I)I6}%CnDfHT_`61@$cnJmHgg=C4EV+ISF51Pu4kzguA<+<S=-%%!zDbmV1JDwI8?gw#{=fD$p2kqrS8{&r}i4?|YE@^{tjz0}QX>&#*rTU@=$skY8_7bfY))I$Ej$29@^7)l`N2)6g8J<Kr}*@%AkH-Sp5GNGYLL7MBaK<+mYE5x=uYqF~H?wLbp^Itwk=IONQOKc{?W$s~z7a9UuXGl1eQ9jxpp+oNN*d$)8Ac)|fzW-`>U*<4~8h=K}#B&+8)-W&R+dSvYcvgtWlpeq9PD1F3wm=mW&U{Wp7@@i=WOvrL9;cJ9h4)5C#v{<%g2jR?w$W*UA_J+DV7-tvo`jq|>R#i8DVPHzj6}F)bk586`RL4hi|}N-Df%Eh139?0mES;DAt{t$iFdNAQ&?+V!q{rRiC8d?x7m^UUo7xp8vUm{0$bx>0RRGd`KFzFEfRZ@8Cf=mx&QlPgI8?BA82Nw@H#;Sk0cMWU@KublbBbrw;Z^9%wLLc>XjfCM~m>ynJXBX4ke_~%Cm?_)mOr&pIBTuFcVCg*+^RrKcq@0Rj6k^@uDu#-2C<SGEanm9-?J*>r2QYatW9+7KV=xneWXNhGW7VGf>i=<~nLENg1uUa0h%ITacOw-KT}De>%Qp&qhv&mQ_J8!71cxN;)gkJh6#i@s7KRs6b)ewxg;L^I(@3ZS%n;&=rgrsgbiVJ7m@_ZYh)!#pl_twgxwCBH7YP`QLfZZ`Xv_ZrmEFzl@Y(K$Q=zot|Pzmanzj>9tlg>q#FzZ0E_7s;)V=q&sK2b`={wE_&=65f0CmluOB|*mDCmmn{>9#e7uvIjksXc_dLL!jGi3JJ-}!H8E&QErBg=`JPYQZvYHT{QP|SQxgchhj98dzldp~=^}V|s{{WXa1XVFW)R3XoB{@@=9w%f*JW?e@9nq&ub-#;Xf73__5kzxK6*4^{`MrR_a;ypwPx|D89Pk9&_3=*@8C^;|32VDzY8s2A`oV#!KXBP5fK2+me}MjWrh9uS9bB-)|iQh`)r`7FMh7h{jV}AJYR9nD&;;CqB*kfp2KN7@|3{R0%Xl$e%S&0g`68TY!c!UjmAE2*7I@Z-w(qzh0kENk`S<$sXT1g3cW^EPJ$9U7B(q?$21KU0$5~mj_v7z8Pol(Xzp>-idoo#v8x?)$1r3uugd*N-M19hdXC`HB`hvB9}vAJv!d)fpbMpE@-vIhy&jn)fp>dW8v%@tE?gFsz23sA0Y(gzp0h7IUt9W9QkE3(Y6(8PZb^c5acaMxE&7t(B=DV;d{<YKg__x8KRqGvX9x$~z!2>WqfeSCE<9pqvXGv43I7nqjX@<#e=nV@k)Z)82}Y7$6}i_C744N9llQq19)qEodan1R1BR@2CVgS#%E{E<N^Pg5)8c;{wy=eCkuDlL1gkB4mdn%TovC1W7J>zq>*_(`<4Y0I4;GOnccw|R|7v6%7%FQl?5Z_g4iq-t%lJ#DI~5-nCNv$Xp3$E3_!jy4yx>+;v1~)m@z9*(-8#xMneC=5r5TS^JN1*Wp4SZypCMa_!u>bDR(W7Il$!flr<iwoh7r)DwTT_1Yc`+gK7->N3P`g%3Vq&?Ci^#ep9XaKrY#RDnz2M3`k_ya8s(84KCvX`Q&9jsC5NUC8ie`FtTjw)=nPO&)-0jeF3lIEygZZtcHn`%N<oWV9uG2RbZlE%n`)Jm1=UZlR1Z%_Qi935BxRY8*$rZ0n#t!&R<!6)P;3A=4XCmtjOK?W7z;2<f4Qjh4EBciok`@*TVwq&>3O+rURu!6_5}k{sS81szy1<xdj+hbosu=YA33NWr|rObXb037{)~PAMfdA~zElqRRdcx%D}^QeQ_?iPqd7$n?PCQRX<nDOg9gQ^Ju+{cr(zsJ9E#qGS)e&dYi)y1AIT{eHI)*#b_rc7K2y5SpXSs+RrF&jaQBLB3Hs0TG_KDyM7%YIo_=4C<*pd9E%QQO7@HYF3<&3r7<U$d1b3m;p$%+PHWNmht;-QSUrShwcOvN_oBt~<`e)msMH>8O`O0Kbyhyx0rc2f`LBinWa*&|@y<R4YoU??WD)F|$U{<nBS03&UtxV%y2>2$GOB!?O!#yHjm~FnIg2hgoD^?2K5C^1RHb)7XRnB7<n&28fs7A2c?xNa=1!SOa-epxI<*J6UdVIG;r#ef-jSy2rI-xXhL=8L9<ALE8G*#w5rVn)>+QJ?nb7;#IKgU>p>KR&NC<Xm<7>UrTLo>&gVM7fb)r0kx1Nb<TO&72QDF-KsG;$dwzB=Fo%&pATW&;z69%thY$*IbT;oJlV(Ic5JbrEf}*@j1%en}}BX*IfyIt@K>k}0oda{F`SYWk*I@`ZP7nH#Dp5}q=lAb`aqr{()BA+r&DR_ENYjDkAiX8LUCUfjqQy{{t@?B63Zrz)zNT`OE_|FC34EHHvo4Azhu%6P~)%b5DRhVw>JhJN3Ehi!}T3iQ@h>grnY%@S%|%T;=1xbVn;H$$@!4^GFnjYe}?mkiC_xo*7W1jFm*GN$rY17D`<*lEW*VXz<9@Zy2Lsv@npx5yZV61EteW;UHPTK2mnR}%eV)bXQKOTY;SNYrpESYU_V(M)+hiy?{O#?#X%9|cyiFwK36S$_Qp=cgm_OBw6S@-2PA<$a$j(r{Q#>TtX`UilOWVWmZn@}qlu6)R4sSvw3ZA2h<&z2t%u86Sl!7&8e{UwHX0MghENIkV5_Ah8>;xM0<-ahgP0?$L6KUm|{XHAihogDq>7?Y%5gvmQE3R22Lt86--+`_d1+H#+Q@#o3HZ@()ac2>PM^c~!iUJ!3quY$X?mgD{}UH_@bXU7fKYZ&Y$-_R`!LPqa6`L;@D(Frnla`N9JHgkawy$?mD=hxObwRM4v0>Un3%7zc(;X0Bdw;xXnxVEdN1_^9t9&f*D>?!m{Dd}ZGbf~ZbAU|}5^m7$Z;9PI2SyX$<Ox7Bt6N=4rEd(^dzf*y9n5|ruWgxKEMapF9@PnfEWR=-@a)T^(aAOj{?Gpg1J;2>J-$=>b!j6C#{QZ!+@Y;EdLVwhO&{R#`~>57T`#oJ~gsR4Jl^j-a=NVc$?(LXh{O`q<VUBaDJ!MYy}w3Ix<{PcUpG<NGlwzds%C6<1$Nrk#Ttk?mJ#fp#1QXS;StcON*{!SCVe(x?lCBl%xaaX_vWZskAxS(Fq$npd^r9Cq3L~7N7vcM%zvUOaj=V>m1aIn|8al0s;q8C+23#b|(0V<lQ?X<|DRywzs4(B|2?CJB(6i5Uytaj@^nEn-HAY^Z5VZtOq{F-9FzZ$2sC3JQ}xBUwzk7&&#_f9v!7!mJb7O+VL0+vZ7QH_{YZ}H{>PhtelRRg)cWG1<46H2-(6Egzo=Ce@lwjxNrlnAc*IvG~kGMfi&kK^$m-5ba~%&zlaT|F}HY*;2{e(dnZ^}Iu`8nRh5-b@3m9l3u$GjfPY$O(axSIVjM57rI=X(hn<SO*gFB8B2^c6mQc5sT|`7>xAr=eSw!holziqpp4LbvUVskVPmgPXPS)mzwpBO#w&#JDFqmAod$0()Tbt^0*iNEcCQAgVSs7T4{gZhtu0UU|niIj^^9*_xNGztx^h;rzcVrzD)>1{{UCb-|cN%!(ynIwFlR1cz1LZU^WEr_B!(WMM|Tv*%lO80`K$#6@_!W#31FtL^?OU?r0oPybj|7A$MY@Pes{-kKFy{Er}EL%@pS%igHar)c$EGGj6r#n9A8~Vt75Z$Yj%4#X`;-^*l~pAXkG(mgF7W@G?w5qIs1dBYe3##p&xfC|B(O*>z_Q$B#^zHacy>B5g4MQlZIyq)YQ_w`)_Rxt(e|Vj3km8ELIiF)ZbAWhRO>y_#RC65L8=Ak5E|!Z3^2y$j}8%nk1*#Lm)}+*;vi|J-w_7-O+)-s*FjV=-JSPQxUB4yLxw2Cw^gbPK$?^OV1^t0R_<D8A+=AUt+C2bsQx?z11ys>Rk;Qej(WIkpnvL6czkAo|<{S!hz&pu6B+#q?s#unD;@9mG-ifw1FI)J`KYs#fEc6X(*ug6d@d9&Z&tE}L5;w^?3r0u%qSKl%|lzSFzP8K!_l!+zLFq)i86<zSePAyg!?u0G_LW*t2)zRI+lYHAd6{)0V*iHe|RB7OCkRupVUgz6wK(Bbw-m_<ul$}$jrg{&X?<hDT%w{wn6q;gYX1mEMIs^EgUfJ+uUUt)k^Yqe7Or37bE1&$va(wiEcN(9BHD!#b<q8w>S?TFrs;1}`OmUTFf+v4t)x%V393Xf}mR~!AaW2QS(vThsAg!-sG_g338I3nNETm0CT#plcicr+sT*IZwCphyi#dK0k!VdVN5K1zorkb^Cv?+9b&kIVN^6P)|3Lb$At6r*d<oqWh=b|7xwY1<Yz-~Q7vuh^Wi@k1wyjuK6;mhfX*JX+#tPapey=ENmKVCYcdC@(8d4M2R2LG>bWQ}JNB639v@KHH(;5vBW_G&t6daPD7Sw7YwBx2r`rKUv|wmKY1iztl$aS5e!sE6bO0J>?<oIP;*0_<bvveFFb<d#ctxY|`?^Bob^a+MT;2|C!cjd<T&SGNiH%K!lH7#!ej7O1r#mlE_J7yNedv6oQBmA~&z<SGL4EoU{U%J3f%qFq-a@m$^XT*dS6PRg6^g)suq`nB=2!j#qe<SdIs_L2+B*UdupE@pziF>xVyi|8~Pzs%;J)U-A)cDyZncjVAl9t%ikwE!C5vdSDv_1uXkcLN{b)eZLbxl}jIty}eu)`%?OP)RH%ZK?01DHwWD|@i5K>iKSr2IMm8S?WZ-}_b5OzR`FK>;x)rJiiAa;0rX(RZV@q)wsqVIq@Q4o*&+%wvtm-ahu`RZCt%o++h4z?JI&vMe9ABeP2t^APJqCe7i4qXz_}~RjPa^x8!T7x2TB6^81+}FeQbfz7o))3rtd0<!IV=oMM*~T&XZeI==RBtosnAj%H9g<)7lBSWdi$t!J!!wCj9~K+!fUz9n$U20c!+Ur<2ZorTcjlD8|!M2TrX5CS%Wo=7w^(m(&*(Zhc0i#bJS_Yf+Tt-4_2bXgBgsW9-(%5(6a9a3_TCDg0zwy?BTlR0`T0iiduHTxa#a#Jb6-T!)q!zNzit2{=L`{LbXL)R3bG*RCQGv^>ZMtF**-rl>&5w@ORHx8!pU{kl!spnsV6a_~?PSEGhRFQW^Y<N4$<fQc$)LMml>iR*+zaq&*;#e-VlZ^k?p-ss~KoirOE3FvWs8+j5|%&uK0{BI4<%+G%i7||fW8;RODJKPH2C|Eps7~YRi(HfM5ADC?SyP;QJwq%<Ki;s!>12W*0GEhaNp6imI<ekj=G47D4KyKc+WC8ky%F*n`(cx=^f7RE+>|Rkpo3!Og`?+B)goE<()4D%B0^8Tc{N?&|Qa7Evh>r0mIqM|#=qm2yE9!A2DpU6Do4{a-a8!p9v<5z8qZI(W%@UuJ5boQnQts8%Y(*d$#mwd|@#2Bf4r;B{T8pq;O>7uy+FEf{skParP6#pqhO0X04E~4R3&3tMYJ>_kfd6;kyn%^J&RKcHd<Gsy*r-Q1!91I35Dr6%R3ctZ4QRy_Rig7!kef1eLl|Ezb?36+{qm_%M#Z5Wc`7;^&mxS^1=ZZDP%K(2ia<j^mz@^Ha4O$D{$w<NAe*!fRI7i@ds@z*!B6v`x7lY>MJUu=&xTW{$Kx}?Y2+NRbivG0cH=8JG{<#9`#a(Rfcxp5#A~($0^}g#w+-YVk{xox1LQ=Wrgj?01}B5f039mxJ%SUf)k|?QS2}n)=;JTBmX$$1i2mraRi_`Z*;&Y_yC{2O0<4Sal-(VlXNILz%<K~2Q~TlzFJ%mHG!*64mHHpdaywluqHWE*s1+l#eUcWWxg`5?#UsCUS(e;XBb{RF@7hG|G@JABsTcPE8s(yawQau4<}4LjIf)mjlqye`2{ojO`;Y&u3;~UL6U#QTbo_ZZ>!qoo0;{6FKYoN_caKCz)2y*q12|jpu2bUBCA1KClelZSmOZ~=5kJum&isglp66C1AnVZqtQ1iLq#N)UaDG7QMg<Ub;frX91Z`0da>Vm!9QE_ZynewmJXrJ^XRVVzR1{k)K6Li{Tx6UKBQ$z&jyW^-NzX(>uP&hXD?Cmn^E&pwAIs7D^{Qh=4bsMlTXXfP*Si1IZNzYAH$s*L^%an{zo?bo17(>_R1c`%#r6=B$qC8QXUxBK&rst{HzKwq;>5j19V&eO1|1_xZ&0}uaBAYalOKKrU{1QV4TP7V_H#*_$cnpAvYD@yJ_s3<y*v~0wlfi}_^UTY_mAC3eFHlwWKR^1OYGgePxrUur-7~9com!h@EH+`<lr?z^y%|wV&NJ@-j##$Q`>vthi;=)pJq<nImdsW!S*;>zCy^wM7d6~nCt|bzOJ(|9p_sT-@76O7w<LTN&}u$v*!*Yn+UR$FFly-O<kj(rzi<f;@F=gT28iOI%n+l1$NQt+>P=soGZwv6j~!qrHL)bn=7myKx(~T5|NiETws?|`du<@hXLM8TI!%0s&}@~3iqtVXj#V9Vx+~ggOVEj71w*@V%yBt0lQ_kuyFTUt`Eb#c^co-SYQuCG2Vpo_a5h7i^`crT>kT|mQp(vYlqb1lWI<g$GQHYf>D%^xeO0Stw>#vVo@Kt>@*Kg(mj`~#c>$Qn%-{h$HQ+Bi0?35hUCyCj#`eeCRd5>8o_dMpA7{9;1aXb7(opXb~WxjBvhiQqpJ^Vr@rn#K&XAb%vsfOlAGhl5`7E@<!s(nZeK!agBKhajR&7!=+4+u=>b}*7C%}F+!v?q)21-`l54YKxrKM~v{YPseFNW`B+1}t=uevW<g>4WyjRVmNG3oPh3iKAT6=*DUBD@-Lak(qev&tARPqtfsWyIjqo(PXgLlRkNuC7e{S{~l@(MCo`bv3B+U@58<#lj3tbB{h7Lg9pM!>01uK2nwCDvyrJnpwg-KmbsT;D{0<ohzB3|yFL?PI-7V#iKM(vpY1erIyXMV<;oNii(<_silz;2+$}exBOt-Z3A>F}aKRIu-11waO-ViWmuL&%=2be_Ry}$3l0iDUZ_O&M8)Z)b6#Nf|H|;WLt;F_4QUbZkh9pD@p%WeevC2y}B6+57G%0I?P1}q)lS+%`5^ZCmuEZs)Ollq||=gZJYLlLTqv?$vNyTw-F40jKGmb;g(}3xO+kTDOeM#oFd$m&dIb^V$*Rz(T>vmgO`&34Er4IB|{*{Y<&hG;-*TXux(Qo;nZ@#U(h5Ixzmma%R5q33}YCk0Lc3NOhP7v+TZF=z-w5qrtEB0O5$~;%*6xA5Mm!1agD3Y(c+jC`4hmiDx~SELs&sZL~(%UdkyFsRT&{Hz}fLr(?!=GP|ysb&)Cd?0qs6D`hrXyf%*?CT?<;Sgt)xEN+m_RqgAWYvcm8l(-MGlFs!>E;^d#!&xV$#zCBoxA01M^+^^-M`Qp27#tRC_KqfB25si)`H@1C+n)J(}wTx~p20jH3yb9SY30P=Yv!|GEf_;zLRIWhhU+*Eseru<8*R+8*xl*1f(XP#*e*o+}9TbhGA-81Kc8xdp=5rnV&#)WdOd2?mL<$^bet+#v%BUHY#2#2*#Dr8X8C&R(T~Z`&jmS#7;QMl}al+<+46Mrr|LBfh(M?*5Sf>91qw7d0n3Arv=^y#a4ha*bNT&_Bp<WLcKGlMt{h#yzZWMz7CUiRJ0ezC^rdv~6xpv$VNeFg)@7I7-K~z>C2;j$+^4mVK$&!Zq>AJ9EP?iLP6sED0`j>wmafxYSsZg$#Imc@CDvB}Dupg5lFzZkse$wk1^H@2Vh8{nzH*1F?{GsD5&n7BFg6acgpqA_09IkPbpzQy!ojvM`W3g|ky}>^;&>*LlZCLC9W68WPp|2I$E|?@y%vg6K=aCb4&v$+1N)KKsEoS-o>d%Lur&bS02a~%a@kQz>+u=%|j?p|0B`SMBIi1WIiyp1eKB;c2mRX#1aMUJjTrL1teIZ>@<Xrc8$XSq8KNK4B1>C_}&J(=@i_-Z!K<dw|gatihsKQInN!4icvT`&OGv|#L%50~>X}~y<?R=90kKse~RlM>o|EdIMB3)uS9@%ZG=4%YwF+R#mwpb9TGO*a0%Y+);REvGk`=ayo!ZedMU)@=c6yq43(h@TAi=~YRL=6166WotwZBGML!3hd>uYCouDXu;t3Y#@L8DZq`Tl(w+^=$M#GGU{chFQ%)AH?dJJVa&vR&!Nw{Jqgp<cWJOPO{qLZ$}%L;f-}=zn4PmGcTov5p!E}0v=xE8v6HGAq!eR;TNmD=JM#lqxl23fP3eD<X)n%1*ZZ!3qZxzO`jgS`4E|T5rA#x_Kc=Hn&(_Fdp%V2j?q+)<cN_xudh?!7w8jrr-RJXd*o4~+NoL-T9b6sZO0OcdS=~<ka%9r&)n^QOk^N$xl3zCb?Bzld_|Sm-ik1JABv>Li=<t5hSf<qrNYa480XWig(HIyGwdHF*>O&vz<_frMbvImm=iQu3`9bTTKAW|kGOwSC<-u6)z+m}NLSDsp`epaux#F6Ht~Bb`~#d2i`-4bAak>&=&C=yzj^+3(M1ntqGC>YW8tdd0gpype?+zNZ9a^CBMQ5>&pD<<4rd=K38wWskL^`DIV+UU%6M8_0gmzHk6=&paVvtnco=N+lafSy_Ss7Znw3>H)JrakzBq#u{;)uhQElb<P>C5aMh&rn-fx8BI}&(7NLIJ1tyW37VYt_M`S980Y_cy1NNvx~;y*2m1j|i^=b$;*R;|>EdSeXOifCR?fyE*48oCxyd(ubA$EhkYnSE-=90TA!5K@W%+Adunt(^#{Fpl1$sm_|l^V9S6>Er{oIVj}8sn?F8GzhzQs{77>&0d$p9MC)`%{pGCsW$W;+l&ItbFg{jx7v>ZB6Qw;re)h>{HQ;THsuh3abl5}PZOi5B_MHvVVC7YEaP5Z$9+)#Y00~0hf|oxhV#+<!K3sXiUg*868f-IYN>Pmb7F@wZTy+I#E<_OZ_<zc6@^JcC1KdpP)?RH5BuzO!+lMEoW-Az)d;&YmhDacW=?xnu(X6S&+gVr#?4{Nu#;$;s@XfZ2EaIef9>FJB^etT4KW}dAeXaMqAW-=b?zrg-2L)MxS7X<2_x5i!9kU|_$Ss4RvyPZc;pBb;ts6nz0_%Wx7wq@2sJIW%;i#=*&OnSt`?jsmn!dhs&OqmTk6)?sa<Ayb(DpEnUP>&i*%KQu$e0RSqfsp%z-&TbAZ|cXzbkrBb%{HiYiX>^J^wPP_N*sp{)B3-zrb-m9Q<5&h{S5Tq?c}f-WXvnu86NW;tq)i@dL-hyn1{zZ-WA?#MW>UvN5gj**8{v@Qc~%aoQQr2u8Rm|UXSr}TjYWLP5|`)m(stOik0gfL-G7i!&C#{#ywvDH(zh<8=R#|xDiOG1V+=BDD>KCrpbvEXTUmskZ=&R6Uv2pk?a>&j7?D&xE@A0w?2i!9G(;?$?umqm-H-OCW{W;XKYCWJW;R=5JB)m3r(k?yKXgY0{K0a1F%|2=^<*~gB@agjJkJQR<;09J)S2Ndb(woF|GjT{6OA-c7a`OuuE)UdcO64GLc{O<jBdi($Wfj4EzQ+z0c+RoUmf8XtEVIzxvVSM`c4W07EPR;P55V$#${r9Y%eNfp{;U{V!V?xCnX~dg%$Zaj|5ldJ8WeQOExG{pLDY?iNpz4WgU<5+I6k@li&_6IFRFJEaPM?LS(`Zk93E{M=KaBMz`b0w}T58I1+YsHj(a*+XurHPW`Up650)lT<uKv`qY46s_J|PN&=REHSkd%|Ut#aZg&0B>iEMgf%1zl!$JXU00{zwQ&3iLG!1kzkEX{7u~i+<K<Fu2XPAhP}3UR&?Q4n(EU59`s7q28^R8E_X{2dM;YE?<)#y&SUVK!W3V!O5Fc&+M&h-kvP4nAON%RQybAQnLc%atkPN&$}k|*?fS9l3lK1+>^k2l?vmZ3JyuTkGX<F4cE0j+aBEdSA!2cr`MVl5F-=TZX1IWwb^RnA7DU8cHExB-5rwkI^ruvam->teEv)b47DwxH$<Sh4IHW)%e|aU(p#55f4QlT)rjkpR5vKwj9XbC+^7OgkNduhEd*l#@lE=&@t??k<-@L6-+2LDyMm3znOZf**=Q~wwShR@QV;rj7%~i-UXbb<6WJ#%J5bJ9fM99z&pN3z9GfjE7JY@6Hp->a^b`3EuF_Oatqy`-?Ozoi(Rf8jXWcMNxlJ#8Z#gXs6~p~&X+VzP-F0a_l|Vgh2SB%F>ALC~!Yur}n21_J?QE=E1T3E92H$Uxh1X4G1xp0_pON(FT--KvpLagpVEscXeq=O7JTU!mO6bU2FR@hbL59D}B>Qj*Kk|rdkVDqNK>_q%ekAoPh21WV4B;fk1i4<iUt>Fj4ws}{2DM&vUUQI=68qe7JO!=7OxML3u(Wz5!=Xt@E%c?p-2%HdBiMySGr&!w&1ThUA$ui}tmH`7H59nI%L@L1fZ=+QLsIj4Hp(fmLRRutLy;xKgkZR!Lf~h9c3Cn{aVwWa5*Nt;K|i*tg`k4fOSaE}snbGuGVkwn$GZz?8xo5F6LE6b4GQvvFP6UY2!cAmEcj;HuW>H^aqr<b@FrV1M$_o_2QcI-E(}+vpm4@bb3R}@AozRojcQ2i<B1N;4M(a2sHE8(HKV4@av^rcX`Hervl#V=!%b_;n|k<<ljt{uZcl8mDC<63QWM2;wbYCNu0Bc_K*Kb^k%Hmzkz3vNYN08{il**OkVdU848@_yyKLHB=l9S$W&LtMK>HQ^U+5!dNo^#acX!6Rwh&{IJk5pkmzc6Shqq|OZYYmcK{Yn9Pp(tgk6_Z)7s@e1)Yn00HjChQlBQ@vs|jFgORIV6z1qzs;3)e$-BU$(_PaA&8xu}TO{D=ODoN>$=<BXX=vsa0Q0yV!`uqhbqs}FQ-Kmq)3V%9k<9=Txb8KLv{slyJ*yV&o2Jzad`(u!2BFI3|)sq=s=RQ|KJHFp`AUh?fR#Bparui3!B$Sve-iWQoe_kEcO`5cfG@q|D$%9VVKE+TX(s7GdZf82uKTjEnqE7PPIBp0rYSHx!9(u40yf8K^$G!kG9PuxkJO2z7vZrGPt|D6Z<)n|3Vi_UYh2?MaFzmLvvSyDC@5Bt4_pTI57y4YN*08Cjz3H4_<Ehap3`AEn`Od|IR1bZ0S$B9+xt^c{!;@p~LORg3S0k}<?oq90=0d?B%B%_K_14y)aTsC9KG#idd(oAy(5xD<gM<)4lIzFX2!Nj}ay_}O{NV9-9+`O4NQsOvfOJo)*f3Su>dw({K{zPy_MRx-`L$VRYJ>NkG9tIn?rt?94jlVK-=|r7IfK0!e0xDOi@(|1FsqMzm(C4Um94lZFUq;=!Q&i0$;HzLZcjSKjM$O74b%{eTBHB~DGGSrjIvzR00F<h0gcE7$E-$}vBYQl0ssI200dcD") if SEAM_SHA else ""
    NBDF_BODY_SHA = "a18e9d6616e5654345ed3cf09fac6c05a772e50bb5bd07ad0bcd6dd4b6275348"
    NBDF_BODY_SRC = _unx("{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;WqbOK3xD2hy*!L{ZEZ$h>Y^w_j6QsV#pxRhB?_~yN4Ne%g^i}bVD49c3@*1YGhtwpL_W;R+>Q6$JPadlM+<yqHvsd1>%bnvAhEXi`DPtiEL5}@!fe>MMO%HRWQk0kLcnGw*0JSndYX{XS@^Dwm7`8;9S-^+1}VAcY9VX+zy8gejg_=-}@V^rXqff$p;!;Xi)?KeYAu@$eur@S#;@KU6-L<`AI3$(cQwhIZ(j`xz`VK<EM}Cka>+o$*6X_ZUJDnhlhFaAMA97MT}xBOd$7e9;Nyu0a&)S8*v@2!z7-}06)Ilo&4Z{`o_zNj6o%q)}?7hfKP&8s$FMY*re#3iLot5v#iIGz4~iBYzSZi@_<t9FQ54-3}$KXGDS==lYR-7VnMj!+Gs7`#FmfrSLCI4uOAOD>LP#=u!}F(ZT?3N8oZUgkc#Mzu?1MSpWc4EHICP1^{3E;?^>8?1Q8QewOMRCF2U2QO6h<6Q@Ny?_V7*_v>Xg}1a4Sjeu(8o_!~1-wQDQzRzbIW`3}N#Xq{wYz&2e7I^7;M3{-?7jwHD|`5D8LT3;H>q<I5Zfnz?HWsJPWRr>c(ofyWnT%_NYp~FbqRFRs_s<#L$Ih|L<=$8%OlDaR3GzitQjx%Bei61B+RwPyCEFB2}Tb>);1qD43{(FTkK4?HKYX1tmPQ~O{qM!P-ZG|H>L33m~avTbLet`=L(*8D!>JeN`8mvKU?FU7i8Qqowq>=s}&A@q~k&qRoi)X5#Cxqn|GhJ(|{@4B)x_OJgId&ah<Od$+9`N6tav<LqpQkw1fex1c=)V}SJ3^G`Y}~PMX*WjqMep`emQFOQ1VQ;DIGpO#Cl`T1yi4+MoR`;&`RyUnRuEiJRsF(556UyQ1cp8(@Xt+|530d}T5D^k0c+TzG<t5nncC#{d4}QI$p!@=IuTN82`~r#kKtY87LPVYV=|vgMVaaa$B1GBH9U@~y)>$JbcDBB9;<YP3Xe)#fP216<=Lc@#o-OKaq+h(>6y2zj1;dW!pay&k-0h}$~9U+&-790ZNx`(?Ub2+sh-gB(nEwz#kN0BA!T^n)|b~vc=y3AI)6|x?xEU{yJm|Y%7>Q7oEFfhEji~F!Q#?1-9q@`H_u4W?g0Z6w)y#UCpjAQpKZ6+%W)&Y{YX7s!UU#6^7M+{gmKj^f?Pgy5nSy{v%KaJ7g+9KL2;&|)d-oTwe5~@87sB3+E0DiDYh2b@aOr`q$!cA^P-lXRHKMjCu#!QwFDngiE&$h!(S)Og^M>PhedrlEOML73NPHl(9P$F4_~qSYVxc=ctR#UcWL4*$k;%|n(9I!DDr7{(sD!#ZN;4DTcM-KMcpqimuCGUE;XrZb)Dncv1hr+yKP$}l5tl2A;)YGY5XP5_U%E%`f2|R>GM?nj9uwJ#+ogNrf(6xny3_FRzx(|VeR<Z7yX8}3a}POR9rZ^gc*25bm6+vW`Rp1ro*V+gOVYCsv&22yYi~H)HrPbw{MtY(7uK?cnvVO@`~5M^zZvk&B>6;*?#vcnW!SL1<z7HPh&*VE!YQ2)|5Dcy!68|8s!kLS$=wIBEfH$s4QcRA$9ywQ|$D~qHf)@h2UQ=wGJU_vre}u$@vhbjLRhNKdcZ(m3~K31lsqbI{NYj8F;R+?iLbYZhnbKjx{sA6LB&K>EYPs&!W!Pt!(VGE8x1YAZ+GqvT$x1&!FyGjD4ckYflTxyGhp$wds%!imK=L=&IyY*9!B`Ve>oH=7|Z(3SR<Ht^`oGoIX!ltWHV9@M$r@nF!}~hJe>*(#(5RM|_q+XnK=dQe)B`9D~Y92;YH?>rGMz1U`)mOLHv8P!v6L<1E*SsCRfQb20W9M0+xwu5_M*@HJOEb!%wT{1nKWgI=$)fJVZ!2%o?A!sL(t!a&!ynpRGyxV6&GdoY0@c}%64&^wwr2NL^lc^kbKdI`-&x!D|eeq*7Ka{1d`-bLUOaTe^6<Mjih?a6VP58-}KPbjbOU}aN;d3;z;V}l0=WzfF5B6n=HLkR7(R-JD6n~q=ygR~5eRc<%>fRL%_T%0DRb0RtnNl2GlnwW-OEq7}~v`ksZw^Pu&Q=!ap%75+h`ZR?g{bf!A`7Xgw<Y;X|u7A(n$&$1*KiT6UzJm0dmPptf`wrG9zc;X=@5oYI68dxPvu6ASkjd$dK<_r?%#|A0+6<TCaGn!XYl4UQ3uIHAA@1x7+9Sp6>XSmR8UEa*8-^3Um%ZKcb?d7hM~P}wIrF3_PXhzZcH%IpqF!zg>;o+O8Lw4CW+-9Pp1ubQ0*Q4naZV-ne*<8xGBVs2R<~KDmJ+`G*xiu$or`wg!yu3@G=P4`<nE*Q8AMdlq{|EPbaQCNYj17NWyD`bCVh!brq$5W-WIJMAH<z6Mj#JxSi(AVZ8&3S%dtxx^MM$%Qd}86p*A;0XU^bzE8Ez4i<`b=LE!N12ZF|WEEIhE7MgbDC0gzIxd2OutzO&#7m=q`yUuK9D^~ygLYa7h>ak6yL4NIjc9gceDnP<g6MkMvJ@xcOoZPO@`imU{(lBew?kv=L`CCAYqNdur%!o^z?aWTkh)s{Gb<o11r9Ma4Vy6j+2k`uvA7x7gbL|{i`wF}Qfc77Y)%QIU!a6Fw&AIney7=BAij0wQv=;;XG}kC{S?LX3++KFGW?H`VL((D#IYuri5i?P^uu>4Q2HShnXq;56j)nrsB`AtLZ;o&+WvkB_!*b3wEe%7Y)2;TwPT4!tArU;+j!;I9m@uIUC512@f(gIPyANa2#NERR0CO_?HECS^xj<Lyej|9026G2H$0Wq2-><>!o)GY#+44CaFvCcFX%9obWn?wVEkbHYrWSnjn{-rzNCg_JQQDL$EvI~V^nJDL1_0QxzZ3oumSn7r@D}!SdorCzq%o3q_dQ#pu-&R(`&tdKogt!FjS&UAk_?RXD0#vkL*c%!9$3su$BLVz>9y;RqUbJk99*lj>N#=;Xhd^k<EO4Ps#)O*Y`kpU2miM8e{P3#6y9kMU=rnJX2rxvJ(F2Fo9$%W%b%-7HjkZ0$_yLH=@_t1Qbg6p!ed77cQNJ7PTy`+cFTVW`S$iN?gdwl0)*Z>WQxcuV(Re7Vu*MvMnOOkNBG*A(wv1*o8h@U&P~RvA$u7I<0+`8(Mxj)x@X@%=x@G}&zVEM%Fj9K4S{zM_t9G0EQKH*>J1`^2^@|rU1&KJ{k&0kHSzsO&^mcG0Q;_oPBh|Nnzmq89hX)itapqXc>MM(uo)?C6o%>JcnAe6vED0)UII>w9^@Iei?iVbM4M@O%2)+O_g&I9o>JNK@bC`~J`qm;TiUB7PVlXxagygV|Jwd}cUgNP5pAlq+*ADv5^A5Q!pW-Tq0aAnhIr|pbanJSd9nnA`LV8~zJJ4Fe2tMoD_}Uv8h9e2mjVkHh%%5P{2c|}zM(<U3MN-4H`xcMS>$~TqrgdUe?4g!wK5R+WKxfUl*CkrCvG?M4;t>i^G^qE`e$zMv;Vf7O6vO~`Q@!FT~itY;q8DChzr$$-Kp<;rY6s`OKAN$+#Vn<kDJcqwOGD>PM!?RXw24b2>Yzm$$8rTr+her(am<l`Fj^5(1}6MZi;_p@y|BmPOeBbevP4wR^|thpwU}M-x~$;hJfYTxC8|i0lCjc`L%)67^;~?a7wCcUz#B~J-q9!T!{JrF*=P&SA~P+mzKJ@NHtG1a1G2`8=ny`V^g%=CPkiv)pcz(G;^Mg$`B)&D^W!LgRkAtpmwf#%0eDBlxhg|xVB_C(3T8xpM)%9$nZpd$lyjBe$H=lw2Sj32Tuib3gu5!rhDcX_Dt%40GGXVK(l-rMdm+V?zM7v>ZOR}!BNk?0tPoPc4bNGw3qPVv$W&7O>x-{skXzS0;wdu)v83GlOTfy@A3+JO#tSBia~U6e^z7vbdxBMwqe~-;?{-1Um_3(+Y<=+I1iMi`V_M<lR$+7E@I|;kZJBBRlk!>c5@V{o-jetVS%v^zoi1&Wi=ir!j_Fkj-StV>C|4NnebHF-BW&X)mI)$?g>&be~cDF5(n@qFl*wMwFou{gbrbGWIW4}q%p@Es^?%`^YH(3X@eJxKn(u!IoU5P&7^LsZ_4r7B>BpIJP)7bou{_ZC6k9e8?)$0>d(+|cL(F)R88uwwl_%k>faN6?$#7ZPyg^oPB=*D7O_YD>3>3OUDaj-YvEvoS=LLs_A<XD<ZQ&Sh9r_OF`|Nin(bYSB*76x&FIzYH|hDivAgX`O_%h;o(5P40PA8IyIu?y>E(@^86sD#;Kl!>LiQbeYxCo{eM4h=(fn0XrxT|wIp@ifoFjHpnVDhQX8my`v=TR^NhnnjW_4o0Io_lALwYYqkUad$fN9p?(LHK!a@yRo#G^C#UI-@Ts~nhpj}IvIwvEb|kRUq3;DfUS)*XD_)X^N{jbuOY4d)fEs5t&qm?0iyI-SR9=r~xMP^eN$qk_X%+M^OSEtN{G<V@-7XmpecQS=&{8|VT2Snp7@3hpSHZLozygwtuA%*+yFHKmeR78~!n)<iXOk0_6gL~Jj1n3S|BY%A}iC8s>j!@7ZXZ+bfPqZewlZCDOQd_iwW%~k-3jyJ_wm%ERn=(o3ci-B#stxEpDAM?#5iMgq{jaByv^<@Ez`Rn=wU|1;=A&o>f2;!B5s}$Ut-3=$P1Xn4|?A|s>RO~v}>l6+6O5H(}LO3Jq>>+=N;$DN~6Vic%mbnx!<F4iZuk$hYVso`w&5p^Uww76A(8Pz9hoYTdB!>jRN$y&h6WUhzcZ^33d-w<nK-fxP1gS$wBb{Q?__^2|*99)>nN=4<ur8oTx8I<YJ;$OW9@@T|8(TYTkT19d?=(YJTCJ+<_yT)A|69Oz&tmHK1Y<^u+(4j&vGA6xtPbwp>I-T239-}tu*Da<kDP$59?qM-2k`?-%2|D}jz}vCzGoWaX&Aph^#w46ZQAT*`xZJiCL$d1P|SnC$hb)s3_RHOeNvdV>nbc@BY?Em?gIs$a2;u=l+rD1ysP#VkpB35nFB9f4TgZQ+_8P%C{dW1E;w>CRu8*hHAh+nbdR(c*ex<Ly+K4FRm~Kd5b9~|9E8tg*RtR~q<xzw=HJsRfKfXmDc#~$<wgMa8-cn4KCgQv{}NBY6~wPSMSw14pT1E>3NTsPyYla#IDbOSe4RU2ATHg`W-F+Kd$*LNELT$M|15$xbKKxnxEogXv)v0%o}9^qi&f@?)tA%vqdka#44<ILC+wSJ+<HZqqHUM^V+8Li5WSL<voMHF5Yf?l3f9<ub&Y!cc$X^>;E(Bg+=J~^phukEUdPNEJzcp~dKu7AWsuJgl3r?59y-<-17)9~I430-{BJ>FKRMWR5mgRDU85YHCx0zrr9Aqh&1{^va`Di6(ag-4ilW937x5GxA$HvCHfSKQepdH_!cYRzYB6(6synqXxp5Wo&zViE8;$g?spZ6ZVtWpRSi{1r1k|^}&OLy8^=J#7GDisX@B<`x9Jav)KMdC0taW%b`v8=;z6$<g3uUe8i7@3DQOX3J3-B?#dKuIL(17AV=VvQ`C=jBZ5Dtv+D4_{fan~fNVUVCIQ_Az-Li()H&~^UW3?coxe*?DGn#+{0$Uj}C;mmDDVz@_o$|JN?Y4blV5AbXJe`6cw-LMaM=zH9tNhYrdv4HJWqR*n+`CYmFT`&xY^3*;aK=IByq?6+jk_^I2al*f<Ox?CFjE>+vSFoQ*i!6LZPRNJnP^3Y%_O0h%+d~98A5PG`L$3ZmCwBmEIWaj6PFL+VEp$2EHBm(S!HJL#$juS~AEhUlI*{Y%W^7wca2B=mfxAUd!a}=~Hq$&>LUeP{Hooc}&D}s}AohNKUsOY6|7sWiR^)hLF%Zw}rsyU}xz$}|s;XDO;1R)}-lH6wMM?{O5!IrCL;Frs)1`lOv?p94@hbbge)E>B@h3?e!HO5DCp<$~juOS}^?4w40^OG>nBIbIQf1QWlJ|{G=<z3Mbor%UXXF?bf#p1e-yTm)9KN2OXPks_o8p;n-zb#X=p|4x0_}vf;<A?(UqhB)!ez*zNmR^Px!BFFcp?E!|6L+)K6D42(S(Oc#&6(McIPTus-7%V)4778R6-eoP88cH3fAfjlXzI}rscIV;<g}v=f3FIWUq_y><YcNEEq^_4<1iUaRazDxQ^|#iC59#A9L7scZS16K`vthjRF!@Po~PkkY8e^W+`EWzwO4q&xD}Pn0kDX*(C%TZI1kd#qjL*^ukmO9gDGU@e?ghtEwCaYr=L<(+7rLwvF?OhQo6!4t?u{c{!T45uKdgl>&6kzYPIP;@@C>zB2{^|N3swQ9K_n{}FWPW}ldWj)B(TYVw<etfu<UTecSACe5(7<%FXn;Uq<Z4bv>Lr=Aw&<I_rw%zVkGM(vV?e3{8|+|2_(RBHhvvUl?CFa**U4W6Vmx6%Mz5==#6-RnDJZZ^;fcr?$!F8ad8lZy1z$}lx0$);lhVFimSNXH}#tzKXh5}Z{x1_Nxcf*fg|_4Hx{N@2hcF@@1qUt7jOs}mo<W?xyZ`*zeD1iK@rUr^E+HEl-#@nmr?V~=f_*8W~dHmSOq-2j-;PHlDE_-xp2h1d46$a|!N3aKc@8~2Z1VE?sLC;XK+TWI)Ap0tXFM?<gDo_&}6)|#T65xY^#83}(WYJ4w*;d2ei?InLu{@QWJrYVY-!pw(SXR|9oW^}5u27$Mr&vtPicmd>C2FiV*adCBB@`U8;oDusBd~CVyC%}-53;MyXAj+An=S?C2lBmJ()`I~dOsGFqx?1nwr$z>d>iwF`@P9SoBJ)q2jd*D+$#quo*t)eeWeNjH*9#pu+Ll4-t!JOj&8Owgm|tliXCwbt;acO-M2UkP)$VcIJKGlFp!UvK8OE)%Pj!m(-*#`Q1m+pd<MUb4@Ww1eYnZ5De%i$|7|!aVLZ3wMhCM0NC7s-yi@dB!41FDnNRtRZ?JHl5&UU@&P3UfwsNVR;?<n=<7TNZOaE!JyRo_Mov<J-k=Jp$ke%A>gw<(Az88?LJO5#N({C9)bM{hc=#bGVO#tM01zbm~eb@-D=B#6MVO-lUK-UgjiGKS-2@NiCb^n3Rn!I1^V9zpxXEYy3B3;6O>zgF@T7reG+x0=;t;&*sPM--97zVhEl8#<BGt6(oWRVH<(iKyz+Jch~N2c5G}gE1FSWdq5_M4U~)P1X%LdG5GNa>~<qb?sdH$$<9l)-oar-RIN%dUg^Er*h61T~pCrXeN@FqS|=o8Ls^_ywWBzhY0Ft(aFc2YY#|D@aOz`UvN$i(f2AgXQFgbr%8yxi7)MC<N2dPGW%MhVi<GAL~84@t9jC>A$@wBQza!m`037HoaKa2Z8;sN{j0%U2i$VxrcA`=mB*RToOdD?wjB<FodXd=+I!P#EqYbL=y;%dv^-+*?dM-MKQ&koIs$7TH?!C>mC^Nk%xy2*AOv;d_4~v;qtuXMZmU4Et-JypWtc2-eE5Ts<vzg&hH{zlSp#nkDLtK3Xg6mOK#Sc*h+uz*5*d4VxXzxZlt&vLaaP-HGfh@25gQ<x4fqHTm7Hin-ve@A2x2~LB?ZHKrlutTleZ}a>k^oAE%%)eoRU-{cxnz!sTO3|IXsLTd}Qiptf<Q)qq}X2k<}fE`8K2bD={cNqeq?Im>RgS3f1gR^Bl)-e6SE@xf57F)53RF1jrmlXk>6w@R>ov;*`Qt39b7)2Xl3$#vVMVp=jtZ)1PYQ{U;+Cw3ca)>_pLGZ29jZM1?YoxN-6V2=izk>%eyw#{Yv$k*3~mP3htc^`q211%+!HMmq+R*XWOqd7&sxyo^Lo;HY)F5uw0A;>Q+|W5oD=jqGylap&)CD>8wTWalUyaZXINPFp$B%DMQhVs-qwO&9p41U`}ytr5o=ZN@h`RJK8;3FsHy1*5EIg&?Bm2t-O$Ul_W)VN#Th<ZRR&16GG?-~b)xAK(jxv}^v`TSncjQ*y^9watMq=N>986NUnLCnj=-M4sk$dSPL44tjjpy@Bvj)5c#6@}l@o_sc-AN<oFNW$v<`C{WzpQ-L;_aq#A+Mv8YFs&zxbwRQih*d;`OgHcnopqfPq3H2cJ5e-JqhqI%Gmk?)IOw=1hX6+|2Fk-=Afa5wg6*1gtywj4yK)!COtS&^D7e%p}<}BKv#Cd$ySZ^P><_P|wS(5Vv6>>0u+s_R_XMESeTeXOdGn8zT8rf<cG!_pBKOaaL3r7`MO}{ev8h1NwktN-a{J|JU62N7dL36D3=$xX(F}0$}kU#6-m<~$V?N>fqVbbHHL~kPo5qmmLTOd{iWFjcWu;EnyNmyQsdQ^Ts5Lh;8fqiT6RY65kEMSQ#zYC~6ITc#4W;a|N@~ey3ws>simm4E1T{}!X(kI~VgYPaDJg0WeA~!G`^yh)g$uI305+wQIpba>8Qqv~pQP#gy23FCtM6v2gawgF)2mvC0LGka-Q(n_`Ij()=&mA094|wI}Z;&`oM<Zt~?C_=Fm&kqZZ^5J{h5q7?D?75hyOJ+kw+K=G6`)y|!vU^FCNAF)3!TA%iOU|&@_vz+ABmD^O`cytIGZ3e4Yt0w9t)Ozhj5nFSv?ShF2m@2pFbQ}-{UJs<Q`HA6l17`qAMt59kiY9nC>ZInK>ZSZuZ1wt7M5Nk9U!GsVQKme{otNt$0#{JfX&gTfr$kDr@9gX@{G_jC*^=D1FjEZU{B{3FAm+e2?Zy1ZcN+Z*q<_4e7>kiV=u*g33z7ieablbR>PmK}d3mx_g)|!%I0>?UACniBahn+7??bHB5WsU^g;JY|G$~mF#4CL<E&%QP+Cgx7#pAo4=n2Mu~?=3kNHb4tFcOSkC_&ND38#LDE<yJmUY%=|MU{1K!M47r{+f>%^nev_GwjBaU$IfNtZFW%(b=BEuo2VkApyJswgy^bZjS`NXhH9$2tvm@nn8&v1b<*_WYqn^DC_Bq-#YhyHy6c;`4=#1n9j;|4al))ZC%jq5j`>K)hxF#ZT(I&|p|%$OSeuDC@C+_D!`BeJN<5JXNbVztm_neOgQ2IT~z0@dNt==4R8nc6Ewt!u*!^Y@p7Tyuyy$rs)UX=t0dyg4B!rMtyp<Io)_w9}9f{I|=CRAl+rKgob2b7*?88G*rphDm8cFdmKVPg4R8j^QpL*9^3$@Hi&)X7of~nqS14{hn{Y+F@=Q-K)dSw2d+I3Q3QvauG@-QwxC{!Tq{t6XEA{^T!>WHY~T;gz^tKxs20t(i-9wg9A~7`=BPaeTEht`HIi<@h;0S(D&gp<AGQ5>2JU!aSgCsCzqL%6E4yvD((<#NZ!Bj{T)%6IvLWT`GfJNJo0<rKTM&MRdTEXoZ{Fb9G$!5Jy%tcn7Skqgxv@eCyJ@R=H?A-Ty$go?7W$de9fOIojo7b)}nw|jjqp@5@bgRgcCVc+LZ0)lGw>kZcjS+JGAk~O(4~ys!ou1_4S}$48IHroTv_UoW>~j!<;A1Iny;3(7)sHz_ixDN`|PSJ@y^LCNl)o*jT1QW!h_oFPbd_a$Ty*A8!zid@fJbo9!1-MA@JhKTw5I6r{E#)h=;7@PuFV%|JH-GtCwO54({eh){T%f_re9h$RZ=um=|gT0z%T1+Omw-y*O^`N!f*)^)>HsBRY1cBFj*fM&oT=%g82`r78GYCgd)7zeN$H*@b+zFW4cGd4`8+dcepywd9)<X!O}B;Em2VUy#aDc5&%@71z=EXV{WV3;B(=CY5zJ3&jDswU=>vqQM37Kb(UaJLVsz<B#2+^8Bz7%ANUK<LgO9nw%jO7Assf7_w4?7z9EBT_RV)<d=&05SH|Lx(e)ZFG+5m!|TfyQnmz_5W2~3&KkD^0#OU-sXx!Stl`Yf~hc`4_d6Mnvc%{@bfxUZObzDtYU;leL?CIP+(hKydZjRD(8*cBo1n(@xNRAj&60eR&x=`@d<p8qvWvb<+gKyn5Y}K4Yon^j?Y}W<3M02`tyMu=?#>#E5^eF6Vb*0$JEaPYqYP{_}_b{h)B7g2aeDfGp}m8n2sWgurc5Gy!NGZOEpuOC9ZDwjrZp&Fko4><<Rn39=Mj1=PDPV(22}Ba|T70q+@rGWDSB>ks&qMgiN1;n$lNFiS5`#%|&O&5s8u#3BQlbrxIG0f4`F24caZcgr%(=ka1OP^+b>_vLImZ&@s_JjyGim@vs8nAS*a;t8zyD#4o}iS317BnY@@8kwg9{Ih>~X>sO0nsBuW62Krfm5t+Qd&6CVc2C}u1#a|(Gn9dbvbmPydqx?ore{imFeQH>*JEt^B-~p2KqLo8rMP5<!2GRVsC7@urN<FqoZ>v5$DIca)-Ty;7N$C4GbHl5`J!PH5>SgjBTRAXa%ON9#x~0aWAV(v+us9zno0W)sH0#5s#ZX8lIDJ)7n%j7K-56}Gd#(I-J>Q1Q8Cd^^t;yeY|BDrV34p#dxY?uL;M6&x6l=imJr7&6Av`FtY-PUCNn(a~Yj~p;fE{OIKPKiCJ{7TTE1hmrFyCAhz-$CFo%=<WW$zubJi3h#?}@+rLjMIR0hcfWC7ef&6HjjIKN!&w%IMqQjtQ_pLT~uT?kvY6eds#DyCTMR9r1f!6dTBMVJhu@7=fdzOk;-KJoP)CePEfcX~$8T=c5)v?jbAg@}`mmY#A0#ni=;?2()@K8qDG}qc>FacozMDFfy!1>bo}_bEh#%<5`eIoL8tRCis=FQS$Z|dn-)>vR%7i+nMHvbW-3m<29cVDY&b07d5jP0I*eB$qS7pv~vNKbyeR06_%W>NL7?6KN&~E8B+uU<GRK_kxt+koq%Yqv_v*mW2D*O%djiZD1moI`1{yY#*(8jtEu89;ngclHje(MoQAA4j1O~qw1Y_ur@NONKCK=A)mZ$EG2}c@e$hcoi0cj!cA_rH_5&j#yd$54th8^Ku2Ohz2fplQM&SsYWO)(*ucV$;5_RTZf7FG(RXBL3#)+aHZF>i=9m;xH%KtJD=b0P7MhjWNId`x@Rxyw?(Z0X&Y{@u{5u=G5is{IrH3XIMQ{Q3nYNbFN);~V+-h@PBZk(rfFFo{h4s1ej{FTM+5PgbBkJlwt&Qv5$wc-I?fw3#0W9`Bk?VXbOu?HUYdPI581H_<*dKT@N%Lq@bCG!7OXQkrp-Rp2$GyVNArf0Xt6N)V6)0#;?zU9mlg+4iPQ(d|ZjL-*UV$>55((*F*_PK|{PhG3k8HSupeiYfQRFl#Q1B3Bfn`4sO$Q-Kbbd^vU3?hStb4fe2)Rug8hd;g<Q<b(|+?)Pq3wFpQXX*OY*gw&A%+ERX66=_30Hiyfhl{BcY#Rqk9tY7+BMH3~fOr3@WozB>8piMi{*(p74iUB(%0iXKY`vYdDWPjhX-%2sIXZ{Sz;R|wSL-b36aPPqFgwc-pFyQ=K<MJUn@WOA+dWUiZ99{UMMGgy<<MvMnl^GJ{O#0634EE)ou02c#=!##Xp=PL8@2fMs62HaJYQSp3W<0iESkTF%R{8^sRiO2PuZ$g<j@V^=KFrnt^%zvN4UJ_+)VP^K8~RG)jy1!weer+PWH($<3%1i|6V;w&3ueppaz>=Xl!yBf*4KSfvzN(1Soc$`9k649_UE4ET-#<?F~vwP^%4b+NHDBE#}xK+c7T(LV3tTOiuoYK&*XFY&WLxHITpoXs99^6&t|K!#9v@oYDD(Z}c?X*>i;j-SFOM8xDXNaQ48-=I#zP%e^=i6ZnL*n8Fapk`OA}hsLjouc8*5{<WM7CY15T%BPfmkJ#oppukBimiA6+@Xt2?LoO%X*tOKLQyUYnPID$gmO556Co1$1;3XJ1tr@UY(r#(n0}!wgGe?H_mY04Zq^^}^KBbuVP|4Ma<76w-d9!rMweL10tYGZXv66JFc*!42)iKW~gLMQp9-64)Z9Y{QRI)pK71gA*tT+`Ftpn>-!%Eygj6VZrhFi0ugR|p$O~xeCnqjwXv#faSGa7iygGD1KAh%9y`;^SIPFiKC?)UWjE2G4aq@)?}>nz5pjAe7UI)aW?n}d2Y6ggIPV3sN?w*U$05cI%9jw4{d@HZ<6#VO}Gk3fvFSzK){%M5jWu(NT;589s3Del-K%?<QbM-+goCFG?6#q?E8RhQJvQJbc_Z|gIhWwgv=6s<eZ1?ooxcq_F`lE;*Rp#%uWfT(DePttRSRT*F&ggU&Ig8Qihn5#e0P_#J|5PUtf+VqvxpT7pk)K6^BJji$SlRKn%Pz<B#*fi2y9-5D`U@jfwMCMR}xltQ5@raxuoFDZoethMvv$O%e$n<d2hSU{ZzXZndA-q1Siqu^Iui1!B)3{^Y&BUR>J;GfVLo(gF#AQCH^gu#pL*pnqL2mR9*>Acv%}!o<`tG0i&;fqYR?<GLoM|VhUP{pc?5UTW0StEM_2w%It#jQREG;(++umjv!5ka@H-l(kNI_zPw|bOf9{^)A7vhIircTxIk|3HgHFSc(v^MFXPWr?75r57Ey$#u(M?l5>>F(KY%B@=LY9|#A@C67uL(4J(z2_v01i<I1f(=)}Iei>_kb27zh@Mr8wA$abeT)<1A%sr0<!^+roppKhFPxWm>1X&WcYgT)hoD1pg>P#%r>6@*qgse3!IToY`0Z~ghMNGVwN~(J0L`6iPWVGyEW-XXuwCSePgBB99l~l4z;(R9eGy;;V8v$ALuhRCh$zZ&<-ygNi`AnjCY@;)+L|UpTCKt?k+MDgcC^vNh=Fjjb9JS+`1e6=h&i{hPP9mw9rqF}W6`@;O#f*jK}m|Fb@)O*9=}Fi-n9iv6~C%^9JFi0N^QyYkaYB*f0k;_FP57`4wo;Jv@!uvd3N(0yJu&C#>F*h;DF<mJ;=_9VV?IfTeg>JDf6~j-9TWr`g2%_2A8ehcR6ErG@+&kaSJF>F|-~I+3J=Lm399WIBT2&%BERwN+0wg@<Ji;X;OsE!DQ!iq5UjF0?JGMRWNymBuukthfSSFqx^rNdzZgzihw3NsI*ZX7ZVwfiI9L*=(jX7(@bkM&WLFa0zBLKjJK0fIOKJNWg`|JyiXj3Po_kTt`h#C0Gs@7qQd+2uslekQnw2DqQj1PQ?afb>M9L5Mcb6kYyKmH@59+usaBFjEC_agS9eoh`u!dweaZz*=+mgAdX{O`Og9seCYc71);%rqI2amjN;{BNJyGRFJWMrW9W~>563Bt)#4QZKlYtX~QGKmz18mZ`#y<O9>Xn6vFKvqqyM!u>A9*hj8jY=*Wl`T_rA9@q{D%`G&yK6+bMd_R$;~@IM1Aa1m;=EV57G~9n6&&|`HN5;Z!i*|iQgf2i<U~|dim4`lHKb&zW?eenSbYUYV-!#4f7ny<)6f;V$$%*%V9r!SW%f!gXurLgVvTuZee#n&Q+N85`Q?BPE+#HLVLpeE$%R*++RB5-d<l%8eHpnqO?_~SGwrWq!)H$(303F`aa6~y05PcTbItq7CmRK!HK%JR1LQgo}nlAU(e$}Kf3%@TNX~8SU*P?5HnQZ!JXIxpV)UsG>2F>BV?K&QHHM8HH~C^kO{A2y+iKwR}zl(oL9_nM&@lM`hnw{;bftd*!%tCl#ymidd!u2!#4xhyT=Z-xM8)KE?}UA>w!Q{QKsMoq_0Ud!-pkSsoSJKTwrJ|C-CI9JGru$kKin5WN_;a*xMGQ$(QJ(%u(DXj&;mhbBv7H=NUvu{VzRh*@gTF*uMg_<vdN_K<-k%1G=usA@de;Zo~aiOz~%+utv|R@lts(BltuYn(kTM1<q19VKnd!`i6?YU5J)>Wx0yC91d(g42PF%jb1*@Wa`pLmMnzQ8oi)&OR6$FbU4h9#qUGhb-A~Z-6Jf{7W^%xFCVqo<^oF)`vd&`9hbm^(Prwt206TH)%J!GEhhXWy9RpeLEY{!?GS@gI(9>PPZBG9-lDBFI0e>D!?hN+E1q_f=@C+RuYEX!>?ER=TxP6Y@Hc#$m8h|~=Hx}Fn{AV>N6j?pIE}*Q=j*>f)uuP>dAD>5lK}9O%A%{zRtx1N9bOrF%Qa8nuf9=D9vDT+$D}#Oi9B#%$;iEVvHFk~Cqu{UO(>0a`VtjfLI`+!#CCwJY0+TX&B5~T#u?PK!g}1-VSdoEZACvb5>wbBVGm#Kd8_m)FT)5*52nw5C)owcnAdnZz{gs2j*2Lo?k^HcdNzZ8s&3}vA3O}I>nZ%d+k9+^e2S{!TM&rbJXfWFA544(5&bMQT&Z34AC(V1d=xnLM@zJr%YB#;*>*DUkVt&~8%Ix~Ru)4#iJcawu~?us;iw-2_RT_}98<|t=2^DizLnq=n|2H-#dqHqLzaTYZbr`*(!320v3yUFcrzyt<$AeLVw^3-VXYS^>GnG83NCu#*Objt8F^lDd6m;!N7(=;>u^UQHQ8Q`6QT7EoSPxJ4-%wXF`F0!H3c+xYK3lBuAJ9))fXd#-Q5|**-7bJ-t^pn5JhAN)6zBUDdhE!Dt~ZC`+W=#|DC(Z7=&T*){X=4xu{ra%6Uey54g9akE!Rwz@0zyTVr{JrwasmdB2Do3>mS*(Z3&RnX)%)B)d6QU6?%=ys`d%bV@EYY0KaH2XAfjHbZIA7GB#{$oEVuM;jlH6we!ii6p~<lMn?kGr$rINoY?O7+CF_sm}*G$S;(O$u0WU_9ra*irn9^&#v`xJOZFvSLU$CzPqz>&z6&9#-h~I0r{!*pzZqs#Vi(;&~8m(Tw#jvHP{1I=WjI8{3OY~T}Ckf5?0AYN%rylI5oJ~;ezwElJG6Pr5uwQW6A=^;}2u9>W{6DSucJxLBQ#bMQ3|Eq^(fD9)4Z_|C-0mf>m)y$L#5Mc54oJ;zB&vD4v1BwVryw#Oo*b*({8f?(MAfJe{%*wJlik9uTv8)~Zy9?pk!tthx#fJS*qh4<0`I1=!|kAR@nv^?FK!QnFL1|Dc7xeajARQ9<3g8|?bexW33%|Me?2980YbkrRJ4z-rnIyxssXil(nth?`%(qTsQgb-^DOpTv$Er$i^6CEuf2JptY6>IaO6+_FRANP@P|i8-9P5sQnTB*2-5leavJ^Wno9Hyb1idCLmn_qW65{5NM=Hsr8QIS~YOq4hL2rH5GTwJBgFnvJ;l2wb#e9^k4{_h)dn{+nJFt?Y8DoqX5XF;luAKy*?7hSMxjmXmfB+&2a@`a2DYJagsO!0{X=Hd5s)iFcILgFV_(m{-J&c@+1D94|;k{qD)4I2*ZbU*UQaSKen<*&7i?6G#bVBYC^!Fm<Je#lff*CMCx5=cO>AieZtaL}p?NnF;j1H3m$d+xZqK<U&ew%MFEdP&ZaMIN&~?#RNDeKvG!0a1WL4C9DR3B;4!f)shWQ0Q|569_UFy1Hi&arvNt<U`GGvh1N*!AX-2CZmUl`P9-n7{(_CUdqF!hwkf4ET)rs5mcezANZNeW?x-etkcLAN(4C=*R=I;OE1iJdVcL$j@h5+6HFCE&$Ht<_Ph#*8y!L3M>yH=Gb7hDm+t)%mPj+j_EWQT%l@zP5%2fyJYb%#I1*i0k613*UXFD_6Bee}pK)7NQmHAajwNPbVEQ!(h-Ah^NjelZ^{3K#~Wfc2Ml<_SswgvV0q!O|7s;@jVFvz_TB{`c2S1B*$`^IhN^p-~MPaRy{^a*9&wN#KS@&zDvZMvvSJ|RdbN$0)ySXEp20yC(T(_U<Fb?Hay+<$l^<sALu>Hoxk|5Y&5ZIa)2!T~eDMT?zvQB@S`ih?bG#6MorcL#ijN*=b9Oo@(dFd$F)p#J#wXT!lM_}h37`+-7*DEZ;ALRLg|BtY_&2;8D|(Q_G7aG7Xt9A`zN(8;R3hY!o<6aL-*>&*8~w|YeSe>4c65&~@u`F$CC*N6<u?*h$~<AkRFLN$J<gZy2q7<8<tZCe~hg|ftzu4rP$!Tx3%TWHksDUIppNFb6>ZM8$C`lvd$PC*i}PjQ9ik9Mwx>7BFSlA=Ad!|O%)g~QVL4+dQ8hz<F0l%BAxZ4Z&H@SBa!6jSmt$a*P3SYC=1=nYInFZ{B{_BSEXrXzU;cpF5s*dR7LScw{?>TY>Cs@-Xle`s^b!%>K3?<692h=8RVYtZo#yWZRQTy3owf9oQIkSK7VdTS<U0wiFlnJWHWn-D_^AfH>5A+T>AElZMw>JXh$`N5~nDC%Nxm3Ny729W7$IN=~Ll%My6IzeQKw-26;9M3S;>eD!J%ui5-&htAjFG;ud!{qQ4w_88}YXlZP?`#5{NYBFRnOU*9njs{0-WoJ#Mwk$J508^`;*>=j8QsL_ol4BFKqdB1Ftvo<!U4Ed<7Z;t<G7PE%W=U$*}<o5$mi>dl0(66wQD5*I1BRaKB6AQxE|zwY0p#Ve!TUno^iiSvuq4KHb@&*-Y%2$;FFG{e$VQ<6I*tHes$LVI(0cyI8yJaggU^r$@kJFj(Q&dzU0|z0n1l@P4f85!!~t=GA4}uY6rR<X<iQrgxJyt9;-X5KG%L-HBY!ujzG&K<gnm|48kXFiE<7`y)oLW!LXAUocrlVZXLz9d1LRG(4MJA3Amh!p)(w$Y3F5jvu;H<jNqV`{X2Wh-7XioNwn9kGypqXJ^QJh>5Do%ytswox4rN4roYO&GGh*a5f>4%)OXr$z<8p+7`A}aozfavrs^f?|IItEE7I`NUY&#p@cYVh40XKEQ#%8D+s7_4VAqd8IdDh8pUJLTN(j3);t_C(rV#|ixn!y`mkkPY7e~_N5epj~SMdB@iO1$NvJ3Mk;OTQ%(Vmd4#?2*{^G&HX3#`7H5fNRU1hB+<DZVZ6r)RIv^CB$z+vyJ!fPgD?O{p04m1@njqaLZU7&9|Yx5ZV@uMY-s_Kc7^dQYaS*bgVV{}H(nxdMUd`WuKBY@h9_i&IGj2N#Z%p$_;IdSAx8zc9j?ewaM3(($j0`>#*FF)nU0`5PWHI;Jeq2hGsX6jsZVZ-&9cMekxy!FasRFq07p+J^0S!P{VL*zXBLdrC%@cxmI+3Tfhs%!PGT4eB{{hwW_t#_znAM5cb}a(5ag*R~ny4@qF;>+V%&iwUWx-xNE$m`sJYn~+Rq!7p%OfXk`z(bcEz_PT1twF@Byt3eTMLLwxsh#&LSpY*&HFo#p-pxIOj<94cRkzP0?(;-ZHg#DTeMuUa-f@cLiy@b>pTG7B*&;nx`QsR+xljK{e^}^)_fz0|_UAeQ){H^*=0G4%+B{F0}2;E7up<nbD)bEEb#!ln=_rEr7R0+-#2IJo>7kM&&!iYjmUjqlE!<teZ=SysD;yV1ymM4ykLJToy&^NDM4AkL|$QCn09YT?^>yH$T&j&cZ<heX!Z}^!k=G-4n0h(odtuek`<tX;B9(q<4+b?Kg8FI`xpkgr(rS1|Z{s)1oPUUAc%YWp)7JfY@>&j>DJW}bvA?$IB4%39{4}*i)LfgLTPeu8uiSi>|NCKshj55m4Gz7HE*Z|w77d0nB`Nb-9LpaDCNG1ad^R=v4i)*lpua_iFVMJ*f5+HH2+DT0)4$-%PG$lGVkZ~)nrIFVjLF~~>Y=Ew^;R<z1*ud1XU!n|bYgX1?#7m`dt6|nR`nWBPlvFHs(L;B>afeuoQ{#T)<$kqb<5Do3dn*xwqDadlxP>LRR~SstwE(n7gu3hLh*TDPjIsl({~5jEnUQ60N}2F$W8C;>)8xfR|B8|7u_`9$YBij_v5vm0l6bS_A`!co1S@IdEncv4oVaO~$uugOBb7>Q<c87$b?EV%kP2A`)DPiA$fWZ#f{t_na6FoG)@;e@$Bor0B3h9x*g6X)A|FeoZR-}y8~NG@oOtD!HDi$l!FjzMB;8V=Fe-g4zTLHJte`j+&#B$I<XFJ|zjFx|W-;0I`AU-F9p96oRb+o=xTTxPL5JEXnId(L&)>>KgiqsYav3KZiV;{t(o;Js*-hmu2U>M$FP?J#z58?+C<U+6M_sXzMCe@KhajZ4@Dlv>W?>G?6z@6eMPh4IOoHIg^t3@*xPZ;_N0>6B<)E>^@{q+ZTH`>5t1Y1^L|*3d)kD7~OGs$Xmk<A_C>!2(71Nb^^8p->;-4229jQet?U407z!%BGz7P?_9O*KuMU#P((u(WAQQ1HLyGh_fpDlBO3b26xe8@whO^EfoQhKBId6K-RI2UG;zpBDrNQ9H<BD#1c%_@c#nGvy5E6|`GZkRh(I!Y<MywpZYOF@K>n7qVKe(?B*-``H+tMc`C*aF~o=@wWZbYOR?yFMz`f-^e!j`_PVR8#z~3O(sfqLr6|z`)g3-3GT%{j{i``+pniUR@OZmeGjbf!r^qI8_(IzWI8NA$<lf3SLp0)$cm5H0C>c_<{YJ&N@0dRh{-&T%ar}E9^3#H=~`7D^KE#m0DGC-%({{i}wW35`bQknggG~Q}Q_&I?ik|s;t*r>aIyaPm_PkA?jeo?ZKjz;G7Y8N?$ol(zIPnS|UxOis-*IxS^9kJ~o8Kbv`oWS-{VqkzqH7S7&8Bi$2WA@8#6^idAB)x-3|wJ$sIvfc&!?X$T1u>qRbk3lo$Cn)P*yA`X;7Z<9(K`{uI_TBanFUFj9cZNkbMbgk6crq4$(0kG%~3>;yP(WHt0(76KSi(%32H6j2`JShl4mquHW^@$BR8qc-6Qo$-Xxz&7P4t3S}?tQY%vqXFO_@1YI(0PCk7Izw17!&2+K(uIu1SGCL286XFtI8|22~CYIjHn`iYNtChtK)4x)JPvgfP~t9#W*d6G=g928olP*2qyYpAyS8|cm4wTG`E3}6=|%)H{VDxA8OW|51!F=%g=YmkkqlCn4EoS-UKJ268^Un$mZ&+MCuC+<t@tU#g@FPs9nY@3KwsIZ^eD5;2qGTW25Y?$hEn}fo)W#ZwMsNyo3m_E_!qRDqw&!DMX653$6%32)x$ezfi>is>z^6Fw`%x9uIE`^tPNqA%j5H;38;8mSDU2*rD>L<JPe;(PKd92Uf--knv?wd`S~Qp|@aVmPg%9gH9ah`UTkabCs0tJC=r-+kIIJ9;C|j6H9+nR!~_69-I<AQ(QqsGdL}1jmNesCo8dEE|t1+3+rlf=8$rY6}TIrqkw}P=S{^u8d?wRlobeMHl1)c7nn_xUc9>_xA29QfbGptMGRpX?FyJPVdLL%rR@Vw>(o1gdY;gU^WkKN4$D~IFgGSuF1uB0_;BTSy6IX9Kv{deiPv>s`^)3a5njz1Tr(}uNt*l({q1DuAbrck_C|*M=hhoF^#TH3);S{*-R2>3VBvFrVX#AQS*k>9kURl<(L~0W&b+^CJ4BCU_a?Q=mdIt58lxY<d5Z+$dbdNd0-y9*C|l=crrEH2%HXPQ^<Dk7l$+^17%q!xr&iM^G=-;U+$-O*<V@8()>BD$ssGs4_A)xa75fr}%U|9wp-6Z`M|azn>cSc>c>$%v$MOCVbWPwYKWvG2a8_vz(XtWSKg%wxM(i5(5N;;P)KdN#1-OD=KQGqI*n1PWFVsuh`^S5aPt^FMZ@;YA=Pq7IH{zE^q1}+Xl9;lUfna`{Nmj_}w>GM>AvBOu!iZMik`fw%Z5*(4lHfk(oqq%5?r4_$&QcT<{Y4VYctNkrQI6EDJ!?bN&UXs&xb|!Cc1Qq{gw}SOvY$u37(edgV!T@UEyt>>e-U|HOCOTEMXo5-4>Y+kePlkf)yM4~sg8cHOlIs+7l4Vyk?e_f<IgBmr|^A7hFz+J@Xs-s>Y0}X1eWR58{(5zVK$Y;xRt|<aVtxcLa&>>YQe?7_>B?TzJcoeMHfBDQTv7<nMV%Yy2f0ut6~A=F_}zPP~M%AshomkX3<uHHBJ57Kap45P`KOuZF&wgl9HRf-p(h9@qj6r7D)8r_+P39cSerBT$J}r-xyWoLil0E<T9y~z5IxG|6G{T?!d8qaGg;fYZXPFbl>BFuU&EY`xYgzO2!;6xCRaWSS9(=9L@PXeH-%@5x4>^eJH}<Vn(`SNYcR8vT=V9D=K5EaXRrCL-cXq@I}{<H$ji{s>V=uSEL!`C39sPiZ{B6v+1h0;>RTjl+PgJ<F_pOqw$n$zezXxbJN873mT|m=~<09%={Oy+ti>H<s*4w^nTw-u{_sQJI>z>bVe{PiZ)tT#Cu<24yRBIXU}<Xt255`&uC5Dvk?3CDX%t5<jY-t_%FE>-zmO^09PFM9O#c+M??U>STW>niA$Y=bP;9aFOKX#y2lu2<wJ;e?PWFJEf|BXtPL*oSb1zHT;fe}xd7MQBOKTQnUPtRc8v+M`^gf#aU#fWN4#3>54&vn9N}>)hi+Mr6aU*7YAYn-)sP3dFQS)%qLP|V!T|wI4h!q(4)AI{34&VC*(0%+aminvEu$lu)EV;N>L(^l+f}*x#AJ#od)F0i>v#rV?}YEcYpuNjV2AR=*$AvAkz8wx3Kb!fUhO*<L_WsuXEyk%laWmo;iK12Fw>0bP)zJD-Paf;a@1)gQ<M&7F_s<10Sbyw0u0|khm<91fNYA<(fD$Zo|E^8>=luoJ}dp0QmHM$QF<qoETKUmrI2H@JTQSVGRI!SzkAal8;x^H)GX`{{%h3gGEu}<krL%ALf;!4ov;y&@f%^d<E?Ot)2V#5j-AEN+c6Inu7DJe$hA3Z!kSjCW*)(j<|7h&cgeA2pWSQ{Ys+{$Z`rPv*JZ3&78tv5*WgDj5EG;Sueq^<hT<J-uX<R3PY2yFjfas%o56^fHs$w__?@=fL6%ooU^Sc9Gc2|FF_eR0KMl_&u;o^?c!tJ#v<DctIG$_6YGGPBT?)&-*1NOHo7R3sw{TM18~{gdL6tgh#ic5-ii;lv@I`)t|Ii7QNW>h1nE8OXf$gyQ$+MqM=$I^Bldv5SSc3NKzyWURV3;iKqmVpter!T|s7|!ypMn__f)%%92eJV+%4*?zQ0Ng(5JKWPH(msxfoPUk!e+;OF0g?oUHo~s?+^wG{dS9dC<G+CcMQl^Rc<Rnsj9p8qS8emX4*~cE>C6WHlc3<L=tB`6Ur5|TIUe00vEVEx(|8gwP!`fE&)*8A2ohgptfR_P%G1i3BsY+URSHc*Z+#jIBgJOM$9B$(uNXwn2-wZb-6u{TM6R3rPS|Ao?pO{=q2(wcdSY`gN(`qBtP=xdi{*in{r<`n$jUQwbAZ_USVEppkp5A`{X%6eZcDmGD2I+T7f!ppXjo32_aUd>57Yz&0zP77bPBVWS~Syr;DV}2^E74MeXTHriyI7P+r9y0-{xky9dTILQd|{goDiy;A+}(QdDnxMTKM+u!m`0sJUsX&nPJzkHiWYE53|$t}k5}1Y+5b_RICso))e_11?v5xByvEu;lCHQT2>G;DrR;Z}aQ5*uxMD$ShWC@9}<|%K&kra9HwBXw&{qC4J_rcfV}yW@h^}XxR#Yzg|<d!%xXRW3dJGyc7)^A#pJnvy!0i!&u)MwcNz90p7-+WYqEnvIqMGEwHEx`WS1#B#88hHuO(?q2LZyK*l=I4(aR)M$go8xJa?NsGp^1L2CNI7b1-{_3mDO!77z(HB=3<mYtEaJX&dy5lnNDap~s{+S8D-ByZ;k_@yc4By31xNqcHg<`!@7_*Zf}8qG*+E#>2{<KI2|s~PMq2+_qx=sNSZM`1$$Et>ZXZ1CNWXVJj5<GO!nI!#;8dhM?P-2tZ}8uiegp8IE(80Tn(>km?+al``|2`KmUB6jxfq7dDt^;&yG2rT}rypTkaLX*>4z}2cRa<4j8|0+GK2{T1ZB`5~L1}?r2;?9QBqG9#;uFvY%E4$D9UH&_|#vRfRaBYz#)Zo<96`Li_3w84Q>Uq<34@F{q4=6jWd%y^+r+f8B2fG>zG-4%Y*}tBzS(Ik?Voj;21#H{40Mt-x|D6_57O5lT02_GgFlz4Q?$1})uJv)PBkT^oEVL$CDJ$gp<wAMnZagaoOk}LzF{tXjlqN48Ovyanmp!j`FHY5#Xo^5@UF+LXDNddRsI?_J3LAsOIhC|I21q*PJ%Lm#k3x38d0!^7avD5}yre)@o&N}S`_kP(r+9w}qY%beNk-D0au7S!t<?B5ux~9tWEL9F=-;JA4BUQHd>`H-iC6q%BGGuHMkq+|lNVMG8$^4!$|~W}Hz=~n6;CwgiL~g8b&`g~rYNv3jHFt=u5s|Dzq4HPsksO#R4mlNh<(r8bO8W2leL;WK5Z@jHb<tSUr$*NGW@>_f!01{$<v=lH8v|zdim$df8Nf6K$Z!12J_c1OP7=Rdym>#uCT|xGLRZae00J((!{zQr#)wmFcihMAx4-lJEsA%)Ak-mM^`BK%M-K5a9?4s;n={zaNp$=H1?dqkCL@HYhg9?3%<cL>*hl;RfA-&=S{LrC%Cw&%seh;W}24wY%X++yLlpfvKOV}go6TpIhl;G;De)JH}tI1IpKyQ3A9EZ%$nRgg;f^Y4qzB>k&9si2%oS+gOjSm$`J#_H;2u7Dz*Z;+YW<9AltILDC7Q*82eBXJ9zfI^lX7;#YmglEcjj$J9QUoT${^hKnn(hw<)4-ZZ`+ix<_2*E74FE4};NNZpL2Z->wfD?U0x6$+`nib7H@5qTT90lamz5J@InNe^lYL{2+hyvQpR*jD<Jvim+7(`#3vJ?R2Y71H<Rca*o<6Op`RWEVU5IFea`YpTs4zB`_6HbT6guWw*e=)?+;VvDucM`3Q#<+X#7TtPA3|8&$wp8m24)@8&$OeEZSL>0uFuk5=w$Mu*Ad47hL747CQ6B2J1&1O5mCXy_I7W`Rw%ngN&B{?3eY#&9l4b1ByKQ2^e|8|-gkMiMF`sancxu#~@XxqMIG9Y1mi`9|RJ<Q0pHKON&o<t7O=G3x(Gd1Nico(&c3TG=!h$}xZ9{=DXFO4~EL0R;rNX7?gx8FG(DV=T}u@r`;oKB11|eSXx=qJ0q!%fnZ0{!8~|^)hi``b1FzhnQ4|UJE|bu%4<JobaYO)5}d9T-~PuKM01&Iw0_Va!Dh>mZ_?v>x-|;+|0EHy#G;Ke(2g%9amoP6QysLxnDufE)r|tcn$z;Gi(ddP}(jNkI#s_Sp?pzrgTHeQ(VNSm~FVQfHF6d9T)bPnq$)Ua^eRfB_&VJ)?5)y*dOxa_FZ=f@l(KbeOc<-n_0OkR7JO?a|<7%qFQVgkOl1YDiK;($%50mfRKLKbDoV@IdT06^kEJcu-(Z1KiSc#_%C#nTDUWAoLJ$$f4ujqiR#1FP_57pZ0nIx7qm4Rh1jvrM!Pd`cJkL4?{bMDuvp7J;RFp)Wu@bm8o}4qOIDA0O*bqSLTG&Jei2jY4y0v*MfFi{<&_S>m`2<p(YhtFfpwaQaT&ObTKir%=2QPchm@u)mlby@eC$EWM7z;L1Msp=1OPHNu-6TF(18Bhp{37m$A@xPiF8<e6Ce9y&~d`iP#jQ|dM%dRY3q@fVbgKT;Ar)bV(MR-w=mR0VQ)ncZba4eomWd2t>%|S{BCZRmHmdx*jT?5cT9J_L<j{=k?It91}>F9w&%UfIQ#VlMXFICwp%;y$?}+bV@)8og?xh{33R$}kdmJ}!#^KC>q0q+9ZY=vTeH^E_FK1Re(I{!WrF*gDPRbq?BgUzTbbVB!&7C*>g;U7C3+c>sX|K^v@#O!2CC;FWf~bDLvV3i_iK~Org19P4TXEJ;<M!)`3<&MMo@~(iCa^kLMLyCQhR3~`n$(-M2@TXCsW>jY+{=`9<v-65Vo-T+rZmo!+ZV)G7%r$9jsHNHE%5F?*aoSXz=PM9BB;&CydbWx4S%hr{HZ{63XTlKALRx{Ncwo(r+5|T%!#d+Fka)ZfRbUg33aRqmsGyFR9K5H1)fe;r<N4jw-l!BgM@SAj1PE#fy3lrY4ZFx>_<Pk#1Fm(EJw|yI8j<c$>W_F4|`q`z~C^E{@woBX#59Mo(jZKb>tBU&IQ)N^~T{aLF&+xH!<tCBwjKqrt~Q6#*>;hk^_k<O$)g0*RqNjO}UQl7}8Acg`lb`{12tz)$ly`ji=9M3qcyq~Z^%<fA*Lk%n?8{mkiL?&0+;R>uixZXxVeouu~I-FXW;obq*646KCQZcNsOG%A8dH(1a&1XogO(`iw{Sv)fkFN!U2Enr|ju)6GLS@ZwS&|-{h@2O{6eKud;aaT36hAsP!LRvm9tU|M8y$yc0);XcyA2O9E#^oOTA1nX<24n~ycT;4Lj(Kov3S|a!n#yk@L)chf_OrFoMZ_n1JFy%XgSO%9L#|9rf!TU02y0YLq?|#{m*p%D$Ecf(AJdj1s#d%dml*}-c(ksvUGJ=bpswx?t{G2mOMjrJ2`U1!$gAs~mC}TF!Z~$BSAh%^!O-)svPz}V)fyDK^7$GUj47;brOeGy>4aJ&Kq=OWn<QBj8TprCjZcNW7D>02ZSyX~A6YF>Cg+p~40S^M^i2hLh>N1^k>Z`noC|oQEU*vHi>@eGF_NQny+(|vmtSb|QV!sQKL!m1c`U>7MMVG7=}Y<c-r|7&6Cc%l-hH<I*L09aX*{q(rEhHFyik|fi4GaR^XFo@UBnNU=#q|vieJM*kx*1(UAS8o%omclv^C9p_TLc>HStN|#SjnoxQeib>1*BL^LQm$bT~b{$l3YJxb_L+>Q0MQ45ig<ATSIs;0~De=_DdHU31+~g1Ve6BR@^|DWmq~&@xf;!x_&x{atZT9S|&Ty<lbiMR}K(6)@edVi>lLvsvH`;dV7j-J$_K7$p($>)b0e23h^NKb>s+Y<$qhH*B=Mbe+aSuhcJO#5!}5ANS~hZb`>gMhrD-MkwfSLNQ@askYDzEkxwGe6V%2XfDVBRd*iC4~*Ezemh~5c#bNv*-GmCRiplBu;vGZa+dyc>mX)m3V;V<Tf$zBX1Htop?SosE?q*H7X4qLjRDswSLm5qsp94Nr;|@>X!lt)YS4h{zMl<aY2mlaBSX2|L;7$Chl~2eUujO~zA-5yZBHFvj1&#Nvv*N%mw}>KkiTZ5qavlvO4LC64>y@MZk*r&=4`S?A(fs+Y9oXSPU_lyyLg`GM8|#-kI<NQW2i(%(%t4Q<pZlWGW+VUOPYkmRhKq(Vb!@HT_kaN>JcHZ`*|q$rq~+%qi;wRg4jN-8Lm<G&U+&cZGl<?T6wZQsC2cjG{r()C=jY_6YLmCwhAaoc5WeD<6%_b@=tfQDCY_X34g<IpcUWS5V$qwZjI&2&)!clhp5_fEuWV1VdNKSdCT88GVRzu0^Ea3E3;@Q5jy|%5}-!4DK;(i;cmHC+XM8>8_R^1h2y*C8*%^|A7+VNhsA@uexR>W{3xS_VCwQZgSTqWw&r0Ol0V$eq)cRAJKYiciq6{?dS(HyiHrgRfT$X5H5Z!uPp@3Pvdl&+Dcdhj4pr=U%NSXKt*Zxu5$Ksss*$4DF-kN3-E*1C`!>1$GApPdcI&DMogCXeTl7rZ%b+BfMC|?{Wg$yOdr2v`hE=3E-qP+trsQ)%hD|G%DIhA6SPy~c<IHa}a~B48l~&oE2)KRp_h^a>=TuyDTRF@BW<^!ZVJyy8rIo{Qj|(w7hfLc~1he;#ID2tRESmhc$ZfzJSBCgt*OP75=Gy^p-|4G}{vkzQL*3HRwDd3x9e;ng+~3rtX<2QYNbal_DZ{;pq_6VCqvQB*9r~wShXsN4wxR27)thB5b3sg82}}DdKV^2Jjt*s^dYg@V<a2*6#P>zP7GarkxkQPmTv`Kyq3?v-x#YheNiVPZGK5XSMq8~1N`iWPZW5`<%F<(!@Rn}+kRBT)=3hNH)R_@~p5nmgtO~PgC95M<iVj?yw$WUsdlxkJl%Be#=PV_}jOyC>Xz9~1>aad73t^fa5!Y@m`rD~zU!l`K69A_BhRT4~jS%2NG}Xb1`EBp_;*Y;O+##L}a-^tH7wasE`{wTQ`KN(KfQ`d541fRg=5k9lL(#<->cd>maIJ&z?qOOmRgG(|U_VJFh!MH&$y%karFzmR^>ElJ=Rr5sDa*U+r}pkmkT`vIt8|j~Pw)x&acbco6+UdEzIBTqlt{XS&c^TiOfLC$%ydo<g?+SN8{6{?NE$a_mvGuVf!71S+yN=bHBbbiMErtW)h$df(h8$y_pCIo|IlG)Mh6!OyqoGwrdtruadrGkP2t5mG&s*zC?{Prgw~Fn?l|fQg!K{&ybP;^Dyg3*P1c6PL_#o?iX+dZr)p&gp;f!5+9KtvrzV~k0%{1L#TEpdA8tUSIhbm9>~SVJs7+Zl7r?YHwjb#{@9XAQf+n;_Tdz5DN=1%I8P^EO9qt;q@KrO<lXROGp~43*Jw3hQ%(^#}<AC-FNZ7ex*;JbDLE=PxzdWpRDdf1fO7bCqO6uuQ3aRD;a+PlK1^4c?<`r6VfZ2*D^o3&hPyi{6#>ZMhkVQlaJd0(&HNmxc{5*0X#DUdNZ#|uRs+HQ9rv6l*tZbx9^AL=z-Ea~x<_U&dNIzf^t2i<)gjli7#dbrMM(+kX1l1xnbkha?&K5I}z!(klOniZ008pE0hbG+GHkl7to2P=AypBq{zyY<;t?bIv>+w82O7PAyB@zx3ztrAX91U3nLnqy-VGFobuVkk%H=b{u3kamq;x$S!6o9hizIAzBV8W0!F5gZuNwXnA$z?>tfN6mQqfofLwil|KpGC`M%XRGYrvJm*`@)1fUn+GjrMy#BS1z`{_yWHQ@jeTmu}}-XRr0EguQPfZX>bf~qblbR-16A0+ELcm{%CS*%cDTrQ~dB5H_oVMb@|rtxT<0kgCEOY<Nmfr*IKZpe$y;me55{oxIq3j^noi1wSS^TAoo(OGPm9aVF0i1(me}lr}*YARZ9HH08$Ej2#-wnCOHQ@rjMeZCIwS1<t|m1FoKGAtn*|^Eq@VWBbjihDYtFh<p|Lq+YQp=?R@k6R?kC#>!^N0=-;GH5zl&U;>C)#_90)_O3EYTB5ER(33&ixgFK(4=H(frMu@8sodooUZ42O5h+>;g?Xx7y0Z|UciAZE?tQi`6{kH#9L=;t9TuhPL-+Oc3Dt~PrwC*a!VFM)$avIe_on}l(6-C$ms4I(@`?9I;aG#`ZJZu1sA}|pv<$~SP+Y+yJxd=e-o=5_@4^Q^*1>n~HT16iSx)Akfj7q317!h@^=61}oGdoh=#R8(VNKN^J&%kSp4&bU*ga2G1CN&crhgaWly^;w6vnq<m_Z*PvSCQ?$FziUNJi$`>S{cE6w>|A;b{b7iGlxcSN1tSB4-Y1ZZ>_TvmKOK@Z~v<m4%(4eJ~bvAQr(jf_N}^oG{Lz`$$7ywRH#1F9?cIcVT^q1B4rCm5bO=UGhmam<S1NT<@5_e47k_eFk<=rR3_L~Pql$Hn0<Sj**1Y9<~Luv9`7$=nvm?hgYe>6nm7h9Q(TBT(6ecUP_qE_)y^~bNPvT8&jD)nsp3$1QZ=_8wy+obTfUS1(32TPNk9R&7GY0ny@9#R0lrvrN{mPMsy?Kwn%Nt~8(uaP<g9Td387$f-hsj7MJnnLk*syKRM##BI{98);u}TwkosCOa9yIWcEZCTO2bRB=vQ4>4+=eQ@YS~d2N>cNzeydfe>w3$vB7Uzp@NZL_t5K+yjuR4%T?^&$lHAzi5~+iU<;h*c>}D9SP^dHPf=MrkdAq|A5Mr;tBX9L=uk;#A$2x}wI?LS!T2*k9>m%?1<?2G83dk=59e`#f}CO3-y)yFW)UJNIzJe_fbO`D{6K4wwOd#yb5}c`xFZp#NlFgi6!dks4HI@5sD-t&E>BF6{C_gv$SuhY!;1`VDHvQ*DM}**8QivhY1-`!dR?Xg%~QJ!Gg~ZcF03|&o(G$k0Mfs0h0zJ)jh>B5i)S(K(Pgg1mWx^)GZ)9y=}CxHnVau@Y@oEU+~fZA&bs+DvgvIhe7H2rN^>gH*#q#cwVUg}OmqsSFtm@VkGef36EHFcml!#YA1G=Y6g?1F%(g}2SL&YrlXS2y`MfZ$?8T|D(AvL0y!711&@Ld#I8}@r<XOtW(Qn5CptR?I()(_HsOz9@^`kGne!BPhPA-iecAlJQ2In+hg&U1>t42{BCLJEw`%9qH-Qb@f1qYj9TdcoT>NxXXA+M+<3i&7>bN$Z5fVk`+YlU+0$?h9&NR_On1c#q7;w(hnaV_diz)|6it*ZjfcfjkG>f?!NrD^PJ;9J}w8Ovl8rLUE9rY@VLiqA_7p_?_#hxLb=#|Qmw?e;nVV#)8B5eOja{D`=0tY}99xuthJq#%KnHQ!leUX<j0i;|DsDcwB;8x690IekGnSQ<ZY+0tM{UyjCgmi-z`Q62`sA8l(sC<^mTbkZ^2)fAz`#A#j4=z~DMmlm*ppX);J{=_3F3-8u!`{F^WTdCkY3!Wl;U}M~}TK6<rJ3Y!r|Dgwx9dwOk(=yE;bXk>o*oWlNprWf5lw5{A`YHBAD%ieGxpQfNfU_9m$SV82L2lU2C^M;fyq)i<-rde7iFfBwtS2qyrp4!B8`S?1V7Z(K>;`P#?lB90FJ_6Uh5=9n&5|};gYaznw|qM|&MI}ktVB(B;ez_g7FW@)9|I|(>iaXF#iZCftCDLf3M>GJniv^ZJ7x-_;U%j1zB`**uC8omsx*C@?Ob>hy)}YaM9N?JqOKCZ{GhNemH;3B5x(X9Q4NZ20A0hIG|L=Xc(T=5VTghdRyxAv*J7WPjAPKYS|Mt|oN2$pigk0@>}b4F!QlKY@GW>iYW$?6^s!(`-okN{tCxnd6=SJ;jRTX+ii;K9xW*6;ZttMr&*7dw#)Ftirwj<Rc?2EcedN|9LtBWX-!xR|Sb+TI0^3$L8bge;M4I<W!+uhU;KVSpe<pXPTU<&$F$*z`@up1i^GGc_HveZHd0m1#vlN`^jO1g71D?&S(t(H8GU5@W!wTwS`*Z-((RimO7Z!lLWDdyh`6x~?v<;z4+aBP>3XAzXQmyF`edJ!$yUC`0_Z4+}#=j5%#c|fdV9c6^-A+E=lFXrB560dX$3!I#x^ufs6SO*4j_b34sIPT}2T)TB{XwdqWh0)PP(%A4qvZ+&%^5gS!fl0Qg5d|Cohz%SC<VY_O##b;i}Sy92jxBh5+RK^m0~`c$BNnImRFO=umfN=@Ys44O$?!K^bX(n9oOs(gou{f62GtBc6~$oezmzzprgT!2?;`nyo!(kU7%@(xy1u6-|4^?mvPKxZ38(JqxS1&$e0&dUS6dLWt_IE(lR<{7BkTH#%~5eo-R?Bcm;x==E&noF}6JSL}<jkJ>yOAGLEZBUe9>l+1)`FKd`<6LG#9=Tq-?_{&E1`Qobl{?ZQqrM%Wih$Qh}7jZdHNPbXSjyogPo<EU_R7DDee)_nmUo=69si8&LFSPjZJRm!MqPS{_RD-CFR8QPUKk3@FnlO4?6=-~W2qftMM3F83!@SX#GCsuUYER3aQomGv<7?x#oh1ysgq>mUbq4S&Ue@u?$tTfY)sIM<N<(daahx$qOGw6wL8wT(I8YywO`mfn=2c05*s$z#HCxL1w$5;WxmbkUI{dE)e;lJn+VkbGo$CGNByN}qmz;@6Ggu;<^E#w63x@`L~5SOt=Ungh#SfC`J-s+XQ2byQ2QPt_uXA0kMHW{M!=74Hfc#J7-%StAe@s}$q%;*&|DUmSp79q8J1OcuWyX^WnT$qbIu`2Zunr2l#*(*q!o}CPyPQhEX0I7sxint(d*PyGKr5dp#J{w_*>F>B+hZbM&-bGB?{{Ft3r&X3g9Rp+}jMyv(594mw_{(|qH`?!1Hll(Wg~9qZ*ZvG(ACNt&%VmSVSZ5KS%R0JNc#q>6PSNN?9@qCrd<UOI(iDo#P8uet7uK+*H*KeteJ+TQC(BYqC8k+d-fhs(jM_8N^+SL4d+f}sU^wR|Be`RI$c9vvuYnLiFTLd#tPU7STvJxrya9mRPpO~krh(ArkqR3C6|=M$PAlRO!>gXwb6Z@<*yy^$w4aU>!Ob2&B$;+*f+WPM@UuRC9DB;~avfR>emj1>#L2#2uQPE|<??Ip5If70dsYpun`u%}F3agvGi?KntQ74`v50QvTo%~@>~@b6<Yo$5b$bl6Rw?&On_hq)OYugH6E5<x!ZR)0<Pv0pZB|-^W@^mHb~-Pzu76AlkdCa1IDgcuVbmkGoiWC~`)z2iysg1Hpa$VqfSKQy3$}8^kYLAhg{U_JnDex4_WEA+`wmRoSfYsThnxtB58+ECzf&emI7bv?HV4#stH_GHAO=gKM?Sb5X4`&C4bgYwG#MLIuI!)>Q)5q<kX1c7>CCSW?LXVeN8_?0#FqXmMm9t#Z5Cd>;x9vO)P7Q{2lea8BTAX<^Nb*B+*aGHO9`Vi-a#AXEpB*fUw5>^|0WG2K`Wv2`ZfoWl9CTV^O_tY5=U8H_+1A130~<zhW4K+8Ia4DZG39%+4v~cyW%rlCF&tUd_eLJ73cWFY6Nb_SxH&Fs0w_tRgkYR1jdaFyW!|2Cg9bxTA;DRisIM<7)|83`>ct65!u?@P4EogjN+_lg3QQ!W8h}1RQRbL_;E~0KsIZ==Z{QZBRE$80Uqwpxu`>9iYNak{2K!l$LG2&D0?elHBkHATm9I~&I|R`fJdiYEI<C~^CPZ;SqHb}gB9&9@y1r1up7poe<tnTSJMYB8I2Ijq3XfJ^BP+pZXvlfb~8~2<l)DboA&z<Tmor_l-T^CPfS`vas*6Av+GR9nVY?@K_MDu$j3C{<JjicTPn{r7t~~D&SfbFraLYMNaIMsMs>b94{G$F_rSA+?frf0hIM=PZkN1~nCuuB{JtET7<MB|AK_Qx8*b=9qDVhaPeg%X@Xia$*Z=?kiug)H!_C7K00G*%0r>3%l=o4^vBYQl0ssI200dcD")
    if hashlib.sha256(NBDF_BODY_SRC.encode("utf-8")).hexdigest() != NBDF_BODY_SHA:
        raise RuntimeError("EMBED_SHA_MISMATCH: the CELL 7 body")
    exec(compile(NBDF_BODY_SRC, "nbdf_cell7_body.py", "exec"), globals())


In [ ]:
if not RERUN:
    # ==================================================================== CELL 8 — report + the identity fallback submission (E18 R2 format)
    if not RERUN and not os.environ.get("RC2_UNIT"):
        write_fallback_submission()
        def _arm_brief(k):
            r = _arm_rec(k) or {}
            return {"status": r.get("status"), "why": r.get("why"), "correctness": r.get("correctness"), "quality_gate": r.get("quality_gate"),
                    "oracle_vs": r.get("oracle_vs"), "long_gate": r.get("long_gate"), "smoke_gate": r.get("smoke_gate"),
                    "prefix_gate": r.get("prefix_gate"), "al_vs_q": r.get("al_vs_q"), "short_exit": r.get("short_exit"),
                    "oracle_acceptance": r.get("oracle_acceptance"), "ingraph_meta": r.get("ingraph_meta"), "prefix_reuse": r.get("prefix_reuse"),
                    "memory_gate_ready": r.get("memory_gate_ready"), "memory_gate_first_decode": r.get("memory_gate_first_decode"),
                    "pin": r.get("pin"), "tries": r.get("tries"), "ready_sec": r.get("ready_sec"),
                    "env_readback": (r.get("env_readback") or {}).get("why"), "config_readback": (r.get("config_readback") or {}).get("why"),
                    "dflash_readback": (r.get("dflash_readback") or {}).get("why"), "dflash_checks": (r.get("dflash_readback") or {}).get("checks"),
                    "seam_inert": (r.get("seam_inert") or {}).get("ok"), "q_readback_final": (r.get("q_readback_final") or {}).get("label"),
                    "q_defer_shares": (r.get("q_readback_final") or {}).get("shares"), "kv_layout": r.get("kv_layout"),
                    "memory_stages": r.get("memory_stages"), "capacity": r.get("capacity"), "long_levels": r.get("long_levels"),
                    "residency_retry": r.get("residency_retry"), "spec_log_all": r.get("spec_log_all"), "graph_counts": r.get("graph_counts"),
                    "graph_spans": r.get("graph_spans"), "stage_compute": r.get("stage_compute"), "parity": r.get("parity_record"),
                    "windows": {k2: T.window_brief(v) | {"prefix_hit_rate": v.get("prefix_hit_rate"), "garbage": (v.get("garbage_review") or {}).get("gate")}
                                for k2, v in (r.get("steady") or {}).items() if isinstance(v, dict) and not k2.startswith("profile")},
                    "responses_saved": r.get("responses_saved"), "launch_error_lines": r.get("launch_error_lines"), "error": r.get("error")}
        R["summary"] = {k: _arm_brief(k) for k in list(ARMS) + list(ISO)}
        save()
        _v = NBDF.get("verdict") or {}
        print("NBDF THROUGHPUT WINNER:", _v.get("best_arm"), _v.get("best_verdict"), _v.get("best_gain_8k"), "|", _v.get("why"))
        print("NBDF TABLE (tok/s, tick, AL per arm):", json.dumps(_v.get("table"), default=str)[:8000])
        print(json.dumps({k: v for k, v in _v.items() if k != "table"}, default=str)[:8000])
        print(json.dumps(R["summary"], default=str)[:16000])
